# install packages

In [ ]:
pip install imbalanced-learn

# load packages

In [ ]:
import pandas as pd
import os
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score
from scipy.stats import zscore
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.linear_model import LogisticRegressionCV
import sys
from imblearn.over_sampling import SMOTE
from sklearn.datasets import make_classification
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

# read in input files

## zip code/SES

In [ ]:
# This query represents dataset "CAD_Afib_IRM" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v8
dataset_60803117_zip_code_socioeconomic_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.zip3_ses_map` zip_code 
    JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) ) 
        AND observation_source_concept_id = 1585250 
        AND observation.value_as_string NOT LIKE 'Res%'"""

ses = pd.read_gbq(
    dataset_60803117_zip_code_socioeconomic_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

print(ses.shape)
ses.head(5)

## demo

In [ ]:
# This query represents dataset "CAD_Afib_IRM" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_60803117_person_sql = """
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 ) )"""

demo = pd.read_gbq(
    dataset_60803117_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

print(demo.shape)
demo.head(5)

## ICD

In [ ]:
# This query represents dataset "CAD_Afib_IRM" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_36851903_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (4064452)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1) 
                OR  condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1326492, 1326493, 1326588, 1326589, 1326590, 1326591, 1326600, 1326601, 1326602, 1326603, 1326604, 1326605, 1326606, 1326607, 1326608, 1326609, 1553751, 1553752, 1553753, 1553754, 1567895, 1567896, 1567901, 1567902, 1567940, 1567943, 1567944, 1567949, 1567956, 1567958, 1567959, 1567960, 1567964, 1567965, 1567966, 1567969, 1567971, 1569124, 1569125, 1569126, 1569127, 1569133, 1569134, 1569135, 1569136, 1569145, 1569168, 1569170, 1569171, 1569172, 1569173, 1569174, 1569179, 1569486, 1569487, 1571687, 1571690, 1571691, 35206867, 35206878, 35206879, 35206881, 35206882, 35207668, 35207669, 35207670, 35207671, 35207672, 35207673, 35207674, 35207675, 35207676, 35207677, 35207678, 35207679, 35207680, 35207681, 35207682, 35207683, 35207684, 35207685, 35207686, 35207687, 35207688, 35207689, 35207691, 35207692, 35207693, 35207694, 35207695, 35207696, 35207697, 35207698, 35207699, 35207700, 35207701, 35207702, 35207703, 35207704, 35207705, 35207706, 35207779,
 35207784, 35207785, 35207792, 35207793, 35207800, 35207801, 35207802, 35208014, 35208015, 35208016, 35208017, 35208018, 35208019, 35208020, 35208021, 35208022, 35208023, 35208024, 35208025, 35210610, 35210623, 35225414, 37200141, 37200142, 37200143, 37200144, 37200145, 37200146, 37200147, 37200148, 37200149, 37200150, 37200151, 37200152, 37200153, 37200154, 37200155, 37200156, 37200157, 37200158, 37200159, 37200160, 37200161, 37200162, 37200163, 37200164, 37200165, 37200166, 37200167, 37200168, 37200170, 37200171, 37200172, 37200175, 37200176, 37200177, 37200180, 37200181, 37200182, 37200185, 37200186, 37200187, 37200188, 37200189, 37200190, 37200191, 37200192, 37200194, 37200195, 37200196, 37200198, 37200199, 37200200, 37200201, 37200202, 37200203, 37200204, 37200205, 37200206, 37200207, 37200208, 37200209, 37200210, 37200211, 37200212, 37200213, 37200214, 37200215, 37200216, 37200217, 37200218, 37200219, 37200220, 37200221, 37200222, 37200223, 37200224, 37200225, 37200227, 37200228,
 37200229, 37200230, 37200232, 37200233, 37200234, 37200235, 37200237, 37200238, 37200239, 37200240, 37200242, 37200243, 37200244, 37200245, 37200246, 37200247, 37200248, 37200249, 37200251, 37200252, 37200253, 37200254, 37200977, 37200979, 37402489, 37402491, 37402492, 37402498, 44819495, 44819496, 44819500, 44819501, 44819502, 44819503, 44819504, 44819702, 44819703, 44820047, 44820678, 44820679, 44820682, 44820683, 44820684, 44820685, 44820861, 44820868, 44820869, 44820888, 44820889, 44821783, 44821787, 44821870, 44821916, 44821951, 44821953, 44821957, 44821986, 44822099, 44822929, 44822930, 44822934, 44822935, 44822936, 44823040, 44824070, 44824071, 44824072, 44824073, 44824074, 44824238, 44824248, 44825258, 44825264, 44825349, 44825430, 44825474, 44826454, 44826455, 44826459, 44826460, 44826461, 44826573, 44826642, 44826677, 44827612, 44827615, 44827616, 44827617, 44827794, 44827796, 44827822, 44828784, 44828785, 44828793, 44828794, 44828795, 44829010, 44829011, 44829013, 44829639,
 44829872, 44829878, 44829879, 44829880, 44829881, 44829882, 44830080, 44830081, 44830086, 44830114, 44830221, 44831045, 44831046, 44831047, 44831148, 44831239, 44831248, 44831280, 44832190, 44832191, 44832192, 44832193, 44832194, 44832299, 44832300, 44832421, 44832422, 44832532, 44832533, 44833365, 44833366, 44833367, 44833368, 44833465, 44833563, 44833573, 44834544, 44834545, 44834548, 44834549, 44834647, 44834726, 44834732, 44835743, 44835928, 44835931, 44835932, 44835980, 44836906, 44836907, 44836908, 44836914, 44836915, 44836916, 44836917, 44836918, 44837134, 44837136, 44837245, 45533017, 45533018, 45533019, 45533020, 45533021, 45533022, 45533023, 45533436, 45533439, 45534188, 45534189, 45537949, 45537951, 45537958, 45537960, 45537961, 45537962, 45538373, 45538374, 45538375, 45538376, 45538377, 45538383, 45538387, 45539105, 45542725, 45542736, 45542737, 45542738, 45543164, 45543167, 45543168, 45543182, 45543921, 45543923, 45547615, 45547621, 45547622, 45547623, 45547624, 45547625,
 45547626, 45547627, 45548010, 45548011, 45548012, 45548013, 45548715, 45552379, 45552381, 45552382, 45552383, 45552385, 45552386, 45553483, 45553484, 45557103, 45557110, 45557111, 45557112, 45557113, 45557536, 45557538, 45557539, 45558214, 45558215, 45561947, 45561948, 45561949, 45562340, 45562343, 45562344, 45563058, 45563060, 45563061, 45566729, 45566731, 45567167, 45567168, 45567180, 45567183, 45567896, 45567897, 45571654, 45572079, 45572080, 45572091, 45572094, 45572770, 45572771, 45576427, 45576428, 45576437, 45576438, 45576439, 45576440, 45576443, 45576865, 45576866, 45576868, 45576876, 45576878, 45577566, 45577567, 45581339, 45581340, 45581349, 45581350, 45581352, 45581353, 45581354, 45581355, 45581766, 45582457, 45582458, 45582459, 45582461, 45586138, 45586139, 45586140, 45586572, 45586573, 45586574, 45586575, 45586587, 45587291, 45587292, 45587294, 45591026, 45591027, 45591029, 45591030, 45591031, 45591456, 45591458, 45591459, 45591460, 45591461, 45592198, 45595787, 45595788,
 45595793, 45595794, 45595795, 45595797, 45595798, 45595799, 45596188, 45596194, 45596197, 45596198, 45596199, 45596929, 45600630, 45600631, 45600636, 45600637, 45600638, 45600639, 45600640, 45600641, 45600642, 45601024, 45601026, 45601027, 45601028, 45601785, 45605390, 45605397, 45605398, 45605401, 45605402, 45605403, 45605404, 45605405, 45605779, 45605781, 45605784, 45605785, 45605786, 45605787, 45605788, 45606547)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) )
            )) c_occurrence"""

icd = pd.read_gbq(
    dataset_36851903_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

print(icd.shape)
icd.head(5)

## observation

In [ ]:
# This query represents dataset "CAD_Afib_IRM" for domain "observation" and was generated for All of Us Controlled Tier Dataset v8
dataset_32928699_observation_sql = """
    SELECT
        observation.person_id,
        observation.observation_datetime,
        observation.observation_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.observation` observation 
        WHERE
            (
                observation_concept_id IN (21492409, 21492410, 21492411, 21492412, 21492413, 21492414, 21492415, 21493945, 2617330, 2617331, 2617829, 2721640, 2721670, 3018063, 3043579, 38001390, 38001391, 4015401, 40217103, 40217330, 40217341, 4027529, 4031367, 4031370, 40483697, 4052051, 4053609, 4053842, 4063124, 40664614, 40664668, 40664817, 4069297, 4074782, 40766231, 40766232, 40766311, 40766360, 40766642, 40766646, 4083596, 4090847, 4098471, 4132133, 4151486, 4165523, 4165539, 4182465, 4182573, 4193014, 4195380, 4206526, 4213783, 4224317, 4237536, 42528763, 42528764, 4261613, 4267497, 4268546, 4270273, 4278461, 4278980, 4282779, 43021337, 4303438, 4304362, 43054909, 4305714, 4307293, 4310138, 4337839, 4338966, 43533208, 443359, 44786552, 44786668, 44792317, 44808782, 45890719, 762525, 763402, 763736, 765735, 801308, 915749) 
                OR  observation_source_concept_id IN (1585375, 1585741, 1585857, 1585860, 1585864, 1585867, 1585870, 1585873, 1585940, 1586159, 1586162, 1586166, 1586169, 1586174, 1586177, 1586182, 1586185, 1586198, 1586201, 1586207, 1586213, 35225414, 37200977, 40192384, 40192386, 40192400, 40192404, 40192410, 40192411, 40192412, 40192414, 40192417, 40192420, 40192431, 40192436, 40192437, 40192440, 40192456, 40192457, 40192458, 40192469, 40192476, 40192492, 40192493, 40192499, 40192500, 40192522, 44829590, 44829639, 45553483, 45571397, 45572770, 45582459, 45582461, 45585890, 45600362)
            )  
            AND (
                observation.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) )
            )) observation"""

observation = pd.read_gbq(
    dataset_32928699_observation_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

print(observation.shape)
observation.head(5)

## labs

In [ ]:
# This query represents dataset "CAD_Afib_IRM" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v8
dataset_55849728_measurement_sql = """
    SELECT
        measurement.person_id,
        m_standard_concept.concept_name as standard_concept_name,
        measurement.measurement_datetime,
        measurement.value_as_number,
        m_unit.concept_name as unit_concept_name,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1002597, 1175729, 1175898, 1175959, 1176069, 1176311, 1176351, 1616654, 1618572, 1619025, 1619026, 21490674, 21490675, 21490678, 21490680, 21490851, 21490852, 21490853, 21490872, 21492239, 21492240, 21492241, 21492391, 21494214, 21494261, 21494263, 3000034, 3000261, 3000333, 3000404, 3000483, 3000637, 3000819, 3000837, 3000845, 3000911, 3000940, 3001308, 3001318, 3001349, 3001501, 3001506, 3001547, 3001582, 3001802, 3001810, 3001975, 3001978, 3002000, 3002009, 3002109, 3002185, 3002240, 3002310, 3002574, 3002590, 3002598, 3002651, 3002666, 3002698, 3002785, 3002812, 3002827, 3003169, 3003201, 3003270, 3003309, 3003403, 3003412, 3003435, 3003453, 3003767, 3003772, 3003932, 3003994, 3004049, 3004077, 3004117, 3004173, 3004195, 3004209, 3004239, 3004249, 3004410, 3004501, 3004529, 3004629, 3004699, 3004764, 3005030, 3005031, 3005071, 3005131, 3005478, 3005570, 3005577, 3005606, 3005673, 3005770, 3005787, 3005802, 3006277, 3006698, 3006717, 3006887,
 3006893, 3007034, 3007070, 3007081, 3007263, 3007295, 3007332, 3007352, 3007393, 3007424, 3007696, 3007794, 3007798, 3007862, 3007943, 3008103, 3008286, 3008392, 3008512, 3008612, 3008631, 3008770, 3008804, 3008846, 3008960, 3009160, 3009224, 3009261, 3009395, 3009397, 3009414, 3009435, 3009456, 3009514, 3009582, 3009596, 3009718, 3009877, 3009913, 3009966, 3010115, 3010279, 3010300, 3010391, 3010617, 3010673, 3010926, 3010956, 3011088, 3011161, 3011163, 3011367, 3011424, 3011498, 3011544, 3011884, 3011972, 3012278, 3012516, 3012526, 3012789, 3012805, 3012888, 3013078, 3013097, 3013104, 3013186, 3013247, 3013290, 3013473, 3013512, 3013604, 3013668, 3013678, 3013702, 3013742, 3013826, 3013940, 3014053, 3014072, 3014145, 3014305, 3014323, 3014444, 3014600, 3014716, 3014737, 3015024, 3015178, 3015204, 3015232, 3015397, 3015544, 3015548, 3015621, 3015739, 3015968, 3016083, 3016087, 3016159, 3016231, 3016494, 3016589, 3016652, 3016662, 3016699, 3016701, 3016723, 3016791, 3016798, 3016945,
 3017188, 3017250, 3017365, 3017439, 3017490, 3017589, 3017925, 3018056, 3018097, 3018104, 3018251, 3018311, 3018336, 3018586, 3018592, 3018822, 3019013, 3019210, 3019240, 3019431, 3019464, 3019493, 3019575, 3019797, 3019876, 3019900, 3019955, 3019962, 3019999, 3020044, 3020058, 3020059, 3020107, 3020222, 3020275, 3020317, 3020324, 3020345, 3020399, 3020491, 3020509, 3020525, 3020564, 3020632, 3020650, 3020660, 3020682, 3020869, 3021194, 3021261, 3021447, 3021492, 3021513, 3021706, 3021737, 3021797, 3021860, 3021924, 3022022, 3022038, 3022192, 3022268, 3022285, 3022314, 3022449, 3022482, 3022548, 3022803, 3022826, 3023024, 3023186, 3023243, 3023306, 3023386, 3023556, 3023572, 3023574, 3023602, 3023752, 3023884, 3024047, 3024268, 3024354, 3024474, 3024561, 3024629, 3024723, 3024762, 3025070, 3025202, 3025232, 3025313, 3025380, 3025398, 3025656, 3025673, 3025750, 3025809, 3025839, 3025866, 3025987, 3026041, 3026071, 3026300, 3026327, 3026453, 3026536, 3026588, 3026677, 3026692, 3026876,
 3027035, 3027074, 3027114, 3027198, 3027315, 3027358, 3027457, 3027468, 3027589, 3027597, 3027598, 3027776, 3027801, 3027936, 3027939, 3027946, 3027997, 3028024, 3028068, 3028074, 3028195, 3028247, 3028286, 3028288, 3028303, 3028437, 3028626, 3028646, 3028737, 3028803, 3028827, 3028944, 3029071, 3029133, 3029139, 3029246, 3029306, 3029335, 3029829, 3029859, 3029937, 3030104, 3030177, 3030260, 3030366, 3030416, 3030437, 3030441, 3030511, 3030997, 3031203, 3031266, 3031412, 3031465, 3031581, 3031681, 3031775, 3031973, 3032230, 3032260, 3032693, 3032710, 3032719, 3032759, 3032771, 3032779, 3032800, 3032987, 3033203, 3033254, 3033364, 3033408, 3033638, 3033819, 3033837, 3033962, 3034207, 3034485, 3034530, 3034639, 3034703, 3034826, 3034884, 3034962, 3035009, 3035125, 3035214, 3035250, 3035352, 3035398, 3035472, 3035521, 3035529, 3035729, 3035759, 3035774, 3035817, 3035856, 3035858, 3035899, 3035982, 3036283, 3036311, 3036406, 3036634, 3036671, 3036807, 3036895, 3037014, 3037052, 3037110,
 3037187, 3037292, 3037524, 3037539, 3037653, 3037791, 3037883, 3038071, 3038276, 3038515, 3038553, 3038609, 3038920, 3038988, 3039422, 3039426, 3039436, 3039720, 3039775, 3039851, 3039873, 3039896, 3039986, 3040151, 3040290, 3040324, 3040820, 3041024, 3041198, 3041253, 3041290, 3042122, 3042145, 3042216, 3042443, 3042462, 3042637, 3043209, 3043687, 3043798, 3043954, 3044242, 3044491, 3044671, 3044889, 3044929, 3044974, 3045156, 3045262, 3045322, 3045462, 3045566, 3045588, 3045700, 3045800, 3045807, 3045989, 3046055, 3046076, 3046405, 3046485, 3046528, 3046549, 3046708, 3046948, 3047023, 3047107, 3047111, 3048516, 3048522, 3048865, 3049187, 3049202, 3049506, 3049783, 3049873, 3050006, 3050449, 3050630, 3050988, 3051825, 3052202, 3052261, 3052566, 3052598, 3053001, 3053190, 3053283, 3053286, 3053341, 35918840, 35976969, 36031550, 36032094, 36032229, 36032294, 36032380, 36032416, 36032787, 36033366, 36033462, 36203185, 36204193, 36303264, 36303797, 36304016, 36304157, 36304326, 36304734,
 36304833, 36304864, 36305059, 36306043, 36306178, 36660257, 36716965, 37019625, 37019651, 37019808, 37019887, 37020015, 37020076, 37020303, 37020736, 37020823, 37020997, 37021192, 37021763, 37021905, 37021968, 37023397, 37024420, 37024641, 37024929, 37025002, 37025537, 37025901, 37025979, 37026560, 37026687, 37026756, 37028490, 37028633, 37029308, 37029357, 37029387, 37029419, 37029979, 37030222, 37030259, 37030713, 37031175, 37031498, 37032253, 37032863, 37033044, 37033544, 37033776, 37034437, 37034761, 37035033, 37035201, 37035650, 37036139, 37036806, 37037336, 37037743, 37038593, 37038731, 37038784, 37039181, 37039374, 37040279, 37040418, 37040621, 37040744, 37040799, 37041042, 37042190, 37043215, 37043257, 37043506, 37044332, 37045117, 37045253, 37045382, 37045508, 37046668, 37047717, 37048053, 37048668, 37048896, 37048990, 37051275, 37051495, 37052309, 37053001, 37053214, 37053398, 37053972, 37054145, 37054380, 37055462, 37055869, 37056546, 37056686, 37056793, 37056881, 37057730,
 37058220, 37058684, 37058704, 37058713, 37058931, 37059330, 37059768, 37060498, 37060543, 37060570, 37060933, 37062148, 37062967, 37063613, 37064108, 37064187, 37064430, 37064590, 37065054, 37065592, 37067978, 37068379, 37069099, 37069481, 37070097, 37071197, 37072167, 37072239, 37072845, 37073196, 37073389, 37073694, 37073796, 37074028, 37074175, 37074494, 37074896, 37076636, 37076649, 37078231, 37078343, 37078633, 37078789, 37078832, 37205200, 37393576, 37398559, 37399119, 37399654, 4008265, 4012477, 4012479, 4013964, 4016950, 4017078, 4017083, 4017497, 4017498, 4017760, 4017787, 4018315, 4018317, 4018318, 4019543, 4027514, 4027681, 4032789, 4036846, 4041556, 4041697, 4041698, 4042059, 4042066, 4042759, 40479187, 40482666, 40482677, 40485039, 4055661, 4055667, 4055668, 4055695, 4060832, 4060833, 4060834, 4068414, 40757369, 40757478, 40757503, 40757565, 40757569, 40758413, 40758569, 40758736, 40759156, 40759279, 40759280, 40760031, 40760483, 40760809, 40761549, 40761610, 40762249,
 40762252, 40762352, 40762636, 40762637, 40762638, 40763912, 40763913, 40764999, 40765014, 40765543, 40766275, 4076704, 40767663, 40768039, 40768057, 40768809, 40769159, 40771054, 40771471, 40771922, 40772572, 40775801, 40782589, 40782761, 40785880, 40789192, 40789378, 40791227, 40795740, 40795765, 40795800, 40796112, 4094447, 4094581, 4097664, 4097882, 4101713, 4108289, 4108431, 4112223, 4116187, 4120298, 4136579, 4136584, 4143633, 4144235, 4149519, 4149883, 4150342, 4150621, 4151414, 4151548, 4152194, 4152686, 4152996, 4153111, 4154347, 4154790, 4155367, 4156045, 4156660, 4182052, 4184637, 4190896, 4191837, 4193718, 4195214, 4195490, 4195503, 4197856, 4197971, 4198718, 4198733, 4209122, 4218282, 4229586, 4230393, 4230629, 4232915, 4236281, 4239021, 4245997, 4248524, 4248525, 4248803, 4249006, 42528717, 42537369, 4260765, 4264765, 4268883, 42868678, 42868692, 4286945, 42869619, 42869628, 42869630, 42869913, 4288601, 4289453, 4292062, 4298391, 4298393, 4302410, 43054914, 43055430,
 43055431, 43055432, 43055485, 4324383, 4331286, 43533980, 4353607, 4353851, 4354252, 4354254, 44786762, 44789161, 44789315, 44789316, 44794807, 44812280, 44816672, 46235168, 46235170, 46235434, 46236280, 46236281, 46236875, 46236952, 46236953, 46237026)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1) 
                OR  measurement_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (903106, 903107, 903109, 903110, 903113, 903114, 903115, 903116, 903118, 903124, 903129, 903130, 903132)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) )
            )) measurement 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
            ON measurement.unit_concept_id = m_unit.concept_id"""

labs = pd.read_gbq(
    dataset_55849728_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

print(labs.shape)
labs.head(5)

## survey

In [ ]:
# This query represents dataset "CAD_Afib_IRM" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_02761836_survey_sql = """
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.question,
        answer.answer  
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.ds_survey` answer   
    WHERE
        (
            question_concept_id IN (1585375, 1585741, 1585857, 1585860, 1585864, 1585867, 1585870, 1585873, 1585940, 1586159, 1586162, 1586166, 1586169, 1586174, 1586177, 1586182, 1586185, 1586198, 1586201, 1586207, 1586213, 40192384, 40192386, 40192400, 40192404, 40192410, 40192411, 40192412, 40192414, 40192417, 40192420, 40192431, 40192436, 40192437, 40192440, 40192456, 40192457, 40192458, 40192469, 40192476, 40192492, 40192493, 40192499, 40192500, 40192522)
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) )
        )"""

survey = pd.read_gbq(
    dataset_02761836_survey_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

print(survey.shape)
survey.head(5)

## medication

In [ ]:
# This query represents dataset "CAD_Afib_IRM" for domain "drug" and was generated for All of Us Controlled Tier Dataset v8
dataset_22809767_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_exposure.drug_exposure_start_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (1123698, 1510813, 1539403, 1545958, 1549686, 1551860, 1592085, 21600315, 21600337, 21600451, 21600452, 21600464, 21600466, 21600470, 21600472, 21600474, 21601461, 21601462, 21601489, 21601516, 21601541, 21601542, 21601553, 21601556, 21601560, 21601561, 21601587, 21601664, 21601665, 21601666, 21601682, 21601698, 21601701, 21601702, 21601709, 21601716, 21601718, 21601719, 21601724, 21601728, 21601730, 21601731, 21601733, 21601738, 21601741, 21601744, 21601745, 21601765, 21601779, 21601780, 21601783, 21601784, 21601801, 21601802, 21601815, 21601822, 21601823, 21601832, 21601833, 21601841, 21601845, 21603698, 40165636, 43534771, 963889)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) )
            )) d_exposure 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
            ON d_exposure.drug_concept_id = d_standard_concept.concept_id"""

drug = pd.read_gbq(
    dataset_22809767_drug_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

print(drug.shape)
drug.head(5)

## PGS

In [ ]:
!mkdir CAD_AFIB

In [ ]:
!gsutil -m cp -R ${WORKSPACE_BUCKET}/Cardio_IRM/AOU/ CAD_AFIB/

In [ ]:
cad_afib_pgs = pd.read_csv('CAD_AFIB/AOU/score/AOU_pgs.txt.gz', sep = '\t', usecols = ['IID', 'PGS', 'Z_norm2'], low_memory = False)
print(len(cad_afib_pgs.index))
cad_afib_pgs.head()

In [ ]:
!mkdir HF

In [ ]:
!gsutil -m cp -R ${WORKSPACE_BUCKET}/HF/AOU/ HF/

In [ ]:
hf_pgs = pd.read_csv('HF/AOU/score/AOU_pgs.txt.gz', sep = '\t', usecols = ['IID', 'PGS', 'Z_norm2'], low_memory = False)
print(len(hf_pgs.index))
hf_pgs.head()

## PREVENT CRS

In [ ]:
cad_prevent_crs = pd.read_csv('CAD.PREVENT_CRS.10_year_risk.txt', sep = '\t')
print(cad_prevent_crs.shape)
cad_prevent_crs.head()

In [ ]:
afib_prevent_crs = pd.read_csv('AFIB.PREVENT_CRS.10_year_risk.txt', sep = '\t')
print(afib_prevent_crs.shape)
afib_prevent_crs.head()

In [ ]:
hf_prevent_crs = pd.read_csv('HF.PREVENT_CRS.10_year_risk.txt', sep = '\t')
print(hf_prevent_crs.shape)
hf_prevent_crs.head()

# clean demo df

## filter to males and females

In [ ]:
demo_sex = demo[demo['sex_at_birth'].isin(['Male', 'Female'])]
demo_sex['sex_at_birth'].unique()

## recode sex column

In [ ]:
demo_sex['sex_at_birth'] = demo_sex['sex_at_birth'].str.replace('Male', '1')
demo_sex['sex_at_birth'] = demo_sex['sex_at_birth'].str.replace('Female', '2')
demo_sex['sex_at_birth'].unique()

## rename sex column

In [ ]:
demo_sex.rename(columns = {'sex_at_birth' : 'SEX'}, inplace = True)
demo_sex.head()

## convert DOB column to datetime

In [ ]:
demo_sex['date_of_birth']=pd.to_datetime(demo_sex['date_of_birth'])

# clean icd/observation/labs/med dfs for ICD based phenotypes

## split by phenotype
- CAD
- AFIB
- HF
- T2D
- T1D
- All Diabetes
- COPD
- Hyperthyroidism

In [ ]:
cad = icd[icd['condition_source_value'].str.contains('410|411|412|414|I20|I21|I22|I23|I24|I25|I46|V45|Z95|Z98')]
cad = cad[~cad['condition_source_value'].str.contains('E11|O24|E10')]
print(cad['condition_source_value'].unique())
print(len(cad['person_id'].unique()))
cad.head()

In [ ]:
afib = icd[icd['condition_source_value'].str.contains('I48|427')]
print(afib['condition_source_value'].unique())
print(len(afib['person_id'].unique()))
afib.head()

In [ ]:
hf = icd[icd['condition_source_value'].str.contains('428.0|428.1|428.9|I50.1|I50.0|I50.9|I13|I51|I52')]
print(hf['condition_source_value'].unique())
print(len(hf['person_id'].unique()))
hf.head()

In [ ]:
t2d = icd[icd['condition_source_value'].str.contains(
    '250.90|250.92|250.80|250.82|250.70|250.72|250.60|250.62|250.50|250.52|250.40|250.42|250.30|250.32|250.20|250.22|250.10|250.12|250.00|250.02|E11|O24.11|O24.12|O24.13')]
print(t2d['condition_source_value'].unique())
print(len(t2d['person_id'].unique()))
t2d.head()

In [ ]:
t1d = icd[icd['condition_source_value'].str.contains(
    '250.91|250.93|250.81|250.83|250.71|250.73|250.61|250.63|250.51|250.53|250.41|250.43|250.31|250.33|250.21|250.23|250.11|250.13|250.01|250.03|E10|O24.0')]
print(t1d['condition_source_value'].unique())
print(len(t1d['person_id'].unique()))
t1d.head()

In [ ]:
diabetes = icd[icd['condition_source_value'].str.contains('O24|Z86.31|Z86.32|P70.2|250|648.0|V12.21|362.0|357.2|775.1|E11|E10')]
print(diabetes['condition_source_value'].unique())
print(len(diabetes['person_id'].unique()))
diabetes.head()

In [ ]:
copd = icd[icd['condition_source_value'].str.contains('J44|496|491|J41|J42|J43|492|493.20|493.21|493.22')]
print(copd['condition_source_value'].unique())
print(len(copd['person_id'].unique()))
copd.head()

In [ ]:
sys_hf = icd[icd['condition_source_value'].str.contains('I50.2|428.2')]
print(sys_hf['condition_source_value'].unique())
print(len(sys_hf['person_id'].unique()))
sys_hf.head()

In [ ]:
hyperthyroidism = icd[icd['condition_source_value'].str.contains('E05|242|376.21|E06.2|P72.1')]
print(hyperthyroidism['condition_source_value'].unique())
print(len(hyperthyroidism['person_id'].unique()))
hyperthyroidism.head()

In [ ]:
hypertension = icd[icd['condition_source_value'].str.contains('I10|I11|I12|I13|I15')]
print(hypertension['condition_source_value'].unique())
print(len(hypertension['person_id'].unique()))
hypertension.head()

## add in codes in observation df

### look at icd codes in observation df

In [ ]:
observation[observation['observation_source_value'].str.contains(r'\.', regex = True, na = False)]['observation_source_value'].unique()

### filter to ICD codes

In [ ]:
cad_observation_codes = observation[observation['observation_source_value'].str.contains('Z95|Z98|V45', na = False)]
print(cad_observation_codes['observation_source_value'].unique())
cad_observation_codes.head()

In [ ]:
diabetes_observation_codes = observation[observation['observation_source_value'].str.contains('Z86|V12|O24', na = False)]
print(diabetes_observation_codes['observation_source_value'].unique())
diabetes_observation_codes.head()

### subset and rename

In [ ]:
cad_observation_codes = cad_observation_codes[['person_id', 'observation_datetime', 'observation_source_value']]
cad_observation_codes.columns = cad_observation_codes.columns.str.replace('observation', 'condition')
cad_observation_codes.columns = cad_observation_codes.columns.str.replace('datetime', 'start_datetime')
cad_observation_codes.head()

In [ ]:
diabetes_observation_codes = diabetes_observation_codes[['person_id', 'observation_datetime', 'observation_source_value']]
diabetes_observation_codes.columns = diabetes_observation_codes.columns.str.replace('observation', 'condition')
diabetes_observation_codes.columns = diabetes_observation_codes.columns.str.replace('datetime', 'start_datetime')
diabetes_observation_codes.head()

### concat

In [ ]:
print(len(cad['person_id'].unique()))
print(len(cad.index))
cad = pd.concat([cad, cad_observation_codes], axis = 0)
print(len(cad['person_id'].unique()))
print(len(cad.index))
cad.head()

In [ ]:
print(len(diabetes['person_id'].unique()))
print(len(diabetes.index))
diabetes = pd.concat([diabetes, diabetes_observation_codes], axis = 0)
print(len(diabetes['person_id'].unique()))
print(len(diabetes.index))
diabetes.head()

## add labs for t2d and ht

### create mad function

In [ ]:
def mad_filter(df, threshold = 5):
    values = df['value_as_number'].dropna()
    
    # Compute median and MAD
    median = values.median()
    mad = np.median(np.abs(values - median))
    
    # Define limits
    lower_limit = median - threshold * mad
    upper_limit = median + threshold * mad

    # Filter and add top/bottom limits
    filtered_df = df[(df['value_as_number'] >= lower_limit) & (df['value_as_number'] <= upper_limit)].copy()

    return filtered_df

### glucose
- diabetes case: 11.1 mmol/L >= glucose
- none meeting criteria after applying mad filter

#### filter by measurement

In [ ]:
glucose = labs[labs['standard_concept_name'].str.contains('glucose', case = False)]
glucose = glucose[~glucose['standard_concept_name'].str.contains('Enzymatic|DBS|Presence|measurement|Cerebral|time|tolerance|Laboratory|level|Ratio|PO|Insulin|meter|Self-monitoring|fluid|qualitative|Specimen|Entitic|test|Urine|Blood', case = False)]
glucose['standard_concept_name'].unique()

#### filter to non fasting

In [ ]:
not_fasting = glucose[glucose['standard_concept_name'].isin(['Glucose [Mass/volume] in Serum or Plasma',
                                                             'Glucose [Moles/volume] in Serum or Plasma'])]
print(len(not_fasting['person_id'].unique()))
print(not_fasting['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

#### check units

In [ ]:
not_fasting.groupby('unit_concept_name')['value_as_number'].describe()

In [ ]:
not_fasting.groupby('unit_source_value')['value_as_number'].describe()

#### convert units

In [ ]:
g_dl = not_fasting[not_fasting['unit_source_value'].isin(['g/dL'])]
print(g_dl['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
g_dl['value_as_number'] = g_dl['value_as_number'] * 55.5
print(g_dl['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

In [ ]:
mg_dl = not_fasting[not_fasting['unit_source_value'].isin(['mg/dL'])]
print(mg_dl['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
mg_dl['value_as_number'] = mg_dl['value_as_number'] * 0.0555
print(mg_dl['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

In [ ]:
mg_ml = not_fasting[not_fasting['unit_source_value'].isin(['mg/mL'])]
print(mg_ml['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
mg_ml['value_as_number'] = mg_ml['value_as_number'] * 5.55
print(mg_ml['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

In [ ]:
ng_dl = not_fasting[not_fasting['unit_source_value'].isin(['ng/dL'])]
print(ng_dl['value_as_number'].describe().apply(lambda x: f'{x:,.5f}'))
ng_dl['value_as_number'] = ng_dl['value_as_number'] * 5.55e-8
print(ng_dl['value_as_number'].describe().apply(lambda x: f'{x:,.5f}'))

In [ ]:
not_fasting_unit_fixed = pd.concat([g_dl, mg_dl, mg_ml, ng_dl], axis = 0)
print(len(not_fasting.index))
print(len(not_fasting['person_id'].unique()))
print(len(not_fasting_unit_fixed.index))
print(len(not_fasting_unit_fixed['person_id'].unique()))
print(not_fasting_unit_fixed['value_as_number'].describe().apply(lambda x: f'{x:,.5f}'))

#### apply mad

In [ ]:
glucose_normal = mad_filter(not_fasting_unit_fixed)
print(len(glucose_normal['person_id'].unique()))
print(glucose_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
glucose_normal.head()

### hba1c

#### filter by measurement name

In [ ]:
hba1c = labs[labs['standard_concept_name'].str.contains('hba1c|hemoglobin|a1c', case = False)]
hba1c = hba1c[hba1c['standard_concept_name'].str.contains('A1c/')]
hba1c = hba1c[~hba1c['standard_concept_name'].str.contains('DBS')]
hba1c['standard_concept_name'].unique()

#### filter to percent only

In [ ]:
hba1c_unit = hba1c[hba1c['unit_concept_name'].str.contains('percent', case = False, na = False)]
print(hba1c_unit['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(hba1c_unit['person_id'].unique()))

#### apply mad

In [ ]:
hba1c_normal = mad_filter(hba1c_unit)
print(len(hba1c_normal['person_id'].unique()))
print(hba1c_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

#### filter to diabetes

In [ ]:
hba1c_diabetes = hba1c_normal[hba1c_normal['value_as_number'] >= 6.5]
print(len(hba1c_diabetes.index))
print(len(hba1c_diabetes['person_id'].unique()))

#### subset and rename

In [ ]:
hba1c_diabetes_sub = hba1c_diabetes[['person_id', 'measurement_datetime', 'value_as_number']]
hba1c_diabetes_sub = hba1c_diabetes_sub.rename(columns = {'measurement_datetime' : 'condition_start_datetime',
                                                          'value_as_number' : 'condition_source_value'})
hba1c_diabetes_sub.head()

#### concat

In [ ]:
print(len(diabetes['person_id'].unique()))
print(len(diabetes.index))
diabetes = pd.concat([diabetes, hba1c_diabetes_sub], axis = 0)
print(len(diabetes['person_id'].unique()))
print(len(diabetes.index))
diabetes.head()

In [ ]:
print(len(t2d['person_id'].unique()))
print(len(t2d.index))
t2d = pd.concat([t2d, hba1c_diabetes_sub], axis = 0)
print(len(t2d['person_id'].unique()))
print(len(t2d.index))
t2d.head()

In [ ]:
print(len(t1d['person_id'].unique()))
print(len(t1d.index))
t1d = pd.concat([t1d, hba1c_diabetes_sub], axis = 0)
print(len(t1d['person_id'].unique()))
print(len(t1d.index))
t1d.head()

### SBP

#### filter by measurement name

In [ ]:
systolic = labs[labs['standard_concept_name'].str.contains('systolic|pressure', case = False)]
systolic = systolic[~systolic['standard_concept_name'].str.contains('diastolic|oxygen|carbon|invasive|Cuff|venous|Arterial|Type|ventricular|toe|Mean|Artery|Atrial|panel|Aorta|taking|ambulatory|anesthesia|exercise', case = False)]
systolic = systolic[~systolic['standard_concept_name'].isin(['Sitting blood pressure',
                                                             'Standing blood pressure'])]
systolic['standard_concept_name'].unique()

#### filter by unit

In [ ]:
systolic_unit = systolic[systolic['unit_concept_name'] == 'millimeter mercury column']
print(systolic_unit['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(systolic_unit['person_id'].unique()))

#### apply mad

In [ ]:
systolic_normal = mad_filter(systolic_unit)
print(len(systolic_normal['person_id'].unique()))
print(systolic_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
systolic_normal.head()

#### filter to HT

In [ ]:
systolic_ht = systolic_normal[systolic_normal['value_as_number'] >= 140]
print(len(systolic_ht.index))
print(len(systolic_ht['person_id'].unique()))

#### subset and rename

In [ ]:
systolic_ht_sub = systolic_ht[['person_id', 'measurement_datetime', 'value_as_number']]
systolic_ht_sub = systolic_ht_sub.rename(columns = {'measurement_datetime' : 'condition_start_datetime',
                                                    'value_as_number' : 'condition_source_value'})
systolic_ht_sub.head()

#### concatenate

In [ ]:
print(len(hypertension['person_id'].unique()))
print(len(hypertension.index))
hypertension = pd.concat([hypertension, systolic_ht_sub], axis = 0)
print(len(hypertension['person_id'].unique()))
print(len(hypertension.index))
hypertension.head()

### diastolic bp

#### filter by measurement name

In [ ]:
diastolic = labs[labs['standard_concept_name'].str.contains('diastolic', case = False)]
diastolic = diastolic[~diastolic['standard_concept_name'].str.contains('artery|systolic|arterial|Invasive', case = False)]
diastolic['standard_concept_name'].unique()

#### filter by unit

In [ ]:
diastolic_unit = diastolic[diastolic['unit_concept_name'] == 'millimeter mercury column']
print(diastolic_unit['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(diastolic_unit['person_id'].unique()))

#### apply mad

In [ ]:
diastolic_normal = mad_filter(diastolic_unit)
print(len(diastolic_normal['person_id'].unique()))
print(diastolic_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
diastolic_normal.head()

#### filter to ht

In [ ]:
diastolic_ht = diastolic_normal[diastolic_normal['value_as_number'] >= 90]
print(len(diastolic_ht.index))
print(len(diastolic_ht['person_id'].unique()))

#### subset and rename

In [ ]:
diastolic_ht_sub = diastolic_ht[['person_id', 'measurement_datetime', 'value_as_number']]
diastolic_ht_sub = diastolic_ht_sub.rename(columns = {'measurement_datetime' : 'condition_start_datetime',
                                                      'value_as_number' : 'condition_source_value'})
diastolic_ht_sub.head()

#### concat

In [ ]:
print(len(hypertension['person_id'].unique()))
print(len(hypertension.index))
hypertension = pd.concat([hypertension, diastolic_ht_sub], axis = 0)
print(len(hypertension['person_id'].unique()))
print(len(hypertension.index))
hypertension.head()

## add meds for ht

### make list of meds

In [ ]:
med_name_df = pd.DataFrame({'med_name' : ['Furosemide',
                                          'Lasix',
                                          'Bumetanide',
                                          'Bumex',
                                          'Torsemide',
                                          'Demadex',
                                          'Amiloride',
                                          'Midamor', 
                                          'Hydro-ride',
                                          'Hydrochlorothiazide',
                                          'HCTZ',
                                          'Esidrix',
                                          'Hydrodiuril',
                                          'Indapamide',
                                          'Lozol',
                                          'Triamterene',
                                          'Dyrenium',
                                          'Chlorthalidone',
                                          'Thalitone',
                                          'Spironolactone',
                                          'Aldactone',
                                          'Eplerenone',
                                          'Inspra',
                                          'Atenolol',
                                          'Tenormin',
                                          'Bisoprolol',
                                          'Zebeta',
                                          'Carvedilol',
                                          'Coreg',
                                          'Carvedilol phosphate',
                                          'Coreg CR',
                                          'Labetalol',
                                          'Trandate',
                                          'Metoprolol succinate',
                                          'Toprol XL',
                                          'Kapspargo Sprinkle',
                                          'Metoprolol tartrate',
                                          'Lopressor',
                                          'Nadolol',
                                          'Corgard',
                                          'Nebivolol',
                                          'Bystolic',
                                          'Pindolol',
                                          'Visken',
                                          'Propranolol',
                                          'Inderal',
                                          'Inderal LA',
                                          'InnoPran XL',
                                          'Captopril',
                                          'Capoten',
                                          'Enalapril',
                                          'Vasotec',
                                          'Fosinopril',
                                          'Monopril',
                                          'Lisinopril',
                                          'Prinivil',
                                          'Zestril',
                                          'Perindopril',
                                          'Aceon',
                                          'Quinapril',
                                          'Accupril',
                                          'Ramipril',
                                          'Altace',
                                          'Trandolapril',
                                          'Mavik',
                                          'Benazepril',
                                          'Lotensin',
                                          'Moexipril',
                                          'Univasc',
                                          'Candesartan',
                                          'Atacand',
                                          'Losartan',
                                          'Cozaar',
                                          'Valsartan',
                                          'Diovan',
                                          'Irbesartan',
                                          'Avapro',
                                          'Olmesartan',
                                          'Benicar',
                                          'Telmisartan',
                                          'Micardis',
                                          'Azilsartan',
                                          'Edarbi',
                                          'Amlodipine',
                                          'Norvasc',
                                          'Diltiazem',
                                          'Cardizem CD',
                                          'Cardizem SR',
                                          'Dilacor XR',
                                          'Tiazac',
                                          'Felodipine',
                                          'Plendil',
                                          'Nicardipine',
                                          'Cardene SR',
                                          'Nifedipine LA',
                                          'Adalat CC',
                                          'Procardia XL',
                                          'Verapamil',
                                          'Calan SR',
                                          'Covera HS',
                                          'Isoptin SR',
                                          'Verelan',
                                          'Doxazosin',
                                          'Cardura',
                                          'Prazosin',
                                          'Minipress',
                                          'Terazosin hydrochloride',
                                          'Hytrin',
                                          'Methyldopa',
                                          'Aldomet',
                                          'Clonidine',
                                          'Catapres',
                                          'Duraclon',
                                          'Kapvay',
                                          'Nexiclon XR',
                                          'Guanfacine',
                                          'Intuniv',
                                          'Tenex',
                                          'Hydralazine',
                                          'Apresoline',
                                          'Minoxidil',
                                          'Loniten',
                                          'Cozaar',
                                          'Calan',
                                          'verapamil hydrochloride',
                                          'metoprolol tartrate',
                                          'spironolactone',
                                          'Cozaar',
                                          'Calan',
                                          'verapamil hydrochloride',
                                          'metoprolol tartrate',
                                          'spironolactone',
                                          'hydralazine hydrochloride',
                                          'Apresoline',
                                          'clonidine hydrochloride',
                                          'diltiazem hydrochloride',
                                          'Cartia',
                                          'losartan',
                                          'irbesartan',
                                          'captopril',
                                          'azilsartan medoxomil',
                                          'Edarbi',
                                          'lisinopril',
                                          'carvedilol phosphate',
                                          'Diltiazem',
                                          'hydrochlorothiazide',
                                          'furosemide',
                                          'olmesartan medoxomil',
                                          'nicardipine hydrochloride',
                                          'verapamil',
                                          'propranolol hydrochloride',
                                          'bisoprolol fumarate',
                                          'quinapril',
                                          'Accupril',
                                          'bendroflumethiazide',
                                          'nadolol',
                                          'Corzide',
                                          'chlorthalidone',
                                          'felodipine',
                                          'amiloride hydrochloride',
                                          'chlorthalidone',
                                          'Thalitone',
                                          'Lotensin',
                                          'benazepril hydrochloride',
                                          'triamterene',
                                          'labetalol hydrochloride',
                                          'valsartan',
                                          'Diovan',
                                          'methyldopa',
                                          'torsemide',
                                          'Torem',
                                          'ramipril',
                                          'bumetanide',
                                          'enalapril',
                                          'prazosin',
                                          'metoprolol succinate',
                                          'bumetanide',
                                          'perindopril',
                                          'labetalol',
                                          'telmisartan',
                                          'Micardis',
                                          'carvedilol',
                                          'Coreg',
                                          'candesartan cilexetil',
                                          'amlodipine',
                                          'trandolapril',
                                          'atenolol',
                                          'amlodipine',
                                          'propranolol',
                                          'pindolol',
                                          'moexipril hydrochloride',
                                          'clonidine',
                                          'nebivolol',
                                          'eplerenone',
                                          'fosinopril sodium',
                                          'candesartan',
                                          'hydralazine',
                                          'amiloride',
                                          'bisoprolol',
                                          'moexipril',
                                          'indapamide',
                                          'nicardipine',
                                          'azilsartan',
                                          'olmesartan',
                                          'fosinopril',
                                          'benazepril']})
    
print(len(med_name_df.index))
med_name_df.head()

### filter meds to these ones

In [ ]:
drug_ht = drug[drug['standard_concept_name'].str.contains('|'.join(med_name_df['med_name']), case = False)]
print(len(drug.index))
print(len(drug['person_id'].unique()))
print(len(drug_ht.index))
print(len(drug_ht['person_id'].unique()))

### rename

In [ ]:
drug_ht_sub = drug_ht.rename(columns = {'drug_exposure_start_datetime' : 'condition_start_datetime',
                                        'standard_concept_name' : 'condition_source_value'})
drug_ht_sub.head()

### concat

In [ ]:
print(len(hypertension['person_id'].unique()))
print(len(hypertension.index))
hypertension = pd.concat([hypertension, drug_ht_sub], axis = 0)
print(len(hypertension['person_id'].unique()))
print(len(hypertension.index))
hypertension.head()

## drop duplicates

In [ ]:
print(len(cad['person_id'].unique()))
print(len(cad.index))
cad = cad.drop_duplicates()
print(len(cad['person_id'].unique()))
print(len(cad.index))

In [ ]:
print(len(afib['person_id'].unique()))
print(len(afib.index))
afib = afib.drop_duplicates()
print(len(afib['person_id'].unique()))
print(len(afib.index))

In [ ]:
print(len(hf['person_id'].unique()))
print(len(hf.index))
hf = hf.drop_duplicates()
print(len(hf['person_id'].unique()))
print(len(hf.index))

In [ ]:
print(len(t2d['person_id'].unique()))
print(len(t2d.index))
t2d = t2d.drop_duplicates()
print(len(t2d['person_id'].unique()))
print(len(t2d.index))

In [ ]:
print(len(t1d['person_id'].unique()))
print(len(t1d.index))
t1d = t1d.drop_duplicates()
print(len(t1d['person_id'].unique()))
print(len(t1d.index))

In [ ]:
print(len(diabetes['person_id'].unique()))
print(len(diabetes.index))
diabetes = diabetes.drop_duplicates()
print(len(diabetes['person_id'].unique()))
print(len(diabetes.index))

In [ ]:
print(len(copd['person_id'].unique()))
print(len(copd.index))
copd = copd.drop_duplicates()
print(len(copd['person_id'].unique()))
print(len(copd.index))

In [ ]:
print(len(sys_hf['person_id'].unique()))
print(len(sys_hf.index))
sys_hf = sys_hf.drop_duplicates()
print(len(sys_hf['person_id'].unique()))
print(len(sys_hf.index))

In [ ]:
print(len(hyperthyroidism['person_id'].unique()))
print(len(hyperthyroidism.index))
hyperthyroidism = hyperthyroidism.drop_duplicates()
print(len(hyperthyroidism['person_id'].unique()))
print(len(hyperthyroidism.index))

In [ ]:
print(len(hypertension['person_id'].unique()))
print(len(hypertension.index))
hypertension = hypertension.drop_duplicates()
print(len(hypertension['person_id'].unique()))
print(len(hypertension.index))

## rule of two

### create id lists

In [ ]:
cad_id_list = cad['person_id'].unique().tolist()
len(cad_id_list)

In [ ]:
afib_id_list = afib['person_id'].unique().tolist()
len(afib_id_list)

In [ ]:
hf_id_list = hf['person_id'].unique().tolist()
len(hf_id_list)

In [ ]:
t2d_id_list = t2d['person_id'].unique().tolist()
len(t2d_id_list)

In [ ]:
t1d_id_list = t1d['person_id'].unique().tolist()
len(t1d_id_list)

In [ ]:
diabetes_id_list = diabetes['person_id'].unique().tolist()
len(diabetes_id_list)

In [ ]:
copd_id_list = copd['person_id'].unique().tolist()
len(copd_id_list)

In [ ]:
sys_hf_id_list = sys_hf['person_id'].unique().tolist()
len(sys_hf_id_list)

In [ ]:
hyperthyroidism_id_list = hyperthyroidism['person_id'].unique().tolist()
len(hyperthyroidism_id_list)

In [ ]:
hypertension_id_list = hypertension['person_id'].unique().tolist()
len(hypertension_id_list)

### for loop

In [ ]:
cad_two_instances = []
for id in cad_id_list:
    id_df = cad[cad['person_id'].isin([id])]
    id_df = id_df.drop_duplicates()
    id_df = id_df.reset_index(drop = True)
    if len(id_df.index) >= 2:
        cad_two_instances.append(id)

In [ ]:
afib_two_instances = []
for id in afib_id_list:
    id_df = afib[afib['person_id'].isin([id])]
    id_df = id_df.drop_duplicates()
    id_df = id_df.reset_index(drop = True)
    if len(id_df.index) >= 2:
        afib_two_instances.append(id)

In [ ]:
hf_two_instances = []
for id in hf_id_list:
    id_df = hf[hf['person_id'].isin([id])]
    id_df = id_df.drop_duplicates()
    id_df = id_df.reset_index(drop = True)
    if len(id_df.index) >= 2:
        hf_two_instances.append(id)

In [ ]:
t2d_two_instances = []
for id in t2d_id_list:
    id_df = t2d[t2d['person_id'].isin([id])]
    id_df = id_df.drop_duplicates()
    id_df = id_df.reset_index(drop = True)
    if len(id_df.index) >= 2:
        t2d_two_instances.append(id)

In [ ]:
t1d_two_instances = []
for id in t1d_id_list:
    id_df = t1d[t1d['person_id'].isin([id])]
    id_df = id_df.drop_duplicates()
    id_df = id_df.reset_index(drop = True)
    if len(id_df.index) >= 2:
        t1d_two_instances.append(id)

In [ ]:
diabetes_two_instances = []
for id in diabetes_id_list:
    id_df = diabetes[diabetes['person_id'].isin([id])]
    id_df = id_df.drop_duplicates()
    id_df = id_df.reset_index(drop = True)
    if len(id_df.index) >= 2:
        diabetes_two_instances.append(id)

In [ ]:
copd_two_instances = []
for id in copd_id_list:
    id_df = copd[copd['person_id'].isin([id])]
    id_df = id_df.drop_duplicates()
    id_df = id_df.reset_index(drop = True)
    if len(id_df.index) >= 2:
        copd_two_instances.append(id)

In [ ]:
sys_hf_two_instances = []
for id in sys_hf_id_list:
    id_df = sys_hf[sys_hf['person_id'].isin([id])]
    id_df = id_df.drop_duplicates()
    id_df = id_df.reset_index(drop = True)
    if len(id_df.index) >= 2:
        sys_hf_two_instances.append(id)

In [ ]:
hyperthyroidism_two_instances = []
for id in hyperthyroidism_id_list:
    id_df = hyperthyroidism[hyperthyroidism['person_id'].isin([id])]
    id_df = id_df.drop_duplicates()
    id_df = id_df.reset_index(drop = True)
    if len(id_df.index) >= 2:
        hyperthyroidism_two_instances.append(id)

In [ ]:
hypertension_two_instances = []
for id in hypertension_id_list:
    id_df = hypertension[hypertension['person_id'].isin([id])]
    id_df = id_df.drop_duplicates()
    id_df = id_df.reset_index(drop = True)
    if len(id_df.index) >= 2:
        hypertension_two_instances.append(id)

### filter dfs

In [ ]:
cad_two = cad[cad['person_id'].isin(cad_two_instances)]
print(len(cad['person_id'].unique()))
print(len(cad_two['person_id'].unique()))
print(len(cad_two_instances))
cad_two.head()

In [ ]:
afib_two = afib[afib['person_id'].isin(afib_two_instances)]
print(len(afib['person_id'].unique()))
print(len(afib_two['person_id'].unique()))
print(len(afib_two_instances))
afib_two.head()

In [ ]:
hf_two = hf[hf['person_id'].isin(hf_two_instances)]
print(len(hf['person_id'].unique()))
print(len(hf_two['person_id'].unique()))
print(len(hf_two_instances))
hf_two.head()

In [ ]:
t2d_two = t2d[t2d['person_id'].isin(t2d_two_instances)]
print(len(t2d['person_id'].unique()))
print(len(t2d_two['person_id'].unique()))
print(len(t2d_two_instances))
t2d_two.head()

In [ ]:
t1d_two = t1d[t1d['person_id'].isin(t1d_two_instances)]
print(len(t1d['person_id'].unique()))
print(len(t1d_two['person_id'].unique()))
print(len(t1d_two_instances))
t1d_two.head()

In [ ]:
diabetes_two = diabetes[diabetes['person_id'].isin(diabetes_two_instances)]
print(len(diabetes['person_id'].unique()))
print(len(diabetes_two['person_id'].unique()))
print(len(diabetes_two_instances))
diabetes_two.head()

In [ ]:
copd_two = copd[copd['person_id'].isin(copd_two_instances)]
print(len(copd['person_id'].unique()))
print(len(copd_two['person_id'].unique()))
print(len(copd_two_instances))
copd_two.head()

In [ ]:
sys_hf_two = sys_hf[sys_hf['person_id'].isin(sys_hf_two_instances)]
print(len(sys_hf['person_id'].unique()))
print(len(sys_hf_two['person_id'].unique()))
print(len(sys_hf_two_instances))
sys_hf_two.head()

In [ ]:
hyperthyroidism_two = hyperthyroidism[hyperthyroidism['person_id'].isin(hyperthyroidism_two_instances)]
print(len(hyperthyroidism['person_id'].unique()))
print(len(hyperthyroidism_two['person_id'].unique()))
print(len(hyperthyroidism_two_instances))
hyperthyroidism_two.head()

In [ ]:
hypertension_two = hypertension[hypertension['person_id'].isin(hypertension_two_instances)]
print(len(hypertension['person_id'].unique()))
print(len(hypertension_two['person_id'].unique()))
print(len(hypertension_two_instances))
hypertension_two.head()

## drop extra columns

In [ ]:
cad_two = cad_two[['person_id', 'condition_start_datetime']]
cad_two.head()

In [ ]:
afib_two = afib_two[['person_id', 'condition_start_datetime']]
afib_two.head()

In [ ]:
hf_two = hf_two[['person_id', 'condition_start_datetime']]
hf_two.head()

In [ ]:
t2d_two = t2d_two[['person_id', 'condition_start_datetime']]
t2d_two.head()

In [ ]:
t1d_two = t1d_two[['person_id', 'condition_start_datetime']]
t1d_two.head()

In [ ]:
diabetes_two = diabetes_two[['person_id', 'condition_start_datetime']]
diabetes_two.head()

In [ ]:
copd_two = copd_two[['person_id', 'condition_start_datetime']]
copd_two.head()

In [ ]:
sys_hf_two = sys_hf_two[['person_id', 'condition_start_datetime']]
sys_hf_two.head()

In [ ]:
hyperthyroidism_two = hyperthyroidism_two[['person_id', 'condition_start_datetime']]
hyperthyroidism_two.head()

In [ ]:
hypertension_two = hypertension_two[['person_id', 'condition_start_datetime']]
hypertension_two.head()

## get first occurance for cad and afib

In [ ]:
cad_two['condition_start_datetime'] = pd.to_datetime(cad_two['condition_start_datetime'])
cad_two = cad_two.sort_values(by = ['person_id', 'condition_start_datetime'])
cad_two = cad_two.drop_duplicates(subset = 'person_id', keep = 'first')
cad_two = cad_two.rename(columns = {'condition_start_datetime' : 'CAD_DATE'})
print(len(cad_two.index))

In [ ]:
afib_two['condition_start_datetime'] = pd.to_datetime(afib_two['condition_start_datetime'])
afib_two = afib_two.sort_values(by = ['person_id', 'condition_start_datetime'])
afib_two = afib_two.drop_duplicates(subset = 'person_id', keep = 'first')
afib_two = afib_two.rename(columns = {'condition_start_datetime' : 'AFIB_DATE'})
print(len(afib_two.index))

In [ ]:
hf_two['condition_start_datetime'] = pd.to_datetime(hf_two['condition_start_datetime'])
hf_two = hf_two.sort_values(by = ['person_id', 'condition_start_datetime'])
hf_two = hf_two.drop_duplicates(subset = 'person_id', keep = 'first')
hf_two = hf_two.rename(columns = {'condition_start_datetime' : 'HF_DATE'})
print(len(hf_two.index))

## drop duplicates for non outcome icd phenotypes and add case column

In [ ]:
t2d_two = t2d_two.sort_values(by = ['person_id', 'condition_start_datetime']).drop_duplicates(
    subset = 'person_id', keep = 'first')
t2d_two['T2D'] = 1
print(len(t2d_two.index))

In [ ]:
t1d_two = t1d_two.sort_values(by = ['person_id', 'condition_start_datetime']).drop_duplicates(
    subset = 'person_id', keep = 'first')
t1d_two['T1D'] = 1
print(len(t1d_two.index))

In [ ]:
diabetes_two = diabetes_two.sort_values(by = ['person_id', 'condition_start_datetime']).drop_duplicates(
    subset = 'person_id', keep = 'first')
diabetes_two['DIABETES'] = 1
print(len(diabetes_two.index))

In [ ]:
copd_two = copd_two.sort_values(by = ['person_id', 'condition_start_datetime']).drop_duplicates(
    subset = 'person_id', keep = 'first')
copd_two['COPD'] = 1
print(len(copd_two.index))

In [ ]:
sys_hf_two = sys_hf_two.sort_values(by = ['person_id', 'condition_start_datetime']).drop_duplicates(
    subset = 'person_id', keep = 'first')
sys_hf_two['SYS_HF'] = 1
print(len(sys_hf_two.index))

In [ ]:
hyperthyroidism_two = hyperthyroidism_two.sort_values(by = ['person_id', 'condition_start_datetime']).drop_duplicates(
    subset = 'person_id', keep = 'first')
hyperthyroidism_two['HYPERTHYROIDISM'] = 1
print(len(hyperthyroidism_two.index))

In [ ]:
hypertension_two = hypertension_two.sort_values(by = ['person_id', 'condition_start_datetime']).drop_duplicates(
    subset = 'person_id', keep = 'first')
hypertension_two['HT'] = 1
print(len(hypertension_two.index))

## filter to cases that occurred before outcome

In [ ]:
cad_before_afib = cad_two.merge(afib_two, on = 'person_id', how = 'left')

cad_before_afib['CAD_AFIB'] = np.nan
cad_before_afib['CAD'] = 1

cad_before_afib.loc[(cad_before_afib['CAD_DATE'] < cad_before_afib['AFIB_DATE']) & (cad_before_afib['AFIB_DATE'].notna()), 'CAD_AFIB'] = cad_before_afib['CAD']

cad_before_afib = cad_before_afib.drop(columns = ['CAD_DATE', 'AFIB_DATE', 'CAD'])

print(cad_before_afib['CAD_AFIB'].value_counts(dropna = False))

In [ ]:
t2d_two = t2d_two.merge(cad_two, on = 'person_id', how = 'left').merge(
    afib_two, on = 'person_id', how = 'left').merge(
    hf_two, on = 'person_id', how = 'left')

t2d_two['T2D_CAD'] = np.nan
t2d_two['T2D_AFIB'] = np.nan
t2d_two['T2D_HF'] = np.nan

t2d_two.loc[(t2d_two['condition_start_datetime'] < t2d_two['CAD_DATE']) & (t2d_two['CAD_DATE'].notna()), 'T2D_CAD'] = t2d_two['T2D']
t2d_two.loc[(t2d_two['condition_start_datetime'] < t2d_two['AFIB_DATE']) & (t2d_two['AFIB_DATE'].notna()), 'T2D_AFIB'] = t2d_two['T2D']
t2d_two.loc[(t2d_two['condition_start_datetime'] < t2d_two['HF_DATE']) & (t2d_two['HF_DATE'].notna()), 'T2D_HF'] = t2d_two['T2D']

t2d_two = t2d_two.drop(columns = ['CAD_DATE', 'AFIB_DATE', 'HF_DATE', 'T2D', 'condition_start_datetime'])

print(t2d_two['T2D_CAD'].value_counts(dropna = False))
print(t2d_two['T2D_AFIB'].value_counts(dropna = False))
print(t2d_two['T2D_HF'].value_counts(dropna = False))

In [ ]:
t1d_two = t1d_two.merge(cad_two, on = 'person_id', how = 'left').merge(
    afib_two, on = 'person_id', how = 'left').merge(
    hf_two, on = 'person_id', how = 'left')

t1d_two['T1D_CAD'] = np.nan
t1d_two['T1D_AFIB'] = np.nan
t1d_two['T1D_HF'] = np.nan

t1d_two.loc[(t1d_two['condition_start_datetime'] < t1d_two['CAD_DATE']) & (t1d_two['CAD_DATE'].notna()), 'T1D_CAD'] = t1d_two['T1D']
t1d_two.loc[(t1d_two['condition_start_datetime'] < t1d_two['AFIB_DATE']) & (t1d_two['AFIB_DATE'].notna()), 'T1D_AFIB'] = t1d_two['T1D']
t1d_two.loc[(t1d_two['condition_start_datetime'] < t1d_two['HF_DATE']) & (t1d_two['HF_DATE'].notna()), 'T1D_HF'] = t1d_two['T1D']

t1d_two = t1d_two.drop(columns = ['CAD_DATE', 'AFIB_DATE', 'HF_DATE', 'T1D', 'condition_start_datetime'])

print(t1d_two['T1D_CAD'].value_counts(dropna = False))
print(t1d_two['T1D_AFIB'].value_counts(dropna = False))
print(t1d_two['T1D_HF'].value_counts(dropna = False))

In [ ]:
diabetes_two = diabetes_two.merge(cad_two, on = 'person_id', how = 'left').merge(
    afib_two, on = 'person_id', how = 'left').merge(
    hf_two, on = 'person_id', how = 'left')

diabetes_two['DIABETES_CAD'] = np.nan
diabetes_two['DIABETES_AFIB'] = np.nan
diabetes_two['DIABETES_HF'] = np.nan

diabetes_two.loc[(diabetes_two['condition_start_datetime'] < diabetes_two['CAD_DATE']) & (diabetes_two['CAD_DATE'].notna()), 'DIABETES_CAD'] = diabetes_two['DIABETES']
diabetes_two.loc[(diabetes_two['condition_start_datetime'] < diabetes_two['AFIB_DATE']) & (diabetes_two['AFIB_DATE'].notna()), 'DIABETES_AFIB'] = diabetes_two['DIABETES']
diabetes_two.loc[(diabetes_two['condition_start_datetime'] < diabetes_two['HF_DATE']) & (diabetes_two['HF_DATE'].notna()), 'DIABETES_HF'] = diabetes_two['DIABETES']

diabetes_two = diabetes_two.drop(columns = ['CAD_DATE', 'AFIB_DATE', 'HF_DATE', 'DIABETES', 'condition_start_datetime'])

print(diabetes_two['DIABETES_CAD'].value_counts(dropna = False))
print(diabetes_two['DIABETES_AFIB'].value_counts(dropna = False))
print(diabetes_two['DIABETES_HF'].value_counts(dropna = False))

In [ ]:
copd_two = copd_two.merge(cad_two, on = 'person_id', how = 'left').merge(
    afib_two, on = 'person_id', how = 'left').merge(
    hf_two, on = 'person_id', how = 'left')

copd_two['COPD_CAD'] = np.nan
copd_two['COPD_AFIB'] = np.nan
copd_two['COPD_HF'] = np.nan

copd_two.loc[(copd_two['condition_start_datetime'] < copd_two['CAD_DATE']) & (copd_two['CAD_DATE'].notna()), 'COPD_CAD'] = copd_two['COPD']
copd_two.loc[(copd_two['condition_start_datetime'] < copd_two['AFIB_DATE']) & (copd_two['AFIB_DATE'].notna()), 'COPD_AFIB'] = copd_two['COPD']
copd_two.loc[(copd_two['condition_start_datetime'] < copd_two['HF_DATE']) & (copd_two['HF_DATE'].notna()), 'COPD_HF'] = copd_two['COPD']

copd_two = copd_two.drop(columns = ['CAD_DATE', 'AFIB_DATE', 'HF_DATE', 'COPD', 'condition_start_datetime'])

print(copd_two['COPD_CAD'].value_counts(dropna = False))
print(copd_two['COPD_AFIB'].value_counts(dropna = False))
print(copd_two['COPD_HF'].value_counts(dropna = False))

In [ ]:
sys_hf_two = sys_hf_two.merge(cad_two, on = 'person_id', how = 'left').merge(
    afib_two, on = 'person_id', how = 'left').merge(
    hf_two, on = 'person_id', how = 'left')

sys_hf_two['SYS_HF_CAD'] = np.nan
sys_hf_two['SYS_HF_AFIB'] = np.nan
sys_hf_two['SYS_HF_HF'] = np.nan

sys_hf_two.loc[(sys_hf_two['condition_start_datetime'] < sys_hf_two['CAD_DATE']) & (sys_hf_two['CAD_DATE'].notna()), 'SYS_HF_CAD'] = sys_hf_two['SYS_HF']
sys_hf_two.loc[(sys_hf_two['condition_start_datetime'] < sys_hf_two['AFIB_DATE']) & (sys_hf_two['AFIB_DATE'].notna()), 'SYS_HF_AFIB'] = sys_hf_two['SYS_HF']
sys_hf_two.loc[(sys_hf_two['condition_start_datetime'] < sys_hf_two['HF_DATE']) & (sys_hf_two['HF_DATE'].notna()), 'SYS_HF_HF'] = sys_hf_two['SYS_HF']

sys_hf_two = sys_hf_two.drop(columns = ['CAD_DATE', 'AFIB_DATE', 'HF_DATE', 'SYS_HF', 'condition_start_datetime'])

print(sys_hf_two['SYS_HF_CAD'].value_counts(dropna = False))
print(sys_hf_two['SYS_HF_AFIB'].value_counts(dropna = False))
print(sys_hf_two['SYS_HF_HF'].value_counts(dropna = False))

In [ ]:
hyperthyroidism_two = hyperthyroidism_two.merge(cad_two, on = 'person_id', how = 'left').merge(
    afib_two, on = 'person_id', how = 'left').merge(
    hf_two, on = 'person_id', how = 'left')

hyperthyroidism_two['HYPERTHYROIDISM_CAD'] = np.nan
hyperthyroidism_two['HYPERTHYROIDISM_AFIB'] = np.nan
hyperthyroidism_two['HYPERTHYROIDISM_HF'] = np.nan

hyperthyroidism_two.loc[(hyperthyroidism_two['condition_start_datetime'] < hyperthyroidism_two['CAD_DATE']) & (hyperthyroidism_two['CAD_DATE'].notna()), 'HYPERTHYROIDISM_CAD'] = hyperthyroidism_two['HYPERTHYROIDISM']
hyperthyroidism_two.loc[(hyperthyroidism_two['condition_start_datetime'] < hyperthyroidism_two['AFIB_DATE']) & (hyperthyroidism_two['AFIB_DATE'].notna()), 'HYPERTHYROIDISM_AFIB'] = hyperthyroidism_two['HYPERTHYROIDISM']
hyperthyroidism_two.loc[(hyperthyroidism_two['condition_start_datetime'] < hyperthyroidism_two['HF_DATE']) & (hyperthyroidism_two['HF_DATE'].notna()), 'HYPERTHYROIDISM_HF'] = hyperthyroidism_two['HYPERTHYROIDISM']

hyperthyroidism_two = hyperthyroidism_two.drop(columns = ['CAD_DATE', 'AFIB_DATE', 'HF_DATE', 'HYPERTHYROIDISM', 'condition_start_datetime'])

print(hyperthyroidism_two['HYPERTHYROIDISM_CAD'].value_counts(dropna = False))
print(hyperthyroidism_two['HYPERTHYROIDISM_AFIB'].value_counts(dropna = False))
print(hyperthyroidism_two['HYPERTHYROIDISM_HF'].value_counts(dropna = False))

In [ ]:
hypertension_two = hypertension_two.merge(cad_two, on = 'person_id', how = 'left').merge(
    afib_two, on = 'person_id', how = 'left').merge(
    hf_two, on = 'person_id', how = 'left')

hypertension_two['HT_CAD'] = np.nan
hypertension_two['HT_AFIB'] = np.nan
hypertension_two['HT_HF'] = np.nan

hypertension_two.loc[(hypertension_two['condition_start_datetime'] < hypertension_two['CAD_DATE']) & (hypertension_two['CAD_DATE'].notna()), 'HT_CAD'] = hypertension_two['HT']
hypertension_two.loc[(hypertension_two['condition_start_datetime'] < hypertension_two['AFIB_DATE']) & (hypertension_two['AFIB_DATE'].notna()), 'HT_AFIB'] = hypertension_two['HT']
hypertension_two.loc[(hypertension_two['condition_start_datetime'] < hypertension_two['HF_DATE']) & (hypertension_two['HF_DATE'].notna()), 'HT_HF'] = hypertension_two['HT']

hypertension_two = hypertension_two.drop(columns = ['CAD_DATE', 'AFIB_DATE', 'HF_DATE', 'HT', 'condition_start_datetime'])

print(hypertension_two['HT_CAD'].value_counts(dropna = False))
print(hypertension_two['HT_AFIB'].value_counts(dropna = False))
print(hypertension_two['HT_HF'].value_counts(dropna = False))

## merge cad, afib, and hf with demo df

In [ ]:
cad_demo = cad_two.merge(demo_sex, on = 'person_id', how = 'inner')
print(len(cad_demo.index))
cad_demo.head()

In [ ]:
afib_demo = afib_two.merge(demo_sex, on = 'person_id', how = 'inner')
print(len(afib_demo.index))
afib_demo.head()

In [ ]:
hf_demo = hf_two.merge(demo_sex, on = 'person_id', how = 'inner')
print(len(hf_demo.index))
hf_demo.head()

## calculate age at first occurance

In [ ]:
cad_demo.loc[:, 'AGE_CAD'] = cad_demo['CAD_DATE'] - cad_demo['date_of_birth']
cad_demo.loc[:, 'AGE_CAD'] = cad_demo['AGE_CAD'].dt.days/365.25
cad_demo.head()

In [ ]:
afib_demo.loc[:, 'AGE_AFIB'] = afib_demo['AFIB_DATE'] - afib_demo['date_of_birth']
afib_demo.loc[:, 'AGE_AFIB'] = afib_demo['AGE_AFIB'].dt.days/365.25
afib_demo.head()

In [ ]:
hf_demo.loc[:, 'AGE_HF'] = hf_demo['HF_DATE'] - hf_demo['date_of_birth']
hf_demo.loc[:, 'AGE_HF'] = hf_demo['AGE_HF'].dt.days/365.25
hf_demo.head()

## drop DOB column and add case column

In [ ]:
cad_demo = cad_demo.drop(columns = ['date_of_birth'])
cad_demo.loc[:, 'CAD'] = 1
cad_demo.head()

In [ ]:
afib_demo = afib_demo.drop(columns = ['date_of_birth'])
afib_demo.loc[:, 'AFIB'] = 1
afib_demo.head()

In [ ]:
hf_demo = hf_demo.drop(columns = ['date_of_birth'])
hf_demo.loc[:, 'HF'] = 1
hf_demo.head()

# create cad/afib/hf controls dataframe

## remove individuals with case icd code

In [ ]:
cad_controls = demo_sex[~demo_sex['person_id'].isin(cad['person_id'])]
print(len(cad_controls.index))
cad_controls.head()

In [ ]:
afib_controls = demo_sex[~demo_sex['person_id'].isin(afib['person_id'])]
print(len(afib_controls.index))
afib_controls.head()

In [ ]:
hf_controls = demo_sex[~demo_sex['person_id'].isin(hf['person_id'])]
print(len(hf_controls.index))
hf_controls.head()

## calculate age at data release

In [ ]:
cad_controls.loc[:, 'CAD_DATE'] = pd.to_datetime('2023-10-01')
cad_controls.loc[:, 'AGE_CAD'] = cad_controls['CAD_DATE'] - cad_controls['date_of_birth'].dt.tz_localize(None)
cad_controls.loc[:, 'AGE_CAD'] = cad_controls['AGE_CAD'].astype(str).str.replace(' days','').astype(float)/365.25
cad_controls.head()

In [ ]:
afib_controls.loc[:, 'AFIB_DATE'] = pd.to_datetime('2023-10-01')
afib_controls.loc[:, 'AGE_AFIB'] = afib_controls['AFIB_DATE'] - afib_controls['date_of_birth'].dt.tz_localize(None)
afib_controls.loc[:, 'AGE_AFIB'] = afib_controls['AGE_AFIB'].astype(str).str.replace(' days','').astype(float)/365.25
afib_controls.head()

In [ ]:
hf_controls.loc[:, 'HF_DATE'] = pd.to_datetime('2023-10-01')
hf_controls.loc[:, 'AGE_HF'] = hf_controls['HF_DATE'] - hf_controls['date_of_birth'].dt.tz_localize(None)
hf_controls.loc[:, 'AGE_HF'] = hf_controls['AGE_HF'].astype(str).str.replace(' days','').astype(float)/365.25
hf_controls.head()

## drop dob column and add CAD/AFIB/HF col

In [ ]:
cad_controls = cad_controls.drop(columns = ['date_of_birth'])
cad_controls.loc[:, 'CAD'] = 0
cad_controls.head()

In [ ]:
afib_controls = afib_controls.drop(columns = ['date_of_birth'])
afib_controls.loc[:, 'AFIB'] = 0
afib_controls.head()

In [ ]:
hf_controls = hf_controls.drop(columns = ['date_of_birth'])
hf_controls.loc[:, 'HF'] = 0
hf_controls.head()

# merge cad/afib/hf cases and controls

In [ ]:
cad_case_control = pd.concat([cad_demo, cad_controls], axis = 0)
print(len(cad_case_control.index))
print(cad_case_control['CAD'].value_counts())
cad_case_control.head()

In [ ]:
afib_case_control = pd.concat([afib_demo, afib_controls], axis = 0)
print(len(afib_case_control.index))
print(afib_case_control['AFIB'].value_counts())
afib_case_control.head()

In [ ]:
hf_case_control = pd.concat([hf_demo, hf_controls], axis = 0)
print(len(hf_case_control.index))
print(hf_case_control['HF'].value_counts())
hf_case_control.head()

# merge cad, afib, and hf

## merge

In [ ]:
cad_afib_hf = cad_case_control.merge(afib_case_control, on = ['person_id', 'SEX'], how = 'outer').merge(
hf_case_control, on = ['person_id', 'SEX'], how = 'outer')
print(len(cad_afib_hf.index))
cad_afib_hf.head()

# merge w icd predictors

## merge

In [ ]:
all_icd = cad_afib_hf.merge(t2d_two, on = 'person_id', how = 'left').merge(
    t1d_two, on = 'person_id', how = 'left').merge(
    diabetes_two, on = 'person_id', how = 'left').merge(
    copd_two, on = 'person_id', how = 'left').merge(
    sys_hf_two, on = 'person_id', how = 'left').merge(
    hyperthyroidism_two, on = 'person_id', how = 'left').merge(
    hypertension_two, on = 'person_id', how = 'left').merge(
    cad_before_afib, on = 'person_id', how = 'left')
print(len(all_icd.index))
all_icd.head()

## identify individuals who are controls and set column as zero

In [ ]:
mask = ~all_icd['person_id'].isin(cad['person_id'])
all_icd.loc[mask, ['CAD_AFIB']] = 0

print(all_icd['CAD_AFIB'].value_counts(dropna = False))

In [ ]:
mask = ~all_icd['person_id'].isin(t2d['person_id'])
all_icd.loc[mask, ['T2D_CAD', 'T2D_AFIB', 'T2D_HF']] = 0

print(all_icd['T2D_CAD'].value_counts(dropna = False))
print(all_icd['T2D_AFIB'].value_counts(dropna = False))
print(all_icd['T2D_HF'].value_counts(dropna = False))

In [ ]:
mask = ~all_icd['person_id'].isin(t1d['person_id'])
all_icd.loc[mask, ['T1D_CAD', 'T1D_AFIB', 'T1D_HF']] = 0

print(all_icd['T1D_CAD'].value_counts(dropna = False))
print(all_icd['T1D_AFIB'].value_counts(dropna = False))
print(all_icd['T1D_HF'].value_counts(dropna = False))

In [ ]:
mask = ~all_icd['person_id'].isin(diabetes['person_id'])
all_icd.loc[mask, ['DIABETES_CAD', 'DIABETES_AFIB', 'DIABETES_HF']] = 0

print(all_icd['DIABETES_CAD'].value_counts(dropna = False))
print(all_icd['DIABETES_AFIB'].value_counts(dropna = False))
print(all_icd['DIABETES_HF'].value_counts(dropna = False))

In [ ]:
mask = ~all_icd['person_id'].isin(copd['person_id'])
all_icd.loc[mask, ['COPD_CAD', 'COPD_AFIB', 'COPD_HF']] = 0

print(all_icd['COPD_CAD'].value_counts(dropna = False))
print(all_icd['COPD_AFIB'].value_counts(dropna = False))
print(all_icd['COPD_HF'].value_counts(dropna = False))

In [ ]:
mask = ~all_icd['person_id'].isin(sys_hf['person_id'])
all_icd.loc[mask, ['SYS_HF_CAD', 'SYS_HF_AFIB', 'SYS_HF_HF']] = 0

print(all_icd['SYS_HF_CAD'].value_counts(dropna = False))
print(all_icd['SYS_HF_AFIB'].value_counts(dropna = False))
print(all_icd['SYS_HF_HF'].value_counts(dropna = False))

In [ ]:
mask = ~all_icd['person_id'].isin(hyperthyroidism['person_id'])
all_icd.loc[mask, ['HYPERTHYROIDISM_CAD', 'HYPERTHYROIDISM_AFIB', 'HYPERTHYROIDISM_HF']] = 0

print(all_icd['HYPERTHYROIDISM_CAD'].value_counts(dropna = False))
print(all_icd['HYPERTHYROIDISM_AFIB'].value_counts(dropna = False))
print(all_icd['HYPERTHYROIDISM_HF'].value_counts(dropna = False))

In [ ]:
mask = ~all_icd['person_id'].isin(hypertension['person_id'])
all_icd.loc[mask, ['HT_CAD', 'HT_AFIB', 'HT_HF']] = 0

print(all_icd['HT_CAD'].value_counts(dropna = False))
print(all_icd['HT_AFIB'].value_counts(dropna = False))
print(all_icd['HT_HF'].value_counts(dropna = False))

## export in progress pheno

In [ ]:
all_icd.to_csv('CVD.IRM.in_progress_pheno.csv', index = None)

# clean labs df

## read in in progress pheno

In [ ]:
all_icd = pd.read_csv('CVD.IRM.in_progress_pheno.csv')
print(all_icd.shape)
all_icd.head()

## create median absolute deviance function

In [ ]:
def mad_filter(df, threshold = 5):
    values = df['value_as_number'].dropna()
    
    # Compute median and MAD
    median = values.median()
    mad = np.median(np.abs(values - median))
    
    # Define limits
    lower_limit = median - threshold * mad
    upper_limit = median + threshold * mad

    # Filter and add top/bottom limits
    filtered_df = df[(df['value_as_number'] >= lower_limit) & (df['value_as_number'] <= upper_limit)].copy()

    return filtered_df

## create date df

In [ ]:
pheno_date = all_icd[['person_id', 'CAD_DATE', 'CAD', 'AFIB_DATE', 'AFIB', 'HF_DATE', 'HF']]

## total cholesterol

### filter by measurement name

In [ ]:
total_chol = labs[labs['standard_concept_name'].str.contains('cholesterol', case = False)]
total_chol = total_chol[~total_chol['standard_concept_name'].str.contains('HDL|VLDL|LDL|crystals|esterase|Triglyceride|Ratio|non-esterified|IDL|density|Percentile|Presence|Stone|fluid|Lipoprotein|DBS|chylomicrons|Specimen', case = False)]
total_chol['standard_concept_name'].unique()

In [ ]:
print(len(total_chol['person_id'].unique()))
print(total_chol['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

### look at units

In [ ]:
total_chol.groupby('unit_concept_name')['value_as_number'].describe()

In [ ]:
total_chol.groupby('unit_source_value')['value_as_number'].describe()

In [ ]:
print(len(total_chol.index))
print(len(total_chol['person_id'].unique()))
mg_dl = total_chol[total_chol['unit_concept_name'].isin(['milligram per deciliter'])]
print(len(mg_dl.index))
print(len(mg_dl['person_id'].unique()))

### filter to mg/dL
- to mg/dL

In [ ]:
total_chol_unit = total_chol[total_chol['unit_concept_name'].isin(['milligram per deciliter'])]
print(len(total_chol_unit['person_id'].unique()))
print(total_chol_unit['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

### apply MAD

In [ ]:
total_chol_normal = mad_filter(total_chol_unit)
print(len(total_chol_normal['person_id'].unique()))
print(total_chol_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
total_chol_normal.head()

### filter by date, rename, and subset

In [ ]:
lab_input = total_chol_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'TOTAL_CHOL'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_CAD')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_AFIB')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_HF')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
total_chol_sub = merge.copy()

### merge

In [ ]:
pheno_total_chol = all_icd.merge(total_chol_sub, on = 'person_id', how = 'left')
print(len(pheno_total_chol.index))
print(len(pheno_total_chol[pheno_total_chol['TOTAL_CHOL_CAD'].isna() == True ]))
print(len(pheno_total_chol[pheno_total_chol['TOTAL_CHOL_AFIB'].isna() == True ]))
print(len(pheno_total_chol[pheno_total_chol['TOTAL_CHOL_HF'].isna() == True ]))
pheno_total_chol.head()

## hdl

### filter by measurement name

In [ ]:
hdl = labs[labs['standard_concept_name'].str.contains('high|density|lipoprotein|hdl|cholesterol', case = False)]
hdl = hdl[~hdl['standard_concept_name'].str.contains('LDL|Triglyceride|measurement|beta|Ratio|Percentile|total|non|3|subparticle|2|length|Presence|electroph|Lipoprotein|fluid|crystals|DBS|esterase|Specimen|chylomicrons|IDL', case = False)]
hdl['standard_concept_name'].unique()

In [ ]:
print(hdl['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(hdl['person_id'].unique()))

### look at units

In [ ]:
hdl.groupby('unit_source_value')['value_as_number'].describe()

### filter units to mg/dL

In [ ]:
print(len(hdl['person_id'].unique()))
hdl_unit = hdl[hdl['unit_concept_name'].isin(['milligram per deciliter'])]
print(len(hdl_unit['person_id'].unique()))
print(hdl_unit['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

### apply mad

In [ ]:
hdl_normal = mad_filter(hdl_unit)
print(len(hdl_normal['person_id'].unique()))
print(hdl_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
hdl_normal.head()

### remove values less than zero

In [ ]:
hdl_positive = hdl_normal[hdl_normal['value_as_number'] > 0]
print(len(hdl_positive['person_id'].unique()))
print(hdl_positive['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

### filter by date, rename and subset

In [ ]:
lab_input = hdl_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'HDL'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_CAD')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_AFIB')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_HF')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
hdl_sub = merge.copy()

### merge

In [ ]:
pheno_hdl = pheno_total_chol.merge(hdl_sub, on = 'person_id', how = 'left')
print(len(pheno_hdl.index))
print(len(pheno_hdl[pheno_hdl['HDL_CAD'].isna() == True ]))
print(len(pheno_hdl[pheno_hdl['HDL_AFIB'].isna() == True ]))
print(len(pheno_hdl[pheno_hdl['HDL_HF'].isna() == True ]))
pheno_hdl.head()

## export and read in progress pheno

In [ ]:
pheno_hdl.to_csv('CVD.IRM.in_progress_pheno.csv', index = None)

In [ ]:
pheno_hdl = pd.read_csv('CVD.IRM.in_progress_pheno.csv')

## hba1c

### filter by measurement name

In [ ]:
hba1c = labs[labs['standard_concept_name'].str.contains('hba1c|hemoglobin|a1c', case = False)]
hba1c['standard_concept_name'].unique()

In [ ]:
hba1c = hba1c[hba1c['standard_concept_name'].str.contains('A1c/')]
hba1c['standard_concept_name'].unique()

In [ ]:
hba1c = hba1c[~hba1c['standard_concept_name'].str.contains('DBS')]
hba1c['standard_concept_name'].unique()

In [ ]:
print(hba1c['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(hba1c['person_id'].unique()))

In [ ]:
(hba1c.groupby("unit_concept_name")["value_as_number"].describe().applymap(lambda x: f"{x:,.2f}"))

### filter to percent only

In [ ]:
hba1c_unit = hba1c[hba1c['unit_concept_name'].str.contains('percent', case = False, na = False)]
print(hba1c_unit['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(hba1c_unit['person_id'].unique()))

### apply MAD

In [ ]:
hba1c_normal = mad_filter(hba1c_unit)
print(len(hba1c_normal['person_id'].unique()))
print(hba1c_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

### filter by date, rename, and subset

In [ ]:
lab_input = hba1c_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'HbA1c'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_CAD')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_AFIB')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_HF')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
hba1c_sub = merge.copy()

### merge

In [ ]:
pheno_hba1c = pheno_hdl.merge(hba1c_sub, on = 'person_id', how = 'left')
print(len(pheno_hba1c.index))
print(len(pheno_hba1c[pheno_hba1c['HbA1c_CAD'].isna() == True ]))
print(len(pheno_hba1c[pheno_hba1c['HbA1c_AFIB'].isna() == True ]))
print(len(pheno_hba1c[pheno_hba1c['HbA1c_HF'].isna() == True ]))
pheno_hba1c.head()

## systolic blood pressure

### filter by measurement name

In [ ]:
systolic = labs[labs['standard_concept_name'].str.contains('systolic|pressure', case = False)]
systolic = systolic[~systolic['standard_concept_name'].str.contains('diastolic|oxygen|carbon|invasive|Cuff|venous|Arterial|Type|ventricular|toe|Mean|Artery|Atrial|panel|Aorta|taking|ambulatory|anesthesia|exercise', case = False)]
systolic = systolic[~systolic['standard_concept_name'].isin(['Sitting blood pressure',
                                                             'Standing blood pressure'])]
systolic['standard_concept_name'].unique()

In [ ]:
print(systolic['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(systolic['person_id'].unique()))

### filter by unit

In [ ]:
systolic_unit = systolic[systolic['unit_concept_name'] == 'millimeter mercury column']
print(systolic_unit['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(systolic_unit['person_id'].unique()))

### apply MAD

In [ ]:
systolic_normal = mad_filter(systolic_unit)
print(len(systolic_normal['person_id'].unique()))
print(systolic_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
systolic_normal.head()

### filter by date, rename and subset

In [ ]:
lab_input = systolic_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'SBP'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_CAD')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_AFIB')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_HF')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
systolic_sub = merge.copy()

### merge

In [ ]:
pheno_systolic = pheno_hba1c.merge(systolic_sub, on = 'person_id', how = 'left')
print(len(pheno_systolic.index))
print(len(pheno_systolic[pheno_systolic['SBP_CAD'].isna() == True]))
print(len(pheno_systolic[pheno_systolic['SBP_AFIB'].isna() == True]))
print(len(pheno_systolic[pheno_systolic['SBP_HF'].isna() == True]))
pheno_systolic.head()

## bmi

### filter by measurement name

In [ ]:
bmi = labs[labs['standard_concept_name'].str.contains('BMI|Body', case = False)]
bmi = bmi[~bmi['standard_concept_name'].str.contains('fluid', case = False)]
bmi['standard_concept_name'].unique()

In [ ]:
print(bmi['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(bmi['person_id'].unique()))

### look at units

In [ ]:
bmi.groupby('unit_concept_name')['value_as_number'].describe()

In [ ]:
bmi.groupby('unit_source_value')['value_as_number'].describe()

In [ ]:
bmi[['unit_concept_name', 'unit_source_value']].drop_duplicates()

### filter units to kg/m2

In [ ]:
print(len(bmi['person_id'].unique()))
bmi_unit = bmi[bmi['unit_concept_name'].isin(['kg/sq. m', 'kilogram per square meter'])]
print(len(bmi_unit['person_id'].unique()))
print(bmi_unit['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

### apply MAD filter

In [ ]:
bmi_normal = mad_filter(bmi_unit)
print(len(bmi_normal['person_id'].unique()))
print(bmi_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))
bmi_normal.head()

### filter by date, rename, and subset

In [ ]:
lab_input = bmi_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'BMI'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_CAD')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_AFIB')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number']].rename(columns = {'value_as_number' : (col + '_HF')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
bmi_sub = merge.copy()

### merge

In [ ]:
pheno_bmi = pheno_systolic.merge(bmi_sub, on = 'person_id', how = 'left')
print(len(pheno_bmi.index))
print(len(pheno_bmi[pheno_bmi['BMI_CAD'].isna() == True]))
print(len(pheno_bmi[pheno_bmi['BMI_AFIB'].isna() == True]))
print(len(pheno_bmi[pheno_bmi['BMI_HF'].isna() == True]))
pheno_bmi.head()

## export and read in in progress pheno

In [ ]:
pheno_bmi.to_csv('CVD.in_progress_pheno.csv', index = False)

In [ ]:
pheno_bmi = pd.read_csv('CVD.in_progress_pheno.csv')

## eGFR

### creatinine

#### filter by measurement name

In [ ]:
cre = labs[labs['standard_concept_name'].str.contains('Creatinine', case = False)]
#cre['standard_concept_name'].unique()

In [ ]:
cre_serum = cre[cre['standard_concept_name'].str.contains('serum', case = False)]
cre_serum = cre_serum[~cre_serum['standard_concept_name'].str.contains('rate|nitrogen|clearance|reduction|dialysis')]
cre_serum['standard_concept_name'].unique()

#### filter units

In [ ]:
cre_serum_unit = cre_serum[~cre_serum['unit_concept_name'].str.contains('no|minute|hours|unit|milliequivalent', case = False, na = True)]
print(len(cre_serum['person_id'].unique()))
print(len(cre_serum_unit['person_id'].unique()))
cre_serum_unit['unit_concept_name'].value_counts(dropna = False)

In [ ]:
print(len(cre_serum_unit['person_id'].unique()))
cre_serum_unit = cre_serum_unit[cre_serum_unit['unit_concept_name'].isin(['milligram per deciliter'])]
print(len(cre_serum_unit['person_id'].unique()))

#### apply mad

In [ ]:
cre_serum_normal = mad_filter(cre_serum_unit)
print(len(cre_serum_normal['person_id'].unique()))
print(cre_serum_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

#### filter by date, subset, and rename

In [ ]:
lab_input = cre_serum_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'CREATININE_SERUM'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_CAD'), 'measurement_datetime' : (col + '_CAD_datetime')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_AFIB'), 'measurement_datetime' : (col + '_AFIB_datetime')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_HF'), 'measurement_datetime' : (col + '_HF_datetime')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
cre_serum_sub = merge.copy()
cre_serum_sub.head()

#### merge with demo

In [ ]:
cre_serum_demo = cre_serum_sub.merge(demo_sex, on = 'person_id', how = 'inner')
print(len(cre_serum_sub.index))
print(len(cre_serum_demo.index))
cre_serum_demo.head()

#### calculate age at measurement

In [ ]:
cre_serum_demo['AGE_CAD'] = cre_serum_demo['CREATININE_SERUM_CAD_datetime'] - cre_serum_demo['date_of_birth']
cre_serum_demo['AGE_CAD'] = cre_serum_demo['AGE_CAD'].astype(str).str.replace(r' .*', '', regex = True)
cre_serum_demo['AGE_CAD'] = cre_serum_demo['AGE_CAD'].replace({'NaT' : np.nan})
cre_serum_demo['AGE_CAD'] = cre_serum_demo['AGE_CAD'].astype(float)
cre_serum_demo['AGE_CAD'] = cre_serum_demo['AGE_CAD']/365.25

In [ ]:
cre_serum_demo['AGE_AFIB'] = cre_serum_demo['CREATININE_SERUM_AFIB_datetime'] - cre_serum_demo['date_of_birth']
cre_serum_demo['AGE_AFIB'] = cre_serum_demo['AGE_AFIB'].astype(str).str.replace(r' .*', '', regex = True)
cre_serum_demo['AGE_AFIB'] = cre_serum_demo['AGE_AFIB'].replace({'NaT' : np.nan})
cre_serum_demo['AGE_AFIB'] = cre_serum_demo['AGE_AFIB'].astype(float)
cre_serum_demo['AGE_AFIB'] = cre_serum_demo['AGE_AFIB']/365.25

In [ ]:
cre_serum_demo['AGE_HF'] = cre_serum_demo['CREATININE_SERUM_HF_datetime'] - cre_serum_demo['date_of_birth']
cre_serum_demo['AGE_HF'] = cre_serum_demo['AGE_HF'].astype(str).str.replace(r' .*', '', regex = True)
cre_serum_demo['AGE_HF'] = cre_serum_demo['AGE_HF'].replace({'NaT' : np.nan})
cre_serum_demo['AGE_HF'] = cre_serum_demo['AGE_HF'].astype(float)
cre_serum_demo['AGE_HF'] = cre_serum_demo['AGE_HF']/365.25

#### convert values zero or less to missing

In [ ]:
cre_serum_demo['CREATININE_SERUM_CAD'] = cre_serum_demo['CREATININE_SERUM_CAD'].mask(cre_serum_demo['CREATININE_SERUM_CAD'] <= 0)

In [ ]:
cre_serum_demo['CREATININE_SERUM_AFIB'] = cre_serum_demo['CREATININE_SERUM_AFIB'].mask(cre_serum_demo['CREATININE_SERUM_AFIB'] <= 0)

In [ ]:
cre_serum_demo['CREATININE_SERUM_HF'] = cre_serum_demo['CREATININE_SERUM_HF'].mask(cre_serum_demo['CREATININE_SERUM_HF'] <= 0)

#### split into males and females

In [ ]:
cre_serum_male = cre_serum_demo[cre_serum_demo['SEX'] == '1']
print(len(cre_serum_male.index))

In [ ]:
cre_serum_female = cre_serum_demo[cre_serum_demo['SEX'] == '2']
print(len(cre_serum_female.index))

#### calculate eGFR

equation
- eGFRcr = 142 x min(Scr/κ, 1)^α x max(Scr/κ, 1)^-1.200 x 0.9938^Age x 1.012 [if female]
- Scr = standardized serum creatinine in mg/dL
- κ = 0.7 (females) or 0.9 (males)
- α = -0.241 (female) or -0.302 (male)
- min(Scr/κ, 1) is the minimum of Scr/κ or 1.0
- max(Scr/κ, 1) is the maximum of Scr/κ or 1.0
- Age (years)

In [ ]:
cre_serum_male['eGFR_CAD'] = 142 * np.minimum(cre_serum_male['CREATININE_SERUM_CAD']/0.9, 1)**-0.302 * np.maximum(cre_serum_male['CREATININE_SERUM_CAD']/0.9, 1)**-1.200 * 0.9938**cre_serum_male['AGE_CAD']
print(cre_serum_male['eGFR_CAD'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cre_serum_male[cre_serum_male['eGFR_CAD'] < 60].index))

In [ ]:
cre_serum_male['eGFR_AFIB'] = 142 * np.minimum(cre_serum_male['CREATININE_SERUM_AFIB']/0.9, 1)**-0.302 * np.maximum(cre_serum_male['CREATININE_SERUM_AFIB']/0.9, 1)**-1.200 * 0.9938**cre_serum_male['AGE_AFIB']
print(cre_serum_male['eGFR_AFIB'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cre_serum_male[cre_serum_male['eGFR_AFIB'] < 60].index))

In [ ]:
cre_serum_male['eGFR_HF'] = 142 * np.minimum(cre_serum_male['CREATININE_SERUM_HF']/0.9, 1)**-0.302 * np.maximum(cre_serum_male['CREATININE_SERUM_HF']/0.9, 1)**-1.200 * 0.9938**cre_serum_male['AGE_HF']
print(cre_serum_male['eGFR_HF'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cre_serum_male[cre_serum_male['eGFR_HF'] < 60].index))

In [ ]:
cre_serum_female['eGFR_CAD'] = 142 * np.minimum(cre_serum_female['CREATININE_SERUM_CAD']/0.7, 1)**-0.241 * np.maximum(cre_serum_female['CREATININE_SERUM_CAD']/0.7, 1)**-1.200 * 0.9938**cre_serum_female['AGE_CAD'] * 1.012
print(cre_serum_female['eGFR_CAD'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cre_serum_female[cre_serum_female['eGFR_CAD'] < 60].index))

In [ ]:
cre_serum_female['eGFR_AFIB'] = 142 * np.minimum(cre_serum_female['CREATININE_SERUM_AFIB']/0.7, 1)**-0.241 * np.maximum(cre_serum_female['CREATININE_SERUM_AFIB']/0.7, 1)**-1.200 * 0.9938**cre_serum_female['AGE_AFIB'] * 1.012
print(cre_serum_female['eGFR_AFIB'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cre_serum_female[cre_serum_female['eGFR_AFIB'] < 60].index))

In [ ]:
cre_serum_female['eGFR_HF'] = 142 * np.minimum(cre_serum_female['CREATININE_SERUM_HF']/0.7, 1)**-0.241 * np.maximum(cre_serum_female['CREATININE_SERUM_HF']/0.7, 1)**-1.200 * 0.9938**cre_serum_female['AGE_HF'] * 1.012
print(cre_serum_female['eGFR_HF'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cre_serum_female[cre_serum_female['eGFR_HF'] < 60].index))

#### reconcat

In [ ]:
cre_serum_egfr = pd.concat([cre_serum_male, cre_serum_female], axis = 0)
print(len(cre_serum_egfr.index))
cre_serum_egfr.head()

#### subset and rename

In [ ]:
cre_serum_egfr_sub = cre_serum_egfr[['person_id',
                                     'eGFR_CAD',
                                     'CREATININE_SERUM_CAD_datetime',
                                     'eGFR_AFIB',
                                     'CREATININE_SERUM_AFIB_datetime',
                                     'eGFR_HF',
                                     'CREATININE_SERUM_HF_datetime']]
cre_serum_egfr_sub = cre_serum_egfr_sub.rename(columns = {'CREATININE_SERUM_CAD_datetime' : 'eGFR_CAD_datetime',
                                                          'CREATININE_SERUM_AFIB_datetime' : 'eGFR_AFIB_datetime',
                                                          'CREATININE_SERUM_HF_datetime' : 'eGFR_HF_datetime'})
cre_serum_egfr_sub.head()

### cystatin C

#### filter by measurement name

In [ ]:
cys_c = labs[labs['standard_concept_name'].str.contains('cystatin', case = False)]
cys_c = cys_c[~cys_c['standard_concept_name'].str.contains('rate|measurement', case = False)]
cys_c['standard_concept_name'].unique()

#### clean units

In [ ]:
cys_c_unit = cys_c[~cys_c['unit_concept_name'].str.contains('no', case = False, na = True)]
print(len(cys_c_unit['person_id'].unique()))
print(len(cys_c['person_id'].unique()))

#### apply mad

In [ ]:
cys_c_normal = mad_filter(cys_c_unit)
print(len(cys_c_normal['person_id'].unique()))
print(cys_c_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

#### filter by date, subset, and rename

In [ ]:
lab_input = cys_c_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'CYSTATIN_C'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_CAD'), 'measurement_datetime' : (col + '_CAD_datetime')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_AFIB'), 'measurement_datetime' : (col + '_AFIB_datetime')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_HF'), 'measurement_datetime' : (col + '_HF_datetime')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
cys_c_sub = merge.copy()
cys_c_sub.head()

#### merge with demo

In [ ]:
cys_c_demo = cys_c_sub.merge(demo_sex, on = 'person_id', how = 'inner')
print(len(cys_c_sub.index))
print(len(cys_c_demo.index))
cys_c_demo.head()

#### calculate age at measurement

In [ ]:
cys_c_demo['AGE_CAD'] = cys_c_demo['CYSTATIN_C_CAD_datetime'] - cys_c_demo['date_of_birth']
cys_c_demo['AGE_CAD'] = cys_c_demo['AGE_CAD'].astype(str).str.replace(r' .*', '', regex = True)
cys_c_demo['AGE_CAD'] = cys_c_demo['AGE_CAD'].replace({'NaT' : np.nan})
cys_c_demo['AGE_CAD'] = cys_c_demo['AGE_CAD'].astype(float)
cys_c_demo['AGE_CAD'] = cys_c_demo['AGE_CAD']/365.25

In [ ]:
cys_c_demo['AGE_AFIB'] = cys_c_demo['CYSTATIN_C_AFIB_datetime'] - cys_c_demo['date_of_birth']
cys_c_demo['AGE_AFIB'] = cys_c_demo['AGE_AFIB'].astype(str).str.replace(r' .*', '', regex = True)
cys_c_demo['AGE_AFIB'] = cys_c_demo['AGE_AFIB'].replace({'NaT' : np.nan})
cys_c_demo['AGE_AFIB'] = cys_c_demo['AGE_AFIB'].astype(float)
cys_c_demo['AGE_AFIB'] = cys_c_demo['AGE_AFIB']/365.25

In [ ]:
cys_c_demo['AGE_HF'] = cys_c_demo['CYSTATIN_C_HF_datetime'] - cys_c_demo['date_of_birth']
cys_c_demo['AGE_HF'] = cys_c_demo['AGE_HF'].astype(str).str.replace(r' .*', '', regex = True)
cys_c_demo['AGE_HF'] = cys_c_demo['AGE_HF'].replace({'NaT' : np.nan})
cys_c_demo['AGE_HF'] = cys_c_demo['AGE_HF'].astype(float)
cys_c_demo['AGE_HF'] = cys_c_demo['AGE_HF']/365.25

#### convert values zero or less to missing

In [ ]:
cys_c_demo['CYSTATIN_C_CAD'] = cys_c_demo['CYSTATIN_C_CAD'].mask(cys_c_demo['CYSTATIN_C_CAD'] <= 0)

In [ ]:
cys_c_demo['CYSTATIN_C_AFIB'] = cys_c_demo['CYSTATIN_C_AFIB'].mask(cys_c_demo['CYSTATIN_C_AFIB'] <= 0)

In [ ]:
cys_c_demo['CYSTATIN_C_HF'] = cys_c_demo['CYSTATIN_C_HF'].mask(cys_c_demo['CYSTATIN_C_HF'] <= 0)

#### split into male and female

In [ ]:
cys_c_male = cys_c_demo[cys_c_demo['SEX'] == '1']
print(len(cys_c_male.index))

In [ ]:
cys_c_female = cys_c_demo[cys_c_demo['SEX'] == '2']
print(len(cys_c_female.index))

#### calculate eGFR
- equation: eGFR = 133 x min(Scys/0.8, 1)^-0.499 x max (Scys/0.8, 1)^-1.328 x 0.996^Age x 0.932 [if female]

In [ ]:
cys_c_male['eGFR_CAD'] = 133 * np.minimum(cys_c_male['CYSTATIN_C_CAD']/0.8, 1)**-0.499 * np.maximum(cys_c_male['CYSTATIN_C_CAD']/0.8, 1)**-1.328 * 0.996**cys_c_male['AGE_CAD']
print(cys_c_male['eGFR_CAD'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cys_c_male[cys_c_male['eGFR_CAD'] < 60].index))

In [ ]:
cys_c_male['eGFR_AFIB'] = 133 * np.minimum(cys_c_male['CYSTATIN_C_AFIB']/0.8, 1)**-0.499 * np.maximum(cys_c_male['CYSTATIN_C_AFIB']/0.8, 1)**-1.328 * 0.996**cys_c_male['AGE_AFIB']
print(cys_c_male['eGFR_AFIB'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cys_c_male[cys_c_male['eGFR_AFIB'] < 60].index))

In [ ]:
cys_c_male['eGFR_HF'] = 133 * np.minimum(cys_c_male['CYSTATIN_C_HF']/0.8, 1)**-0.499 * np.maximum(cys_c_male['CYSTATIN_C_HF']/0.8, 1)**-1.328 * 0.996**cys_c_male['AGE_HF']
print(cys_c_male['eGFR_HF'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cys_c_male[cys_c_male['eGFR_HF'] < 60].index))

In [ ]:
cys_c_female['eGFR_CAD'] = 133 * np.minimum(cys_c_female['CYSTATIN_C_CAD']/0.8, 1)**-0.499 * np.maximum(cys_c_female['CYSTATIN_C_CAD']/0.8, 1)**-1.328 * 0.996**cys_c_female['AGE_CAD'] * 0.932
print(cys_c_female['eGFR_CAD'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cys_c_female[cys_c_female['eGFR_CAD'] < 60].index))

In [ ]:
cys_c_female['eGFR_AFIB'] = 133 * np.minimum(cys_c_female['CYSTATIN_C_AFIB']/0.8, 1)**-0.499 * np.maximum(cys_c_female['CYSTATIN_C_AFIB']/0.8, 1)**-1.328 * 0.996**cys_c_female['AGE_AFIB'] * 0.932
print(cys_c_female['eGFR_AFIB'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cys_c_female[cys_c_female['eGFR_AFIB'] < 60].index))

In [ ]:
cys_c_female['eGFR_HF'] = 133 * np.minimum(cys_c_female['CYSTATIN_C_HF']/0.8, 1)**-0.499 * np.maximum(cys_c_female['CYSTATIN_C_HF']/0.8, 1)**-1.328 * 0.996**cys_c_female['AGE_HF'] * 0.932
print(cys_c_female['eGFR_HF'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(cys_c_female[cys_c_female['eGFR_HF'] < 60].index))

#### reconcat

In [ ]:
cys_c_egfr = pd.concat([cys_c_male, cys_c_female], axis = 0)
print(len(cys_c_egfr.index))
cys_c_egfr.head()

#### subset and rename

In [ ]:
cys_c_egfr_sub = cys_c_egfr[['person_id',
                             'eGFR_CAD',
                             'CYSTATIN_C_CAD_datetime',
                             'eGFR_AFIB',
                             'CYSTATIN_C_AFIB_datetime',
                             'eGFR_HF',
                             'CYSTATIN_C_HF_datetime']]
cys_c_egfr_sub = cys_c_egfr_sub.rename(columns = {'CYSTATIN_C_CAD_datetime' : 'eGFR_CAD_datetime',
                                                  'CYSTATIN_C_AFIB_datetime' : 'eGFR_AFIB_datetime',
                                                  'CYSTATIN_C_HF_datetime' : 'eGFR_HF_datetime'})
cys_c_egfr_sub.head()

### egfr

#### filter by measurement name

In [ ]:
egfr = labs[labs['standard_concept_name'].str.contains('Glomerular|filtration|rate|eGFR', case = False)]
egfr = egfr[~egfr['standard_concept_name'].str.contains('gene|MDRD|Schwartz', case = False)]
egfr = egfr[egfr['standard_concept_name'].str.contains('CKD-EPI', case = False)]
egfr['standard_concept_name'].unique()

#### apply mad

In [ ]:
egfr_normal = mad_filter(egfr)
print(len(egfr_normal['person_id'].unique()))
print(egfr_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

#### filter by date, subset, and rename

In [ ]:
lab_input = egfr_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'eGFR'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_CAD'), 'measurement_datetime' : (col + '_CAD_datetime')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_AFIB'), 'measurement_datetime' : (col + '_AFIB_datetime')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_HF'), 'measurement_datetime' : (col + '_HF_datetime')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
egfr_sub = merge.copy()
egfr_sub.head()

### concatenate and aggregate eGFR

#### concatenate

In [ ]:
egfr_all = pd.concat([egfr_sub, cre_serum_egfr_sub, cys_c_egfr_sub], axis = 0)
print(len(egfr_all.index))
print(len(egfr_all['person_id'].unique()))
egfr_all.head()

#### get value closest to case date

In [ ]:
lab_input = egfr_all.copy()
print(len(lab_input['person_id'].unique()))
col = 'eGFR'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['eGFR_CAD_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['eGFR_AFIB_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['eGFR_HF_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'eGFR_CAD']]
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'eGFR_AFIB']]
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'eGFR_HF']]
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
egfr_first = merge.copy()
egfr_first.head()

### merge

In [ ]:
pheno_egfr = pheno_bmi.merge(egfr_first, on = 'person_id', how = 'left')
print(len(pheno_egfr.index))
print(len(pheno_egfr[pheno_egfr['eGFR_CAD'].isna() == True]))
print(len(pheno_egfr[pheno_egfr['eGFR_AFIB'].isna() == True]))
print(len(pheno_egfr[pheno_egfr['eGFR_HF'].isna() == True]))
pheno_egfr.head()

## uACR

### urine creatinine

#### filter by measurement name

In [ ]:
cre_urine = cre[cre['standard_concept_name'].str.contains('Urine', case = False)]
cre_urine = cre_urine[~cre_urine['standard_concept_name'].str.contains('albumin|dopamine|calcium|epinephrine|cortisol|protein|time|ratio|interpretation|24', case = False)]
cre_urine['standard_concept_name'].unique()

#### filter by unit

In [ ]:
cre_urine_unit = cre_urine[~cre_urine['unit_concept_name'].str.contains('no|minute|hour|arbitrary|min|percent|day|total|spec|creatinine', case = False, na = True)]
cre_urine_unit = cre_urine_unit[cre_urine_unit['unit_concept_name'].isin(['milligram per deciliter'])]
print(len(cre_urine['person_id'].unique()))
print(len(cre_urine_unit['person_id'].unique()))
cre_urine_unit['unit_concept_name'].value_counts(dropna = False)

#### apply mad

In [ ]:
cre_urine_normal = mad_filter(cre_urine_unit)
print(len(cre_urine_normal['person_id'].unique()))
print(cre_urine_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

#### filter by date, subset, and rename

In [ ]:
lab_input = cre_urine_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'CREATININE_URINE'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_CAD'), 'measurement_datetime' : (col + '_CAD_datetime')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_AFIB'), 'measurement_datetime' : (col + '_AFIB_datetime')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_HF'), 'measurement_datetime' : (col + '_HF_datetime')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
cre_urine_sub = merge.copy()
cre_urine_sub.head()

### albumin

#### filter by measurement name

In [ ]:
alb = labs[labs['standard_concept_name'].str.contains('Albumin', case = False)]
alb = alb[alb['standard_concept_name'].str.contains('Urine', case = False)]
alb['standard_concept_name'].unique()

In [ ]:
acr_alb = alb[alb['standard_concept_name'].str.contains('Creatinine', case = False)]
alb_raw = alb[~alb['standard_concept_name'].isin(acr_alb['standard_concept_name'])]
alb_raw = alb_raw[~alb_raw['standard_concept_name'].str.contains('Globulin|time|total|hour|duration|creatinine', case = False)]
alb_raw['standard_concept_name'].unique()

#### filter by units

In [ ]:
alb_raw_unit = alb_raw[~alb_raw['unit_concept_name'].str.contains('No|percent|dimer|creatinine|minute', case = False, na = True)]
print(len(alb_raw_unit['person_id'].unique()))
print(len(alb_raw['person_id'].unique()))
alb_raw_unit['unit_concept_name'].value_counts(dropna = False)

In [ ]:
print(len(alb_raw_unit['person_id'].unique()))
alb_raw_unit = alb_raw_unit[~alb_raw_unit['unit_concept_name'].isin(['milligram per gram', 'microgram per milligram', 'million per microliter'])]
print(len(alb_raw_unit['person_id'].unique()))
alb_raw_unit['unit_concept_name'].value_counts(dropna = False)

In [ ]:
alb_raw_g_dL = alb_raw_unit[alb_raw_unit['unit_concept_name'].isin(['gram per deciliter','g/dL'])]
print(alb_raw_g_dL['unit_concept_name'].unique())
alb_raw_g_dL['value_as_number'] = alb_raw_g_dL['value_as_number']*1000
alb_raw_g_dL['value_as_number'].describe().map(lambda x: f'{x:,.2f}')

In [ ]:
alb_raw_mg_L = alb_raw_unit[alb_raw_unit['unit_concept_name'].isin(['milligram per liter','mg/L'])]
print(alb_raw_mg_L['unit_concept_name'].unique())
alb_raw_mg_L['value_as_number'] = alb_raw_mg_L['value_as_number']/10
alb_raw_mg_L['value_as_number'].describe().map(lambda x: f'{x:,.2f}')

In [ ]:
alb_raw_ug_mL = alb_raw_unit[alb_raw_unit['unit_concept_name'].isin(['microgram per milliliter'])]
print(alb_raw_ug_mL['unit_concept_name'].unique())
alb_raw_ug_mL['value_as_number'] = alb_raw_ug_mL['value_as_number']*0.1
alb_raw_ug_mL['value_as_number'].describe().map(lambda x: f'{x:,.2f}')

In [ ]:
alb_raw_ug_dL = alb_raw_unit[alb_raw_unit['unit_concept_name'].isin(['milligram per deciliter'])]
print(alb_raw_ug_dL['unit_concept_name'].unique())
alb_raw_ug_dL['value_as_number'] = alb_raw_ug_dL['value_as_number']/1000
alb_raw_ug_dL['value_as_number'].describe().map(lambda x: f'{x:,.2f}')

In [ ]:
alb_raw_mg_mL = alb_raw_unit[alb_raw_unit['unit_concept_name'].isin(['milligram per milliliter'])]
print(alb_raw_mg_mL['unit_concept_name'].unique())
alb_raw_mg_mL['value_as_number'] = alb_raw_mg_mL['value_as_number']*100
alb_raw_mg_mL['value_as_number'].describe().map(lambda x: f'{x:,.2f}')

In [ ]:
alb_raw_mg_dL = alb_raw_unit[alb_raw_unit['unit_concept_name'].isin(['milligram per deciliter', 'mg/dL'])]
print(alb_raw_mg_dL['unit_concept_name'].unique())
alb_raw_mg_dL['value_as_number'].describe().map(lambda x: f'{x:,.2f}')

In [ ]:
alb_raw_clean_unit = pd.concat([alb_raw_mg_dL, alb_raw_mg_L, alb_raw_ug_mL], axis = 0)
print(len(alb_raw_clean_unit['person_id'].unique()))
alb_raw_clean_unit['value_as_number'].describe().map(lambda x: f'{x:,.2f}')

#### apply mad

In [ ]:
alb_raw_normal = mad_filter(alb_raw_clean_unit)
print(len(alb_raw_normal['person_id'].unique()))
print(alb_raw_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

#### filter by date, subset, and rename

In [ ]:
lab_input = alb_raw_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'ALBUMIN_URINE'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_CAD'), 'measurement_datetime' : (col + '_CAD_datetime')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_AFIB'), 'measurement_datetime' : (col + '_AFIB_datetime')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_HF'), 'measurement_datetime' : (col + '_HF_datetime')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
alb_raw_sub = merge.copy()
alb_raw_sub.head()

### uACR

#### filter by measurement name

In [ ]:
acr_alb = acr_alb[~acr_alb['standard_concept_name'].str.contains('Presence')]
acr_alb['standard_concept_name'].unique()

#### filter by units

In [ ]:
acr_alb_unit = acr_alb[~acr_alb['unit_concept_name'].str.contains('no|percent|second|times|day|ratio|liter|dL', case = False, na = True)]
print(len(acr_alb_unit['person_id'].unique()))
print(len(acr_alb['person_id'].unique()))
acr_alb_unit['unit_concept_name'].value_counts(dropna = False)

In [ ]:
acr_alb_mg_g = acr_alb_unit[acr_alb_unit['unit_concept_name'].isin(['mcg/mg',
                                                                    'mg/g',
                                                                    'microgram per milligram of creatinine',
                                                                    'milligram per gram',
                                                                    'milligram per gram of creatinine'])]
print(acr_alb_mg_g['unit_concept_name'].unique())
acr_alb_mg_g['value_as_number'] = acr_alb_mg_g['value_as_number']
acr_alb_mg_g.groupby('unit_concept_name')['value_as_number'].describe().applymap(lambda x: f'{x:,.2f}')

In [ ]:
acr_alb_mg_mg = acr_alb_unit[acr_alb_unit['unit_concept_name'].isin(['milligram per milligram',
                                                                     'milligram per milligram of creatinine'])]
print(acr_alb_mg_mg['unit_concept_name'].unique())
acr_alb_mg_mg['value_as_number'] = acr_alb_mg_mg['value_as_number']*1000
acr_alb_mg_mg.groupby('unit_concept_name')['value_as_number'].describe().applymap(lambda x: f'{x:,.2f}')

In [ ]:
acr_alb_mg_mg_clean = acr_alb_mg_mg[acr_alb_mg_mg['unit_concept_name'].isin(['milligram per milligram'])]
print(acr_alb_mg_mg_clean['unit_concept_name'].unique())

In [ ]:
acr_alb_mg_mmol = acr_alb_unit[acr_alb_unit['unit_concept_name'].isin(['milligram per millimole'])]
print(acr_alb_mg_mmol['unit_concept_name'].unique())
acr_alb_mg_mmol['value_as_number'] = acr_alb_mg_mmol['value_as_number']*8.84
acr_alb_mg_mmol.groupby('unit_concept_name')['value_as_number'].describe().applymap(lambda x: f'{x:,.2f}')

In [ ]:
acr_alb_mcg_mmol = acr_alb_unit[acr_alb_unit['unit_concept_name'].isin(['microgram per millimole of creatinine'])]
print(acr_alb_mcg_mmol['unit_concept_name'].unique())
acr_alb_mcg_mmol['value_as_number'] = acr_alb_mcg_mmol['value_as_number']/113.12
acr_alb_mcg_mmol.groupby('unit_concept_name')['value_as_number'].describe().applymap(lambda x: f'{x:,.2f}')

In [ ]:
acr_alb_mcg_g = acr_alb_unit[acr_alb_unit['unit_concept_name'].isin(['microgram per gram'])]
print(acr_alb_mcg_g['unit_concept_name'].unique())
acr_alb_mcg_g['value_as_number'] = acr_alb_mcg_g['value_as_number']/1000
acr_alb_mcg_g.groupby('unit_concept_name')['value_as_number'].describe().applymap(lambda x: f'{x:,.2f}')

In [ ]:
acr_alb_clean_unit = pd.concat([acr_alb_mg_g, acr_alb_mg_mg_clean], axis = 0)
print(len(acr_alb_unit['person_id'].unique()))
print(len(acr_alb_clean_unit['person_id'].unique()))

#### apply mad

In [ ]:
acr_alb_normal = mad_filter(acr_alb_clean_unit)
print(len(acr_alb_normal['person_id'].unique()))
print(acr_alb_normal['value_as_number'].describe().apply(lambda x: f'{x:,.2f}'))

#### filter by date, subset, and rename

In [ ]:
lab_input = acr_alb_normal.copy()
print(len(lab_input['person_id'].unique()))
col = 'uACR'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['measurement_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_CAD'), 'measurement_datetime' : (col + '_CAD_datetime')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_AFIB'), 'measurement_datetime' : (col + '_AFIB_datetime')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'value_as_number', 'measurement_datetime']].rename(columns = {'value_as_number' : (col + '_HF'), 'measurement_datetime' : (col + '_HF_datetime')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
acr_alb_sub = merge.copy()
acr_alb_sub.head()

### calculate uACR
- urine albumin / urine creatinine

#### merge

In [ ]:
alb_cre = alb_raw_sub.merge(cre_urine_sub, on = 'person_id', how = 'inner')
print(len(alb_raw_sub.index))
print(len(cre_urine_sub.index))
print(len(alb_cre.index))
alb_cre.head()

#### replace zeros with missing

In [ ]:
alb_cre['ALBUMIN_URINE_CAD'] = alb_cre['ALBUMIN_URINE_CAD'].mask(alb_cre['ALBUMIN_URINE_CAD'] <= 0)
alb_cre['CREATININE_URINE_CAD'] = alb_cre['CREATININE_URINE_CAD'].mask(alb_cre['CREATININE_URINE_CAD'] <= 0)

In [ ]:
alb_cre['ALBUMIN_URINE_AFIB'] = alb_cre['ALBUMIN_URINE_AFIB'].mask(alb_cre['ALBUMIN_URINE_AFIB'] <= 0)
alb_cre['CREATININE_URINE_AFIB'] = alb_cre['CREATININE_URINE_AFIB'].mask(alb_cre['CREATININE_URINE_AFIB'] <= 0)

In [ ]:
alb_cre['ALBUMIN_URINE_HF'] = alb_cre['ALBUMIN_URINE_HF'].mask(alb_cre['ALBUMIN_URINE_HF'] <= 0)
alb_cre['CREATININE_URINE_HF'] = alb_cre['CREATININE_URINE_HF'].mask(alb_cre['CREATININE_URINE_HF'] <= 0)

#### calculate

In [ ]:
alb_cre['uACR_CAD'] = (alb_cre['ALBUMIN_URINE_CAD'] / alb_cre['CREATININE_URINE_CAD']) * 1000
print(alb_cre['uACR_CAD'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(alb_cre[alb_cre['uACR_CAD'] > 30].index))

In [ ]:
alb_cre['uACR_AFIB'] = (alb_cre['ALBUMIN_URINE_AFIB'] / alb_cre['CREATININE_URINE_AFIB']) * 1000
print(alb_cre['uACR_AFIB'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(alb_cre[alb_cre['uACR_AFIB'] > 30].index))

In [ ]:
alb_cre['uACR_HF'] = (alb_cre['ALBUMIN_URINE_HF'] / alb_cre['CREATININE_URINE_HF']) * 1000
print(alb_cre['uACR_HF'].describe().apply(lambda x: f'{x:,.2f}'))
print(len(alb_cre[alb_cre['uACR_HF'] > 30].index))

#### get uACR date
- date closest to dx

In [ ]:
lab_input = alb_cre.copy()
date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')

date['ALBUMIN_URINE_CAD_datetime'] = pd.to_datetime(date['ALBUMIN_URINE_CAD_datetime'], errors = 'coerce').dt.tz_localize(None)
date['CREATININE_URINE_CAD_datetime'] = pd.to_datetime(date['CREATININE_URINE_CAD_datetime'], errors = 'coerce').dt.tz_localize(None)
date['CAD_DATE'] = pd.to_datetime(date['CAD_DATE'], errors = 'coerce').dt.tz_localize(None)

date['ALBUMIN_URINE_AFIB_datetime'] = pd.to_datetime(date['ALBUMIN_URINE_AFIB_datetime'], errors = 'coerce').dt.tz_localize(None)
date['CREATININE_URINE_AFIB_datetime'] = pd.to_datetime(date['CREATININE_URINE_AFIB_datetime'], errors = 'coerce').dt.tz_localize(None)
date['AFIB_DATE'] = pd.to_datetime(date['AFIB_DATE'], errors = 'coerce').dt.tz_localize(None)

date['ALBUMIN_URINE_HF_datetime'] = pd.to_datetime(date['ALBUMIN_URINE_HF_datetime'], errors = 'coerce').dt.tz_localize(None)
date['CREATININE_URINE_HF_datetime'] = pd.to_datetime(date['CREATININE_URINE_HF_datetime'], errors = 'coerce').dt.tz_localize(None)
date['HF_DATE'] = pd.to_datetime(date['HF_DATE'], errors = 'coerce').dt.tz_localize(None)

cad_diff_albumin = (date['ALBUMIN_URINE_CAD_datetime'] - date['CAD_DATE']).abs()
cad_diff_creatinine = (date['CREATININE_URINE_CAD_datetime'] - date['CAD_DATE']).abs()

afib_diff_albumin = (date['ALBUMIN_URINE_AFIB_datetime'] - date['AFIB_DATE']).abs()
afib_diff_creatinine = (date['CREATININE_URINE_AFIB_datetime'] - date['AFIB_DATE']).abs()

hf_diff_albumin = (date['ALBUMIN_URINE_HF_datetime'] - date['HF_DATE']).abs()
hf_diff_creatinine = (date['CREATININE_URINE_HF_datetime'] - date['HF_DATE']).abs()

date['uACR_CAD_datetime'] = pd.to_datetime(np.where(cad_diff_albumin.notna() & ((cad_diff_albumin <= cad_diff_creatinine) | cad_diff_creatinine.isna()), date['ALBUMIN_URINE_CAD_datetime'], np.where(cad_diff_creatinine.notna(), date['CREATININE_URINE_CAD_datetime'], pd.NaT)))
date['uACR_AFIB_datetime'] = pd.to_datetime(np.where(afib_diff_albumin.notna() & ((afib_diff_albumin <= afib_diff_creatinine) | afib_diff_creatinine.isna()), date['ALBUMIN_URINE_AFIB_datetime'], np.where(afib_diff_creatinine.notna(), date['CREATININE_URINE_AFIB_datetime'], pd.NaT)))
date['uACR_HF_datetime'] = pd.to_datetime(np.where(hf_diff_albumin.notna() & ((hf_diff_albumin <= hf_diff_creatinine) | hf_diff_creatinine.isna()), date['ALBUMIN_URINE_HF_datetime'], np.where(hf_diff_creatinine.notna(), date['CREATININE_URINE_HF_datetime'], pd.NaT)))

date = date[['person_id', 'uACR_CAD', 'uACR_CAD_datetime', 'uACR_AFIB', 'uACR_AFIB_datetime', 'uACR_HF', 'uACR_HF_datetime']]

date.loc[date['uACR_CAD_datetime'].isna(), 'uACR_CAD'] = np.nan
date.loc[date['uACR_AFIB_datetime'].isna(), 'uACR_AFIB'] = np.nan
date.loc[date['uACR_HF_datetime'].isna(), 'uACR_HF'] = np.nan

alb_cre_first = date.copy()
alb_cre_first.head()

### concatenate uACR

#### concatenate

In [ ]:
uacr_all = pd.concat([alb_cre_first, acr_alb_sub], axis = 0)
print(len(uacr_all.index))
print(len(uacr_all['person_id'].unique()))
uacr_all.head()

#### get value closest to case date

In [ ]:
lab_input = uacr_all.copy()
print(len(lab_input['person_id'].unique()))
col = 'uACR'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - pd.to_datetime(date['uACR_CAD_datetime'], utc = True, format = 'ISO8601')).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - pd.to_datetime(date['uACR_AFIB_datetime'], utc = True, format = 'ISO8601')).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - pd.to_datetime(date['uACR_HF_datetime'], utc = True, format = 'ISO8601')).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first = cad_first[['person_id', 'uACR_CAD']]
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'uACR_AFIB']]
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'uACR_HF']]
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')
uacr_first = merge.copy()
uacr_first.head()

### merge

In [ ]:
pheno_uacr = pheno_egfr.merge(uacr_first, on = 'person_id', how = 'left')
print(len(pheno_uacr.index))
print(len(pheno_uacr[pheno_uacr['uACR_CAD'].isna() == True]))
print(len(pheno_uacr[pheno_uacr['uACR_AFIB'].isna() == True]))
print(len(pheno_uacr[pheno_uacr['uACR_HF'].isna() == True]))
pheno_uacr.head()

## export and import in progress pheno

In [ ]:
pheno_uacr.to_csv('in_progress_pheno.csv', index = None)

In [ ]:
pheno_uacr = pd.read_csv('in_progress_pheno.csv')

# clean medication df

## BP meds

### make df with list of meds

In [ ]:
med_name_df = pd.DataFrame({'med_name' : ['Furosemide',
                                          'Lasix',
                                          'Bumetanide',
                                          'Bumex',
                                          'Torsemide',
                                          'Demadex',
                                          'Amiloride',
                                          'Midamor', 
                                          'Hydro-ride',
                                          'Hydrochlorothiazide',
                                          'HCTZ',
                                          'Esidrix',
                                          'Hydrodiuril',
                                          'Indapamide',
                                          'Lozol',
                                          'Triamterene',
                                          'Dyrenium',
                                          'Chlorthalidone',
                                          'Thalitone',
                                          'Spironolactone',
                                          'Aldactone',
                                          'Eplerenone',
                                          'Inspra',
                                          'Atenolol',
                                          'Tenormin',
                                          'Bisoprolol',
                                          'Zebeta',
                                          'Carvedilol',
                                          'Coreg',
                                          'Carvedilol phosphate',
                                          'Coreg CR',
                                          'Labetalol',
                                          'Trandate',
                                          'Metoprolol succinate',
                                          'Toprol XL',
                                          'Kapspargo Sprinkle',
                                          'Metoprolol tartrate',
                                          'Lopressor',
                                          'Nadolol',
                                          'Corgard',
                                          'Nebivolol',
                                          'Bystolic',
                                          'Pindolol',
                                          'Visken',
                                          'Propranolol',
                                          'Inderal',
                                          'Inderal LA',
                                          'InnoPran XL',
                                          'Captopril',
                                          'Capoten',
                                          'Enalapril',
                                          'Vasotec',
                                          'Fosinopril',
                                          'Monopril',
                                          'Lisinopril',
                                          'Prinivil',
                                          'Zestril',
                                          'Perindopril',
                                          'Aceon',
                                          'Quinapril',
                                          'Accupril',
                                          'Ramipril',
                                          'Altace',
                                          'Trandolapril',
                                          'Mavik',
                                          'Benazepril',
                                          'Lotensin',
                                          'Moexipril',
                                          'Univasc',
                                          'Candesartan',
                                          'Atacand',
                                          'Losartan',
                                          'Cozaar',
                                          'Valsartan',
                                          'Diovan',
                                          'Irbesartan',
                                          'Avapro',
                                          'Olmesartan',
                                          'Benicar',
                                          'Telmisartan',
                                          'Micardis',
                                          'Azilsartan',
                                          'Edarbi',
                                          'Amlodipine',
                                          'Norvasc',
                                          'Diltiazem',
                                          'Cardizem CD',
                                          'Cardizem SR',
                                          'Dilacor XR',
                                          'Tiazac',
                                          'Felodipine',
                                          'Plendil',
                                          'Nicardipine',
                                          'Cardene SR',
                                          'Nifedipine LA',
                                          'Adalat CC',
                                          'Procardia XL',
                                          'Verapamil',
                                          'Calan SR',
                                          'Covera HS',
                                          'Isoptin SR',
                                          'Verelan',
                                          'Doxazosin',
                                          'Cardura',
                                          'Prazosin',
                                          'Minipress',
                                          'Terazosin hydrochloride',
                                          'Hytrin',
                                          'Methyldopa',
                                          'Aldomet',
                                          'Clonidine',
                                          'Catapres',
                                          'Duraclon',
                                          'Kapvay',
                                          'Nexiclon XR',
                                          'Guanfacine',
                                          'Intuniv',
                                          'Tenex',
                                          'Hydralazine',
                                          'Apresoline',
                                          'Minoxidil',
                                          'Loniten',
                                          'Cozaar',
                                          'Calan',
                                          'verapamil hydrochloride',
                                          'metoprolol tartrate',
                                          'spironolactone',
                                          'Cozaar',
                                          'Calan',
                                          'verapamil hydrochloride',
                                          'metoprolol tartrate',
                                          'spironolactone',
                                          'hydralazine hydrochloride',
                                          'Apresoline',
                                          'clonidine hydrochloride',
                                          'diltiazem hydrochloride',
                                          'Cartia',
                                          'losartan',
                                          'irbesartan',
                                          'captopril',
                                          'azilsartan medoxomil',
                                          'Edarbi',
                                          'lisinopril',
                                          'carvedilol phosphate',
                                          'Diltiazem',
                                          'hydrochlorothiazide',
                                          'furosemide',
                                          'olmesartan medoxomil',
                                          'nicardipine hydrochloride',
                                          'verapamil',
                                          'propranolol hydrochloride',
                                          'bisoprolol fumarate',
                                          'quinapril',
                                          'Accupril',
                                          'bendroflumethiazide',
                                          'nadolol',
                                          'Corzide',
                                          'chlorthalidone',
                                          'felodipine',
                                          'amiloride hydrochloride',
                                          'chlorthalidone',
                                          'Thalitone',
                                          'Lotensin',
                                          'benazepril hydrochloride',
                                          'triamterene',
                                          'labetalol hydrochloride',
                                          'valsartan',
                                          'Diovan',
                                          'methyldopa',
                                          'torsemide',
                                          'Torem',
                                          'ramipril',
                                          'bumetanide',
                                          'enalapril',
                                          'prazosin',
                                          'metoprolol succinate',
                                          'bumetanide',
                                          'perindopril',
                                          'labetalol',
                                          'telmisartan',
                                          'Micardis',
                                          'carvedilol',
                                          'Coreg',
                                          'candesartan cilexetil',
                                          'amlodipine',
                                          'trandolapril',
                                          'atenolol',
                                          'amlodipine',
                                          'propranolol',
                                          'pindolol',
                                          'moexipril hydrochloride',
                                          'clonidine',
                                          'nebivolol',
                                          'eplerenone',
                                          'fosinopril sodium',
                                          'candesartan',
                                          'hydralazine',
                                          'amiloride',
                                          'bisoprolol',
                                          'moexipril',
                                          'indapamide',
                                          'nicardipine',
                                          'azilsartan',
                                          'olmesartan',
                                          'fosinopril',
                                          'benazepril']})
    
print(len(med_name_df.index))
med_name_df.head()

### filter meds df to these ones

In [ ]:
drug_ht = drug[drug['standard_concept_name'].str.contains('|'.join(med_name_df['med_name']), case = False)]
print(len(drug.index))
print(len(drug['person_id'].unique()))
print(len(drug_ht.index))
print(len(drug_ht['person_id'].unique()))

### subset and make case column

In [ ]:
drug_ht_sub = drug_ht[['person_id']].drop_duplicates()
drug_ht_sub['HT_MED'] = 1

### merge

In [ ]:
pheno_drug_ht = pheno_uacr.merge(drug_ht_sub, on = 'person_id', how = 'left')
pheno_drug_ht['HT_MED'] = pheno_drug_ht['HT_MED'].fillna(0)
print(len(pheno_drug_ht.index))
print(pheno_drug_ht['HT_MED'].value_counts(dropna = False))
pheno_drug_ht.head()

## cholesterol meds (statins)

### make med list

In [ ]:
med_name_df = pd.DataFrame({'med_name' : ['Lipitor',
                                          'Lescol',
                                          'atorvastatin',
                                          'fluvastatin',
                                          'lovastatin',
                                          'Altoprev',
                                          'Livalo',
                                          'Zypitamag',
                                          'pitavastatin',
                                          'pravastatin',
                                          'rosuvastatin',
                                          'Crestor',
                                          'Zocor',
                                          'simvastatin']})
print(len(med_name_df.index))
med_name_df.head()

### filter meds to these ones

In [ ]:
drug_chol = drug[drug['standard_concept_name'].str.contains('|'.join(med_name_df['med_name']), case = False)]
print(len(drug.index))
print(len(drug['person_id'].unique()))
print(len(drug_chol.index))
print(len(drug_chol['person_id'].unique()))

### subset and make case column

In [ ]:
drug_chol_sub = drug_chol[['person_id']].drop_duplicates()
drug_chol_sub['CHOL_MED'] = 1

### merge

In [ ]:
pheno_drug_chol = pheno_drug_ht.merge(drug_chol_sub, on = 'person_id', how = 'left')
pheno_drug_chol['CHOL_MED'] = pheno_drug_chol['HT_MED'].fillna(0)
print(len(pheno_drug_chol.index))
print(pheno_drug_chol['CHOL_MED'].value_counts(dropna = False))
pheno_drug_chol.head()

## export and import in progress pheno

In [ ]:
pheno_drug_chol.to_csv('CVD.in_progress_pheno.csv', index = False)

In [ ]:
pheno_drug_chol = pd.read_csv('CVD.in_progress_pheno.csv')

# clean survey data

## smoking

**variable encodings**

All Questions
- 'Smoking: 100 Cigs Lifetime'
- 'Smoking: Smoke Frequency'
- 'Smoking: Daily Smoke Starting Age'
- 'Smoking: Serious Quit Attempt'
- 'Attempt Quit Smoking: Completely Quit Age'
- 'Smoking: Number Of Years'
- 'Smoking: Current Daily Cigarette Number'
- 'Smoking: Average Daily Cigarette Number'
- 'Electronic Smoking: Electric Smoke Participant'
- 'Electronic Smoking: Electric Smoke Frequency'
- 'Cigar Smoking: Cigar Smoke Participant'
- 'Cigar Smoking: Current Cigar Frequency'
- 'Hookah Smoking: Hookah Smoke Participant'
- 'Hookah Smoking: Current Hookah Frequency'

Cigarette questions
- 'Smoking: 100 Cigs Lifetime'
- 'Smoking: Smoke Frequency'
- 'Smoking: Daily Smoke Starting Age'
- 'Smoking: Serious Quit Attempt'
- 'Attempt Quit Smoking: Completely Quit Age'
- 'Smoking: Number Of Years'
- 'Smoking: Current Daily Cigarette Number'
- 'Smoking: Average Daily Cigarette Number'

Electronic smoking
- 'Electronic Smoking: Electric Smoke Participant'
- 'Electronic Smoking: Electric Smoke Frequency'

Cigar smoking
- 'Cigar Smoking: Cigar Smoke Participant'
- 'Cigar Smoking: Current Cigar Frequency'

Hookah smoking
- 'Hookah Smoking: Hookah Smoke Participant'
- 'Hookah Smoking: Current Hookah Frequency'

**question encodings**

***Cigarette questions***

Smoking: Number Of Years
- 'PMI: Dont Know' : np.nan (missing)
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- rest are integers- no recoding

Smoking: Smoke Frequency
- 'Smoke Frequency: Not At All' : 0
- 'Smoke Frequency: Every Day' : 2
- 'Smoke Frequency: Some Days' : 1
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- 'PMI: Dont Know' : np.nan (missing)

Smoking: 100 Cigs Lifetime
- '100 Cigs Lifetime: No' : 0
- '100 Cigs Lifetime: Yes' : 1
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- 'PMI: Dont Know' : np.nan (missing)

Smoking: Current Daily Cigarette Number
- 'PMI: Dont Know' : np.nan (missing)
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- rest are integers- no recoding

Attempt Quit Smoking: Completely Quit Age
- 'PMI: Dont Know' : np.nan (missing)
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- rest are integers- no recoding

Smoking: Daily Smoke Starting Age
- 'PMI: Dont Know' : np.nan (missing)
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- rest are integers- no recoding

Smoking: Average Daily Cigarette Number
- 'PMI: Dont Know' : np.nan (missing)
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- rest are integers- no recoding

***cigar questions***

Cigar Smoking: Cigar Smoke Participant
- 'Cigar Smoke Participant: No' : 0
- 'Cigar Smoke Participant: Yes' : 1
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- 'PMI: Dont Know' : np.nan (missing)

Cigar Smoking: Current Cigar Frequency
- 'Current Cigar Frequency: Not At All' : 0
- 'Current Cigar Frequency: Some Days' : 1
- 'Current Cigar Frequency: Every Day' : 2
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- 'PMI: Dont Know' : np.nan (missing)

***electronic smoking questions***

Electronic Smoking: Electric Smoke Participant
- 'Electric Smoke Participant: No' : 0
- 'Electric Smoke Participant: Yes' : 1
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- 'PMI: Dont Know' : np.nan (missing)

Electronic Smoking: Electric Smoke Frequency
- 'Electric Smoke Frequency: Not At All' : 0
- 'Electric Smoke Frequency: Some Days' : 1
- 'Electric Smoke Frequency: Every Day' : 2
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- 'PMI: Dont Know' : np.nan (missing)

***hookah smoking questions***

Hookah Smoking: Hookah Smoke Participant
- 'Hookah Smoke Participant: No' : 0
- 'Hookah Smoke Participant: Yes' : 1
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- 'PMI: Dont Know' : np.nan (missing)

Hookah Smoking: Current Hookah Frequency
- 'Current Hookah Frequency: Not At All' : 0
- 'Current Hookah Frequency: Some Days' : 1
- 'Current Hookah Frequency: Every Day' : 2
- 'PMI: Skip' : np.nan (missing)
- 'PMI: Prefer Not To Answer' : np.nan (missing)
- 'PMI: Dont Know' : np.nan (missing)

If they did not answer the question, they are not a smoker. Questions were asked in a flexible way in the survey

In [ ]:
smoking_survey = survey[survey['question'].str.contains('smoking', case = False)]
smoking_survey['question'].unique()

### cigarette questions

#### Smoking: Number Of Years

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Smoking: Number Of Years'])]
print(smoking_survey_sub[smoking_survey_sub['answer'].str.contains('PMI')]['answer'].value_counts())
smoking_survey_sub[~smoking_survey_sub['answer'].str.contains('PMI')]['answer'].astype(int).describe()

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_CIG_YEARS'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_CIG_YEARS']]
smoking_survey_sub['SMOKING_CIG_YEARS'] = smoking_survey_sub['SMOKING_CIG_YEARS'].replace({'PMI: Dont Know' : '-9',
                                                                                           'PMI: Skip' : '-9',
                                                                                           'PMI: Prefer Not To Answer' : '-9'})
smoking_survey_sub['SMOKING_CIG_YEARS'] = smoking_survey_sub['SMOKING_CIG_YEARS'].astype('Int64')
print(smoking_survey_sub['SMOKING_CIG_YEARS'].describe())
print(len(smoking_survey_sub.index))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_sub

#### smoke frequency

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Smoking: Smoke Frequency'])]
print(smoking_survey_sub['answer'].value_counts())

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_CIG_FREQ'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_CIG_FREQ']]
smoking_survey_sub['SMOKING_CIG_FREQ'] = smoking_survey_sub['SMOKING_CIG_FREQ'].replace({'Smoke Frequency: Not At All' : 0,
                                                                                         'Smoke Frequency: Every Day' : 2,
                                                                                         'Smoke Frequency: Some Days' : 1,
                                                                                         'PMI: Skip' : -9,
                                                                                         'PMI: Prefer Not To Answer' : -9,
                                                                                         'PMI: Dont Know' : -9})
smoking_survey_sub['SMOKING_CIG_FREQ'] = smoking_survey_sub['SMOKING_CIG_FREQ'].astype('Int64')
print(smoking_survey_sub['SMOKING_CIG_FREQ'].value_counts(dropna = False))
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

#### 100 cigs in lifetome

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Smoking: 100 Cigs Lifetime'])]
print(len(smoking_survey_sub['person_id'].unique()))
print(len(smoking_survey['person_id'].unique()))
print(smoking_survey_sub['answer'].value_counts())

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_CIG_100_LIFETIME'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_CIG_100_LIFETIME']]
smoking_survey_sub['SMOKING_CIG_100_LIFETIME'] = smoking_survey_sub['SMOKING_CIG_100_LIFETIME'].replace({'100 Cigs Lifetime: No' : 0,
                                                                                                           '100 Cigs Lifetime: Yes' : 1,
                                                                                                           'PMI: Skip' : np.nan,
                                                                                                           'PMI: Prefer Not To Answer' : np.nan,
                                                                                                           'PMI: Dont Know' : np.nan})
smoking_survey_sub['SMOKING_CIG_100_LIFETIME'] = smoking_survey_sub['SMOKING_CIG_100_LIFETIME'].astype('Int64')
print(smoking_survey_sub['SMOKING_CIG_100_LIFETIME'].value_counts(dropna = False))
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

#### current daily cig number

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Smoking: Current Daily Cigarette Number'])]
print(smoking_survey_sub[smoking_survey_sub['answer'].str.contains('PMI')]['answer'].value_counts())
smoking_survey_sub[~smoking_survey_sub['answer'].str.contains('PMI')]['answer'].astype(int).describe()

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_CIG_NUM_DAILY_NOW'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_CIG_NUM_DAILY_NOW']]
smoking_survey_sub['SMOKING_CIG_NUM_DAILY_NOW'] = smoking_survey_sub['SMOKING_CIG_NUM_DAILY_NOW'].replace({'PMI: Dont Know' : '-9',
                                                                                                           'PMI: Skip' : '-9',
                                                                                                           'PMI: Prefer Not To Answer' : '-9'})
smoking_survey_sub['SMOKING_CIG_NUM_DAILY_NOW'] = smoking_survey_sub['SMOKING_CIG_NUM_DAILY_NOW'].astype('Int64')
print(smoking_survey_sub['SMOKING_CIG_NUM_DAILY_NOW'].describe())
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

#### age completely quit smoking

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Attempt Quit Smoking: Completely Quit Age'])]
print(smoking_survey_sub[smoking_survey_sub['answer'].str.contains('PMI')]['answer'].value_counts())
print(smoking_survey_sub[~smoking_survey_sub['answer'].str.contains('PMI')]['answer'].astype(int).describe())
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_CIG_QUIT_AGE'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_CIG_QUIT_AGE']]
smoking_survey_sub['SMOKING_CIG_QUIT_AGE'] = smoking_survey_sub['SMOKING_CIG_QUIT_AGE'].replace({'PMI: Dont Know' : '-9',
                                                                                                 'PMI: Skip' : '-9',
                                                                                                 'PMI: Prefer Not To Answer' : '-9'})
smoking_survey_sub['SMOKING_CIG_QUIT_AGE'] = smoking_survey_sub['SMOKING_CIG_QUIT_AGE'].astype('Int64')
print(smoking_survey_sub['SMOKING_CIG_QUIT_AGE'].describe())
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

#### daily smoking starting age

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Smoking: Daily Smoke Starting Age'])]
print(smoking_survey_sub[smoking_survey_sub['answer'].str.contains('PMI')]['answer'].value_counts())
print(smoking_survey_sub[~smoking_survey_sub['answer'].str.contains('PMI')]['answer'].astype(int).describe())
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_CIG_DAILY_START_AGE'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_CIG_DAILY_START_AGE']]
smoking_survey_sub['SMOKING_CIG_DAILY_START_AGE'] = smoking_survey_sub['SMOKING_CIG_DAILY_START_AGE'].replace({'PMI: Dont Know' : '-9',
                                                                                                               'PMI: Skip' : '-9',
                                                                                                               'PMI: Prefer Not To Answer' : '-9'})
smoking_survey_sub['SMOKING_CIG_DAILY_START_AGE'] = smoking_survey_sub['SMOKING_CIG_DAILY_START_AGE'].astype('Int64')
print(smoking_survey_sub['SMOKING_CIG_DAILY_START_AGE'].describe())
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

#### avg cig number over entire smoking time

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Smoking: Average Daily Cigarette Number'])]
print(smoking_survey_sub[smoking_survey_sub['answer'].str.contains('PMI')]['answer'].value_counts())
print(smoking_survey_sub[~smoking_survey_sub['answer'].str.contains('PMI')]['answer'].astype(int).describe())
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_CIG_NUM_DAILY_LIFETIME'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_CIG_NUM_DAILY_LIFETIME']]
smoking_survey_sub['SMOKING_CIG_NUM_DAILY_LIFETIME'] = smoking_survey_sub['SMOKING_CIG_NUM_DAILY_LIFETIME'].replace({'PMI: Dont Know' : '-9',
                                                                                                                     'PMI: Skip' : '-9',
                                                                                                                     'PMI: Prefer Not To Answer' : '-9'})
smoking_survey_sub['SMOKING_CIG_NUM_DAILY_LIFETIME'] = smoking_survey_sub['SMOKING_CIG_NUM_DAILY_LIFETIME'].astype('Int64')
print(smoking_survey_sub['SMOKING_CIG_NUM_DAILY_LIFETIME'].describe())
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

### cigar questions

#### cigar smoke participant

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Cigar Smoking: Cigar Smoke Participant'])]
print(len(smoking_survey_sub['person_id'].unique()))
print(len(smoking_survey['person_id'].unique()))
print(smoking_survey_sub['answer'].value_counts())

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_CIGAR_STATUS'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_CIGAR_STATUS']]
smoking_survey_sub['SMOKING_CIGAR_STATUS'] = smoking_survey_sub['SMOKING_CIGAR_STATUS'].replace({'Cigar Smoke Participant: No' : 0,
                                                                                                 'Cigar Smoke Participant: Yes' : 1,
                                                                                                 'PMI: Skip' : np.nan,
                                                                                                 'PMI: Prefer Not To Answer' : np.nan,
                                                                                                 'PMI: Dont Know' : np.nan})
smoking_survey_sub['SMOKING_CIGAR_STATUS'] = smoking_survey_sub['SMOKING_CIGAR_STATUS'].astype('Int64')
print(smoking_survey_sub['SMOKING_CIGAR_STATUS'].value_counts(dropna = False))
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

#### cigar freq

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Cigar Smoking: Current Cigar Frequency'])]
print(len(smoking_survey_sub['person_id'].unique()))
print(len(smoking_survey['person_id'].unique()))
print(smoking_survey_sub['answer'].value_counts())

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_CIGAR_FREQ'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_CIGAR_FREQ']]
smoking_survey_sub['SMOKING_CIGAR_FREQ'] = smoking_survey_sub['SMOKING_CIGAR_FREQ'].replace({'Current Cigar Frequency: Not At All' : 0,
                                                                                             'Current Cigar Frequency: Some Days' : 1,
                                                                                             'Current Cigar Frequency: Every Day' : 2,
                                                                                             'PMI: Skip' : -9,
                                                                                             'PMI: Prefer Not To Answer' : -9,
                                                                                             'PMI: Dont Know' : -9})
smoking_survey_sub['SMOKING_CIGAR_FREQ'] = smoking_survey_sub['SMOKING_CIGAR_FREQ'].astype('Int64')
print(smoking_survey_sub['SMOKING_CIGAR_FREQ'].value_counts(dropna = False))
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

### electronic smoking questions

#### status

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Electronic Smoking: Electric Smoke Participant'])]
print(smoking_survey_sub['answer'].value_counts())

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_ELECTRONIC_STATUS'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_ELECTRONIC_STATUS']]
smoking_survey_sub['SMOKING_ELECTRONIC_STATUS'] = smoking_survey_sub['SMOKING_ELECTRONIC_STATUS'].replace({'Electric Smoke Participant: No' : 0,
                                                                                                           'Electric Smoke Participant: Yes' : 1,
                                                                                                           'PMI: Skip' : np.nan,
                                                                                                           'PMI: Prefer Not To Answer' : np.nan,
                                                                                                           'PMI: Dont Know' : np.nan})
smoking_survey_sub['SMOKING_ELECTRONIC_STATUS'] = smoking_survey_sub['SMOKING_ELECTRONIC_STATUS'].astype('Int64')
print(smoking_survey_sub['SMOKING_ELECTRONIC_STATUS'].value_counts(dropna = False))
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

#### frequency

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Electronic Smoking: Electric Smoke Frequency'])]
print(smoking_survey_sub['answer'].value_counts())

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_ELECTRONIC_FREQ'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_ELECTRONIC_FREQ']]
smoking_survey_sub['SMOKING_ELECTRONIC_FREQ'] = smoking_survey_sub['SMOKING_ELECTRONIC_FREQ'].replace({'Electric Smoke Frequency: Not At All' : 0,
                                                                                                       'Electric Smoke Frequency: Some Days' : 1,
                                                                                                       'Electric Smoke Frequency: Every Day' : 2,
                                                                                                       'PMI: Skip' : -9,
                                                                                                       'PMI: Prefer Not To Answer' : -9,
                                                                                                       'PMI: Dont Know' : -9})
smoking_survey_sub['SMOKING_ELECTRONIC_FREQ'] = smoking_survey_sub['SMOKING_ELECTRONIC_FREQ'].astype('Int64')
print(smoking_survey_sub['SMOKING_ELECTRONIC_FREQ'].value_counts(dropna = False))
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

### hookah questions

#### status

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Hookah Smoking: Hookah Smoke Participant'])]
print(smoking_survey_sub['answer'].value_counts())

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_HOOKAH_STATUS'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_HOOKAH_STATUS']]
smoking_survey_sub['SMOKING_HOOKAH_STATUS'] = smoking_survey_sub['SMOKING_HOOKAH_STATUS'].replace({'Hookah Smoke Participant: No' : 0,
                                                                                                   'Hookah Smoke Participant: Yes' : 1,
                                                                                                   'PMI: Skip' : np.nan,
                                                                                                   'PMI: Prefer Not To Answer' : np.nan,
                                                                                                   'PMI: Dont Know' : np.nan})
smoking_survey_sub['SMOKING_HOOKAH_STATUS'] = smoking_survey_sub['SMOKING_HOOKAH_STATUS'].astype('Int64')
print(smoking_survey_sub['SMOKING_HOOKAH_STATUS'].value_counts(dropna = False))
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

#### frequency

In [ ]:
smoking_survey_sub = smoking_survey[smoking_survey['question'].isin(['Hookah Smoking: Current Hookah Frequency'])]
print(smoking_survey_sub['answer'].value_counts())

In [ ]:
smoking_survey_sub.rename(columns = {'answer' : 'SMOKING_HOOKAH_FREQ'}, inplace = True)
smoking_survey_sub = smoking_survey_sub[['person_id', 'SMOKING_HOOKAH_FREQ']]
smoking_survey_sub['SMOKING_HOOKAH_FREQ'] = smoking_survey_sub['SMOKING_HOOKAH_FREQ'].replace({'Current Hookah Frequency: Not At All' : 0,
                                                                                               'Current Hookah Frequency: Some Days' : 1,
                                                                                               'Current Hookah Frequency: Every Day' : 2,
                                                                                               'PMI: Skip' : -9,
                                                                                               'PMI: Prefer Not To Answer' : -9,
                                                                                               'PMI: Dont Know' : -9})
smoking_survey_sub['SMOKING_HOOKAH_FREQ'] = smoking_survey_sub['SMOKING_HOOKAH_FREQ'].astype('Int64')
print(smoking_survey_sub['SMOKING_HOOKAH_FREQ'].value_counts(dropna = False))
print(len(smoking_survey_sub.index))
print(len(smoking_survey_sub['person_id'].unique()))
smoking_survey_sub.head()

In [ ]:
smoking_survey_clean = smoking_survey_clean.merge(smoking_survey_sub, on = 'person_id', how = 'outer')
smoking_survey_clean.head()

### fill in missing values

#### cigs

In [ ]:
smoking_cigs_non_smoker = smoking_survey_clean[smoking_survey_clean['SMOKING_CIG_100_LIFETIME'] == 0]
smoking_cigs_smoker = smoking_survey_clean[~smoking_survey_clean.index.isin(smoking_cigs_non_smoker.index)]
print(len(smoking_cigs_non_smoker.index))
print(len(smoking_cigs_smoker.index))
smoking_cigs_non_smoker.head()

In [ ]:
cig_cols = ['SMOKING_CIG_YEARS',
            'SMOKING_CIG_FREQ',
            'SMOKING_CIG_NUM_DAILY_NOW',
            'SMOKING_CIG_QUIT_AGE',
            'SMOKING_CIG_DAILY_START_AGE',
            'SMOKING_CIG_NUM_DAILY_LIFETIME']

In [ ]:
smoking_cigs_non_smoker[cig_cols] = smoking_cigs_non_smoker[cig_cols].fillna(0)
smoking_cigs_non_smoker.head()

In [ ]:
smoking_cigs_clean = pd.concat([smoking_cigs_non_smoker, smoking_cigs_smoker], axis = 0)
print(len(smoking_cigs_clean.index))
print(len(smoking_survey_clean.index))

#### cigar

In [ ]:
smoking_cigar_non_smoker = smoking_cigs_clean[smoking_cigs_clean['SMOKING_CIGAR_STATUS'] == 0]
smoking_cigar_smoker = smoking_cigs_clean[~smoking_cigs_clean.index.isin(smoking_cigar_non_smoker.index)]
print(len(smoking_cigar_non_smoker.index))
print(len(smoking_cigar_smoker.index))
smoking_cigar_non_smoker[['person_id', 'SMOKING_CIGAR_FREQ']].head()

In [ ]:
smoking_cigar_non_smoker['SMOKING_CIGAR_FREQ'] = smoking_cigar_non_smoker['SMOKING_CIGAR_FREQ'].fillna(0)
smoking_cigar_non_smoker[['person_id', 'SMOKING_CIGAR_FREQ']].head()

In [ ]:
smoking_cigar_clean = pd.concat([smoking_cigar_non_smoker, smoking_cigar_smoker], axis = 0)
print(len(smoking_cigar_clean.index))
print(len(smoking_survey_clean.index))

#### electronic

In [ ]:
smoking_electronic_non_smoker = smoking_cigar_clean[smoking_cigar_clean['SMOKING_ELECTRONIC_STATUS'] == 0]
smoking_electronic_smoker = smoking_cigar_clean[~smoking_cigar_clean.index.isin(smoking_electronic_non_smoker.index)]
print(len(smoking_electronic_non_smoker.index))
print(len(smoking_electronic_smoker.index))
smoking_electronic_non_smoker[['person_id', 'SMOKING_ELECTRONIC_FREQ']].head()

In [ ]:
smoking_electronic_non_smoker['SMOKING_ELECTRONIC_FREQ'] = smoking_electronic_non_smoker['SMOKING_ELECTRONIC_FREQ'].fillna(0)
smoking_electronic_non_smoker[['person_id', 'SMOKING_ELECTRONIC_FREQ']].head()

In [ ]:
smoking_electronic_clean = pd.concat([smoking_electronic_non_smoker, smoking_electronic_smoker], axis = 0)
print(len(smoking_electronic_clean.index))
print(len(smoking_survey_clean.index))

#### hookah

In [ ]:
smoking_hookah_non_smoker = smoking_electronic_clean[smoking_electronic_clean['SMOKING_HOOKAH_STATUS'] == 0]
smoking_hookah_smoker = smoking_electronic_clean[~smoking_electronic_clean.index.isin(smoking_hookah_non_smoker.index)]
print(len(smoking_hookah_non_smoker.index))
print(len(smoking_hookah_smoker.index))
smoking_hookah_non_smoker[['person_id', 'SMOKING_HOOKAH_FREQ']].head()

In [ ]:
smoking_hookah_non_smoker['SMOKING_HOOKAH_FREQ'] = smoking_hookah_non_smoker['SMOKING_HOOKAH_FREQ'].fillna(0)
smoking_hookah_non_smoker[['person_id', 'SMOKING_HOOKAH_FREQ']].head()

In [ ]:
smoking_hookah_clean = pd.concat([smoking_hookah_non_smoker, smoking_hookah_smoker], axis = 0)
print(len(smoking_hookah_clean.index))
print(len(smoking_survey_clean.index))

### replace -9 with missing

In [ ]:
smoking_all_clean = smoking_hookah_clean.replace(-9, np.nan)

### create combined cig column

column encoding
- 0 = never
- 1 = previous
- 2 = current

Cigarette questions
- 'Smoking: 100 Cigs Lifetime'
- 'Smoking: Smoke Frequency'
- 'Smoking: Daily Smoke Starting Age'
- 'Smoking: Serious Quit Attempt'
- 'Attempt Quit Smoking: Completely Quit Age'
- 'Smoking: Number Of Years'
- 'Smoking: Current Daily Cigarette Number'
- 'Smoking: Average Daily Cigarette Number'

Any value > 0 = current smoker (2)
- 'Smoking: 100 Cigs Lifetime'
- 'Smoking: Smoke Frequency'
- 'Smoking: Daily Smoke Starting Age'
- 'Smoking: Serious Quit Attempt'
- 'Smoking: Number Of Years'
- 'Smoking: Current Daily Cigarette Number'
- 'Smoking: Average Daily Cigarette Number'

Disease age > quit age = previous smoker (1)
- 'Smoking: Serious Quit Attempt'

All values 0 or missing = non-smoker (0)

#### make age df

In [ ]:
pheno_pgs = pd.read_csv('CVD.IRM.phenotype.csv')

In [ ]:
pheno_age = pheno_drug_chol[['person_id', 'CAD', 'AGE_CAD', 'AFIB', 'AGE_AFIB', 'HF', 'AGE_HF']]
pheno_age.head()

In [ ]:
pheno_age = pheno_pgs[['person_id', 'CAD', 'AGE_CAD', 'AFIB', 'AGE_AFIB', 'HF', 'AGE_HF']]
pheno_age.head()

#### merge with pheno date

In [ ]:
smoking_survey_age = pheno_age.merge(smoking_all_clean, on = 'person_id', how = 'inner')
print(len(smoking_survey_clean.index))
print(len(pheno_age.index))
print(len(smoking_survey_age.index))
smoking_survey_age.head()

#### initialize smoking columns

In [ ]:
smoking_survey_age["SMOKING_CIG_STATUS_CAD"] = 2
smoking_survey_age["SMOKING_CIG_STATUS_AFIB"] = 2
smoking_survey_age["SMOKING_CIG_STATUS_HF"] = 2

#### identify former smokers

In [ ]:
cad_mask = (smoking_survey_age["SMOKING_CIG_QUIT_AGE"].notna()) & (smoking_survey_age["SMOKING_CIG_QUIT_AGE"] < smoking_survey_age["AGE_CAD"])
smoking_survey_age.loc[cad_mask, "SMOKING_CIG_STATUS_CAD"] = 1

smoking_survey_age[smoking_survey_age['SMOKING_CIG_QUIT_AGE'].notna()][['person_id', 'AGE_CAD', 'SMOKING_CIG_QUIT_AGE', 'SMOKING_CIG_STATUS_CAD']].head()

In [ ]:
afib_mask = (smoking_survey_age["SMOKING_CIG_QUIT_AGE"].notna()) & (smoking_survey_age["SMOKING_CIG_QUIT_AGE"] < smoking_survey_age["AGE_AFIB"])
smoking_survey_age.loc[afib_mask, "SMOKING_CIG_STATUS_AFIB"] = 1

smoking_survey_age[smoking_survey_age['SMOKING_CIG_QUIT_AGE'].notna()][['person_id', 'AGE_AFIB', 'SMOKING_CIG_QUIT_AGE', 'SMOKING_CIG_STATUS_AFIB']].head()

In [ ]:
hf_mask = (smoking_survey_age["SMOKING_CIG_QUIT_AGE"].notna()) & (smoking_survey_age["SMOKING_CIG_QUIT_AGE"] < smoking_survey_age["AGE_HF"])
smoking_survey_age.loc[hf_mask, "SMOKING_CIG_STATUS_HF"] = 1

smoking_survey_age[smoking_survey_age['SMOKING_CIG_QUIT_AGE'].notna()][['person_id', 'AGE_HF', 'SMOKING_CIG_QUIT_AGE', 'SMOKING_CIG_STATUS_HF']].head()

#### identify non-smokers

In [ ]:
smoking_cols = [col for col in smoking_survey_clean.columns if col.startswith('SMOKING_CIG_')]
smoking_cols.remove('SMOKING_CIG_QUIT_AGE')
print(smoking_cols)

In [ ]:
smoking_survey_age.loc[(smoking_survey_age[smoking_cols] == 0).all(axis = 1), "SMOKING_CIG_STATUS_CAD"] = 0
print(smoking_survey_age['SMOKING_CIG_STATUS_CAD'].value_counts(dropna = False))
smoking_survey_age.loc[(smoking_survey_age[smoking_cols] == 0).all(axis = 1), "SMOKING_CIG_STATUS_AFIB"] = 0
print(smoking_survey_age['SMOKING_CIG_STATUS_AFIB'].value_counts(dropna = False))
smoking_survey_age.loc[(smoking_survey_age[smoking_cols] == 0).all(axis = 1), "SMOKING_CIG_STATUS_HF"] = 0
print(smoking_survey_age['SMOKING_CIG_STATUS_HF'].value_counts(dropna = False))

#### subset

In [ ]:
smoking_survey_final = smoking_survey_age.drop(columns = ['CAD', 'AGE_CAD', 'AFIB', 'AGE_AFIB', 'HF', 'AGE_HF'])

### merge

In [ ]:
pheno_smoking = pheno_drug_chol.merge(smoking_survey_final, on = 'person_id', how = 'left')
print(len(pheno_smoking.index))
pheno_smoking.head()

### update current pheno file

In [ ]:
smoking_cols = pheno_pgs.columns[pheno_pgs.columns.str.contains('SMOKING_')].tolist()
print(smoking_cols)

In [ ]:
pheno_pgs_sub = pheno_pgs.drop(columns = smoking_cols)

In [ ]:
pheno_pgs = pheno_pgs_sub.merge(smoking_survey_final, on = 'person_id', how = 'left')
print(len(pheno_pgs.index))
pheno_pgs.head()

### export in progress pheno

In [ ]:
pheno_smoking.to_csv('CVD.in_progress_pheno.csv', index = None)

## income

**encoding**

Income: Annual Income
- 'PMI: Prefer Not To Answer' : np.nan
- 'PMI: Skip' : np.nan
- 'Annual Income: less 10k' : 8
- 'Annual Income: 10k 25k' : 7
- 'Annual Income: 25k 35k' : 6
- 'Annual Income: 35k 50k' : 5
- 'Annual Income: 50k 75k' : 4
- 'Annual Income: 75k 100k' : 3
- 'Annual Income: 100k 150k' : 2
- 'Annual Income: 150k 200k' : 1
- 'Annual Income: more 200k' : 0

In [ ]:
income_survey = survey[survey['question'].str.contains('income', case = False)]
income_survey['question'].unique()

### annual income

In [ ]:
income_survey_sub = income_survey[income_survey['question'].isin(['Income: Annual Income'])]
income_survey_sub['answer'].value_counts()

In [ ]:
income_survey_sub.rename(columns = {'answer' : 'INCOME_ANNUAL'}, inplace = True)
income_survey_sub = income_survey_sub[['person_id', 'INCOME_ANNUAL']]
income_survey_sub['INCOME_ANNUAL'] = income_survey_sub['INCOME_ANNUAL'].replace({'PMI: Prefer Not To Answer' : np.nan,
                                                                                 'PMI: Skip' : np.nan,
                                                                                 'Annual Income: less 10k' : 8,
                                                                                 'Annual Income: 10k 25k' : 7,
                                                                                 'Annual Income: 25k 35k' : 6,
                                                                                 'Annual Income: 35k 50k' : 5,
                                                                                 'Annual Income: 50k 75k' : 4,
                                                                                 'Annual Income: 75k 100k' : 3,
                                                                                 'Annual Income: 100k 150k' : 2,
                                                                                 'Annual Income: 150k 200k' : 1,
                                                                                 'Annual Income: more 200k' : 0})
income_survey_sub['INCOME_ANNUAL'] = income_survey_sub['INCOME_ANNUAL'].astype('Int64')
print(income_survey_sub['INCOME_ANNUAL'].value_counts(dropna = False))
print(len(income_survey_sub.index))
print(len(income_survey_sub['person_id'].unique()))
income_survey_sub.head()

In [ ]:
income_survey_clean = income_survey_sub

### merge

In [ ]:
pheno_income = pheno_smoking.merge(income_survey_clean, on = 'person_id', how = 'left')
print(len(pheno_income.index))
print(pheno_income['INCOME_ANNUAL'].value_counts(dropna = False))
pheno_income.head()

## education
**encoding**

Education Level: Highest Grade
- 'Highest Grade: Never Attended' : 7
- 'Highest Grade: One Through Four' : 6
- 'Highest Grade: Five Through Eight' : 5
- 'Highest Grade: Nine Through Eleven' : 4
- 'Highest Grade: Twelve Or GED' : 3
- 'Highest Grade: College One to Three' : 2
- 'Highest Grade: College Graduate' : 1
- 'Highest Grade: Advanced Degree' : 0
- 'PMI: Skip' : np.nan
- 'PMI: Prefer Not To Answer' : np.nan

In [ ]:
education_survey = survey[survey['question'].str.contains('education', case = False)]
education_survey['question'].unique()

### highest level education

In [ ]:
education_survey_sub = education_survey[education_survey['question'].isin(['Education Level: Highest Grade'])]
education_survey_sub['answer'].value_counts()

In [ ]:
education_survey_sub.rename(columns = {'answer' : 'EDUCATION_HIGHEST'}, inplace = True)
education_survey_sub = education_survey_sub[['person_id', 'EDUCATION_HIGHEST']]
education_survey_sub['EDUCATION_HIGHEST'] = education_survey_sub['EDUCATION_HIGHEST'].replace({'Highest Grade: Never Attended' : 7,
                                                                                                             'Highest Grade: One Through Four' : 6,
                                                                                                             'Highest Grade: Five Through Eight' : 5,
                                                                                                             'Highest Grade: Nine Through Eleven' : 4,
                                                                                                             'Highest Grade: Twelve Or GED' : 3,
                                                                                                             'Highest Grade: College One to Three' : 2,
                                                                                                             'Highest Grade: College Graduate' : 1,
                                                                                                             'Highest Grade: Advanced Degree' : 0,
                                                                                                             'PMI: Skip' : np.nan,
                                                                                                             'PMI: Prefer Not To Answer' : np.nan})
education_survey_sub['EDUCATION_HIGHEST'] = education_survey_sub['EDUCATION_HIGHEST'].astype('Int64')
print(education_survey_sub['EDUCATION_HIGHEST'].value_counts(dropna = False))
print(len(education_survey_sub.index))
print(len(education_survey_sub['person_id'].unique()))
education_survey_sub.head()

In [ ]:
education_survey_clean = education_survey_sub

### merge

In [ ]:
pheno_education = pheno_income.merge(education_survey_clean, on = 'person_id', how = 'left')
print(len(pheno_education.index))
print(pheno_education['EDUCATION_HIGHEST'].value_counts(dropna = False))
pheno_education.head()

## neighborhood

**encoding**

What is the main type of housing in your neighborhood?
- 'Detached single-family housing' : 0
- 'Townhouses' : 1
- 'Mix of single-family residences and townhouses' : 2
- 'Apartments or condos of 4-12 stories' : 3
- 'Apartments or condos of more than 12 stories' : 4
- 'PMI: Dont Know' : np.nan
- 'PMI: Skip' : np.nan

How much you agree or disagree that your neighborhood is safe?
- 'Strongly agree' : 0
- 'Agree' : 1
- 'Disagree' : 2
- 'Strongly disagree' : 3
- 'PMI: Skip' : np.nan

How much you agree or disagree that your neighborhood is clean?
- 'Strongly agree' : 0,
- 'Agree' : 1,
- 'Disagree' : 2,
- 'Strongly disagree' : 3,
- 'PMI: Skip' : np.nan

How much you agree or disagree that your neighborhood is noisy?
- 'Strongly agree' : 3,
- 'Agree' : 2,
- 'Disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan

How much you agree or disagree that vandalism is common in your neighborhood?
- 'Strongly agree' : 3,
- 'Agree' : 2,
- 'Disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan

How much you agree or disagree that people in your neighborhood can be trusted?
- 'Strongly agree' : 0,
- 'Agree' : 1,
- 'Neutral (neither agree nor disagree)' : 2,
- 'Disagree' : 3,
- 'Strongly disagree' : 4,
- 'PMI: Skip' : np.nan

How much you agree or disagree that there is a lot of crime in your neighborhood?
- 'Strongly agree' : 3,
- 'Agree' : 2,
- 'Disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan

How much you agree or disagree that there is a lot of graffiti in your neighborhood?
- 'Strongly agree' : 3,
- 'Agree' : 2,
- 'Disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan

How much you agree or disagree that there is too much drug use in your neighborhood?
- 'Strongly agree' : 3,
- 'Agree' : 2,
- 'Disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan

How much you agree or disagree that people in your neighborhood share the same values?
- 'Strongly agree' : 0,
- 'Agree' : 1,
- 'Neutral (neither agree nor disagree)' : 2,
- 'Disagree' : 3,
- 'Strongly disagree' : 4,
- 'PMI: Skip' : np.nan

How much you agree or disagree that there is too much alcohol use in your neighborhood?
- 'Strongly agree' : 3,
- 'Agree' : 2,
- 'Disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan

There are sidewalks on most of the streets in my neighborhood. Would you say that you...
- 'Strongly agree' : 0,
- 'Somewhat agree' : 1,
- 'Somewhat disagree' : 2,
- 'Strongly disagree' : 3,
- 'Does not apply to my neighborhood' : np.nan,
- 'PMI: Skip' : np.nan,
- 'PMI: Dont Know': np.nan

How much you agree or disagree that in your neighborhood people watch out for each other?
- 'Strongly agree' : 0,
- 'Agree' : 1,
- 'Disagree' : 2,
- 'Strongly disagree' : 3,
- 'PMI: Skip' : np.nan

How much you agree or disagree that there are lot of abandoned buildings in your neighborhood?
- 'Strongly agree' : 3,
- 'Agree' : 2,
- 'Disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan

How much you agree or disagree that people in your neighborhood generally get along with each other?
- 'Strongly agree' : 0,
- 'Agree' : 1,
- 'Neutral (neither agree nor disagree)' : 2,
- 'Disagree' : 3,
- 'Strongly disagree' : 4,
- 'PMI: Skip' : np.nan

The crime rate in my neighborhood makes it unsafe to go on walks at night. Would you say that you...
- 'Strongly agree' : 3,
- 'Somewhat agree' : 2,
- 'Somewhat disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan,
- 'PMI: Dont Know': np.nan

The crime rate in my neighborhood makes it unsafe to go on walks during the day. Would you say that you...
- 'Strongly agree' : 3,
- 'Somewhat agree' : 2,
- 'Somewhat disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan,
- 'PMI: Dont Know': np.nan

How much you agree or disagree that people in your neighborhood take good care of their houses and apartments?
- 'Strongly agree' : 0,
- 'Agree' : 1,
- 'Disagree' : 2,
- 'Strongly disagree' : 3,
- 'PMI: Skip' : np.nan

There are facilities to bicycle in or near my neighborhood, such as special lanes, separate paths or trails, or shared use paths for cycles and pedestrians. Would you say that you...
- 'Strongly agree' : 0,
- 'Somewhat agree' : 1,
- 'Somewhat disagree' : 2,
- 'Strongly disagree' : 3,
- 'Does not apply to my neighborhood' : np.nan,
- 'PMI: Skip' : np.nan,
- 'PMI: Dont Know': np.nan

My neighborhood has several free or low-cost recreation facilities, such as parks, walking trails, bike paths, recreation centers, playgrounds, public swimming pools, etc. Would you say that you...
- 'Strongly agree' : 0,
- 'Somewhat agree' : 1,
- 'Somewhat disagree' : 2,
- 'Strongly disagree' : 3,
- 'PMI: Skip' : np.nan,
- 'PMI: Dont Know': np.nan

How much you agree or disagree that there are too many people hanging around on the streets near your home?
- 'Strongly agree' : 3,
- 'Agree' : 2,
- 'Disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan

How much you agree or disagree that you are always having trouble with your neighbors?
- 'Strongly agree' : 3,
- 'Agree' : 2,
- 'Disagree' : 1,
- 'Strongly disagree' : 0,
- 'PMI: Skip' : np.nan

Many shops, stores, markets or other places to buy things I need are within easy walking distance of my home. Would you say that you...
- 'Strongly agree' : 0,
- 'Somewhat agree' : 1,
- 'Somewhat disagree' : 2,
- 'Strongly disagree' : 3,
- 'PMI: Skip' : np.nan,
- 'PMI: Dont Know': np.nan

It is within a 10-15 minute walk to a transit stop (such as bus, train, trolley, or tram) from my home. Would you say that you...
- 'Strongly agree' : 0,
- 'Somewhat agree' : 1,
- 'Somewhat disagree' : 2,
- 'Strongly disagree' : 3,
- 'PMI: Skip' : np.nan,
- 'PMI: Dont Know': np.nan

In [ ]:
neighbor_survey = survey[survey['question'].str.contains('neighbor|transit|store|street', case = False)]
#neighbor_survey['question'].unique()

### type of housing

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['What is the main type of housing in your neighborhood?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_HOUSING'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_HOUSING']]
neighbor_survey_sub['NEIGHBORHOOD_HOUSING'] = neighbor_survey_sub['NEIGHBORHOOD_HOUSING'].replace({'Detached single-family housing' : 0,
                                                                                                   'Townhouses' : 1,
                                                                                                   'Mix of single-family residences and townhouses' : 2,
                                                                                                   'Apartments or condos of 4-12 stories' : 3,
                                                                                                   'Apartments or condos of more than 12 stories' : 4,
                                                                                                   'PMI: Dont Know' : np.nan,
                                                                                                   'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_HOUSING'] = neighbor_survey_sub['NEIGHBORHOOD_HOUSING'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_HOUSING'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_sub

### safe

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that your neighborhood is safe?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_SAFE'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_SAFE']]
neighbor_survey_sub['NEIGHBORHOOD_SAFE'] = neighbor_survey_sub['NEIGHBORHOOD_SAFE'].replace({'Strongly agree' : 0,
                                                                                             'Agree' : 1,
                                                                                             'Disagree' : 2,
                                                                                             'Strongly disagree' : 3,
                                                                                             'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_SAFE'] = neighbor_survey_sub['NEIGHBORHOOD_SAFE'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_SAFE'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### clean

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that your neighborhood is clean?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_CLEAN'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_CLEAN']]
neighbor_survey_sub['NEIGHBORHOOD_CLEAN'] = neighbor_survey_sub['NEIGHBORHOOD_CLEAN'].replace({'Strongly agree' : 0,
                                                                                               'Agree' : 1,
                                                                                               'Disagree' : 2,
                                                                                               'Strongly disagree' : 3,
                                                                                               'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_CLEAN'] = neighbor_survey_sub['NEIGHBORHOOD_CLEAN'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_CLEAN'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### noisy

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that your neighborhood is noisy?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_NOISE'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_NOISE']]
neighbor_survey_sub['NEIGHBORHOOD_NOISE'] = neighbor_survey_sub['NEIGHBORHOOD_NOISE'].replace({'Strongly agree' : 3,
                                                                                               'Agree' : 2,
                                                                                               'Disagree' : 1,
                                                                                               'Strongly disagree' : 0,
                                                                                               'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_NOISE'] = neighbor_survey_sub['NEIGHBORHOOD_NOISE'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_NOISE'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### vandalism

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that vandalism is common in your neighborhood?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_VANDALISM'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_VANDALISM']]
neighbor_survey_sub['NEIGHBORHOOD_VANDALISM'] = neighbor_survey_sub['NEIGHBORHOOD_VANDALISM'].replace({'Strongly agree' : 3,
                                                                                                     'Agree' : 2,
                                                                                                     'Disagree' : 1,
                                                                                                     'Strongly disagree' : 0,
                                                                                                     'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_VANDALISM'] = neighbor_survey_sub['NEIGHBORHOOD_VANDALISM'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_VANDALISM'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### people can be trusted

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that people in your neighborhood can be trusted?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_TRUST'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_TRUST']]
neighbor_survey_sub['NEIGHBORHOOD_TRUST'] = neighbor_survey_sub['NEIGHBORHOOD_TRUST'].replace({'Strongly agree' : 0,
                                                                                               'Agree' : 1,
                                                                                               'Neutral (neither agree nor disagree)' : 2,
                                                                                               'Disagree' : 3,
                                                                                               'Strongly disagree' : 4,
                                                                                               'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_TRUST'] = neighbor_survey_sub['NEIGHBORHOOD_TRUST'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_TRUST'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### a lot of crime

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that there is a lot of crime in your neighborhood?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_ALOT_CRIME'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_ALOT_CRIME']]
neighbor_survey_sub['NEIGHBORHOOD_ALOT_CRIME'] = neighbor_survey_sub['NEIGHBORHOOD_ALOT_CRIME'].replace({'Strongly agree' : 3,
                                                                                                         'Agree' : 2,
                                                                                                         'Disagree' : 1,
                                                                                                         'Strongly disagree' : 0,
                                                                                                         'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_ALOT_CRIME'] = neighbor_survey_sub['NEIGHBORHOOD_ALOT_CRIME'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_ALOT_CRIME'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### graffiti

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that there is a lot of graffiti in your neighborhood?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_GRAFFITI'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_GRAFFITI']]
neighbor_survey_sub['NEIGHBORHOOD_GRAFFITI'] = neighbor_survey_sub['NEIGHBORHOOD_GRAFFITI'].replace({'Strongly agree' : 3,
                                                                                                     'Agree' : 2,
                                                                                                     'Disagree' : 1,
                                                                                                     'Strongly disagree' : 0,
                                                                                                     'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_GRAFFITI'] = neighbor_survey_sub['NEIGHBORHOOD_GRAFFITI'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_GRAFFITI'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### drug use

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that there is too much drug use in your neighborhood?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_DRUG_USE'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_DRUG_USE']]
neighbor_survey_sub['NEIGHBORHOOD_DRUG_USE'] = neighbor_survey_sub['NEIGHBORHOOD_DRUG_USE'].replace({'Strongly agree' : 3,
                                                                                                     'Agree' : 2,
                                                                                                     'Disagree' : 1,
                                                                                                     'Strongly disagree' : 0,
                                                                                                     'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_DRUG_USE'] = neighbor_survey_sub['NEIGHBORHOOD_DRUG_USE'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_DRUG_USE'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### share same values

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that people in your neighborhood share the same values?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_SAME_VALUES'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_SAME_VALUES']]
neighbor_survey_sub['NEIGHBORHOOD_SAME_VALUES'] = neighbor_survey_sub['NEIGHBORHOOD_SAME_VALUES'].replace({'Strongly agree' : 0,
                                                                                                           'Agree' : 1,
                                                                                                           'Neutral (neither agree nor disagree)' : 2,
                                                                                                           'Disagree' : 3,
                                                                                                           'Strongly disagree' : 4,
                                                                                                           'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_SAME_VALUES'] = neighbor_survey_sub['NEIGHBORHOOD_SAME_VALUES'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_SAME_VALUES'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### alcohol use

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that there is too much alcohol use in your neighborhood?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_ALCOHOL'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_ALCOHOL']]
neighbor_survey_sub['NEIGHBORHOOD_ALCOHOL'] = neighbor_survey_sub['NEIGHBORHOOD_ALCOHOL'].replace({'Strongly agree' : 3,
                                                                                                   'Agree' : 2,
                                                                                                   'Disagree' : 1,
                                                                                                   'Strongly disagree' : 0,
                                                                                                   'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_ALCOHOL'] = neighbor_survey_sub['NEIGHBORHOOD_ALCOHOL'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_ALCOHOL'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### sidewalks

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['There are sidewalks on most of the streets in my neighborhood. Would you say that you...'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_SIDEWALK'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_SIDEWALK']]
neighbor_survey_sub['NEIGHBORHOOD_SIDEWALK'] = neighbor_survey_sub['NEIGHBORHOOD_SIDEWALK'].replace({'Strongly agree' : 0,
                                                                                                     'Somewhat agree' : 1,
                                                                                                     'Somewhat disagree' : 2,
                                                                                                     'Strongly disagree' : 3,
                                                                                                     'Does not apply to my neighborhood' : np.nan,
                                                                                                     'PMI: Skip' : np.nan,
                                                                                                     'PMI: Dont Know': np.nan})
neighbor_survey_sub['NEIGHBORHOOD_SIDEWALK'] = neighbor_survey_sub['NEIGHBORHOOD_SIDEWALK'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_SIDEWALK'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### people watch out for each other

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that in your neighborhood people watch out for each other?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_WATCH'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_WATCH']]
neighbor_survey_sub['NEIGHBORHOOD_WATCH'] = neighbor_survey_sub['NEIGHBORHOOD_WATCH'].replace({'Strongly agree' : 0,
                                                                                               'Agree' : 1,
                                                                                               'Disagree' : 2,
                                                                                               'Strongly disagree' : 3,
                                                                                               'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_WATCH'] = neighbor_survey_sub['NEIGHBORHOOD_WATCH'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_WATCH'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### abandoned buildings

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that there are lot of abandoned buildings in your neighborhood?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_BUILDING'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_BUILDING']]
neighbor_survey_sub['NEIGHBORHOOD_BUILDING'] = neighbor_survey_sub['NEIGHBORHOOD_BUILDING'].replace({'Strongly agree' : 3,
                                                                                                     'Agree' : 2,
                                                                                                     'Disagree' : 1,
                                                                                                     'Strongly disagree' : 0,
                                                                                                     'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_BUILDING'] = neighbor_survey_sub['NEIGHBORHOOD_BUILDING'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_BUILDING'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### get along

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that people in your neighborhood generally get along with each other?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_GET_ALONG'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_GET_ALONG']]
neighbor_survey_sub['NEIGHBORHOOD_GET_ALONG'] = neighbor_survey_sub['NEIGHBORHOOD_GET_ALONG'].replace({'Strongly agree' : 0,
                                                                                                       'Agree' : 1,
                                                                                                       'Neutral (neither agree nor disagree)' : 2,
                                                                                                       'Disagree' : 3,
                                                                                                       'Strongly disagree' : 4,
                                                                                                       'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_GET_ALONG'] = neighbor_survey_sub['NEIGHBORHOOD_GET_ALONG'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_GET_ALONG'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### unsafe to walk at night

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['The crime rate in my neighborhood makes it unsafe to go on walks at night. Would you say that you...'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_UNSAFE_WALK_NIGHT'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_UNSAFE_WALK_NIGHT']]
neighbor_survey_sub['NEIGHBORHOOD_UNSAFE_WALK_NIGHT'] = neighbor_survey_sub['NEIGHBORHOOD_UNSAFE_WALK_NIGHT'].replace({'Strongly agree' : 3,
                                                                                                                       'Somewhat agree' : 2,
                                                                                                                       'Somewhat disagree' : 1,
                                                                                                                       'Strongly disagree' : 0,
                                                                                                                       'PMI: Skip' : np.nan,
                                                                                                                       'PMI: Dont Know': np.nan})
neighbor_survey_sub['NEIGHBORHOOD_UNSAFE_WALK_NIGHT'] = neighbor_survey_sub['NEIGHBORHOOD_UNSAFE_WALK_NIGHT'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_UNSAFE_WALK_NIGHT'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### unsafe to walk during the day

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['The crime rate in my neighborhood makes it unsafe to go on walks during the day. Would you say that you...'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_UNSAFE_WALK_DAY'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_UNSAFE_WALK_DAY']]
neighbor_survey_sub['NEIGHBORHOOD_UNSAFE_WALK_DAY'] = neighbor_survey_sub['NEIGHBORHOOD_UNSAFE_WALK_DAY'].replace({'Strongly agree' : 3,
                                                                                                                   'Somewhat agree' : 2,
                                                                                                                   'Somewhat disagree' : 1,
                                                                                                                   'Strongly disagree' : 0,
                                                                                                                   'PMI: Skip' : np.nan,
                                                                                                                   'PMI: Dont Know': np.nan})
neighbor_survey_sub['NEIGHBORHOOD_UNSAFE_WALK_DAY'] = neighbor_survey_sub['NEIGHBORHOOD_UNSAFE_WALK_DAY'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_UNSAFE_WALK_DAY'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### take care of homes

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that people in your neighborhood take good care of their houses and apartments?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_CARE'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_CARE']]
neighbor_survey_sub['NEIGHBORHOOD_CARE'] = neighbor_survey_sub['NEIGHBORHOOD_CARE'].replace({'Strongly agree' : 0,
                                                                                             'Agree' : 1,
                                                                                             'Disagree' : 2,
                                                                                             'Strongly disagree' : 3,
                                                                                             'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_CARE'] = neighbor_survey_sub['NEIGHBORHOOD_CARE'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_CARE'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### bike facilities

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['There are facilities to bicycle in or near my neighborhood, such as special lanes, separate paths or trails, or shared use paths for cycles and pedestrians. Would you say that you...'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_BIKE'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_BIKE']]
neighbor_survey_sub['NEIGHBORHOOD_BIKE'] = neighbor_survey_sub['NEIGHBORHOOD_BIKE'].replace({'Strongly agree' : 0,
                                                                                             'Somewhat agree' : 1,
                                                                                             'Somewhat disagree' : 2,
                                                                                             'Strongly disagree' : 3,
                                                                                             'Does not apply to my neighborhood' : np.nan,
                                                                                             'PMI: Skip' : np.nan,
                                                                                             'PMI: Dont Know': np.nan})
neighbor_survey_sub['NEIGHBORHOOD_BIKE'] = neighbor_survey_sub['NEIGHBORHOOD_BIKE'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_BIKE'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### free recreational amenities

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['My neighborhood has several free or low-cost recreation facilities, such as parks, walking trails, bike paths, recreation centers, playgrounds, public swimming pools, etc. Would you say that you...'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_FREE_AMENITIES'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_FREE_AMENITIES']]
neighbor_survey_sub['NEIGHBORHOOD_FREE_AMENITIES'] = neighbor_survey_sub['NEIGHBORHOOD_FREE_AMENITIES'].replace({'Strongly agree' : 0,
                                                                                                                 'Somewhat agree' : 1,
                                                                                                                 'Somewhat disagree' : 2,
                                                                                                                 'Strongly disagree' : 3,
                                                                                                                 'PMI: Skip' : np.nan,
                                                                                                                 'PMI: Dont Know': np.nan})
neighbor_survey_sub['NEIGHBORHOOD_FREE_AMENITIES'] = neighbor_survey_sub['NEIGHBORHOOD_FREE_AMENITIES'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_FREE_AMENITIES'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### people hanging around streets

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that there are too many people hanging around on the streets near your home?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_PPL_HANGING_AROUND'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_PPL_HANGING_AROUND']]
neighbor_survey_sub['NEIGHBORHOOD_PPL_HANGING_AROUND'] = neighbor_survey_sub['NEIGHBORHOOD_PPL_HANGING_AROUND'].replace({'Strongly agree' : 3,
                                                                                                                         'Agree' : 2,
                                                                                                                         'Disagree' : 1,
                                                                                                                         'Strongly disagree' : 0,
                                                                                                                         'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_PPL_HANGING_AROUND'] = neighbor_survey_sub['NEIGHBORHOOD_PPL_HANGING_AROUND'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_PPL_HANGING_AROUND'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### trouble

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['How much you agree or disagree that you are always having trouble with your neighbors?'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_TROUBLE'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_TROUBLE']]
neighbor_survey_sub['NEIGHBORHOOD_TROUBLE'] = neighbor_survey_sub['NEIGHBORHOOD_TROUBLE'].replace({'Strongly agree' : 3,
                                                                                                   'Agree' : 2,
                                                                                                   'Disagree' : 1,
                                                                                                   'Strongly disagree' : 0,
                                                                                                   'PMI: Skip' : np.nan})
neighbor_survey_sub['NEIGHBORHOOD_TROUBLE'] = neighbor_survey_sub['NEIGHBORHOOD_TROUBLE'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_TROUBLE'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### stores

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['Many shops, stores, markets or other places to buy things I need are within easy walking distance of my home. Would you say that you...'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_STORES'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_STORES']]
neighbor_survey_sub['NEIGHBORHOOD_STORES'] = neighbor_survey_sub['NEIGHBORHOOD_STORES'].replace({'Strongly agree' : 0,
                                                                                                 'Somewhat agree' : 1,
                                                                                                 'Somewhat disagree' : 2,
                                                                                                 'Strongly disagree' : 3,
                                                                                                 'PMI: Skip' : np.nan,
                                                                                                 'PMI: Dont Know': np.nan})
neighbor_survey_sub['NEIGHBORHOOD_STORES'] = neighbor_survey_sub['NEIGHBORHOOD_STORES'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_STORES'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### transit

In [ ]:
neighbor_survey_sub = neighbor_survey[neighbor_survey['question'].isin(['It is within a 10-15 minute walk to a transit stop (such as bus, train, trolley, or tram) from my home. Would you say that you...'])]
neighbor_survey_sub['answer'].value_counts()

In [ ]:
neighbor_survey_sub.rename(columns = {'answer' : 'NEIGHBORHOOD_TRANSIT'}, inplace = True)
neighbor_survey_sub = neighbor_survey_sub[['person_id', 'NEIGHBORHOOD_TRANSIT']]
neighbor_survey_sub['NEIGHBORHOOD_TRANSIT'] = neighbor_survey_sub['NEIGHBORHOOD_TRANSIT'].replace({'Strongly agree' : 0,
                                                                                                   'Somewhat agree' : 1,
                                                                                                   'Somewhat disagree' : 2,
                                                                                                   'Strongly disagree' : 3,
                                                                                                   'PMI: Skip' : np.nan,
                                                                                                   'PMI: Dont Know': np.nan})
neighbor_survey_sub['NEIGHBORHOOD_TRANSIT'] = neighbor_survey_sub['NEIGHBORHOOD_TRANSIT'].astype('Int64')
print(neighbor_survey_sub['NEIGHBORHOOD_TRANSIT'].value_counts(dropna = False))
print(len(neighbor_survey_sub.index))
print(len(neighbor_survey_sub['person_id'].unique()))
neighbor_survey_sub.head()

In [ ]:
neighbor_survey_clean = neighbor_survey_clean.merge(neighbor_survey_sub, on = 'person_id', how = 'outer')
neighbor_survey_clean.head()

### merge

In [ ]:
pheno_neighbor = pheno_education.merge(neighbor_survey_clean, on = 'person_id', how = 'left')
print(len(pheno_neighbor.index))
pheno_neighbor.head()

### export in progress pheno

In [ ]:
pheno_neighbor.to_csv('CVD.in_progress_pheno.csv', index = None)

## physical activity

**encoding**

Overall Health: Everyday Activities
- 'Everyday Activities: Completely' : 0,
- 'Everyday Activities: Mostly' : 1,
- 'Everyday Activities: Moderately' : 2, 
- 'Everyday Activities: A Little' : 3, 
- 'Everyday Activities: Not At All' : 4,
- 'PMI: Skip' : np.nan

### make pheno date df

In [ ]:
pheno_date = pheno_neighbor[['person_id', 'CAD', 'CAD_DATE', 'AFIB', 'AFIB_DATE', 'HF', 'HF_DATE']]

### everyday physical activities

In [ ]:
physical_survey_sub = survey[survey['question'].isin(['Overall Health: Everyday Activities'])]
physical_survey_sub['answer'].value_counts()

In [ ]:
lab_input = physical_survey_sub.copy()
print(len(lab_input['person_id'].unique()))
col = 'PA_EVERYDAY'

date = lab_input.merge(pheno_date, on = 'person_id', how = 'inner')
date['CAD_DIFF'] = ((pd.to_datetime(date['CAD_DATE'], utc = True, format = 'ISO8601') - date['survey_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['AFIB_DIFF'] = ((pd.to_datetime(date['AFIB_DATE'], utc = True, format = 'ISO8601') - date['survey_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)
date['HF_DIFF'] = ((pd.to_datetime(date['HF_DATE'], utc = True, format = 'ISO8601') - date['survey_datetime']).astype(str).str.replace(r' days.*','', regex = True).replace('NaT', np.nan)).astype(float)

cad_first = date[date['CAD_DIFF'] >= 0].sort_values(by = ['person_id', 'CAD_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
afib_first = date[date['AFIB_DIFF'] >= 0].sort_values(by = ['person_id', 'AFIB_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')
hf_first = date[date['HF_DIFF'] >= 0].sort_values(by = ['person_id', 'HF_DIFF']).drop_duplicates(subset = ['person_id'], keep = 'first')

cad_first['answer'] = cad_first['answer'].replace({'Everyday Activities: Completely' : 0,
                                                   'Everyday Activities: Mostly' : 1,
                                                   'Everyday Activities: Moderately' : 2,
                                                   'Everyday Activities: A Little' : 3,
                                                   'Everyday Activities: Not At All' : 4,
                                                   'PMI: Skip' : np.nan})
afib_first['answer'] = afib_first['answer'].replace({'Everyday Activities: Completely' : 0,
                                                     'Everyday Activities: Mostly' : 1,
                                                     'Everyday Activities: Moderately' : 2,
                                                     'Everyday Activities: A Little' : 3,
                                                     'Everyday Activities: Not At All' : 4,
                                                     'PMI: Skip' : np.nan})
hf_first['answer'] = hf_first['answer'].replace({'Everyday Activities: Completely' : 0,
                                                 'Everyday Activities: Mostly' : 1,
                                                 'Everyday Activities: Moderately' : 2,
                                                 'Everyday Activities: A Little' : 3,
                                                 'Everyday Activities: Not At All' : 4,
                                                 'PMI: Skip' : np.nan})

cad_first = cad_first[['person_id', 'answer']].rename(columns = {'answer' : (col + '_CAD')})
print(len(cad_first['person_id'].unique()))
afib_first = afib_first[['person_id', 'answer']].rename(columns = {'answer' : (col + '_AFIB')})
print(len(afib_first['person_id'].unique()))
hf_first = hf_first[['person_id', 'answer']].rename(columns = {'answer' : (col + '_HF')})
print(len(hf_first['person_id'].unique()))

merge = cad_first.merge(afib_first, how = 'outer', on = 'person_id').merge(hf_first, how = 'outer', on = 'person_id')

physical_survey_sub = merge.copy()
physical_survey_sub.head()

In [ ]:
physical_survey_clean = physical_survey_sub
physical_survey_clean.head()

### merge

In [ ]:
pheno_pa = pheno_neighbor.merge(physical_survey_clean, on = 'person_id', how = 'left')
print(len(pheno_pa.index))
pheno_pa.head()

## alcohol

**encoding**

'Alcohol: Alcohol Participant'
- 'Alcohol Participant: Yes' : 1,
- 'Alcohol Participant: No' : 0
- 'PMI: Skip' : np.nan,
- 'PMI: Prefer Not To Answer' : np.nan

'Alcohol: Drink Frequency Past Year'
- 'Drink Frequency Past Year: Never' : 0,
- 'Drink Frequency Past Year: Monthly Or Less' : 1,
- 'Drink Frequency Past Year: 2 to 4 Per Month' : 2,
- 'Drink Frequency Past Year: 2 to 3 Per Week' : 3,
- 'Drink Frequency Past Year: 4 or More Per Week' : 4,
- 'PMI: Skip' : np.nan,
- 'PMI: Prefer Not To Answer' : np.nan

'Alcohol: Average Daily Drink Count'
- 'Average Daily Drink Count: 1 or 2' : 0,
- 'Average Daily Drink Count: 3 or 4' : 1,
- 'Average Daily Drink Count: 5 or 6' : 2,
- 'Average Daily Drink Count: 7 to 9' : 3,
- 'Average Daily Drink Count: 10 or More' : 4,
- 'PMI: Skip' : np.nan,
- 'PMI: Prefer Not To Answer' : np.nan

'Alcohol: 6 or More Drinks Occurrence'
- '6 or More Drinks Occurrence: Never In Last Year' : 0,
- '6 or More Drinks Occurrence: Less Than Monthly' : 1,
- '6 or More Drinks Occurrence: Monthly' : 2,
- '6 or More Drinks Occurrence: Weekly' : 3,
- '6 or More Drinks Occurrence: Daily' : 4,
- 'PMI: Skip' : np.nan,
- 'PMI: Prefer Not To Answer' : np.nan

### filter survey df

In [ ]:
alcohol_survey = survey[survey['question'].str.contains('Alcohol')]
print(alcohol_survey['question'].unique())

### alcohol participant

In [ ]:
alcohol_survey_sub = survey[survey['question'].isin(['Alcohol: Alcohol Participant'])]
alcohol_survey_sub['answer'].value_counts()

In [ ]:
alcohol_survey_sub.rename(columns = {'answer' : 'ALCOHOL_STATUS'}, inplace = True)
alcohol_survey_sub = alcohol_survey_sub[['person_id', 'ALCOHOL_STATUS']]
alcohol_survey_sub['ALCOHOL_STATUS'] = alcohol_survey_sub['ALCOHOL_STATUS'].replace({'Alcohol Participant: Yes' : 1,
                                                                                     'Alcohol Participant: No' : 0,
                                                                                     'PMI: Skip' : np.nan,
                                                                                     'PMI: Prefer Not To Answer' : np.nan})
alcohol_survey_sub['ALCOHOL_STATUS'] = alcohol_survey_sub['ALCOHOL_STATUS'].astype('Int64')
print(alcohol_survey_sub['ALCOHOL_STATUS'].value_counts(dropna = False))
print(len(alcohol_survey_sub.index))
print(len(alcohol_survey_sub['person_id'].unique()))
alcohol_survey_sub.head()

In [ ]:
alcohol_survey_clean = alcohol_survey_sub

### freq

In [ ]:
alcohol_survey_sub = survey[survey['question'].isin(['Alcohol: Drink Frequency Past Year'])]
alcohol_survey_sub['answer'].value_counts()

In [ ]:
alcohol_survey_sub.rename(columns = {'answer' : 'ALCOHOL_FREQ'}, inplace = True)
alcohol_survey_sub = alcohol_survey_sub[['person_id', 'ALCOHOL_FREQ']]
alcohol_survey_sub['ALCOHOL_FREQ'] = alcohol_survey_sub['ALCOHOL_FREQ'].replace({'Drink Frequency Past Year: Never' : 0,
                                                                                 'Drink Frequency Past Year: Monthly Or Less' : 1,
                                                                                 'Drink Frequency Past Year: 2 to 4 Per Month' : 2,
                                                                                 'Drink Frequency Past Year: 2 to 3 Per Week' : 3,
                                                                                 'Drink Frequency Past Year: 4 or More Per Week' : 4,
                                                                                 'PMI: Skip' : np.nan,
                                                                                 'PMI: Prefer Not To Answer' : np.nan})
alcohol_survey_sub['ALCOHOL_FREQ'] = alcohol_survey_sub['ALCOHOL_FREQ'].astype('Int64')
print(alcohol_survey_sub['ALCOHOL_FREQ'].value_counts(dropna = False))
print(len(alcohol_survey_sub.index))
print(len(alcohol_survey_sub['person_id'].unique()))
alcohol_survey_sub.head()

In [ ]:
alcohol_survey_clean = alcohol_survey_clean.merge(alcohol_survey_sub, on = 'person_id', how = 'outer')
alcohol_survey_clean.head()

### avg daily drink count

In [ ]:
alcohol_survey_sub = survey[survey['question'].isin(['Alcohol: Average Daily Drink Count'])]
alcohol_survey_sub['answer'].value_counts()

In [ ]:
alcohol_survey_sub.rename(columns = {'answer' : 'ALCOHOL_AVG_NUM_DAILY'}, inplace = True)
alcohol_survey_sub = alcohol_survey_sub[['person_id', 'ALCOHOL_AVG_NUM_DAILY']]
alcohol_survey_sub['ALCOHOL_AVG_NUM_DAILY'] = alcohol_survey_sub['ALCOHOL_AVG_NUM_DAILY'].replace({'Average Daily Drink Count: 1 or 2' : 0,
                                                                                                   'Average Daily Drink Count: 3 or 4' : 1,
                                                                                                   'Average Daily Drink Count: 5 or 6' : 2,
                                                                                                   'Average Daily Drink Count: 7 to 9' : 3,
                                                                                                   'Average Daily Drink Count: 10 or More' : 4,
                                                                                                   'PMI: Skip' : np.nan,
                                                                                                   'PMI: Prefer Not To Answer' : np.nan})
alcohol_survey_sub['ALCOHOL_AVG_NUM_DAILY'] = alcohol_survey_sub['ALCOHOL_AVG_NUM_DAILY'].astype('Int64')
print(alcohol_survey_sub['ALCOHOL_AVG_NUM_DAILY'].value_counts(dropna = False))
print(len(alcohol_survey_sub.index))
print(len(alcohol_survey_sub['person_id'].unique()))
alcohol_survey_sub.head()

In [ ]:
alcohol_survey_clean = alcohol_survey_clean.merge(alcohol_survey_sub, on = 'person_id', how = 'outer')
alcohol_survey_clean.head()

### 6 or more

In [ ]:
alcohol_survey_sub = survey[survey['question'].isin(['Alcohol: 6 or More Drinks Occurrence'])]
alcohol_survey_sub['answer'].value_counts()

In [ ]:
alcohol_survey_sub.rename(columns = {'answer' : 'ALCOHOL_SIX_MORE'}, inplace = True)
alcohol_survey_sub = alcohol_survey_sub[['person_id', 'ALCOHOL_SIX_MORE']]
alcohol_survey_sub['ALCOHOL_SIX_MORE'] = alcohol_survey_sub['ALCOHOL_SIX_MORE'].replace({'6 or More Drinks Occurrence: Never In Last Year' : 0,
                                                                                         '6 or More Drinks Occurrence: Less Than Monthly' : 1,
                                                                                         '6 or More Drinks Occurrence: Monthly' : 2,
                                                                                         '6 or More Drinks Occurrence: Weekly' : 3,
                                                                                         '6 or More Drinks Occurrence: Daily' : 4,
                                                                                         'PMI: Skip' : np.nan,
                                                                                         'PMI: Prefer Not To Answer' : np.nan})
alcohol_survey_sub['ALCOHOL_SIX_MORE'] = alcohol_survey_sub['ALCOHOL_SIX_MORE'].astype('Int64')
print(alcohol_survey_sub['ALCOHOL_SIX_MORE'].value_counts(dropna = False))
print(len(alcohol_survey_sub.index))
print(len(alcohol_survey_sub['person_id'].unique()))
alcohol_survey_sub.head()

In [ ]:
alcohol_survey_clean = alcohol_survey_clean.merge(alcohol_survey_sub, on = 'person_id', how = 'outer')
alcohol_survey_clean.head()

### merge

In [ ]:
pheno_alcohol = pheno_pa.merge(alcohol_survey_clean, on = 'person_id', how = 'left')
print(len(pheno_alcohol.index))
pheno_alcohol.head()

# clean AOU SDOH file
- this data is based on Census block

## check for duplicates

In [ ]:
print(len(ses.index))
print(len(ses['person_id'].unique()))

In [ ]:
ses.head()

## drop extra columns and rename

In [ ]:
ses_sub = ses.drop(columns = ['observation_datetime', 'zip_code', 'american_community_survey_year'])
ses_sub.columns = [c if c == 'person_id' else c.upper() for c in ses_sub.columns]
ses_sub.head()

## merge

In [ ]:
pheno_ses = pheno_alcohol.merge(ses_sub, how = 'left', on = 'person_id')
print(len(pheno_ses.index))
pheno_ses.head()

# clean PGS

## remove non-AOU IDs and convert column to numeric

In [ ]:
cad_afib_pgs_filt = cad_afib_pgs[~cad_afib_pgs['IID'].str.contains('NA|HG|SS')]
cad_afib_pgs_filt['IID'] = cad_afib_pgs_filt['IID'].astype('Int64')
print(len(cad_afib_pgs.index))
print(len(cad_afib_pgs_filt.index))

In [ ]:
hf_pgs_filt = hf_pgs[~hf_pgs['IID'].str.contains('NA|HG|SS')]
hf_pgs_filt['IID'] = hf_pgs_filt['IID'].astype('Int64')
print(len(hf_pgs.index))
print(len(hf_pgs_filt.index))

## split CAD and AFIB PGS, subset, and rename

In [ ]:
print(cad_afib_pgs['PGS'].unique())

In [ ]:
pgs_cad = cad_afib_pgs_filt[cad_afib_pgs_filt['PGS'].isin(['PGS005092_hmPOS_GRCh38'])].drop(columns = ['PGS']).rename(columns = {'IID' : 'person_id',
                                                                                                                                 'Z_norm2' : 'PGS_CAD'})
print(len(pgs_cad.index))
pgs_cad.head()

In [ ]:
pgs_afib = cad_afib_pgs_filt[cad_afib_pgs_filt['PGS'].isin(['PGS005313_hmPOS_GRCh38'])].drop(columns = ['PGS']).rename(columns = {'IID' : 'person_id',
                                                                                                                                  'Z_norm2' : 'PGS_AFIB'})
print(len(pgs_afib.index))
pgs_afib.head()

In [ ]:
pgs_hf = hf_pgs_filt.drop(columns = ['PGS']).rename(columns = {'IID' : 'person_id',
                                                               'Z_norm2' : 'PGS_HF'})

## merge and drop date columns

In [ ]:
pheno_pgs = pheno_ses.merge(pgs_cad, on = 'person_id', how = 'left').merge(
    pgs_afib, on = 'person_id', how = 'left').merge(
    pgs_hf, on = 'person_id', how = 'left').drop(
    columns = ['CAD_DATE', 'AFIB_DATE', 'HF_DATE'])
print(len(pheno_pgs.dropna(subset = ['PGS_CAD']).index))
print(len(pheno_pgs.dropna(subset = ['PGS_AFIB']).index))
print(len(pheno_pgs.dropna(subset = ['PGS_HF']).index))
print(len(pheno_pgs.index))

In [ ]:
pheno_pgs.columns

## export and import final pheno

In [ ]:
pheno_pgs.to_csv('CVD.IRM.phenotype.csv', index = None)

In [ ]:
pheno_pgs = pd.read_csv('CVD.IRM.phenotype.csv')

# build CRS

**AFIB CRS 10 year risk**

Reference: FHS

Predictors:
- Age
- Sex
- BMI
- SBP
- HT treatment
- PR interval
- Significant murmur
- Prevalent HF

**ASCVD (for CAD) CRS 10 year risk**

Reference: AHA PREVENT

Required Predictors:
- Sex
- Age
- Statins
- Smoking
- HDL
- BMI
- Total cholesterol
- SBP
- eGFR
- diabetes
- HT med

Optional predictors:
- uACR
- HbA1c
- SDI

## AFIB

### remove missing

In [ ]:
print(len(pheno_pgs.index))
print(pheno_pgs['AFIB'].value_counts(dropna = False))
afib_crs_input = pheno_pgs.dropna(subset = ['person_id',
                                            'AFIB',
                                            'CAD_AFIB',
                                            'COPD_AFIB',
                                            'AGE_AFIB',
                                            'SYS_HF_AFIB',
                                            'HT_AFIB',
                                            'HYPERTHYROIDISM_AFIB'])
print(len(afib_crs_input.index))
print(afib_crs_input['AFIB'].value_counts(dropna = False))

### create function

In [ ]:
def c2hest(df, cad_col, copd_col, ht_col, age_col, sys_df_col, hyperthyroidism_col):
    # initialize score
    score = pd.Series(0, index = df.index)

    # Add 1 point for CAD
    score += df[cad_col].astype(int)

    # Add 1 point for COPD
    score += df[copd_col].astype(int)

    # Add 1 point for Hypertension
    score += df[ht_col].astype(int)

    # Add 2 points if age ≥ 75
    score += (df[age_col] >= 75).astype(int) * 2

    # Add 2 points if systolic heart failure
    score += df[sys_df_col].astype(int) * 2

    # Add 1 point for hyperthyroidism
    score += df[hyperthyroidism_col].astype(int)

    return score

### apply function to AFIB

In [ ]:
print(afib_crs_input.shape)
afib_crs_input["C2HEST_score"] = c2hest(afib_crs_input,
                                        'CAD_AFIB',
                                        'COPD_AFIB',
                                        'HT_AFIB',
                                        'AGE_AFIB',
                                        'SYS_HF_AFIB',
                                        'HYPERTHYROIDISM_AFIB')
print(afib_crs_input.shape)
print(afib_crs_input["C2HEST_score"].value_counts(dropna = False))

In [ ]:
print(afib_crs_input.groupby('AFIB')["C2HEST_score"].value_counts(dropna = False))

### subset and rename

In [ ]:
afib_crs_input_sub = afib_crs_input[['person_id', 'C2HEST_score']]
afib_crs_input_sub = afib_crs_input_sub.rename(columns = {'C2HEST_score' : 'AFIB_C2HEST_score'})

### merge

In [ ]:
pheno_crs = pheno_pgs.merge(afib_crs_input_sub, on = 'person_id', how = 'left')
print(len(pheno_crs.index))

## add prevent CRS- computed in an R notebook

### subset and rename

In [ ]:
cad_prevent_crs_sub = cad_prevent_crs.drop(columns = ['CAD', 'age', 'over_years'])
cad_prevent_crs_sub = cad_prevent_crs_sub.add_prefix('CAD_PREVENT_CRS_')
cad_prevent_crs_sub = cad_prevent_crs_sub.rename(columns = {'CAD_PREVENT_CRS_person_id' : 'person_id'})
print(len(cad_prevent_crs_sub.index))
cad_prevent_crs_sub.head()

In [ ]:
afib_prevent_crs_sub = afib_prevent_crs.drop(columns = ['AFIB', 'age', 'over_years'])
afib_prevent_crs_sub = afib_prevent_crs_sub.add_prefix('AFIB_PREVENT_CRS_')
afib_prevent_crs_sub = afib_prevent_crs_sub.rename(columns = {'AFIB_PREVENT_CRS_person_id' : 'person_id'})
print(len(afib_prevent_crs_sub.index))
afib_prevent_crs_sub.head()

In [ ]:
hf_prevent_crs_sub = hf_prevent_crs.drop(columns = ['HF', 'age', 'over_years'])
hf_prevent_crs_sub = hf_prevent_crs_sub.add_prefix('HF_PREVENT_CRS_')
hf_prevent_crs_sub = hf_prevent_crs_sub.rename(columns = {'HF_PREVENT_CRS_person_id' : 'person_id'})
print(len(hf_prevent_crs_sub.index))
hf_prevent_crs_sub.head()

### merge

In [ ]:
pheno_crs = pheno_crs.merge(cad_prevent_crs_sub, how = 'left', on = 'person_id').merge(
    afib_prevent_crs_sub, how = 'left', on = 'person_id').merge(
    hf_prevent_crs_sub, how = 'left', on = 'person_id')
print(pheno_crs.shape)
pheno_crs.head()

## export and import in progress pheno

In [ ]:
pheno_crs.to_csv('CVD.IRM.CRS.phenotype.csv', index = None)

In [ ]:
pheno_crs = pd.read_csv('CVD.IRM.CRS.phenotype.csv')

# normalize PXS variables

## define inverse normal transformation function

In [ ]:
def inverse_normal_transform(x):
    ranks = x.rank(method = 'average', na_option = 'keep')
    n = ranks.notna().sum()
    transformed = norm.ppf((ranks - 0.5) / n)
    return transformed

## define scalers

In [ ]:
pxs_scaler =  MinMaxScaler(feature_range = (0, 1))

## physical activity

In [ ]:
col = 'PA_EVERYDAY'

cad_col = col + '_CAD'
afib_col = col + '_AFIB'
hf_col = col + '_HF'

print(pheno_crs.shape)
df = pheno_crs[['person_id', cad_col, afib_col, hf_col]]

cad_no_missing = df.dropna(subset = cad_col)
cad_missing = df[~df.index.isin(cad_no_missing.index)]
cad_missing[(cad_col + '_SCALE')] = np.nan

afib_no_missing = df.dropna(subset = afib_col)
afib_missing = df[~df.index.isin(afib_no_missing.index)]
afib_missing[(afib_col + '_SCALE')] = np.nan

hf_no_missing = df.dropna(subset = hf_col)
hf_missing = df[~df.index.isin(hf_no_missing.index)]
hf_missing[(hf_col + '_SCALE')] = np.nan

cad_no_missing[(cad_col + '_SCALE')] = pxs_scaler.fit_transform(cad_no_missing[[cad_col]])
afib_no_missing[(afib_col + '_SCALE')] = pxs_scaler.fit_transform(afib_no_missing[[afib_col]])
hf_no_missing[(hf_col + '_SCALE')] = pxs_scaler.fit_transform(hf_no_missing[[hf_col]])

cad = pd.concat([cad_no_missing, cad_missing], axis = 0)
afib = pd.concat([afib_no_missing, afib_missing], axis = 0)
hf = pd.concat([hf_no_missing, hf_missing], axis = 0)

cad_sub = cad[['person_id', (cad_col + '_SCALE')]]
afib_sub = afib[['person_id', (afib_col + '_SCALE')]]
hf_sub = hf[['person_id', (hf_col + '_SCALE')]]

new_df = cad_sub.merge(afib_sub, on = 'person_id', how = 'inner').merge(hf_sub, on = 'person_id', how = 'inner')

pheno_crs = pheno_crs.merge(new_df, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

## neighborhood

In [ ]:
col = 'NEIGHBORHOOD_DRUG_USE'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_SAFE'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_TRUST'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_BUILDING'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_ALCOHOL'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_VANDALISM'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_SIDEWALK'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_BIKE'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_CLEAN'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_WATCH'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_HOUSING'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_GET_ALONG'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_UNSAFE_WALK_DAY'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_UNSAFE_WALK_NIGHT'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_CARE'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_ALOT_CRIME'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_SAME_VALUES'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_GRAFFITI'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_NOISE'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_FREE_AMENITIES'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_PPL_HANGING_AROUND'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_TROUBLE'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_STORES'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NEIGHBORHOOD_TRANSIT'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

## income

In [ ]:
col = 'INCOME_ANNUAL'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

## highest education

In [ ]:
col = 'EDUCATION_HIGHEST'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

## alcohol

In [ ]:
col = 'ALCOHOL_FREQ'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'ALCOHOL_AVG_NUM_DAILY'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'ALCOHOL_SIX_MORE'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

## SES variables

In [ ]:
col = 'MEDIAN_INCOME'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
missing[(col + '_INV_NORMAL_SCALE')] = np.nan

no_missing[(col + '_INV_NORMAL_SCALE')] = inverse_normal_transform(no_missing[col]) * -1
no_missing[(col + '_INV_NORMAL_SCALE')] = pxs_scaler.fit_transform(no_missing[[(col + '_INV_NORMAL_SCALE')]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_INV_NORMAL_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'ASSISTED_INCOME'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
missing[(col + '_INV_NORMAL_SCALE')] = np.nan

no_missing[(col + '_INV_NORMAL_SCALE')] = inverse_normal_transform(no_missing[col]) * -1
no_missing[(col + '_INV_NORMAL_SCALE')] = pxs_scaler.fit_transform(no_missing[[(col + '_INV_NORMAL_SCALE')]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_INV_NORMAL_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'HIGH_SCHOOL_EDUCATION'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
missing[(col + '_INV_NORMAL_SCALE')] = np.nan

no_missing[(col + '_INV_NORMAL_SCALE')] = inverse_normal_transform(no_missing[col]) * -1
no_missing[(col + '_INV_NORMAL_SCALE')] = pxs_scaler.fit_transform(no_missing[[(col + '_INV_NORMAL_SCALE')]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_INV_NORMAL_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'NO_HEALTH_INSURANCE'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
missing[(col + '_INV_NORMAL_SCALE')] = np.nan

no_missing[(col + '_INV_NORMAL_SCALE')] = inverse_normal_transform(no_missing[col]) * -1
no_missing[(col + '_INV_NORMAL_SCALE')] = pxs_scaler.fit_transform(no_missing[[(col + '_INV_NORMAL_SCALE')]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_INV_NORMAL_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'POVERTY'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
missing[(col + '_INV_NORMAL_SCALE')] = np.nan

no_missing[(col + '_INV_NORMAL_SCALE')] = inverse_normal_transform(no_missing[col]) * -1
no_missing[(col + '_INV_NORMAL_SCALE')] = pxs_scaler.fit_transform(no_missing[[(col + '_INV_NORMAL_SCALE')]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_INV_NORMAL_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'VACANT_HOUSING'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
missing[(col + '_INV_NORMAL_SCALE')] = np.nan

no_missing[(col + '_INV_NORMAL_SCALE')] = inverse_normal_transform(no_missing[col]) * -1
no_missing[(col + '_INV_NORMAL_SCALE')] = pxs_scaler.fit_transform(no_missing[[(col + '_INV_NORMAL_SCALE')]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_INV_NORMAL_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'DEPRIVATION_INDEX'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
missing[(col + '_INV_NORMAL_SCALE')] = np.nan

no_missing[(col + '_INV_NORMAL_SCALE')] = inverse_normal_transform(no_missing[col]) * -1
no_missing[(col + '_INV_NORMAL_SCALE')] = pxs_scaler.fit_transform(no_missing[[(col + '_INV_NORMAL_SCALE')]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_INV_NORMAL_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

## smoking

In [ ]:
col = 'SMOKING_CIG_YEARS'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
missing[(col + '_INV_NORMAL_SCALE')] = np.nan

no_missing[(col + '_INV_NORMAL_SCALE')] = inverse_normal_transform(no_missing[col]) * -1
no_missing[(col + '_INV_NORMAL_SCALE')] = pxs_scaler.fit_transform(no_missing[[(col + '_INV_NORMAL_SCALE')]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_INV_NORMAL_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'SMOKING_CIG_FREQ'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'SMOKING_CIG_NUM_DAILY_NOW'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
missing[(col + '_INV_NORMAL_SCALE')] = np.nan

no_missing[(col + '_INV_NORMAL_SCALE')] = inverse_normal_transform(no_missing[col]) * -1
no_missing[(col + '_INV_NORMAL_SCALE')] = pxs_scaler.fit_transform(no_missing[[(col + '_INV_NORMAL_SCALE')]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_INV_NORMAL_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'SMOKING_CIG_NUM_DAILY_LIFETIME'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
missing[(col + '_INV_NORMAL_SCALE')] = np.nan

no_missing[(col + '_INV_NORMAL_SCALE')] = inverse_normal_transform(no_missing[col]) * -1
no_missing[(col + '_INV_NORMAL_SCALE')] = pxs_scaler.fit_transform(no_missing[[(col + '_INV_NORMAL_SCALE')]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_INV_NORMAL_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'SMOKING_CIGAR_FREQ'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'SMOKING_ELECTRONIC_FREQ'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'SMOKING_HOOKAH_FREQ'

print(pheno_crs.shape)
df = pheno_crs[['person_id', col]]

no_missing = df.dropna(subset = col)
missing = df[~df.index.isin(no_missing.index)]
cad_missing[(col + '_SCALE')] = np.nan

no_missing[(col + '_SCALE')] = pxs_scaler.fit_transform(no_missing[[col]])

cat = pd.concat([no_missing, missing], axis = 0)

cat_sub = cat[['person_id', (col + '_SCALE')]]

pheno_crs = pheno_crs.merge(cat_sub, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

In [ ]:
col = 'SMOKING_CIG_STATUS'

cad_col = col + '_CAD'
afib_col = col + '_AFIB'
hf_col = col + '_HF'

print(pheno_crs.shape)
df = pheno_crs[['person_id', cad_col, afib_col, hf_col]]

cad_no_missing = df.dropna(subset = cad_col)
cad_missing = df[~df.index.isin(cad_no_missing.index)]
cad_missing[(cad_col + '_SCALE')] = np.nan

afib_no_missing = df.dropna(subset = afib_col)
afib_missing = df[~df.index.isin(afib_no_missing.index)]
afib_missing[(afib_col + '_SCALE')] = np.nan

hf_no_missing = df.dropna(subset = hf_col)
hf_missing = df[~df.index.isin(hf_no_missing.index)]
hf_missing[(hf_col + '_SCALE')] = np.nan

cad_no_missing[(cad_col + '_SCALE')] = pxs_scaler.fit_transform(cad_no_missing[[cad_col]])
afib_no_missing[(afib_col + '_SCALE')] = pxs_scaler.fit_transform(afib_no_missing[[afib_col]])
hf_no_missing[(hf_col + '_SCALE')] = pxs_scaler.fit_transform(hf_no_missing[[hf_col]])

cad = pd.concat([cad_no_missing, cad_missing], axis = 0)
afib = pd.concat([afib_no_missing, afib_missing], axis = 0)
hf = pd.concat([hf_no_missing, hf_missing], axis = 0)

cad_sub = cad[['person_id', (cad_col + '_SCALE')]]
afib_sub = afib[['person_id', (afib_col + '_SCALE')]]
hf_sub = hf[['person_id', (hf_col + '_SCALE')]]

new_df = cad_sub.merge(afib_sub, on = 'person_id', how = 'inner').merge(hf_sub, on = 'person_id', how = 'inner')

pheno_crs = pheno_crs.merge(new_df, on = 'person_id', how = 'inner')
print(pheno_crs.shape)

## rename smoking column

In [ ]:
pheno_transform = pheno_crs.rename(columns = {'SMOKING_CIGS_100_LIFETIME' : 'SMOKING_CIG_100_LIFETIME'})

## export pheno

In [ ]:
pheno_transform.to_csv('CVD.IRM.CRS.phenotype.variable_transformation.csv', index = None)

## get summary numbers

In [ ]:
print(len(pheno_transform.index))
print(len(pheno_transform.dropna(subset = ['CAD']).index))
print(len(pheno_transform.dropna(subset = ['AFIB']).index))
print(len(pheno_transform.dropna(subset = ['HF']).index))

# subset and remove missing

## select columns

In [ ]:
irm_cols = pheno_transform.columns[pheno_transform.columns.str.contains('person_id|PGS|CRS|score|SCALE')].tolist()
irm_cols = [c for c in irm_cols if 'ntile' not in c]
irm_cols = [c for c in irm_cols if 'model' not in c]
irm_cols.append('SMOKING_HOOKAH_STATUS')
irm_cols.append('SMOKING_ELECTRONIC_STATUS')
irm_cols.append('SMOKING_CIGAR_STATUS')
irm_cols.append('SMOKING_CIG_100_LIFETIME')
irm_cols.append('ALCOHOL_STATUS')
irm_cols.append('SEX')
irm_cols.append('CAD')
irm_cols.append('AFIB')
irm_cols.append('HF')
irm_cols.append('AGE_CAD')
irm_cols.append('AGE_AFIB')
irm_cols.append('AGE_HF')
print(len(irm_cols))
irm_cols

## subset entire df

In [ ]:
pheno_irm = pheno_transform[irm_cols]
pheno_irm.head()

## split by phenotype

### make col lists

In [ ]:
cad_cols = pheno_irm.columns[~pheno_irm.columns.str.contains('AFIB|HF')].tolist()
print(len(cad_cols))

In [ ]:
afib_cols = pheno_irm.columns[~pheno_irm.columns.str.contains('CAD|HF')].tolist()
print(len(afib_cols))

In [ ]:
hf_cols = pheno_irm.columns[~pheno_irm.columns.str.contains('CAD|AFIB')].tolist()
print(len(hf_cols))
hf_cols

### subset

In [ ]:
cad_irm = pheno_irm[cad_cols]
cad_irm.head()

In [ ]:
afib_irm = pheno_irm[afib_cols]
afib_irm.head()

In [ ]:
hf_irm = pheno_irm[hf_cols]
hf_irm.head()

## removing missing and people younger than 18

In [ ]:
pgs_crs_cols = cad_irm.columns[cad_irm.columns.str.contains('PGS|CRS')].tolist()
pgs_crs_cols.append('person_id')
pgs_crs_cols.append('CAD')

cad_irm_no_missing = cad_irm.dropna(subset = pgs_crs_cols)
cad_irm_no_missing = cad_irm_no_missing[cad_irm_no_missing['AGE_CAD'] >= 18]
print(len(cad_irm.index))
print(len(cad_irm_no_missing.index))
cad_irm_no_missing['CAD'].value_counts(dropna = False)

In [ ]:
crs_cols = afib_irm.columns[afib_irm.columns.str.contains('score')].tolist()
print(crs_cols)

afib_irm_no_missing = afib_irm.dropna(subset = ['person_id', 'PGS_AFIB'])
afib_irm_no_missing = afib_irm_no_missing.dropna(subset = crs_cols, how = 'all')
afib_irm_no_missing = afib_irm_no_missing[afib_irm_no_missing['AGE_AFIB'] >= 18]
print(len(afib_irm.index))
print(len(afib_irm_no_missing.index))
afib_irm_no_missing['AFIB'].value_counts(dropna = False)

In [ ]:
pgs_crs_cols = hf_irm.columns[hf_irm.columns.str.contains('PGS|CRS')].tolist()
pgs_crs_cols.append('person_id')
pgs_crs_cols.append('HF')

hf_irm_no_missing = hf_irm.dropna(subset = pgs_crs_cols)
hf_irm_no_missing = hf_irm_no_missing[hf_irm_no_missing['AGE_HF'] >= 18]
print(len(hf_irm.index))
print(len(hf_irm_no_missing.index))
hf_irm_no_missing['HF'].value_counts(dropna = False)

## export

In [ ]:
cad_irm_no_missing.to_csv('CAD.IRM.PREVENT_CRS.no_missing.eval_input.csv', index = None)

In [ ]:
afib_irm_no_missing.to_csv('AFIB.IRM.C2HEST_PREVENT_CRS.no_missing.eval_input.csv', index = None)

In [ ]:
hf_irm_no_missing.to_csv('HF.IRM.PREVENT_CRS.no_missing.eval_input.csv', index = None)

## select individuals who are not in any of these groups
- by phenotype

In [ ]:
print(len(pheno_ntile.index))
cad_feature_selection = pheno_ntile[~pheno_ntile['person_id'].isin(cad_irm_no_missing['person_id'])]
print(len(cad_feature_selection.index))
print(cad_feature_selection['CAD'].value_counts(dropna = False))

In [ ]:
print(len(pheno_ntile.index))
afib_feature_selection = pheno_ntile[~pheno_ntile['person_id'].isin(afib_irm_no_missing['person_id'])]
print(len(afib_feature_selection.index))
print(afib_feature_selection['AFIB'].value_counts(dropna = False))

In [ ]:
print(len(pheno_ntile.index))
hf_feature_selection = pheno_ntile[~pheno_ntile['person_id'].isin(hf_irm_no_missing['person_id'])]
print(len(hf_feature_selection.index))
print(hf_feature_selection['HF'].value_counts(dropna = False))

## drop columns not needed for feature selection

### select cols

In [ ]:
feature_selection_cols = pheno_ntile.columns[pheno_ntile.columns.str.contains('person_id|SCALE')].tolist()
feature_selection_cols.append('SMOKING_HOOKAH_STATUS')
feature_selection_cols.append('SMOKING_ELECTRONIC_STATUS')
feature_selection_cols.append('SMOKING_CIGAR_STATUS')
feature_selection_cols.append('SMOKING_CIG_100_LIFETIME')
feature_selection_cols.append('ALCOHOL_STATUS')
feature_selection_cols.append('CAD')
feature_selection_cols.append('AFIB')
feature_selection_cols.append('HF')
feature_selection_cols.append('AGE_CAD')
feature_selection_cols.append('AGE_AFIB')
feature_selection_cols.append('AGE_HF')
feature_selection_cols.append('SEX')
print(len(feature_selection_cols))
feature_selection_cols

### get number of columns per variable

In [ ]:
smoking_cols = [col for col in feature_selection_cols if 'SMOKING_' in col]
print(len(smoking_cols))

In [ ]:
alcohol_cols = [col for col in feature_selection_cols if 'ALCOHOL_' in col]
print(len(alcohol_cols))

In [ ]:
neighborhood_cols = [col for col in feature_selection_cols if 'NEIGHBORHOOD_' in col]
print(len(neighborhood_cols))

### subset

In [ ]:
cad_feature_selection_sub = cad_feature_selection[feature_selection_cols]
print(cad_feature_selection_sub.shape)

In [ ]:
afib_feature_selection_sub = afib_feature_selection[feature_selection_cols]
print(afib_feature_selection_sub.shape)

In [ ]:
hf_feature_selection_sub = hf_feature_selection[feature_selection_cols]
print(hf_feature_selection_sub.shape)

### remove phenotype specific cols

In [ ]:
cad_feature_selection_sub = cad_feature_selection_sub.drop(columns = ['AFIB',
                                                                      'HF',
                                                                      'AGE_AFIB',
                                                                      'AGE_HF',
                                                                      'PA_EVERYDAY_AFIB_SCALE',
                                                                      'PA_EVERYDAY_HF_SCALE',
                                                                      'SMOKING_CIG_STATUS_AFIB_SCALE',
                                                                      'SMOKING_CIG_STATUS_HF_SCALE'])
print(len(cad_feature_selection_sub.columns))

In [ ]:
afib_feature_selection_sub = afib_feature_selection_sub.drop(columns = ['CAD',
                                                                        'HF',
                                                                        'AGE_CAD',
                                                                        'AGE_HF',
                                                                        'PA_EVERYDAY_CAD_SCALE',
                                                                        'PA_EVERYDAY_HF_SCALE',
                                                                        'SMOKING_CIG_STATUS_CAD_SCALE',
                                                                        'SMOKING_CIG_STATUS_HF_SCALE'])
print(len(afib_feature_selection_sub.columns))

In [ ]:
hf_feature_selection_sub = hf_feature_selection_sub.drop(columns = ['CAD',
                                                                    'AFIB',
                                                                    'AGE_CAD',
                                                                    'AGE_AFIB',
                                                                    'PA_EVERYDAY_CAD_SCALE',
                                                                    'PA_EVERYDAY_AFIB_SCALE',
                                                                    'SMOKING_CIG_STATUS_CAD_SCALE',
                                                                    'SMOKING_CIG_STATUS_AFIB_SCALE'])
print(len(hf_feature_selection_sub.columns))

## remove missing and people younger than 18

In [ ]:
cad_feature_selection_no_missing = cad_feature_selection_sub.dropna(subset = ['CAD', 'AGE_CAD'])
cad_feature_selection_no_missing = cad_feature_selection_no_missing[cad_feature_selection_no_missing['AGE_CAD'] >= 18]
print(len(cad_feature_selection_no_missing.index))
print(cad_feature_selection_no_missing['CAD'].value_counts(dropna = False))

In [ ]:
afib_feature_selection_no_missing = afib_feature_selection_sub.dropna(subset = ['AFIB', 'AGE_AFIB'])
afib_feature_selection_no_missing = afib_feature_selection_no_missing[afib_feature_selection_no_missing['AGE_AFIB'] >= 18]
print(len(afib_feature_selection_no_missing.index))
print(afib_feature_selection_no_missing['AFIB'].value_counts(dropna = False))

In [ ]:
hf_feature_selection_no_missing = hf_feature_selection_sub.dropna(subset = ['HF', 'AGE_HF'])
hf_feature_selection_no_missing = hf_feature_selection_no_missing[hf_feature_selection_no_missing['AGE_HF'] >= 18]
print(len(hf_feature_selection_no_missing.index))
print(hf_feature_selection_no_missing['HF'].value_counts(dropna = False))

# compare feature selection and IRM groups

### define inv normal transformation function

In [ ]:
def inverse_normal_transform(x):
    ranks = x.rank(method = 'average', na_option = 'keep')
    n = ranks.notna().sum()
    transformed = norm.ppf((ranks - 0.5) / n)
    return transformed

### cad

In [ ]:
cad_sex_feature_selection_summary = pd.DataFrame(cad_feature_selection_no_missing["SEX"].value_counts(dropna = False, normalize = True)).transpose()
cad_case_feature_selection_summary = pd.DataFrame(cad_feature_selection_no_missing['CAD'].value_counts(dropna = False, normalize = True)).transpose()
cad_age_feature_selection_summary = pd.DataFrame(cad_feature_selection_no_missing['AGE_CAD'].describe()).transpose()
cad_inv_norm_age_feature_selection_summary = pd.DataFrame(inverse_normal_transform(cad_feature_selection_no_missing['AGE_CAD'].describe())).transpose()

In [ ]:
cad_sex_irm_summary = pd.DataFrame(cad_irm_no_missing["SEX"].value_counts(dropna = False, normalize = True)).transpose()
cad_case_irm_summary = pd.DataFrame(cad_irm_no_missing['CAD'].value_counts(dropna = False, normalize = True)).transpose()
cad_age_irm_summary = pd.DataFrame(cad_irm_no_missing['AGE_CAD'].describe()).transpose()
cad_inv_norm_age_irm_summary = pd.DataFrame(inverse_normal_transform(cad_irm_no_missing['AGE_CAD'].describe())).transpose()

In [ ]:
cad_sex_summary = pd.concat([cad_sex_feature_selection_summary, cad_sex_irm_summary], axis = 0)
cad_sex_summary = cad_sex_summary.rename(columns = {2 : 'Female', 1 : 'Male'})
cad_sex_summary.insert(0, 'Group', ['Feature Selection', 'IRM Computation'])
cad_sex_summary.to_csv('AOU.CAD.SEX.csv', index = None)
cad_sex_summary

In [ ]:
cad_case_summary = pd.concat([cad_case_feature_selection_summary, cad_case_irm_summary], axis = 0)
cad_case_summary = cad_case_summary.rename(columns = {0 : 'Control', 1 : 'Case'})
cad_case_summary.insert(0, 'Group', ['Feature Selection', 'IRM Computation'])
cad_case_summary.to_csv('AOU.CAD.CASE.csv', index = None)
cad_case_summary

In [ ]:
cad_inv_norm_age_feature_selection_summary.columns = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
cad_inv_norm_age_irm_summary.columns = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
cad_age_summary = pd.concat([cad_age_feature_selection_summary,
                             cad_age_irm_summary,
                             cad_inv_norm_age_feature_selection_summary,
                             cad_inv_norm_age_irm_summary], axis = 0)
cad_age_summary.insert(0, 'Age', ['Raw Age', 'Raw Age', 'Inverse Normally Transformed Age', 'Inverse Normally Transformed Age'])
cad_age_summary.insert(0, 'Group', ['Feature Selection', 'IRM Computation', 'Feature Selection', 'IRM Computation'])
cad_age_summary = cad_age_summary.drop(columns = ['count'])
cad_age_summary.to_csv('AOU.CAD.AGE.csv', index = None)
cad_age_summary

In [ ]:
sns.color_palette("colorblind")

In [ ]:
print(sns.color_palette("colorblind").as_hex())

In [ ]:
palette = ['#029e73', '#56b4e9']

df1_plot_raw = cad_feature_selection_no_missing[['AGE_CAD']].assign(group = 'Feature Selection')
df2_plot_raw = cad_irm_no_missing[['AGE_CAD']].assign(group = 'IRM Computation')
age_df_raw = pd.concat([df1_plot_raw, df2_plot_raw], axis = 0)

# Prepare the second dataset (inverse normal transformed)
df1_plot_inv = cad_feature_selection_no_missing[['AGE_CAD']].assign(group = 'Feature Selection')
df2_plot_inv = cad_irm_no_missing[['AGE_CAD']].assign(group = 'IRM Computation')

df1_plot_inv['AGE_CAD'] = inverse_normal_transform(df1_plot_inv['AGE_CAD'])
df2_plot_inv['AGE_CAD'] = inverse_normal_transform(df2_plot_inv['AGE_CAD'])
age_df_inv = pd.concat([df1_plot_inv, df2_plot_inv], axis = 0)

# Create subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot raw age distribution
sns.kdeplot(data = age_df_raw, x = 'AGE_CAD', hue = 'group', fill = True, alpha = 0.4, ax = axes[0], legend = False, palette = palette)
axes[0].set_title("Raw Age")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Density")

# Plot inverse-normal-transformed age distribution
sns.kdeplot(data = age_df_inv, x = 'AGE_CAD', hue = 'group', fill = True, alpha = 0.4, ax = axes[1], legend = False, palette = palette)
axes[1].set_title("Inverse Normally Transformed Age")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Density")

handles = [mpatches.Patch(color = palette[0], label = "Feature Selection"),
           mpatches.Patch(color = palette[1], label = "IRM Computation")]
axes[0].legend(handles = handles, title = 'Group')
axes[1].legend(handles = handles, title = 'Group')

axes[0].text(-0.1, 1.1, 'A', transform = axes[0].transAxes, 
            fontsize = 20, fontweight = 'bold', va = 'top', ha = 'right')

axes[1].text(-0.1, 1.1, 'B', transform = axes[1].transAxes, 
            fontsize = 20, fontweight = 'bold', va = 'top', ha = 'right')


plot_table = cad_age_summary.rename(columns = {'mean' : 'Mean', 'std' : 'Stdev', 'min' : 'Min', 'max' : 'Max'})
plot_table = plot_table.round(2)

cell_colors = []

header_fill = '#d9d9d9'
row_fill_1 = '#ffffff'
row_fill_2 = '#f2f2f2'

for i in range(len(plot_table)):
    row_color = row_fill_1 if i % 2 == 0 else row_fill_2
    cell_colors.append([row_color] * len(plot_table.columns))
    
the_table = axes[0].table(cellText = plot_table.values,
                          colLabels = plot_table.columns,
                          loc = 'bottom',
                          cellLoc = 'center',
                          colColours = [header_fill] * len(plot_table.columns),
                          cellColours = cell_colors,
                          bbox = [0.0, -0.6, 2, 0.4]) 

the_table.auto_set_font_size(False)
the_table.set_fontsize(11)
for (row, col), cell in the_table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight = 'bold')
        
col_widths = {
    0 : 0.2,
    1 : 0.3,
    2 : 0.1,
    3 : 0.1,
    4 : 0.1,
    5 : 0.1,
    6 : 0.1,
    7 : 0.1,
    8 : 0.1
}

for col, width in col_widths.items():
    for row in range(len(plot_table) + 1):
        the_table[row, col].set_width(width)

plt.subplots_adjust(bottom = 0.3)

fig.suptitle("AOU CAD Age Distribution", fontsize = 16, fontweight = 'bold')

plt.savefig('AOU.CAD.AGE.png', dpi = 600, bbox_inches = 'tight')
plt.show()

In [ ]:
palette = ['#029e73', '#56b4e9']

df1_counts = cad_feature_selection_no_missing['SEX'].value_counts(normalize = True) * 100
df2_counts = cad_irm_no_missing['SEX'].value_counts(normalize = True) * 100

stacked = pd.DataFrame({'Feature Selection': df1_counts,
                        'IRM Computation': df2_counts}).fillna(0)

fig, ax = plt.subplots(figsize = (14, 6))

stacked.T.plot(kind = 'bar', stacked = True, color = palette, rot = 0, ax = ax, width = 0.6)

plot_table = cad_sex_summary.copy()
plot_table['Female'] = plot_table['Female'] * 100
plot_table['Male'] = plot_table['Male'] * 100
plot_table = plot_table.round(2)

cell_colors = []

header_fill = '#d9d9d9'
row_fill_1 = '#ffffff'
row_fill_2 = '#f2f2f2'

for i in range(len(plot_table)):
    row_color = row_fill_1 if i % 2 == 0 else row_fill_2
    cell_colors.append([row_color] * len(plot_table.columns))

the_table = plt.table(cellText = plot_table.values,
                      colLabels = plot_table.columns,
                      loc = 'right',
                      cellLoc = 'center',
                      bbox = [1.25, 0.15, 0.7, 0.6],
                      colColours = [header_fill] * len(plot_table.columns),
                      cellColours = cell_colors)

for (row, col), cell in the_table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight = 'bold')

the_table.auto_set_font_size(False)
the_table.set_fontsize(11)
the_table.scale(1.5, 1.8)

plt.subplots_adjust(right = 0.55)
ax.set_ylabel("Percentage (%)", fontsize = 12)
ax.set_xlabel("Subset", fontsize = 12)

plt.legend(labels = ['Female', 'Male'], title = "Group")

fig.suptitle("AOU CAD Sex Distribution", fontsize = 16, fontweight = 'bold')

plt.savefig('AOU.CAD.SEX.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
palette = ['#029e73', '#56b4e9']

df1_counts = cad_feature_selection_no_missing['CAD'].value_counts(normalize = True) * 100
df2_counts = cad_irm_no_missing['CAD'].value_counts(normalize = True) * 100

stacked = pd.DataFrame({'Feature Selection': df1_counts,
                        'IRM Computation': df2_counts}).fillna(0)

fig, ax = plt.subplots(figsize = (14, 6))

stacked.T.plot(kind = 'bar', stacked = True, color = palette, rot = 0, ax = ax, width = 0.6)

plot_table = cad_case_summary.copy()
plot_table['Control'] = plot_table['Control'] * 100
plot_table['Case'] = plot_table['Case'] * 100
plot_table = plot_table.round(2)

cell_colors = []

header_fill = '#d9d9d9'
row_fill_1 = '#ffffff'
row_fill_2 = '#f2f2f2'

for i in range(len(plot_table)):
    row_color = row_fill_1 if i % 2 == 0 else row_fill_2
    cell_colors.append([row_color] * len(plot_table.columns))

the_table = plt.table(cellText = plot_table.values,
                      colLabels = plot_table.columns,
                      loc = 'right',
                      cellLoc = 'center',
                      bbox = [1.25, 0.15, 0.7, 0.6],
                      colColours = [header_fill] * len(plot_table.columns),
                      cellColours = cell_colors)

for (row, col), cell in the_table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight = 'bold')

the_table.auto_set_font_size(False)
the_table.set_fontsize(11)
the_table.scale(1.5, 1.8)

plt.subplots_adjust(right = 0.55)
ax.set_ylabel("Percentage (%)", fontsize = 12)
ax.set_xlabel("Subset", fontsize = 12)

plt.legend(labels = ['Control', 'Case'], title = "Group")

fig.suptitle("AOU CAD Case/Control Distribution", fontsize = 16, fontweight = 'bold')

plt.savefig('AOU.CAD.CASE.png', dpi = 300, bbox_inches = 'tight')
plt.show()

### afib

In [ ]:
afib_sex_feature_selection_summary = pd.DataFrame(afib_feature_selection_no_missing["SEX"].value_counts(dropna = False, normalize = True)).transpose()
afib_case_feature_selection_summary = pd.DataFrame(afib_feature_selection_no_missing['AFIB'].value_counts(dropna = False, normalize = True)).transpose()
afib_age_feature_selection_summary = pd.DataFrame(afib_feature_selection_no_missing['AGE_AFIB'].describe()).transpose()
afib_inv_norm_age_feature_selection_summary = pd.DataFrame(inverse_normal_transform(afib_feature_selection_no_missing['AGE_AFIB'].describe())).transpose()

In [ ]:
afib_sex_irm_summary = pd.DataFrame(afib_irm_no_missing["SEX"].value_counts(dropna = False, normalize = True)).transpose()
afib_case_irm_summary = pd.DataFrame(afib_irm_no_missing['AFIB'].value_counts(dropna = False, normalize = True)).transpose()
afib_age_irm_summary = pd.DataFrame(afib_irm_no_missing['AGE_AFIB'].describe()).transpose()
afib_inv_norm_age_irm_summary = pd.DataFrame(inverse_normal_transform(afib_irm_no_missing['AGE_AFIB'].describe())).transpose()

In [ ]:
afib_sex_summary = pd.concat([afib_sex_feature_selection_summary, afib_sex_irm_summary], axis = 0)
afib_sex_summary = afib_sex_summary.rename(columns = {2 : 'Female', 1 : 'Male'})
afib_sex_summary.insert(0, 'Group', ['Feature Selection', 'IRM Computation'])
afib_sex_summary.to_csv('AOU.AFIB.SEX.csv', index = None)
afib_sex_summary

In [ ]:
afib_case_summary = pd.concat([afib_case_feature_selection_summary, afib_case_irm_summary], axis = 0)
afib_case_summary = afib_case_summary.rename(columns = {0 : 'Control', 1 : 'Case'})
afib_case_summary.insert(0, 'Group', ['Feature Selection', 'IRM Computation'])
afib_case_summary.to_csv('AOU.AFIB.CASE.csv', index = None)
afib_case_summary

In [ ]:
afib_inv_norm_age_feature_selection_summary.columns = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
afib_inv_norm_age_irm_summary.columns = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
afib_age_summary = pd.concat([afib_age_feature_selection_summary,
                             afib_age_irm_summary,
                             afib_inv_norm_age_feature_selection_summary,
                             afib_inv_norm_age_irm_summary], axis = 0)
afib_age_summary.insert(0, 'Age', ['Raw Age', 'Raw Age', 'Inverse Normally Transformed Age', 'Inverse Normally Transformed Age'])
afib_age_summary.insert(0, 'Group', ['Feature Selection', 'IRM Computation', 'Feature Selection', 'IRM Computation'])
afib_age_summary = afib_age_summary.drop(columns = ['count'])
afib_age_summary.to_csv('AOU.AFIB.AGE.csv', index = None)
afib_age_summary

In [ ]:
palette = ['#029e73', '#56b4e9']

df1_plot_raw = afib_feature_selection_no_missing[['AGE_AFIB']].assign(group = 'Feature Selection')
df2_plot_raw = afib_irm_no_missing[['AGE_AFIB']].assign(group = 'IRM Computation')
age_df_raw = pd.concat([df1_plot_raw, df2_plot_raw], axis = 0)

df1_plot_inv = afib_feature_selection_no_missing[['AGE_AFIB']].assign(group = 'Feature Selection')
df2_plot_inv = afib_irm_no_missing[['AGE_AFIB']].assign(group = 'IRM Computation')

df1_plot_inv['AGE_AFIB'] = inverse_normal_transform(df1_plot_inv['AGE_AFIB'])
df2_plot_inv['AGE_AFIB'] = inverse_normal_transform(df2_plot_inv['AGE_AFIB'])
age_df_inv = pd.concat([df1_plot_inv, df2_plot_inv], axis = 0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot raw age distribution
sns.kdeplot(data = age_df_raw, x = 'AGE_AFIB', hue = 'group', fill = True, alpha = 0.4, ax = axes[0], legend = False, palette = palette)
axes[0].set_title("Raw Age")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Density")

sns.kdeplot(data = age_df_inv, x = 'AGE_AFIB', hue = 'group', fill = True, alpha = 0.4, ax = axes[1], legend = False, palette = palette)
axes[1].set_title("Inverse Normally Transformed Age")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Density")

handles = [mpatches.Patch(color = palette[0], label = "Feature Selection"),
           mpatches.Patch(color = palette[1], label = "IRM Computation")]
axes[0].legend(handles = handles, title = 'Group')
axes[1].legend(handles = handles, title = 'Group')

axes[0].text(-0.1, 1.1, 'A', transform = axes[0].transAxes, 
            fontsize = 20, fontweight = 'bold', va = 'top', ha = 'right')

axes[1].text(-0.1, 1.1, 'B', transform = axes[1].transAxes, 
            fontsize = 20, fontweight = 'bold', va = 'top', ha = 'right')


plot_table = afib_age_summary.rename(columns = {'mean' : 'Mean', 'std' : 'Stdev', 'min' : 'Min', 'max' : 'Max'})
plot_table = plot_table.round(2)

cell_colors = []

header_fill = '#d9d9d9'
row_fill_1 = '#ffffff'
row_fill_2 = '#f2f2f2'

for i in range(len(plot_table)):
    row_color = row_fill_1 if i % 2 == 0 else row_fill_2
    cell_colors.append([row_color] * len(plot_table.columns))
    
the_table = axes[0].table(cellText = plot_table.values,
                          colLabels = plot_table.columns,
                          loc = 'bottom',
                          cellLoc = 'center',
                          colColours = [header_fill] * len(plot_table.columns),
                          cellColours = cell_colors,
                          bbox = [0.0, -0.6, 2, 0.4]) 

the_table.auto_set_font_size(False)
the_table.set_fontsize(11)
for (row, col), cell in the_table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight = 'bold')
        
col_widths = {
    0 : 0.2,
    1 : 0.3,
    2 : 0.1,
    3 : 0.1,
    4 : 0.1,
    5 : 0.1,
    6 : 0.1,
    7 : 0.1,
    8 : 0.1
}

for col, width in col_widths.items():
    for row in range(len(plot_table) + 1):
        the_table[row, col].set_width(width)

plt.subplots_adjust(bottom = 0.3)

fig.suptitle("AOU AFIB Age Distribution", fontsize = 16, fontweight = 'bold')

plt.savefig('AOU.AFIB.AGE.png', dpi = 600, bbox_inches = 'tight')
plt.show()

In [ ]:
palette = ['#029e73', '#56b4e9']

df1_counts = afib_feature_selection_no_missing['SEX'].value_counts(normalize = True) * 100
df2_counts = afib_irm_no_missing['SEX'].value_counts(normalize = True) * 100

stacked = pd.DataFrame({'Feature Selection': df1_counts,
                        'IRM Computation': df2_counts}).fillna(0)

fig, ax = plt.subplots(figsize = (14, 6))

stacked.T.plot(kind = 'bar', stacked = True, color = palette, rot = 0, ax = ax, width = 0.6)

plot_table = afib_sex_summary.copy()
plot_table['Female'] = plot_table['Female'] * 100
plot_table['Male'] = plot_table['Male'] * 100
plot_table = plot_table.round(2)

cell_colors = []

header_fill = '#d9d9d9'
row_fill_1 = '#ffffff'
row_fill_2 = '#f2f2f2'

for i in range(len(plot_table)):
    row_color = row_fill_1 if i % 2 == 0 else row_fill_2
    cell_colors.append([row_color] * len(plot_table.columns))

the_table = plt.table(cellText = plot_table.values,
                      colLabels = plot_table.columns,
                      loc = 'right',
                      cellLoc = 'center',
                      bbox = [1.25, 0.15, 0.7, 0.6],
                      colColours = [header_fill] * len(plot_table.columns),
                      cellColours = cell_colors)

for (row, col), cell in the_table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight = 'bold')

the_table.auto_set_font_size(False)
the_table.set_fontsize(11)
the_table.scale(1.5, 1.8)

plt.subplots_adjust(right = 0.55)
ax.set_ylabel("Percentage (%)", fontsize = 12)
ax.set_xlabel("Subset", fontsize = 12)

plt.legend(labels = ['Female', 'Male'], title = "Group")

fig.suptitle("AOU AFIB Sex Distribution", fontsize = 16, fontweight = 'bold')

plt.savefig('AOU.AFIB.SEX.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
palette = ['#029e73', '#56b4e9']

df1_counts = afib_feature_selection_no_missing['AFIB'].value_counts(normalize = True) * 100
df2_counts = afib_irm_no_missing['AFIB'].value_counts(normalize = True) * 100

stacked = pd.DataFrame({'Feature Selection': df1_counts,
                        'IRM Computation': df2_counts}).fillna(0)

fig, ax = plt.subplots(figsize = (14, 6))

stacked.T.plot(kind = 'bar', stacked = True, color = palette, rot = 0, ax = ax, width = 0.6)

plot_table = afib_case_summary.copy()
plot_table['Control'] = plot_table['Control'] * 100
plot_table['Case'] = plot_table['Case'] * 100
plot_table = plot_table.round(2)

cell_colors = []

header_fill = '#d9d9d9'
row_fill_1 = '#ffffff'
row_fill_2 = '#f2f2f2'

for i in range(len(plot_table)):
    row_color = row_fill_1 if i % 2 == 0 else row_fill_2
    cell_colors.append([row_color] * len(plot_table.columns))

the_table = plt.table(cellText = plot_table.values,
                      colLabels = plot_table.columns,
                      loc = 'right',
                      cellLoc = 'center',
                      bbox = [1.25, 0.15, 0.7, 0.6],
                      colColours = [header_fill] * len(plot_table.columns),
                      cellColours = cell_colors)

for (row, col), cell in the_table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight = 'bold')

the_table.auto_set_font_size(False)
the_table.set_fontsize(11)
the_table.scale(1.5, 1.8)

plt.subplots_adjust(right = 0.55)
ax.set_ylabel("Percentage (%)", fontsize = 12)
ax.set_xlabel("Subset", fontsize = 12)

plt.legend(labels = ['Control', 'Case'], title = "Group")

fig.suptitle("AOU AFIB Case/Control Distribution", fontsize = 16, fontweight = 'bold')

plt.savefig('AOU.AFIB.CASE.png', dpi = 300, bbox_inches = 'tight')
plt.show()

### HF

In [ ]:
hf_sex_feature_selection_summary = pd.DataFrame(hf_feature_selection_no_missing["SEX"].value_counts(dropna = False, normalize = True)).transpose()
hf_case_feature_selection_summary = pd.DataFrame(hf_feature_selection_no_missing['HF'].value_counts(dropna = False, normalize = True)).transpose()
hf_age_feature_selection_summary = pd.DataFrame(hf_feature_selection_no_missing['AGE_HF'].describe()).transpose()
hf_inv_norm_age_feature_selection_summary = pd.DataFrame(inverse_normal_transform(hf_feature_selection_no_missing['AGE_HF'].describe())).transpose()

In [ ]:
hf_sex_irm_summary = pd.DataFrame(hf_irm_no_missing["SEX"].value_counts(dropna = False, normalize = True)).transpose()
hf_case_irm_summary = pd.DataFrame(hf_irm_no_missing['HF'].value_counts(dropna = False, normalize = True)).transpose()
hf_age_irm_summary = pd.DataFrame(hf_irm_no_missing['AGE_HF'].describe()).transpose()
hf_inv_norm_age_irm_summary = pd.DataFrame(inverse_normal_transform(hf_irm_no_missing['AGE_HF'].describe())).transpose()

In [ ]:
hf_sex_summary = pd.concat([hf_sex_feature_selection_summary, hf_sex_irm_summary], axis = 0)
hf_sex_summary = hf_sex_summary.rename(columns = {2 : 'Female', 1 : 'Male'})
hf_sex_summary.insert(0, 'Group', ['Feature Selection', 'IRM Computation'])
hf_sex_summary.to_csv('AOU.HF.SEX.csv', index = None)
hf_sex_summary

In [ ]:
hf_case_summary = pd.concat([hf_case_feature_selection_summary, hf_case_irm_summary], axis = 0)
hf_case_summary = hf_case_summary.rename(columns = {0 : 'Control', 1 : 'Case'})
hf_case_summary.insert(0, 'Group', ['Feature Selection', 'IRM Computation'])
hf_case_summary.to_csv('AOU.HF.CASE.csv', index = None)
hf_case_summary

In [ ]:
hf_inv_norm_age_feature_selection_summary.columns = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
hf_inv_norm_age_irm_summary.columns = ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
hf_age_summary = pd.concat([hf_age_feature_selection_summary,
                             hf_age_irm_summary,
                             hf_inv_norm_age_feature_selection_summary,
                             hf_inv_norm_age_irm_summary], axis = 0)
hf_age_summary.insert(0, 'Age', ['Raw Age', 'Raw Age', 'Inverse Normally Transformed Age', 'Inverse Normally Transformed Age'])
hf_age_summary.insert(0, 'Group', ['Feature Selection', 'IRM Computation', 'Feature Selection', 'IRM Computation'])
hf_age_summary = hf_age_summary.drop(columns = ['count'])
hf_age_summary.to_csv('AOU.HF.AGE.csv', index = None)
hf_age_summary

In [ ]:
palette = ['#029e73', '#56b4e9']

df1_plot_raw = hf_feature_selection_no_missing[['AGE_HF']].assign(group = 'Feature Selection')
df2_plot_raw = hf_irm_no_missing[['AGE_HF']].assign(group = 'IRM Computation')
age_df_raw = pd.concat([df1_plot_raw, df2_plot_raw], axis = 0)

# Prepare the second dataset (inverse normal transformed)
df1_plot_inv = hf_feature_selection_no_missing[['AGE_HF']].assign(group = 'Feature Selection')
df2_plot_inv = hf_irm_no_missing[['AGE_HF']].assign(group = 'IRM Computation')

df1_plot_inv['AGE_HF'] = inverse_normal_transform(df1_plot_inv['AGE_HF'])
df2_plot_inv['AGE_HF'] = inverse_normal_transform(df2_plot_inv['AGE_HF'])
age_df_inv = pd.concat([df1_plot_inv, df2_plot_inv], axis = 0)

# Create subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot raw age distribution
sns.kdeplot(data = age_df_raw, x = 'AGE_HF', hue = 'group', fill = True, alpha = 0.4, ax = axes[0], legend = False, palette = palette)
axes[0].set_title("Raw Age")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Density")

# Plot inverse-normal-transformed age distribution
sns.kdeplot(data = age_df_inv, x = 'AGE_HF', hue = 'group', fill = True, alpha = 0.4, ax = axes[1], legend = False, palette = palette)
axes[1].set_title("Inverse Normally Transformed Age")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Density")

handles = [mpatches.Patch(color = palette[0], label = "Feature Selection"),
           mpatches.Patch(color = palette[1], label = "IRM Computation")]
axes[0].legend(handles = handles, title = 'Group')
axes[1].legend(handles = handles, title = 'Group')

axes[0].text(-0.1, 1.1, 'A', transform = axes[0].transAxes, 
            fontsize = 20, fontweight = 'bold', va = 'top', ha = 'right')

axes[1].text(-0.1, 1.1, 'B', transform = axes[1].transAxes, 
            fontsize = 20, fontweight = 'bold', va = 'top', ha = 'right')


plot_table = hf_age_summary.rename(columns = {'mean' : 'Mean', 'std' : 'Stdev', 'min' : 'Min', 'max' : 'Max'})
plot_table = plot_table.round(2)

cell_colors = []

header_fill = '#d9d9d9'
row_fill_1 = '#ffffff'
row_fill_2 = '#f2f2f2'

for i in range(len(plot_table)):
    row_color = row_fill_1 if i % 2 == 0 else row_fill_2
    cell_colors.append([row_color] * len(plot_table.columns))
    
the_table = axes[0].table(cellText = plot_table.values,
                          colLabels = plot_table.columns,
                          loc = 'bottom',
                          cellLoc = 'center',
                          colColours = [header_fill] * len(plot_table.columns),
                          cellColours = cell_colors,
                          bbox = [0.0, -0.6, 2, 0.4]) 

the_table.auto_set_font_size(False)
the_table.set_fontsize(11)
for (row, col), cell in the_table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight = 'bold')
        
col_widths = {
    0 : 0.2,
    1 : 0.3,
    2 : 0.1,
    3 : 0.1,
    4 : 0.1,
    5 : 0.1,
    6 : 0.1,
    7 : 0.1,
    8 : 0.1
}

for col, width in col_widths.items():
    for row in range(len(plot_table) + 1):
        the_table[row, col].set_width(width)

plt.subplots_adjust(bottom = 0.3)

fig.suptitle("AOU HF Age Distribution", fontsize = 16, fontweight = 'bold')

plt.savefig('AOU.HF.AGE.png', dpi = 600, bbox_inches = 'tight')
plt.show()

In [ ]:
palette = ['#029e73', '#56b4e9']

df1_counts = hf_feature_selection_no_missing['SEX'].value_counts(normalize = True) * 100
df2_counts = hf_irm_no_missing['SEX'].value_counts(normalize = True) * 100

stacked = pd.DataFrame({'Feature Selection': df1_counts,
                        'IRM Computation': df2_counts}).fillna(0)

fig, ax = plt.subplots(figsize = (14, 6))

stacked.T.plot(kind = 'bar', stacked = True, color = palette, rot = 0, ax = ax, width = 0.6)

plot_table = hf_sex_summary.copy()
plot_table['Female'] = plot_table['Female'] * 100
plot_table['Male'] = plot_table['Male'] * 100
plot_table = plot_table.round(2)

cell_colors = []

header_fill = '#d9d9d9'
row_fill_1 = '#ffffff'
row_fill_2 = '#f2f2f2'

for i in range(len(plot_table)):
    row_color = row_fill_1 if i % 2 == 0 else row_fill_2
    cell_colors.append([row_color] * len(hf_sex_summary.columns))

the_table = plt.table(cellText = plot_table.values,
                      colLabels = plot_table.columns,
                      loc = 'right',
                      cellLoc = 'center',
                      bbox = [1.25, 0.15, 0.7, 0.6],
                      colColours = [header_fill] * len(plot_table.columns),
                      cellColours = cell_colors)

for (row, col), cell in the_table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight = 'bold')

the_table.auto_set_font_size(False)
the_table.set_fontsize(11)
the_table.scale(1.5, 1.8)

plt.subplots_adjust(right = 0.55)
ax.set_ylabel("Percentage (%)", fontsize = 12)
ax.set_xlabel("Subset", fontsize = 12)

plt.legend(labels = ['Female', 'Male'], title = "Group")

fig.suptitle("AOU HF Sex Distribution", fontsize = 16, fontweight = 'bold')

plt.savefig('AOU.HF.SEX.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
palette = ['#029e73', '#56b4e9']

df1_counts = hf_feature_selection_no_missing['HF'].value_counts(normalize = True) * 100
df2_counts = hf_irm_no_missing['HF'].value_counts(normalize = True) * 100

stacked = pd.DataFrame({'Feature Selection': df1_counts,
                        'IRM Computation': df2_counts}).fillna(0)

fig, ax = plt.subplots(figsize = (14, 6))

stacked.T.plot(kind = 'bar', stacked = True, color = palette, rot = 0, ax = ax, width = 0.6)

plot_table = hf_case_summary.copy()
plot_table['Control'] = plot_table['Control'] * 100
plot_table['Case'] = plot_table['Case'] * 100
plot_table = plot_table.round(2)

cell_colors = []

header_fill = '#d9d9d9'
row_fill_1 = '#ffffff'
row_fill_2 = '#f2f2f2'

for i in range(len(plot_table)):
    row_color = row_fill_1 if i % 2 == 0 else row_fill_2
    cell_colors.append([row_color] * len(plot_table.columns))

the_table = plt.table(cellText = plot_table.values,
                      colLabels = plot_table.columns,
                      loc = 'right',
                      cellLoc = 'center',
                      bbox = [1.25, 0.15, 0.7, 0.6],
                      colColours = [header_fill] * len(plot_table.columns),
                      cellColours = cell_colors)

for (row, col), cell in the_table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight = 'bold')

the_table.auto_set_font_size(False)
the_table.set_fontsize(11)
the_table.scale(1.5, 1.8)

plt.subplots_adjust(right = 0.55)
ax.set_ylabel("Percentage (%)", fontsize = 12)
ax.set_xlabel("Subset", fontsize = 12)

plt.legend(labels = ['Control', 'Case'], title = "Group")

fig.suptitle("AOU HF Case/Control Distribution", fontsize = 16, fontweight = 'bold')

plt.savefig('AOU.HF.CASE.png', dpi = 300, bbox_inches = 'tight')
plt.show()

## apply inv normal transformation of age

In [ ]:
cad_irm_no_missing['AGE_CAD'] = inverse_normal_transform(cad_irm_no_missing['AGE_CAD'])
cad_feature_selection_no_missing['AGE_CAD'] = inverse_normal_transform(cad_feature_selection_no_missing['AGE_CAD'])

In [ ]:
afib_irm_no_missing['AGE_AFIB'] = inverse_normal_transform(afib_irm_no_missing['AGE_AFIB'])
afib_feature_selection_no_missing['AGE_AFIB'] = inverse_normal_transform(afib_feature_selection_no_missing['AGE_AFIB'])

In [ ]:
hf_irm_no_missing['AGE_HF'] = inverse_normal_transform(hf_irm_no_missing['AGE_HF'])
hf_feature_selection_no_missing['AGE_HF'] = inverse_normal_transform(hf_feature_selection_no_missing['AGE_HF'])

## export

In [ ]:
cad_feature_selection_no_missing.to_csv('CAD.IRM.PXS_feature_selection_input.csv', index = None)

In [ ]:
afib_feature_selection_no_missing.to_csv('AFIB.IRM.PXS_feature_selection_input.csv', index = None)

In [ ]:
hf_feature_selection_no_missing.to_csv('HF.IRM.PXS_feature_selection_input.csv', index = None)

In [ ]:
cad_irm_no_missing.to_csv('CAD.IRM.PREVENT_CRS.no_missing.eval_input.csv', index = None)

In [ ]:
afib_irm_no_missing.to_csv('AFIB.IRM.C2HEST_PREVENT_CRS.no_missing.eval_input.csv', index = None)

In [ ]:
hf_irm_no_missing.to_csv('HF.IRM.PREVENT_CRS.no_missing.eval_input.csv', index = None)

# case proportions by risk groups

## read in input files

### eval input

In [ ]:
cad_irm_no_missing = pd.read_csv('CAD.IRM.PREVENT_CRS.no_missing.eval_input.csv', low_memory = False)

In [ ]:
afib_irm_no_missing = pd.read_csv('AFIB.IRM.C2HEST_PREVENT_CRS.no_missing.eval_input.csv', low_memory = False)

In [ ]:
hf_irm_no_missing = pd.read_csv('HF.IRM.PREVENT_CRS.no_missing.eval_input.csv', low_memory = False)

### weights

In [ ]:
cad_shap_selected = pd.read_csv('CAD.catboost.shap.selected_features.txt', sep = '\t', index_col = 'feature')

In [ ]:
afib_shap_selected = pd.read_csv('AFIB.catboost.shap.selected_features.txt', sep = '\t', index_col = 'feature')

In [ ]:
hf_shap_selected = pd.read_csv('HF.catboost.shap.selected_features.txt', sep = '\t', index_col = 'feature')

## calculate PXS

In [ ]:
pxs_cols = cad_shap_selected.index.tolist()
pxs_cols.remove(('AGE_CAD'))
pxs_cols.remove('SEX')

cad_irm_no_missing['PXS_AVG'] = cad_irm_no_missing[pxs_cols].mean(axis = 1, skipna = True)

weights = cad_shap_selected['WEIGHT']
w = pd.Series(0, index = cad_irm_no_missing.columns, dtype = float)
w.loc[pxs_cols] = weights.loc[pxs_cols]
weighted_sum = (cad_irm_no_missing * w).sum(axis = 1)
effective_weight_sum = (cad_irm_no_missing.notna() * w).sum(axis = 1)
weighted_avg = weighted_sum / effective_weight_sum
cad_irm_no_missing['PXS_WEIGHTED_AVG'] = weighted_avg

In [ ]:
pxs_cols = afib_shap_selected.index.tolist()
pxs_cols.remove(('AGE_AFIB'))
pxs_cols.remove('SEX')

afib_irm_no_missing['PXS_AVG'] = afib_irm_no_missing[pxs_cols].mean(axis = 1, skipna = True)

weights = afib_shap_selected['WEIGHT']
w = pd.Series(0, index = afib_irm_no_missing.columns, dtype = float)
w.loc[pxs_cols] = weights.loc[pxs_cols]
weighted_sum = (afib_irm_no_missing * w).sum(axis = 1)
effective_weight_sum = (afib_irm_no_missing.notna() * w).sum(axis = 1)
weighted_avg = weighted_sum / effective_weight_sum
afib_irm_no_missing['PXS_WEIGHTED_AVG'] = weighted_avg

In [ ]:
pxs_cols = hf_shap_selected.index.tolist()
pxs_cols.remove(('AGE_HF'))
pxs_cols.remove('SEX')

hf_irm_no_missing['PXS_AVG'] = hf_irm_no_missing[pxs_cols].mean(axis = 1, skipna = True)

weights = hf_shap_selected['WEIGHT']
w = pd.Series(0, index = hf_irm_no_missing.columns, dtype = float)
w.loc[pxs_cols] = weights.loc[pxs_cols]
weighted_sum = (hf_irm_no_missing * w).sum(axis = 1)
effective_weight_sum = (hf_irm_no_missing.notna() * w).sum(axis = 1)
weighted_avg = weighted_sum / effective_weight_sum
hf_irm_no_missing['PXS_WEIGHTED_AVG'] = weighted_avg

## define inv normal transformation function

In [ ]:
def inverse_normal_transform(x):
    ranks = x.rank(method = 'average', na_option = 'keep')
    n = ranks.notna().sum()
    transformed = norm.ppf((ranks - 0.5) / n)
    return transformed

## compute PGS and PXS ntile

In [ ]:
cad_irm_no_missing['PGS_CAD_ntile'] = 100 * norm.cdf(cad_irm_no_missing['PGS_CAD'])
cad_irm_no_missing['PXS_AVG_ntile'] = 100 * norm.cdf(inverse_normal_transform(cad_irm_no_missing['PXS_AVG']))
cad_irm_no_missing['PXS_WEIGHTED_AVG_ntile'] = 100 * norm.cdf(inverse_normal_transform(cad_irm_no_missing['PXS_WEIGHTED_AVG']))

In [ ]:
afib_irm_no_missing['PGS_AFIB_ntile'] = 100 * norm.cdf(afib_irm_no_missing['PGS_AFIB'])
afib_irm_no_missing['PXS_AVG_ntile'] = 100 * norm.cdf(inverse_normal_transform(afib_irm_no_missing['PXS_AVG']))
afib_irm_no_missing['PXS_WEIGHTED_AVG_ntile'] = 100 * norm.cdf(inverse_normal_transform(afib_irm_no_missing['PXS_WEIGHTED_AVG']))

In [ ]:
hf_irm_no_missing['PGS_HF_ntile'] = 100 * norm.cdf(hf_irm_no_missing['PGS_HF'])
hf_irm_no_missing['PXS_AVG_ntile'] = 100 * norm.cdf(inverse_normal_transform(hf_irm_no_missing['PXS_AVG']))
hf_irm_no_missing['PXS_WEIGHTED_AVG_ntile'] = 100 * norm.cdf(inverse_normal_transform(hf_irm_no_missing['PXS_WEIGHTED_AVG']))

## bin individuals by risk threshold

PRS and PXS thresholds
- 0-40%
- 40-80%
- 80-90%
- 90-100%

CRS thresholds
- 0-5%
- 5-7.5 %
- 7.5 - 20%
- \>= 20%

### cad

In [ ]:
df = cad_irm_no_missing.copy()
pheno = 'CAD'
crs = 'CAD_PREVENT_CRS_chd'

# pgs
## 0-10
pgs_0_10 = df[df['PGS_' + pheno + '_ntile'] < 10]
pgs_0_10_case = pgs_0_10[pgs_0_10[pheno] == 1]
percent_pgs_0_10_case = (len(pgs_0_10_case.index) / len(pgs_0_10.index)) * 100

## 10-90
pgs_10_90 = df[(df['PGS_' + pheno + '_ntile'] >= 10) & (df['PGS_' + pheno + '_ntile'] < 90)]
pgs_10_90_case = pgs_10_90[pgs_10_90[pheno] == 1]
percent_pgs_10_90_case = (len(pgs_10_90_case.index) / len(pgs_10_90.index)) * 100
## 90-100
pgs_90_100 = df[(df['PGS_' + pheno + '_ntile'] >= 90)]
pgs_90_100_case = pgs_90_100[pgs_90_100[pheno] == 1]
percent_pgs_90_100_case = (len(pgs_90_100_case.index) / len(pgs_90_100.index)) * 100

# pxs
## 0-10
pxs_0_10 = df[df['PXS_AVG_ntile'] < 10]
pxs_0_10_case = pxs_0_10[pxs_0_10[pheno] == 1]
percent_pxs_0_10_case = (len(pxs_0_10_case.index) / len(pxs_0_10.index)) * 100
## 10-90
pxs_10_90 = df[(df['PXS_AVG_ntile'] >= 10) & (df['PXS_AVG_ntile'] < 90)]
pxs_10_90_case = pxs_10_90[pxs_10_90[pheno] == 1]
percent_pxs_10_90_case = (len(pxs_10_90_case.index) / len(pxs_10_90.index)) * 100
## 90-100
pxs_90_100 = df[df['PXS_AVG_ntile'] >= 90]
pxs_90_100_case = pxs_90_100[pxs_90_100[pheno] == 1]
percent_pxs_90_100_case = (len(pxs_90_100_case.index) / len(pxs_90_100.index)) * 100

# crs
## 0-5
crs_0_5 = df[df[crs] < 0.05]
crs_0_5_case = crs_0_5[crs_0_5[pheno] == 1]
percent_crs_0_5_case = (len(crs_0_5_case.index) / len(crs_0_5.index)) * 100
## 5-7.5
crs_5_7 = df[(df[crs] >= 0.05) & (df[crs] < 0.075)]
crs_5_7_case = crs_5_7[crs_5_7[pheno] == 1]
percent_crs_5_7_case = (len(crs_5_7_case.index) / len(crs_5_7.index)) * 100
## 7.5+
crs_7_100 = df[df[crs] >= 0.075]
crs_7_100_case = crs_7_100[crs_7_100[pheno] == 1]
percent_crs_7_100_case = (len(crs_7_100_case.index) / len(crs_7_100.index)) * 100

# pgs + crs
## low
pgs_crs_1 = pgs_0_10[pgs_0_10['person_id'].isin(crs_0_5['person_id'])]
pgs_crs_1_case = pgs_crs_1[pgs_crs_1[pheno] == 1]
percent_pgs_crs_1_case = (len(pgs_crs_1_case.index) / len(pgs_crs_1.index)) * 100
## medium
pgs_crs_2 = pgs_10_90[pgs_10_90['person_id'].isin(crs_5_7['person_id'])]
pgs_crs_2_case = pgs_crs_2[pgs_crs_2[pheno] == 1]
percent_pgs_crs_2_case = (len(pgs_crs_2_case.index) / len(pgs_crs_2.index)) * 100
## high
pgs_crs_3 = pgs_90_100[pgs_90_100['person_id'].isin(crs_7_100['person_id'])]
pgs_crs_3_case = pgs_crs_3[pgs_crs_3[pheno] == 1]
percent_pgs_crs_3_case = (len(pgs_crs_3_case.index) / len(pgs_crs_3.index)) * 100

# pgs + pxs
## low
pgs_pxs_1 = pgs_0_10[pgs_0_10['person_id'].isin(pxs_0_10['person_id'])]
pgs_pxs_1_case = pgs_pxs_1[pgs_pxs_1[pheno] == 1]
percent_pgs_pxs_1_case = (len(pgs_pxs_1_case.index) / len(pgs_pxs_1.index)) * 100
## medium
pgs_pxs_2 = pgs_10_90[pgs_10_90['person_id'].isin(pxs_10_90['person_id'])]
pgs_pxs_2_case = pgs_pxs_2[pgs_pxs_2[pheno] == 1]
percent_pgs_pxs_2_case = (len(pgs_pxs_2_case.index) / len(pgs_pxs_2.index)) * 100
## high
pgs_pxs_3 = pgs_90_100[pgs_90_100['person_id'].isin(pxs_90_100['person_id'])]
pgs_pxs_3_case = pgs_pxs_3[pgs_pxs_3[pheno] == 1]
percent_pgs_pxs_3_case = (len(pgs_pxs_3_case.index) / len(pgs_pxs_3.index)) * 100

# crs + pxs
## low
pxs_crs_1 = pxs_0_10[pxs_0_10['person_id'].isin(crs_0_5['person_id'])]
pxs_crs_1_case = pxs_crs_1[pxs_crs_1[pheno] == 1]
percent_pxs_crs_1_case = (len(pxs_crs_1_case.index) / len(pxs_crs_1.index)) * 100
## medium
pxs_crs_2 = pxs_10_90[pxs_10_90['person_id'].isin(crs_5_7['person_id'])]
pxs_crs_2_case = pxs_crs_2[pxs_crs_2[pheno] == 1]
percent_pxs_crs_2_case = (len(pxs_crs_2_case.index) / len(pxs_crs_2.index)) * 100
## high
pxs_crs_3 = pxs_90_100[pxs_90_100['person_id'].isin(crs_7_100['person_id'])]
pxs_crs_3_case = pxs_crs_3[pxs_crs_3[pheno] == 1]
percent_pxs_crs_3_case = (len(pxs_crs_3_case.index) / len(pxs_crs_3.index)) * 100

# pgs + crs + pxs
## low
pgs_pxs_crs_1 = pgs_pxs_1[pgs_pxs_1['person_id'].isin(crs_0_5['person_id'])]
pgs_pxs_crs_1_case = pgs_pxs_crs_1[pgs_pxs_crs_1[pheno] == 1]
percent_pgs_pxs_crs_1_case = (len(pgs_pxs_crs_1_case.index) / len(pgs_pxs_crs_1.index)) * 100
## medium
pgs_pxs_crs_2 = pgs_pxs_2[pgs_pxs_2['person_id'].isin(crs_5_7['person_id'])]
pgs_pxs_crs_2_case = pgs_pxs_crs_2[pgs_pxs_crs_2[pheno] == 1]
percent_pgs_pxs_crs_2_case = (len(pgs_pxs_crs_2_case.index) / len(pgs_pxs_crs_2.index)) * 100
## high
pgs_pxs_crs_3 = pgs_pxs_3[pgs_pxs_3['person_id'].isin(crs_7_100['person_id'])]
pgs_pxs_crs_3_case = pgs_pxs_crs_3[pgs_pxs_crs_3[pheno] == 1]
percent_pgs_pxs_crs_3_case = (len(pgs_pxs_crs_3_case.index) / len(pgs_pxs_crs_3.index)) * 100

# make dataframe
cad_df = pd.DataFrame({'Feature' : ['PGS', 'CRS', 'PXS', 'PGS + CRS', 'PGS + PXS', 'CRS + PXS', 'PGS + CRS + PXS'],
                       'low_risk' : [percent_pgs_0_10_case, percent_crs_0_5_case, percent_pxs_0_10_case, percent_pgs_crs_1_case, percent_pgs_pxs_1_case, percent_pxs_crs_1_case, percent_pgs_pxs_crs_1_case],
                       'medium_risk' : [percent_pgs_10_90_case, percent_crs_5_7_case, percent_pxs_10_90_case, percent_pgs_crs_2_case, percent_pgs_pxs_2_case, percent_pxs_crs_2_case, percent_pgs_pxs_crs_2_case],
                       'high_risk' : [percent_pgs_90_100_case, percent_crs_7_100_case, percent_pxs_90_100_case, percent_pgs_crs_3_case, percent_pgs_pxs_3_case, percent_pxs_crs_3_case, percent_pgs_pxs_crs_3_case]})
cad_df

In [ ]:
print('PGS')
print(len(pgs_0_10.index))
print(len(pgs_0_10_case.index))

print(len(pgs_10_90.index))
print(len(pgs_10_90_case.index))

print(len(pgs_90_100.index))
print(len(pgs_90_100_case.index))

print('')
print('CRS')
print(len(crs_0_5.index))
print(len(crs_0_5_case.index))

print(len(crs_5_7.index))
print(len(crs_5_7_case.index))

print(len(crs_7_100.index))
print(len(crs_7_100_case.index))

print('')
print('PXS')
print(len(pxs_0_10.index))
print(len(pxs_0_10_case.index))

print(len(pxs_10_90.index))
print(len(pxs_10_90_case.index))

print(len(pxs_90_100.index))
print(len(pxs_90_100_case.index))

print('')
print('PGS + CRS')
print(len(pgs_crs_1.index))
print(len(pgs_crs_1_case.index))

print(len(pgs_crs_2.index))
print(len(pgs_crs_2_case.index))

print(len(pgs_crs_3.index))
print(len(pgs_crs_3.index))

print('')
print('PGS + PXS')
print(len(pgs_pxs_1.index))
print(len(pgs_pxs_1_case.index))

print(len(pgs_pxs_2.index))
print(len(pgs_pxs_2_case.index))

print(len(pgs_pxs_3.index))
print(len(pgs_pxs_3.index))

print('')
print('CRS + PXS')
print(len(pxs_crs_1.index))
print(len(pxs_crs_1_case.index))

print(len(pxs_crs_2.index))
print(len(pxs_crs_2_case.index))

print(len(pxs_crs_3.index))
print(len(pxs_crs_3.index))

print('')
print('PGS + CRS + PXS')
print(len(pgs_pxs_crs_1.index))
print(len(pgs_pxs_crs_1_case.index))

print(len(pgs_pxs_crs_2.index))
print(len(pgs_pxs_crs_2_case.index))

print(len(pgs_pxs_crs_3.index))
print(len(pgs_pxs_crs_3.index))

### afib

In [ ]:
df = afib_irm_no_missing.copy()
pheno = 'AFIB'
crs = 'AFIB_C2HEST_score'

# pgs
## 0-10
pgs_0_10 = df[df['PGS_' + pheno + '_ntile'] < 10]
pgs_0_10_case = pgs_0_10[pgs_0_10[pheno] == 1]
percent_pgs_0_10_case = (len(pgs_0_10_case.index) / len(pgs_0_10.index)) * 100
## 10-90
pgs_10_90 = df[(df['PGS_' + pheno + '_ntile'] >= 10) & (df['PGS_' + pheno + '_ntile'] < 90)]
pgs_10_90_case = pgs_10_90[pgs_10_90[pheno] == 1]
percent_pgs_10_90_case = (len(pgs_10_90_case.index) / len(pgs_10_90.index)) * 100
## 90-100
pgs_90_100 = df[(df['PGS_' + pheno + '_ntile'] >= 90)]
pgs_90_100_case = pgs_90_100[pgs_90_100[pheno] == 1]
percent_pgs_90_100_case = (len(pgs_90_100_case.index) / len(pgs_90_100.index)) * 100

# pxs
## 0-10
pxs_0_10 = df[df['PXS_AVG_ntile'] < 10]
pxs_0_10_case = pxs_0_10[pxs_0_10[pheno] == 1]
percent_pxs_0_10_case = (len(pxs_0_10_case.index) / len(pxs_0_10.index)) * 100
## 10-90
pxs_10_90 = df[(df['PXS_AVG_ntile'] >= 10) & (df['PXS_AVG_ntile'] < 90)]
pxs_10_90_case = pxs_10_90[pxs_10_90[pheno] == 1]
percent_pxs_10_90_case = (len(pxs_10_90_case.index) / len(pxs_10_90.index)) * 100
## 90-100
pxs_90_100 = df[df['PXS_AVG_ntile'] >= 90]
pxs_90_100_case = pxs_90_100[pxs_90_100[pheno] == 1]
percent_pxs_90_100_case = (len(pxs_90_100_case.index) / len(pxs_90_100.index)) * 100

# crs
## 0-5
crs_0_5 = df[df[crs] <= 1]
crs_0_5_case = crs_0_5[crs_0_5[pheno] == 1]
percent_crs_0_5_case = (len(crs_0_5_case.index) / len(crs_0_5.index)) * 100
## 5-7.5
crs_5_7 = df[(df[crs] >= 2) & (df[crs] <= 3)]
crs_5_7_case = crs_5_7[crs_5_7[pheno] == 1]
percent_crs_5_7_case = (len(crs_5_7_case.index) / len(crs_5_7.index)) * 100
## 7.5+
crs_7_100 = df[df[crs] >= 4]
crs_7_100_case = crs_7_100[crs_7_100[pheno] == 1]
percent_crs_7_100_case = (len(crs_7_100_case.index) / len(crs_7_100.index)) * 100

# pgs + crs
## low
pgs_crs_1 = pgs_0_10[pgs_0_10['person_id'].isin(crs_0_5['person_id'])]
pgs_crs_1_case = pgs_crs_1[pgs_crs_1[pheno] == 1]
percent_pgs_crs_1_case = (len(pgs_crs_1_case.index) / len(pgs_crs_1.index)) * 100
## medium
pgs_crs_2 = pgs_10_90[pgs_10_90['person_id'].isin(crs_5_7['person_id'])]
pgs_crs_2_case = pgs_crs_2[pgs_crs_2[pheno] == 1]
percent_pgs_crs_2_case = (len(pgs_crs_2_case.index) / len(pgs_crs_2.index)) * 100
## high
pgs_crs_3 = pgs_90_100[pgs_90_100['person_id'].isin(crs_7_100['person_id'])]
pgs_crs_3_case = pgs_crs_3[pgs_crs_3[pheno] == 1]
percent_pgs_crs_3_case = (len(pgs_crs_3_case.index) / len(pgs_crs_3.index)) * 100

# pgs + pxs
## low
pgs_pxs_1 = pgs_0_10[pgs_0_10['person_id'].isin(pxs_0_10['person_id'])]
pgs_pxs_1_case = pgs_pxs_1[pgs_pxs_1[pheno] == 1]
percent_pgs_pxs_1_case = (len(pgs_pxs_1_case.index) / len(pgs_pxs_1.index)) * 100
## medium
pgs_pxs_2 = pgs_10_90[pgs_10_90['person_id'].isin(pxs_10_90['person_id'])]
pgs_pxs_2_case = pgs_pxs_2[pgs_pxs_2[pheno] == 1]
percent_pgs_pxs_2_case = (len(pgs_pxs_2_case.index) / len(pgs_pxs_2.index)) * 100
## high
pgs_pxs_3 = pgs_90_100[pgs_90_100['person_id'].isin(pxs_90_100['person_id'])]
pgs_pxs_3_case = pgs_pxs_3[pgs_pxs_3[pheno] == 1]
percent_pgs_pxs_3_case = (len(pgs_pxs_3_case.index) / len(pgs_pxs_3.index)) * 100

# crs + pxs
## low
pxs_crs_1 = pxs_0_10[pxs_0_10['person_id'].isin(crs_0_5['person_id'])]
pxs_crs_1_case = pxs_crs_1[pxs_crs_1[pheno] == 1]
percent_pxs_crs_1_case = (len(pxs_crs_1_case.index) / len(pxs_crs_1.index)) * 100
## medium
pxs_crs_2 = pxs_10_90[pxs_10_90['person_id'].isin(crs_5_7['person_id'])]
pxs_crs_2_case = pxs_crs_2[pxs_crs_2[pheno] == 1]
percent_pxs_crs_2_case = (len(pxs_crs_2_case.index) / len(pxs_crs_2.index)) * 100
## high
pxs_crs_3 = pxs_90_100[pxs_90_100['person_id'].isin(crs_7_100['person_id'])]
pxs_crs_3_case = pxs_crs_3[pxs_crs_3[pheno] == 1]
percent_pxs_crs_3_case = (len(pxs_crs_3_case.index) / len(pxs_crs_3.index)) * 100

# pgs + crs + pxs
## low
pgs_pxs_crs_1 = pgs_pxs_1[pgs_pxs_1['person_id'].isin(crs_0_5['person_id'])]
pgs_pxs_crs_1_case = pgs_pxs_crs_1[pgs_pxs_crs_1[pheno] == 1]
percent_pgs_pxs_crs_1_case = (len(pgs_pxs_crs_1_case.index) / len(pgs_pxs_crs_1.index)) * 100
## medium
pgs_pxs_crs_2 = pgs_pxs_2[pgs_pxs_2['person_id'].isin(crs_5_7['person_id'])]
pgs_pxs_crs_2_case = pgs_pxs_crs_2[pgs_pxs_crs_2[pheno] == 1]
percent_pgs_pxs_crs_2_case = (len(pgs_pxs_crs_2_case.index) / len(pgs_pxs_crs_2.index)) * 100
## high
pgs_pxs_crs_3 = pgs_pxs_3[pgs_pxs_3['person_id'].isin(crs_7_100['person_id'])]
pgs_pxs_crs_3_case = pgs_pxs_crs_3[pgs_pxs_crs_3[pheno] == 1]
percent_pgs_pxs_crs_3_case = (len(pgs_pxs_crs_3_case.index) / len(pgs_pxs_crs_3.index)) * 100

# make dataframe
afib_df = pd.DataFrame({'Feature' : ['PGS', 'CRS', 'PXS', 'PGS + CRS', 'PGS + PXS', 'CRS + PXS', 'PGS + CRS + PXS'],
                       'low_risk' : [percent_pgs_0_10_case, percent_crs_0_5_case, percent_pxs_0_10_case, percent_pgs_crs_1_case, percent_pgs_pxs_1_case, percent_pxs_crs_1_case, percent_pgs_pxs_crs_1_case],
                       'medium_risk' : [percent_pgs_10_90_case, percent_crs_5_7_case, percent_pxs_10_90_case, percent_pgs_crs_2_case, percent_pgs_pxs_2_case, percent_pxs_crs_2_case, percent_pgs_pxs_crs_2_case],
                       'high_risk' : [percent_pgs_90_100_case, percent_crs_7_100_case, percent_pxs_90_100_case, percent_pgs_crs_3_case, percent_pgs_pxs_3_case, percent_pxs_crs_3_case, percent_pgs_pxs_crs_3_case]})
afib_df

In [ ]:
print('PGS')
print(len(pgs_0_10.index))
print(len(pgs_0_10_case.index))

print(len(pgs_10_90.index))
print(len(pgs_10_90_case.index))

print(len(pgs_90_100.index))
print(len(pgs_90_100_case.index))

print('')
print('CRS')
print(len(crs_0_5.index))
print(len(crs_0_5_case.index))

print(len(crs_5_7.index))
print(len(crs_5_7_case.index))

print(len(crs_7_100.index))
print(len(crs_7_100_case.index))

print('')
print('PXS')
print(len(pxs_0_10.index))
print(len(pxs_0_10_case.index))

print(len(pxs_10_90.index))
print(len(pxs_10_90_case.index))

print(len(pxs_90_100.index))
print(len(pxs_90_100_case.index))

print('')
print('PGS + CRS')
print(len(pgs_crs_1.index))
print(len(pgs_crs_1_case.index))

print(len(pgs_crs_2.index))
print(len(pgs_crs_2_case.index))

print(len(pgs_crs_3.index))
print(len(pgs_crs_3.index))

print('')
print('PGS + PXS')
print(len(pgs_pxs_1.index))
print(len(pgs_pxs_1_case.index))

print(len(pgs_pxs_2.index))
print(len(pgs_pxs_2_case.index))

print(len(pgs_pxs_3.index))
print(len(pgs_pxs_3.index))

print('')
print('CRS + PXS')
print(len(pxs_crs_1.index))
print(len(pxs_crs_1_case.index))

print(len(pxs_crs_2.index))
print(len(pxs_crs_2_case.index))

print(len(pxs_crs_3.index))
print(len(pxs_crs_3.index))

print('')
print('PGS + CRS + PXS')
print(len(pgs_pxs_crs_1.index))
print(len(pgs_pxs_crs_1_case.index))

print(len(pgs_pxs_crs_2.index))
print(len(pgs_pxs_crs_2_case.index))

print(len(pgs_pxs_crs_3.index))
print(len(pgs_pxs_crs_3.index))

### hf

In [ ]:
df = hf_irm_no_missing.copy()
pheno = 'HF'
crs = 'HF_PREVENT_CRS_heart_failure'

# pgs
## 0-10
pgs_0_10 = df[df['PGS_' + pheno + '_ntile'] < 10]
pgs_0_10_case = pgs_0_10[pgs_0_10[pheno] == 1]
percent_pgs_0_10_case = (len(pgs_0_10_case.index) / len(pgs_0_10.index)) * 100
## 10-90
pgs_10_90 = df[(df['PGS_' + pheno + '_ntile'] >= 10) & (df['PGS_' + pheno + '_ntile'] < 90)]
pgs_10_90_case = pgs_10_90[pgs_10_90[pheno] == 1]
percent_pgs_10_90_case = (len(pgs_10_90_case.index) / len(pgs_10_90.index)) * 100
## 90-100
pgs_90_100 = df[(df['PGS_' + pheno + '_ntile'] >= 90)]
pgs_90_100_case = pgs_90_100[pgs_90_100[pheno] == 1]
percent_pgs_90_100_case = (len(pgs_90_100_case.index) / len(pgs_90_100.index)) * 100

# pxs
## 0-10
pxs_0_10 = df[df['PXS_AVG_ntile'] < 10]
pxs_0_10_case = pxs_0_10[pxs_0_10[pheno] == 1]
percent_pxs_0_10_case = (len(pxs_0_10_case.index) / len(pxs_0_10.index)) * 100
## 10-90
pxs_10_90 = df[(df['PXS_AVG_ntile'] >= 10) & (df['PXS_AVG_ntile'] < 90)]
pxs_10_90_case = pxs_10_90[pxs_10_90[pheno] == 1]
percent_pxs_10_90_case = (len(pxs_10_90_case.index) / len(pxs_10_90.index)) * 100
## 90-100
pxs_90_100 = df[df['PXS_AVG_ntile'] >= 90]
pxs_90_100_case = pxs_90_100[pxs_90_100[pheno] == 1]
percent_pxs_90_100_case = (len(pxs_90_100_case.index) / len(pxs_90_100.index)) * 100

# crs
## 0-5
crs_0_5 = df[df[crs] < 0.05]
crs_0_5_case = crs_0_5[crs_0_5[pheno] == 1]
percent_crs_0_5_case = (len(crs_0_5_case.index) / len(crs_0_5.index)) * 100
## 5-7.5
crs_5_7 = df[(df[crs] >= 0.05) & (df[crs] < 0.075)]
crs_5_7_case = crs_5_7[crs_5_7[pheno] == 1]
percent_crs_5_7_case = (len(crs_5_7_case.index) / len(crs_5_7.index)) * 100
## 7.5+
crs_7_100 = df[df[crs] >= 0.075]
crs_7_100_case = crs_7_100[crs_7_100[pheno] == 1]
percent_crs_7_100_case = (len(crs_7_100_case.index) / len(crs_7_100.index)) * 100

# pgs + crs
## low
pgs_crs_1 = pgs_0_10[pgs_0_10['person_id'].isin(crs_0_5['person_id'])]
pgs_crs_1_case = pgs_crs_1[pgs_crs_1[pheno] == 1]
percent_pgs_crs_1_case = (len(pgs_crs_1_case.index) / len(pgs_crs_1.index)) * 100
## medium
pgs_crs_2 = pgs_10_90[pgs_10_90['person_id'].isin(crs_5_7['person_id'])]
pgs_crs_2_case = pgs_crs_2[pgs_crs_2[pheno] == 1]
percent_pgs_crs_2_case = (len(pgs_crs_2_case.index) / len(pgs_crs_2.index)) * 100
## high
pgs_crs_3 = pgs_90_100[pgs_90_100['person_id'].isin(crs_7_100['person_id'])]
pgs_crs_3_case = pgs_crs_3[pgs_crs_3[pheno] == 1]
percent_pgs_crs_3_case = (len(pgs_crs_3_case.index) / len(pgs_crs_3.index)) * 100

# pgs + pxs
## low
pgs_pxs_1 = pgs_0_10[pgs_0_10['person_id'].isin(pxs_0_10['person_id'])]
pgs_pxs_1_case = pgs_pxs_1[pgs_pxs_1[pheno] == 1]
percent_pgs_pxs_1_case = (len(pgs_pxs_1_case.index) / len(pgs_pxs_1.index)) * 100
## medium
pgs_pxs_2 = pgs_10_90[pgs_10_90['person_id'].isin(pxs_10_90['person_id'])]
pgs_pxs_2_case = pgs_pxs_2[pgs_pxs_2[pheno] == 1]
percent_pgs_pxs_2_case = (len(pgs_pxs_2_case.index) / len(pgs_pxs_2.index)) * 100
## high
pgs_pxs_3 = pgs_90_100[pgs_90_100['person_id'].isin(pxs_90_100['person_id'])]
pgs_pxs_3_case = pgs_pxs_3[pgs_pxs_3[pheno] == 1]
percent_pgs_pxs_3_case = (len(pgs_pxs_3_case.index) / len(pgs_pxs_3.index)) * 100

# crs + pxs
## low
pxs_crs_1 = pxs_0_10[pxs_0_10['person_id'].isin(crs_0_5['person_id'])]
pxs_crs_1_case = pxs_crs_1[pxs_crs_1[pheno] == 1]
percent_pxs_crs_1_case = (len(pxs_crs_1_case.index) / len(pxs_crs_1.index)) * 100
## medium
pxs_crs_2 = pxs_10_90[pxs_10_90['person_id'].isin(crs_5_7['person_id'])]
pxs_crs_2_case = pxs_crs_2[pxs_crs_2[pheno] == 1]
percent_pxs_crs_2_case = (len(pxs_crs_2_case.index) / len(pxs_crs_2.index)) * 100
## high
pxs_crs_3 = pxs_90_100[pxs_90_100['person_id'].isin(crs_7_100['person_id'])]
pxs_crs_3_case = pxs_crs_3[pxs_crs_3[pheno] == 1]
percent_pxs_crs_3_case = (len(pxs_crs_3_case.index) / len(pxs_crs_3.index)) * 100

# pgs + crs + pxs
## low
pgs_pxs_crs_1 = pgs_pxs_1[pgs_pxs_1['person_id'].isin(crs_0_5['person_id'])]
pgs_pxs_crs_1_case = pgs_pxs_crs_1[pgs_pxs_crs_1[pheno] == 1]
percent_pgs_pxs_crs_1_case = (len(pgs_pxs_crs_1_case.index) / len(pgs_pxs_crs_1.index)) * 100
## medium
pgs_pxs_crs_2 = pgs_pxs_2[pgs_pxs_2['person_id'].isin(crs_5_7['person_id'])]
pgs_pxs_crs_2_case = pgs_pxs_crs_2[pgs_pxs_crs_2[pheno] == 1]
percent_pgs_pxs_crs_2_case = (len(pgs_pxs_crs_2_case.index) / len(pgs_pxs_crs_2.index)) * 100
## high
pgs_pxs_crs_3 = pgs_pxs_3[pgs_pxs_3['person_id'].isin(crs_7_100['person_id'])]
pgs_pxs_crs_3_case = pgs_pxs_crs_3[pgs_pxs_crs_3[pheno] == 1]
percent_pgs_pxs_crs_3_case = (len(pgs_pxs_crs_3_case.index) / len(pgs_pxs_crs_3.index)) * 100

# make dataframe
hf_df = pd.DataFrame({'Feature' : ['PGS', 'CRS', 'PXS', 'PGS + CRS', 'PGS + PXS', 'CRS + PXS', 'PGS + CRS + PXS'],
                       'low_risk' : [percent_pgs_0_10_case, percent_crs_0_5_case, percent_pxs_0_10_case, percent_pgs_crs_1_case, percent_pgs_pxs_1_case, percent_pxs_crs_1_case, percent_pgs_pxs_crs_1_case],
                       'medium_risk' : [percent_pgs_10_90_case, percent_crs_5_7_case, percent_pxs_10_90_case, percent_pgs_crs_2_case, percent_pgs_pxs_2_case, percent_pxs_crs_2_case, percent_pgs_pxs_crs_2_case],
                       'high_risk' : [percent_pgs_90_100_case, percent_crs_7_100_case, percent_pxs_90_100_case, percent_pgs_crs_3_case, percent_pgs_pxs_3_case, percent_pxs_crs_3_case, percent_pgs_pxs_crs_3_case]})
hf_df

In [ ]:
print('PGS')
print(len(pgs_0_10.index))
print(len(pgs_0_10_case.index))

print(len(pgs_10_90.index))
print(len(pgs_10_90_case.index))

print(len(pgs_90_100.index))
print(len(pgs_90_100_case.index))

print('')
print('CRS')
print(len(crs_0_5.index))
print(len(crs_0_5_case.index))

print(len(crs_5_7.index))
print(len(crs_5_7_case.index))

print(len(crs_7_100.index))
print(len(crs_7_100_case.index))

print('')
print('PXS')
print(len(pxs_0_10.index))
print(len(pxs_0_10_case.index))

print(len(pxs_10_90.index))
print(len(pxs_10_90_case.index))

print(len(pxs_90_100.index))
print(len(pxs_90_100_case.index))

print('')
print('PGS + CRS')
print(len(pgs_crs_1.index))
print(len(pgs_crs_1_case.index))

print(len(pgs_crs_2.index))
print(len(pgs_crs_2_case.index))

print(len(pgs_crs_3.index))
print(len(pgs_crs_3.index))

print('')
print('PGS + PXS')
print(len(pgs_pxs_1.index))
print(len(pgs_pxs_1_case.index))

print(len(pgs_pxs_2.index))
print(len(pgs_pxs_2_case.index))

print(len(pgs_pxs_3.index))
print(len(pgs_pxs_3.index))

print('')
print('CRS + PXS')
print(len(pxs_crs_1.index))
print(len(pxs_crs_1_case.index))

print(len(pxs_crs_2.index))
print(len(pxs_crs_2_case.index))

print(len(pxs_crs_3.index))
print(len(pxs_crs_3.index))

print('')
print('PGS + CRS + PXS')
print(len(pgs_pxs_crs_1.index))
print(len(pgs_pxs_crs_1_case.index))

print(len(pgs_pxs_crs_2.index))
print(len(pgs_pxs_crs_2_case.index))

print(len(pgs_pxs_crs_3.index))
print(len(pgs_pxs_crs_3.index))

## update tables with missing for participant count < 20
- in compliance with data dissemination policy

In [ ]:
cad_df_pub = cad_df.copy()
cad_df_pub.loc[4, 'low_risk'] = np.nan
cad_df_pub.loc[6, 'low_risk'] = np.nan
cad_df_pub.loc[6, 'high_risk'] = np.nan
cad_df_pub

In [ ]:
afib_df_pub = afib_df.copy()
afib_df_pub.loc[4, 'low_risk'] = np.nan
afib_df_pub.loc[6, 'low_risk'] = np.nan
afib_df_pub.loc[6, 'high_risk'] = np.nan
afib_df_pub

In [ ]:
hf_df_pub = hf_df.copy()
hf_df_pub.loc[4, 'low_risk'] = np.nan
hf_df_pub.loc[6, 'low_risk'] = np.nan
hf_df_pub.loc[6, 'high_risk'] = np.nan
hf_df_pub

## make line graphs

In [ ]:
sns.set_palette("colorblind")

In [ ]:
risk_levels = ["low_risk", "medium_risk", "high_risk"]

fig, axes = plt.subplots(1, 3, figsize = (18, 8), sharey = True)

datasets = [
    (cad_df_pub, "CAD", "A"),
    (afib_df_pub, "AFIB", "B"),
    (hf_df_pub, "HF", "C")
]

for ax, (df, title, label) in zip(axes, datasets):

    for _, row in df.iterrows():
        y = [row[level] for level in risk_levels]
        ax.plot(risk_levels, y, marker = 'o', label = row["Feature"])

    ax.set_title(title)
    ax.set_xlabel("Risk Category")

    ax.text(-0.15, 1.05, label,
            transform = ax.transAxes,
            fontsize = 14,
            fontweight = 'bold',
            va = 'top')
    
    table_df = df[["Feature"] + risk_levels]
    
    table_df = table_df.rename(columns = {
        "low_risk" : "Low Risk",
        "medium_risk" : "Medium Risk",
        "high_risk" : "High Risk"
    })
    
    table_df = table_df.round(2)
    table_df = table_df.replace({np.nan : 'Count ≤ 20'})
    
    cell_colors = []

    header_fill = '#d9d9d9'
    row_fill_1 = '#ffffff'
    row_fill_2 = '#f2f2f2'
    
    for i in range(len(table_df)):
        row_color = row_fill_1 if i % 2 == 0 else row_fill_2
        cell_colors.append([row_color] * len(table_df.columns))

    table = ax.table(
        cellText = table_df.values,
        colLabels = table_df.columns,
        loc = 'bottom',
        colColours = [header_fill] * len(table_df.columns),
        cellColours = cell_colors,
        cellLoc = 'center',
        colLoc = 'center',
        bbox = [-0.1, -0.45, 1.15, 0.35]
    )
    
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight = 'bold')
    
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    
    col_widths = {
        0 : 0.5,
        1 : 0.3,
        2 : 0.35,
        3 : 0.3
    }

    for col, width in col_widths.items():
        for row in range(len(table_df) + 1):
            table[row, col].set_width(width)

    ax.set_position([ax.get_position().x0,
                     ax.get_position().y0 + 0.2,
                     ax.get_position().width,
                     ax.get_position().height * 0.8])

axes[0].set_ylabel("Percentage of Cases")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels,
           title = "Feature",
           bbox_to_anchor = (0.9, 0.6),
           loc = "center left")

fig.suptitle("AOU Case Proportions Across Risk Thresholds",
             fontsize = 16,
             fontweight = 'bold',
             y = 1.01)

plt.savefig("AOU.all_pheno.case_proportions_by_risk_thresholds.png", dpi = 300, bbox_inches = 'tight')
plt.show()

## export percentage dfs

In [ ]:
cad_df.to_csv('CAD.case_proportions_by_risk_thresholds.csv', index = None)

In [ ]:
afib_df.to_csv('AFIB.case_proportions_by_risk_thresholds.csv', index = None)

In [ ]:
hf_df.to_csv('HF.case_proportions_by_risk_thresholds.csv', index = None)

# process catboost feature selection outputs

## read in inputs

### CAD

In [ ]:
!mkdir CAD

In [ ]:
!gsutil -m cp -R ${WORKSPACE_BUCKET}/Cardio_IRM/CAD/output/feature_selection/catboost/ CAD/

In [ ]:
pheno = 'CAD'

gain = []
shap = []

for iter in list(range(1, 1001)):
    gain_filename = pheno + '/catboost/' + pheno + '.catboost.gain.feature_importance.ITER_' + str(iter) + '.txt'
    shap_filename = pheno + '/catboost/' + pheno + '.catboost.shap.feature_importance.ITER_' + str(iter) + '.txt'
    
    gain.append(pd.read_csv(gain_filename, sep = '\t', index_col = 'feature').drop(columns = ['Unnamed: 0']))
    shap.append(pd.read_csv(shap_filename, sep = '\t', index_col = 'feature').drop(columns = ['Unnamed: 0']))

cad_gain_df = pd.concat(gain, axis = 1)
cad_shap_df = pd.concat(shap, axis = 1)

### AFIB

In [ ]:
!mkdir AFIB

In [ ]:
!gsutil -m cp -R ${WORKSPACE_BUCKET}/Cardio_IRM/AFIB/output/feature_selection/catboost/ AFIB/

In [ ]:
pheno = 'AFIB'

gain = []
shap = []

for iter in list(range(1, 1001)):
    gain_filename = pheno + '/catboost/' + pheno + '.catboost.gain.feature_importance.ITER_' + str(iter) + '.txt'
    shap_filename = pheno + '/catboost/' + pheno + '.catboost.shap.feature_importance.ITER_' + str(iter) + '.txt'
    
    gain.append(pd.read_csv(gain_filename, sep = '\t', index_col = 'feature').drop(columns = ['Unnamed: 0']))
    shap.append(pd.read_csv(shap_filename, sep = '\t', index_col = 'feature').drop(columns = ['Unnamed: 0']))

afib_gain_df = pd.concat(gain, axis = 1)
afib_shap_df = pd.concat(shap, axis = 1)

### HF

In [ ]:
!gsutil -m cp -R ${WORKSPACE_BUCKET}/Cardio_IRM/HF/output/feature_selection/catboost/ HF/

In [ ]:
pheno = 'HF'

gain = []
shap = []

for iter in list(range(1, 1001)):
    gain_filename = pheno + '/catboost/' + pheno + '.catboost.gain.feature_importance.ITER_' + str(iter) + '.txt'
    shap_filename = pheno + '/catboost/' + pheno + '.catboost.shap.feature_importance.ITER_' + str(iter) + '.txt'
    
    gain.append(pd.read_csv(gain_filename, sep = '\t', index_col = 'feature').drop(columns = ['Unnamed: 0']))
    shap.append(pd.read_csv(shap_filename, sep = '\t', index_col = 'feature').drop(columns = ['Unnamed: 0']))

hf_gain_df = pd.concat(gain, axis = 1)
hf_shap_df = pd.concat(shap, axis = 1)

## calculate means

### cad

In [ ]:
cad_gain_df["MEAN"] = cad_gain_df.mean(axis = 1)
cad_gain_df[["MEAN"]].sort_values(by = ['MEAN'], ascending = False)

In [ ]:
cad_shap_df["MEAN"] = cad_shap_df.mean(axis = 1)
cad_shap_df[["MEAN"]].sort_values(by = ['MEAN'], ascending = False)

### afib

In [ ]:
afib_gain_df["MEAN"] = afib_gain_df.mean(axis = 1)
afib_gain_df[["MEAN"]].sort_values(by = ['MEAN'], ascending = False)

In [ ]:
afib_shap_df["MEAN"] = afib_shap_df.mean(axis = 1)
afib_shap_df[["MEAN"]].sort_values(by = ['MEAN'], ascending = False)

### hf

In [ ]:
hf_gain_df["MEAN"] = hf_gain_df.mean(axis = 1)
hf_gain_df[["MEAN"]].sort_values(by = ['MEAN'], ascending = False)

In [ ]:
hf_shap_df["MEAN"] = hf_shap_df.mean(axis = 1)
hf_shap_df[["MEAN"]].sort_values(by = ['MEAN'], ascending = False)

## select features based on percentile threshold

### cad

In [ ]:
vals = cad_gain_df['MEAN'].values
thresh = np.percentile(vals, 50)
cad_gain_selected = cad_gain_df[cad_gain_df["MEAN"] >= thresh]
print(len(cad_gain_df.index))
print(len(cad_gain_selected.index))
cad_gain_selected[['MEAN']].sort_values(by = ['MEAN'], ascending = False)

In [ ]:
vals = cad_shap_df['MEAN'].values
thresh = np.percentile(vals, 50)
cad_shap_selected = cad_shap_df[cad_shap_df["MEAN"] >= thresh]
print(len(cad_shap_df.index))
print(len(cad_shap_selected.index))
cad_shap_selected[['MEAN']].sort_values(by = ['MEAN'], ascending = False)

### AFIB

In [ ]:
vals = afib_gain_df['MEAN'].values
thresh = np.percentile(vals, 50)
afib_gain_selected = afib_gain_df[afib_gain_df["MEAN"] >= thresh]
print(len(afib_gain_df.index))
print(len(afib_gain_selected.index))
afib_gain_selected[['MEAN']].sort_values(by = ['MEAN'], ascending = False)

In [ ]:
vals = afib_shap_df['MEAN'].values
thresh = np.percentile(vals, 50)
afib_shap_selected = afib_shap_df[afib_shap_df["MEAN"] >= thresh]
print(len(afib_shap_df.index))
print(len(afib_shap_selected.index))
afib_shap_selected[['MEAN']].sort_values(by = ['MEAN'], ascending = False)

### HF

In [ ]:
vals = hf_gain_df['MEAN'].values
thresh = np.percentile(vals, 50)
hf_gain_selected = hf_gain_df[hf_gain_df["MEAN"] >= thresh]
print(len(hf_gain_df.index))
print(len(hf_gain_selected.index))
hf_gain_selected[['MEAN']].sort_values(by = ['MEAN'], ascending = False)

In [ ]:
vals = hf_shap_df['MEAN'].values
thresh = np.percentile(vals, 50)
hf_shap_selected = hf_shap_df[hf_shap_df["MEAN"] >= thresh]
print(len(hf_shap_df.index))
print(len(hf_shap_selected.index))
hf_shap_selected[['MEAN']].sort_values(by = ['MEAN'], ascending = False)

## check feature differences between phenotypes

### remove pheno labels

In [ ]:
cad_shap_clean = cad_shap_selected.copy()
cad_shap_clean.index = cad_shap_selected.index.str.replace('_CAD', '')

In [ ]:
afib_shap_clean = afib_shap_selected.copy()
afib_shap_clean.index = afib_shap_selected.index.str.replace('_AFIB', '')

In [ ]:
hf_shap_clean = hf_shap_selected.copy()
hf_shap_clean.index = hf_shap_selected.index.str.replace('_HF', '')

### compare

#### cad

In [ ]:
cad_shap_clean[~cad_shap_clean.index.isin(afib_shap_clean.index)][['MEAN']]

In [ ]:
cad_shap_clean[~cad_shap_clean.index.isin(hf_shap_clean.index)][['MEAN']]

#### afib

In [ ]:
afib_shap_clean[~afib_shap_clean.index.isin(cad_shap_clean.index)][['MEAN']]

In [ ]:
afib_shap_clean[~afib_shap_clean.index.isin(hf_shap_clean.index)][['MEAN']]

#### hf

In [ ]:
hf_shap_clean[~hf_shap_clean.index.isin(cad_shap_clean.index)][['MEAN']]

In [ ]:
hf_shap_clean[~hf_shap_clean.index.isin(afib_shap_clean.index)][['MEAN']]

## check feature differences between shap and gain

### cad

In [ ]:
cad_shap_selected[~cad_shap_selected.index.isin(cad_gain_selected.index)][['MEAN']]

In [ ]:
cad_gain_selected[~cad_gain_selected.index.isin(cad_shap_selected.index)][['MEAN']]

### afib

In [ ]:
afib_shap_selected[~afib_shap_selected.index.isin(afib_gain_selected.index)][['MEAN']]

In [ ]:
afib_gain_selected[~afib_gain_selected.index.isin(afib_shap_selected.index)][['MEAN']]

### hf

In [ ]:
hf_shap_selected[~hf_shap_selected.index.isin(hf_gain_selected.index)][['MEAN']]

In [ ]:
hf_gain_selected[~hf_gain_selected.index.isin(hf_shap_selected.index)][['MEAN']]

## compute weights
- mean / sum of all means

### cad

In [ ]:
cad_shap_selected['WEIGHT'] = cad_shap_selected['MEAN'] / sum(cad_shap_selected['MEAN'])
cad_shap_selected[['MEAN', 'WEIGHT']].sort_values(by = ['MEAN'], ascending = False)

In [ ]:
cad_gain_selected['WEIGHT'] = cad_gain_selected['MEAN'] / sum(cad_gain_selected['MEAN'])
cad_gain_selected[['MEAN', 'WEIGHT']].sort_values(by = ['MEAN'], ascending = False)

### afib

In [ ]:
afib_shap_selected['WEIGHT'] = afib_shap_selected['MEAN'] / sum(afib_shap_selected['MEAN'])
afib_shap_selected[['MEAN', 'WEIGHT']].sort_values(by = ['MEAN'], ascending = False)

In [ ]:
afib_gain_selected['WEIGHT'] = afib_gain_selected['MEAN'] / sum(afib_gain_selected['MEAN'])
afib_gain_selected[['MEAN', 'WEIGHT']].sort_values(by = ['MEAN'], ascending = False)

### hf

In [ ]:
hf_shap_selected['WEIGHT'] = hf_shap_selected['MEAN'] / sum(hf_shap_selected['MEAN'])
hf_shap_selected[['MEAN', 'WEIGHT']].sort_values(by = ['MEAN'], ascending = False)

In [ ]:
hf_gain_selected['WEIGHT'] = hf_gain_selected['MEAN'] / sum(hf_gain_selected['MEAN'])
hf_gain_selected[['MEAN', 'WEIGHT']].sort_values(by = ['MEAN'], ascending = False)

## export

### cad

In [ ]:
cad_shap_selected.to_csv('CAD.catboost.shap.selected_features.txt', sep = '\t')

In [ ]:
cad_shap_selected[['MEAN']].sort_values(by = ['MEAN'], ascending = False).to_csv('CAD.catboost.shap.selected_features.csv')

In [ ]:
cad_gain_selected.to_csv('CAD.catboost.gain.selected_features.txt', sep = '\t')

### afib

In [ ]:
afib_shap_selected.to_csv('AFIB.catboost.shap.selected_features.txt', sep = '\t')

In [ ]:
afib_shap_selected[['MEAN']].sort_values(by = ['MEAN'], ascending = False).to_csv('AFIB.catboost.shap.selected_features.csv')

In [ ]:
afib_gain_selected.to_csv('AFIB.catboost.gain.selected_features.txt', sep = '\t')

### hf

In [ ]:
hf_shap_selected.to_csv('HF.catboost.shap.selected_features.txt', sep = '\t')

In [ ]:
hf_shap_selected[['MEAN']].sort_values(by = ['MEAN'], ascending = False).to_csv('HF.catboost.shap.selected_features.csv')

In [ ]:
hf_gain_selected.to_csv('HF.catboost.gain.selected_features.txt', sep = '\t')

# process outputs from eval 1000 iterations

## read in outputs

### cad

In [ ]:
!gsutil -m cp -R ${WORKSPACE_BUCKET}/Cardio_IRM/CAD/output/regressions/eval/ CAD/

In [ ]:
pheno = 'CAD'

total_cvd_auroc = []
total_cvd_auprc = []
total_cvd_f1 = []
total_cvd_balanced_acc = []
ascvd_auroc = []
ascvd_auprc = []
ascvd_f1 = []
ascvd_balanced_acc = []
heart_failure_auroc = []
heart_failure_auprc = []
heart_failure_f1 = []
heart_failure_balanced_acc = []
chd_auroc = []
chd_auprc = []
chd_f1 = []
chd_balanced_acc = []
stroke_auroc = []
stroke_auprc = []
stroke_f1 = []
stroke_balanced_acc = []

for iter in list(range(1, 1001)):
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    total_cvd_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    total_cvd_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    total_cvd_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    total_cvd_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    ascvd_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    ascvd_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    ascvd_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    ascvd_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    heart_failure_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    heart_failure_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    heart_failure_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    heart_failure_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    chd_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    chd_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    chd_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    chd_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    stroke_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    stroke_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    stroke_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    stroke_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))


cad_total_cvd_auroc_df = pd.concat(total_cvd_auroc, axis = 1)
cad_total_cvd_auprc_df = pd.concat(total_cvd_auprc, axis = 1)
cad_total_cvd_f1_df = pd.concat(total_cvd_f1, axis = 1)
cad_total_cvd_balanced_acc_df = pd.concat(total_cvd_balanced_acc, axis = 1)
cad_ascvd_auroc_df = pd.concat(ascvd_auroc, axis = 1)
cad_ascvd_auprc_df = pd.concat(ascvd_auprc, axis = 1)
cad_ascvd_f1_df = pd.concat(ascvd_f1, axis = 1)
cad_ascvd_balanced_acc_df = pd.concat(ascvd_balanced_acc, axis = 1)
cad_heart_failure_auroc_df = pd.concat(heart_failure_auroc, axis = 1)
cad_heart_failure_auprc_df = pd.concat(heart_failure_auprc, axis = 1)
cad_heart_failure_f1_df = pd.concat(heart_failure_f1, axis = 1)
cad_heart_failure_balanced_acc_df = pd.concat(heart_failure_balanced_acc, axis = 1)
cad_chd_auroc_df = pd.concat(chd_auroc, axis = 1)
cad_chd_auprc_df = pd.concat(chd_auprc, axis = 1)
cad_chd_f1_df = pd.concat(chd_f1, axis = 1)
cad_chd_balanced_acc_df = pd.concat(chd_balanced_acc, axis = 1)
cad_stroke_auroc_df = pd.concat(stroke_auroc, axis = 1)
cad_stroke_auprc_df = pd.concat(stroke_auprc, axis = 1)
cad_stroke_f1_df = pd.concat(stroke_f1, axis = 1)
cad_stroke_balanced_acc_df = pd.concat(stroke_balanced_acc, axis = 1)

In [ ]:
pheno = 'CAD'

pgs_beta = []
pgs_pval = []
pxs_beta = []
pxs_pval = []
wt_pxs_beta = []
wt_pxs_pval = []
pgs_pxs_beta = []
pgs_pxs_pval = []
pgs_wt_pxs_beta = []
pgs_wt_pxs_pval = []

total_cvd_crs_beta = []
total_cvd_crs_pval = []
total_cvd_pgs_crs_beta = []
total_cvd_pgs_crs_pval = []
total_cvd_crs_pxs_beta = []
total_cvd_crs_pxs_pval = []
total_cvd_crs_wt_pxs_beta = []
total_cvd_crs_wt_pxs_pval = []
total_cvd_pgs_crs_pxs_beta = []
total_cvd_pgs_crs_pxs_pval = []
total_cvd_pgs_crs_wt_pxs_beta = []
total_cvd_pgs_crs_wt_pxs_pval = []

ascvd_crs_beta = []
ascvd_crs_pval = []
ascvd_pgs_crs_beta = []
ascvd_pgs_crs_pval = []
ascvd_crs_pxs_beta = []
ascvd_crs_pxs_pval = []
ascvd_crs_wt_pxs_beta = []
ascvd_crs_wt_pxs_pval = []
ascvd_pgs_crs_pxs_beta = []
ascvd_pgs_crs_pxs_pval = []
ascvd_pgs_crs_wt_pxs_beta = []
ascvd_pgs_crs_wt_pxs_pval = []

heart_failure_crs_beta = []
heart_failure_crs_pval = []
heart_failure_pgs_crs_beta = []
heart_failure_pgs_crs_pval = []
heart_failure_crs_pxs_beta = []
heart_failure_crs_pxs_pval = []
heart_failure_crs_wt_pxs_beta = []
heart_failure_crs_wt_pxs_pval = []
heart_failure_pgs_crs_pxs_beta = []
heart_failure_pgs_crs_pxs_pval = []
heart_failure_pgs_crs_wt_pxs_beta = []
heart_failure_pgs_crs_wt_pxs_pval = []

stroke_crs_beta = []
stroke_crs_pval = []
stroke_pgs_crs_beta = []
stroke_pgs_crs_pval = []
stroke_crs_pxs_beta = []
stroke_crs_pxs_pval = []
stroke_crs_wt_pxs_beta = []
stroke_crs_wt_pxs_pval = []
stroke_pgs_crs_pxs_beta = []
stroke_pgs_crs_pxs_pval = []
stroke_pgs_crs_wt_pxs_beta = []
stroke_pgs_crs_wt_pxs_pval = []

chd_crs_beta = []
chd_crs_pval = []
chd_pgs_crs_beta = []
chd_pgs_crs_pval = []
chd_crs_pxs_beta = []
chd_crs_pxs_pval = []
chd_crs_wt_pxs_beta = []
chd_crs_wt_pxs_pval = []
chd_pgs_crs_pxs_beta = []
chd_pgs_crs_pxs_pval = []
chd_pgs_crs_wt_pxs_beta = []
chd_pgs_crs_wt_pxs_pval = []

for iter in list(range(1, 1001)):
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.PVAL.ITER_' + str(iter) + '.txt'
    
    pgs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 1]).dropna().rename(columns = {'PGS_CAD' : ('ITER_' + str(iter))}))
    pgs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 1]).dropna().rename(columns = {'PGS_CAD' : ('ITER_' + str(iter))}))
    
    pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 3]).dropna().rename(columns = {'PXS_AVG' : ('ITER_' + str(iter))}))
    pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 3]).dropna().rename(columns = {'PXS_AVG' : ('ITER_' + str(iter))}))
    
    wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 4]).dropna().rename(columns = {'PXS_WEIGHTED_AVG' : ('ITER_' + str(iter))}))
    wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 4]).dropna().rename(columns = {'PXS_WEIGHTED_AVG' : ('ITER_' + str(iter))}))
    
    pgs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 8]).dropna().rename(columns = {"('PGS_CAD', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    pgs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 8]).dropna().rename(columns = {"('PGS_CAD', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    pgs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 11]).dropna().rename(columns = {"('PGS_CAD', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    pgs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 11]).dropna().rename(columns = {"('PGS_CAD', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'CAD_PREVENT_CRS_total_cvd' : ('ITER_' + str(iter))}))
    total_cvd_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'CAD_PREVENT_CRS_total_cvd' : ('ITER_' + str(iter))}))
    
    total_cvd_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_total_cvd')" : ('ITER_' + str(iter))}))
    total_cvd_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_total_cvd')" : ('ITER_' + str(iter))}))
    
    total_cvd_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('CAD_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('CAD_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('CAD_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('CAD_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.PVAL.ITER_' + str(iter) + '.txt'
    
    ascvd_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'CAD_PREVENT_CRS_ascvd' : ('ITER_' + str(iter))}))
    ascvd_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'CAD_PREVENT_CRS_ascvd' : ('ITER_' + str(iter))}))
    
    ascvd_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_ascvd')" : ('ITER_' + str(iter))}))
    ascvd_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_ascvd')" : ('ITER_' + str(iter))}))
    
    ascvd_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('CAD_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    ascvd_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('CAD_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    ascvd_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('CAD_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    ascvd_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('CAD_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    ascvd_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    ascvd_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    ascvd_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    ascvd_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))

    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.PVAL.ITER_' + str(iter) + '.txt'
    
    heart_failure_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'CAD_PREVENT_CRS_heart_failure' : ('ITER_' + str(iter))}))
    heart_failure_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'CAD_PREVENT_CRS_heart_failure' : ('ITER_' + str(iter))}))
    
    heart_failure_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_heart_failure')" : ('ITER_' + str(iter))}))
    heart_failure_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_heart_failure')" : ('ITER_' + str(iter))}))
    
    heart_failure_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('CAD_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('CAD_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    heart_failure_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('CAD_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('CAD_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    heart_failure_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    heart_failure_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.PVAL.ITER_' + str(iter) + '.txt'
    
    stroke_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'CAD_PREVENT_CRS_stroke' : ('ITER_' + str(iter))}))
    stroke_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'CAD_PREVENT_CRS_stroke' : ('ITER_' + str(iter))}))
    
    stroke_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_stroke')" : ('ITER_' + str(iter))}))
    stroke_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_stroke')" : ('ITER_' + str(iter))}))
    
    stroke_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('CAD_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    stroke_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('CAD_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    stroke_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('CAD_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    stroke_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('CAD_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    stroke_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    stroke_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    stroke_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    stroke_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.PVAL.ITER_' + str(iter) + '.txt'
    
    chd_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'CAD_PREVENT_CRS_chd' : ('ITER_' + str(iter))}))
    chd_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'CAD_PREVENT_CRS_chd' : ('ITER_' + str(iter))}))
    
    chd_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_chd')" : ('ITER_' + str(iter))}))
    chd_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_chd')" : ('ITER_' + str(iter))}))
    
    chd_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('CAD_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    chd_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('CAD_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    chd_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('CAD_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    chd_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('CAD_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    chd_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    chd_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    chd_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    chd_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_CAD', 'CAD_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
cad_pgs_beta_df = pd.concat(pgs_beta, axis = 1)
cad_pgs_pval_df = pd.concat(pgs_pval, axis = 1)

cad_pxs_beta_df = pd.concat(pxs_beta, axis = 1)
cad_pxs_pval_df = pd.concat(pxs_pval, axis = 1)

cad_wt_pxs_beta_df = pd.concat(wt_pxs_beta, axis = 1)
cad_wt_pxs_pval_df = pd.concat(wt_pxs_pval, axis = 1)

cad_pgs_pxs_beta_df = pd.concat(pgs_pxs_beta, axis = 1)
cad_pgs_pxs_pval_df = pd.concat(pgs_pxs_pval, axis = 1)

cad_pgs_wt_pxs_beta_df = pd.concat(pgs_wt_pxs_beta, axis = 1)
cad_pgs_wt_pxs_pval_df = pd.concat(pgs_wt_pxs_pval, axis = 1)

cad_total_cvd_crs_beta_df = pd.concat(total_cvd_crs_beta, axis = 1)
cad_total_cvd_crs_pval_df = pd.concat(total_cvd_crs_pval, axis = 1)

cad_total_cvd_pgs_crs_beta_df = pd.concat(total_cvd_pgs_crs_beta, axis = 1)
cad_total_cvd_pgs_crs_pval_df = pd.concat(total_cvd_pgs_crs_pval, axis = 1)

cad_total_cvd_crs_pxs_beta_df = pd.concat(total_cvd_crs_pxs_beta, axis = 1)
cad_total_cvd_crs_pxs_pval_df = pd.concat(total_cvd_crs_pxs_pval, axis = 1)

cad_total_cvd_crs_wt_pxs_beta_df = pd.concat(total_cvd_crs_wt_pxs_beta, axis = 1)
cad_total_cvd_crs_wt_pxs_pval_df = pd.concat(total_cvd_crs_wt_pxs_pval, axis = 1)

cad_total_cvd_pgs_crs_pxs_beta_df = pd.concat(total_cvd_pgs_crs_pxs_beta, axis = 1)
cad_total_cvd_pgs_crs_pxs_pval_df = pd.concat(total_cvd_pgs_crs_pxs_pval, axis = 1)

cad_total_cvd_pgs_crs_wt_pxs_beta_df = pd.concat(total_cvd_pgs_crs_wt_pxs_beta, axis = 1)
cad_total_cvd_pgs_crs_wt_pxs_pval_df = pd.concat(total_cvd_pgs_crs_wt_pxs_pval, axis = 1)

cad_ascvd_crs_beta_df = pd.concat(ascvd_crs_beta, axis = 1)
cad_ascvd_crs_pval_df = pd.concat(ascvd_crs_pval, axis = 1)

cad_ascvd_pgs_crs_beta_df = pd.concat(ascvd_pgs_crs_beta, axis = 1)
cad_ascvd_pgs_crs_pval_df = pd.concat(ascvd_pgs_crs_pval, axis = 1)

cad_ascvd_crs_pxs_beta_df = pd.concat(ascvd_crs_pxs_beta, axis = 1)
cad_ascvd_crs_pxs_pval_df = pd.concat(ascvd_crs_pxs_pval, axis = 1)

cad_ascvd_crs_wt_pxs_beta_df = pd.concat(ascvd_crs_wt_pxs_beta, axis = 1)
cad_ascvd_crs_wt_pxs_pval_df = pd.concat(ascvd_crs_wt_pxs_pval, axis = 1)

cad_ascvd_pgs_crs_pxs_beta_df = pd.concat(ascvd_pgs_crs_pxs_beta, axis = 1)
cad_ascvd_pgs_crs_pxs_pval_df = pd.concat(ascvd_pgs_crs_pxs_pval, axis = 1)

cad_ascvd_pgs_crs_wt_pxs_beta_df = pd.concat(ascvd_pgs_crs_wt_pxs_beta, axis = 1)
cad_ascvd_pgs_crs_wt_pxs_pval_df = pd.concat(ascvd_pgs_crs_wt_pxs_pval, axis = 1)

cad_heart_failure_crs_beta_df = pd.concat(heart_failure_crs_beta, axis = 1)
cad_heart_failure_crs_pval_df = pd.concat(heart_failure_crs_pval, axis = 1)

cad_heart_failure_pgs_crs_beta_df = pd.concat(heart_failure_pgs_crs_beta, axis = 1)
cad_heart_failure_pgs_crs_pval_df = pd.concat(heart_failure_pgs_crs_pval, axis = 1)

cad_heart_failure_crs_pxs_beta_df = pd.concat(heart_failure_crs_pxs_beta, axis = 1)
cad_heart_failure_crs_pxs_pval_df = pd.concat(heart_failure_crs_pxs_pval, axis = 1)

cad_heart_failure_crs_wt_pxs_beta_df = pd.concat(heart_failure_crs_wt_pxs_beta, axis = 1)
cad_heart_failure_crs_wt_pxs_pval_df = pd.concat(heart_failure_crs_wt_pxs_pval, axis = 1)

cad_heart_failure_pgs_crs_pxs_beta_df = pd.concat(heart_failure_pgs_crs_pxs_beta, axis = 1)
cad_heart_failure_pgs_crs_pxs_pval_df = pd.concat(heart_failure_pgs_crs_pxs_pval, axis = 1)

cad_heart_failure_pgs_crs_wt_pxs_beta_df = pd.concat(heart_failure_pgs_crs_wt_pxs_beta, axis = 1)
cad_heart_failure_pgs_crs_wt_pxs_pval_df = pd.concat(heart_failure_pgs_crs_wt_pxs_pval, axis = 1)

cad_stroke_crs_beta_df = pd.concat(stroke_crs_beta, axis = 1)
cad_stroke_crs_pval_df = pd.concat(stroke_crs_pval, axis = 1)

cad_stroke_pgs_crs_beta_df = pd.concat(stroke_pgs_crs_beta, axis = 1)
cad_stroke_pgs_crs_pval_df = pd.concat(stroke_pgs_crs_pval, axis = 1)

cad_stroke_crs_pxs_beta_df = pd.concat(stroke_crs_pxs_beta, axis = 1)
cad_stroke_crs_pxs_pval_df = pd.concat(stroke_crs_pxs_pval, axis = 1)

cad_stroke_crs_wt_pxs_beta_df = pd.concat(stroke_crs_wt_pxs_beta, axis = 1)
cad_stroke_crs_wt_pxs_pval_df = pd.concat(stroke_crs_wt_pxs_pval, axis = 1)

cad_stroke_pgs_crs_pxs_beta_df = pd.concat(stroke_pgs_crs_pxs_beta, axis = 1)
cad_stroke_pgs_crs_pxs_pval_df = pd.concat(stroke_pgs_crs_pxs_pval, axis = 1)

cad_stroke_pgs_crs_wt_pxs_beta_df = pd.concat(stroke_pgs_crs_wt_pxs_beta, axis = 1)
cad_stroke_pgs_crs_wt_pxs_pval_df = pd.concat(stroke_pgs_crs_wt_pxs_pval, axis = 1)

cad_chd_crs_beta_df = pd.concat(chd_crs_beta, axis = 1)
cad_chd_crs_pval_df = pd.concat(chd_crs_pval, axis = 1)

cad_chd_pgs_crs_beta_df = pd.concat(chd_pgs_crs_beta, axis = 1)
cad_chd_pgs_crs_pval_df = pd.concat(chd_pgs_crs_pval, axis = 1)

cad_chd_crs_pxs_beta_df = pd.concat(chd_crs_pxs_beta, axis = 1)
cad_chd_crs_pxs_pval_df = pd.concat(chd_crs_pxs_pval, axis = 1)

cad_chd_crs_wt_pxs_beta_df = pd.concat(chd_crs_wt_pxs_beta, axis = 1)
cad_chd_crs_wt_pxs_pval_df = pd.concat(chd_crs_wt_pxs_pval, axis = 1)

cad_chd_pgs_crs_pxs_beta_df = pd.concat(chd_pgs_crs_pxs_beta, axis = 1)
cad_chd_pgs_crs_pxs_pval_df = pd.concat(chd_pgs_crs_pxs_pval, axis = 1)

cad_chd_pgs_crs_wt_pxs_beta_df = pd.concat(chd_pgs_crs_wt_pxs_beta, axis = 1)
cad_chd_pgs_crs_wt_pxs_pval_df = pd.concat(chd_pgs_crs_wt_pxs_pval, axis = 1)

### afib

In [ ]:
!gsutil -m cp -R ${WORKSPACE_BUCKET}/Cardio_IRM/AFIB/output/regressions/eval/ AFIB/

In [ ]:
pheno = 'AFIB'

total_cvd_auroc = []
total_cvd_auprc = []
total_cvd_f1 = []
total_cvd_balanced_acc = []
ascvd_auroc = []
ascvd_auprc = []
ascvd_f1 = []
ascvd_balanced_acc = []
heart_failure_auroc = []
heart_failure_auprc = []
heart_failure_f1 = []
heart_failure_balanced_acc = []
chd_auroc = []
chd_auprc = []
chd_f1 = []
chd_balanced_acc = []
stroke_auroc = []
stroke_auprc = []
stroke_f1 = []
stroke_balanced_acc = []
c2hest_auroc = []
c2hest_auprc = []
c2hest_f1 = []
c2hest_balanced_acc = []

for iter in list(range(1, 1001)):
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    total_cvd_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    total_cvd_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    total_cvd_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    total_cvd_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    ascvd_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    ascvd_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    ascvd_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    ascvd_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    heart_failure_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    heart_failure_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    heart_failure_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    heart_failure_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    chd_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    chd_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    chd_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    chd_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    stroke_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    stroke_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    stroke_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    stroke_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.C2HEST_CRS.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.C2HEST_CRS.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.C2HEST_CRS.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.C2HEST_CRS.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    c2hest_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    c2hest_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    c2hest_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    c2hest_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))


afib_total_cvd_auroc_df = pd.concat(total_cvd_auroc, axis = 1)
afib_total_cvd_auprc_df = pd.concat(total_cvd_auprc, axis = 1)
afib_total_cvd_f1_df = pd.concat(total_cvd_f1, axis = 1)
afib_total_cvd_balanced_acc_df = pd.concat(total_cvd_balanced_acc, axis = 1)
afib_ascvd_auroc_df = pd.concat(ascvd_auroc, axis = 1)
afib_ascvd_auprc_df = pd.concat(ascvd_auprc, axis = 1)
afib_ascvd_f1_df = pd.concat(ascvd_f1, axis = 1)
afib_ascvd_balanced_acc_df = pd.concat(ascvd_balanced_acc, axis = 1)
afib_heart_failure_auroc_df = pd.concat(heart_failure_auroc, axis = 1)
afib_heart_failure_auprc_df = pd.concat(heart_failure_auprc, axis = 1)
afib_heart_failure_f1_df = pd.concat(heart_failure_f1, axis = 1)
afib_heart_failure_balanced_acc_df = pd.concat(heart_failure_balanced_acc, axis = 1)
afib_chd_auroc_df = pd.concat(chd_auroc, axis = 1)
afib_chd_auprc_df = pd.concat(chd_auprc, axis = 1)
afib_chd_f1_df = pd.concat(chd_f1, axis = 1)
afib_chd_balanced_acc_df = pd.concat(chd_balanced_acc, axis = 1)
afib_stroke_auroc_df = pd.concat(stroke_auroc, axis = 1)
afib_stroke_auprc_df = pd.concat(stroke_auprc, axis = 1)
afib_stroke_f1_df = pd.concat(stroke_f1, axis = 1)
afib_stroke_balanced_acc_df = pd.concat(stroke_balanced_acc, axis = 1)
afib_c2hest_auroc_df = pd.concat(c2hest_auroc, axis = 1)
afib_c2hest_auprc_df = pd.concat(c2hest_auprc, axis = 1)
afib_c2hest_f1_df = pd.concat(c2hest_f1, axis = 1)
afib_c2hest_balanced_acc_df = pd.concat(c2hest_balanced_acc, axis = 1)

In [ ]:
pheno = 'AFIB'

pgs_beta = []
pgs_pval = []
pxs_beta = []
pxs_pval = []
wt_pxs_beta = []
wt_pxs_pval = []
pgs_pxs_beta = []
pgs_pxs_pval = []
pgs_wt_pxs_beta = []
pgs_wt_pxs_pval = []

total_cvd_crs_beta = []
total_cvd_crs_pval = []
total_cvd_pgs_crs_beta = []
total_cvd_pgs_crs_pval = []
total_cvd_crs_pxs_beta = []
total_cvd_crs_pxs_pval = []
total_cvd_crs_wt_pxs_beta = []
total_cvd_crs_wt_pxs_pval = []
total_cvd_pgs_crs_pxs_beta = []
total_cvd_pgs_crs_pxs_pval = []
total_cvd_pgs_crs_wt_pxs_beta = []
total_cvd_pgs_crs_wt_pxs_pval = []

ascvd_crs_beta = []
ascvd_crs_pval = []
ascvd_pgs_crs_beta = []
ascvd_pgs_crs_pval = []
ascvd_crs_pxs_beta = []
ascvd_crs_pxs_pval = []
ascvd_crs_wt_pxs_beta = []
ascvd_crs_wt_pxs_pval = []
ascvd_pgs_crs_pxs_beta = []
ascvd_pgs_crs_pxs_pval = []
ascvd_pgs_crs_wt_pxs_beta = []
ascvd_pgs_crs_wt_pxs_pval = []

heart_failure_crs_beta = []
heart_failure_crs_pval = []
heart_failure_pgs_crs_beta = []
heart_failure_pgs_crs_pval = []
heart_failure_crs_pxs_beta = []
heart_failure_crs_pxs_pval = []
heart_failure_crs_wt_pxs_beta = []
heart_failure_crs_wt_pxs_pval = []
heart_failure_pgs_crs_pxs_beta = []
heart_failure_pgs_crs_pxs_pval = []
heart_failure_pgs_crs_wt_pxs_beta = []
heart_failure_pgs_crs_wt_pxs_pval = []

stroke_crs_beta = []
stroke_crs_pval = []
stroke_pgs_crs_beta = []
stroke_pgs_crs_pval = []
stroke_crs_pxs_beta = []
stroke_crs_pxs_pval = []
stroke_crs_wt_pxs_beta = []
stroke_crs_wt_pxs_pval = []
stroke_pgs_crs_pxs_beta = []
stroke_pgs_crs_pxs_pval = []
stroke_pgs_crs_wt_pxs_beta = []
stroke_pgs_crs_wt_pxs_pval = []

chd_crs_beta = []
chd_crs_pval = []
chd_pgs_crs_beta = []
chd_pgs_crs_pval = []
chd_crs_pxs_beta = []
chd_crs_pxs_pval = []
chd_crs_wt_pxs_beta = []
chd_crs_wt_pxs_pval = []
chd_pgs_crs_pxs_beta = []
chd_pgs_crs_pxs_pval = []
chd_pgs_crs_wt_pxs_beta = []
chd_pgs_crs_wt_pxs_pval = []

c2hest_crs_beta = []
c2hest_crs_pval = []
c2hest_pgs_crs_beta = []
c2hest_pgs_crs_pval = []
c2hest_crs_pxs_beta = []
c2hest_crs_pxs_pval = []
c2hest_crs_wt_pxs_beta = []
c2hest_crs_wt_pxs_pval = []
c2hest_pgs_crs_pxs_beta = []
c2hest_pgs_crs_pxs_pval = []
c2hest_pgs_crs_wt_pxs_beta = []
c2hest_pgs_crs_wt_pxs_pval = []

for iter in list(range(1, 1001)):
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.PVAL.ITER_' + str(iter) + '.txt'
    
    pgs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 1]).dropna().rename(columns = {'PGS_AFIB' : ('ITER_' + str(iter))}))
    pgs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 1]).dropna().rename(columns = {'PGS_AFIB' : ('ITER_' + str(iter))}))
    
    pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 3]).dropna().rename(columns = {'PXS_AVG' : ('ITER_' + str(iter))}))
    pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 3]).dropna().rename(columns = {'PXS_AVG' : ('ITER_' + str(iter))}))
    
    wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 4]).dropna().rename(columns = {'PXS_WEIGHTED_AVG' : ('ITER_' + str(iter))}))
    wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 4]).dropna().rename(columns = {'PXS_WEIGHTED_AVG' : ('ITER_' + str(iter))}))
    
    pgs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 8]).dropna().rename(columns = {"('PGS_AFIB', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    pgs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 8]).dropna().rename(columns = {"('PGS_AFIB', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    pgs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 11]).dropna().rename(columns = {"('PGS_AFIB', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    pgs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 11]).dropna().rename(columns = {"('PGS_AFIB', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_PREVENT_CRS_total_cvd' : ('ITER_' + str(iter))}))
    total_cvd_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_PREVENT_CRS_total_cvd' : ('ITER_' + str(iter))}))
    
    total_cvd_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_total_cvd')" : ('ITER_' + str(iter))}))
    total_cvd_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_total_cvd')" : ('ITER_' + str(iter))}))
    
    total_cvd_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.PVAL.ITER_' + str(iter) + '.txt'
    
    ascvd_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_PREVENT_CRS_ascvd' : ('ITER_' + str(iter))}))
    ascvd_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_PREVENT_CRS_ascvd' : ('ITER_' + str(iter))}))
    
    ascvd_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_ascvd')" : ('ITER_' + str(iter))}))
    ascvd_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_ascvd')" : ('ITER_' + str(iter))}))
    
    ascvd_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    ascvd_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    ascvd_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    ascvd_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    ascvd_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    ascvd_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    ascvd_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    ascvd_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))

    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.PVAL.ITER_' + str(iter) + '.txt'
    
    heart_failure_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_PREVENT_CRS_heart_failure' : ('ITER_' + str(iter))}))
    heart_failure_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_PREVENT_CRS_heart_failure' : ('ITER_' + str(iter))}))
    
    heart_failure_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_heart_failure')" : ('ITER_' + str(iter))}))
    heart_failure_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_heart_failure')" : ('ITER_' + str(iter))}))
    
    heart_failure_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    heart_failure_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    heart_failure_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    heart_failure_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.PVAL.ITER_' + str(iter) + '.txt'
    
    stroke_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_PREVENT_CRS_stroke' : ('ITER_' + str(iter))}))
    stroke_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_PREVENT_CRS_stroke' : ('ITER_' + str(iter))}))
    
    stroke_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_stroke')" : ('ITER_' + str(iter))}))
    stroke_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_stroke')" : ('ITER_' + str(iter))}))
    
    stroke_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    stroke_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    stroke_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    stroke_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    stroke_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    stroke_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    stroke_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    stroke_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.PVAL.ITER_' + str(iter) + '.txt'
    
    chd_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_PREVENT_CRS_chd' : ('ITER_' + str(iter))}))
    chd_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_PREVENT_CRS_chd' : ('ITER_' + str(iter))}))
    
    chd_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_chd')" : ('ITER_' + str(iter))}))
    chd_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_chd')" : ('ITER_' + str(iter))}))
    
    chd_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    chd_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    chd_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    chd_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    chd_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    chd_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    chd_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    chd_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    beta_filename = pheno + '/eval/' + pheno + '.C2HEST_CRS.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.C2HEST_CRS.PVAL.ITER_' + str(iter) + '.txt'
    
    c2hest_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_C2HEST_CRS' : ('ITER_' + str(iter))}))
    c2hest_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'AFIB_C2HEST_CRS' : ('ITER_' + str(iter))}))
    
    c2hest_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_C2HEST_CRS')" : ('ITER_' + str(iter))}))
    c2hest_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_C2HEST_CRS')" : ('ITER_' + str(iter))}))
    
    c2hest_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_C2HEST_CRS', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    c2hest_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('AFIB_C2HEST_CRS', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    c2hest_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_C2HEST_CRS', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    c2hest_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('AFIB_C2HEST_CRS', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    c2hest_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_C2HEST_CRS', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    c2hest_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_C2HEST_CRS', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    c2hest_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_C2HEST_CRS', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    c2hest_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_AFIB', 'AFIB_C2HEST_CRS', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
afib_pgs_beta_df = pd.concat(pgs_beta, axis = 1)
afib_pgs_pval_df = pd.concat(pgs_pval, axis = 1)

afib_pxs_beta_df = pd.concat(pxs_beta, axis = 1)
afib_pxs_pval_df = pd.concat(pxs_pval, axis = 1)

afib_wt_pxs_beta_df = pd.concat(wt_pxs_beta, axis = 1)
afib_wt_pxs_pval_df = pd.concat(wt_pxs_pval, axis = 1)

afib_pgs_pxs_beta_df = pd.concat(pgs_pxs_beta, axis = 1)
afib_pgs_pxs_pval_df = pd.concat(pgs_pxs_pval, axis = 1)

afib_pgs_wt_pxs_beta_df = pd.concat(pgs_wt_pxs_beta, axis = 1)
afib_pgs_wt_pxs_pval_df = pd.concat(pgs_wt_pxs_pval, axis = 1)

afib_total_cvd_crs_beta_df = pd.concat(total_cvd_crs_beta, axis = 1)
afib_total_cvd_crs_pval_df = pd.concat(total_cvd_crs_pval, axis = 1)

afib_total_cvd_pgs_crs_beta_df = pd.concat(total_cvd_pgs_crs_beta, axis = 1)
afib_total_cvd_pgs_crs_pval_df = pd.concat(total_cvd_pgs_crs_pval, axis = 1)

afib_total_cvd_crs_pxs_beta_df = pd.concat(total_cvd_crs_pxs_beta, axis = 1)
afib_total_cvd_crs_pxs_pval_df = pd.concat(total_cvd_crs_pxs_pval, axis = 1)

afib_total_cvd_crs_wt_pxs_beta_df = pd.concat(total_cvd_crs_wt_pxs_beta, axis = 1)
afib_total_cvd_crs_wt_pxs_pval_df = pd.concat(total_cvd_crs_wt_pxs_pval, axis = 1)

afib_total_cvd_pgs_crs_pxs_beta_df = pd.concat(total_cvd_pgs_crs_pxs_beta, axis = 1)
afib_total_cvd_pgs_crs_pxs_pval_df = pd.concat(total_cvd_pgs_crs_pxs_pval, axis = 1)

afib_total_cvd_pgs_crs_wt_pxs_beta_df = pd.concat(total_cvd_pgs_crs_wt_pxs_beta, axis = 1)
afib_total_cvd_pgs_crs_wt_pxs_pval_df = pd.concat(total_cvd_pgs_crs_wt_pxs_pval, axis = 1)

afib_ascvd_crs_beta_df = pd.concat(ascvd_crs_beta, axis = 1)
afib_ascvd_crs_pval_df = pd.concat(ascvd_crs_pval, axis = 1)

afib_ascvd_pgs_crs_beta_df = pd.concat(ascvd_pgs_crs_beta, axis = 1)
afib_ascvd_pgs_crs_pval_df = pd.concat(ascvd_pgs_crs_pval, axis = 1)

afib_ascvd_crs_pxs_beta_df = pd.concat(ascvd_crs_pxs_beta, axis = 1)
afib_ascvd_crs_pxs_pval_df = pd.concat(ascvd_crs_pxs_pval, axis = 1)

afib_ascvd_crs_wt_pxs_beta_df = pd.concat(ascvd_crs_wt_pxs_beta, axis = 1)
afib_ascvd_crs_wt_pxs_pval_df = pd.concat(ascvd_crs_wt_pxs_pval, axis = 1)

afib_ascvd_pgs_crs_pxs_beta_df = pd.concat(ascvd_pgs_crs_pxs_beta, axis = 1)
afib_ascvd_pgs_crs_pxs_pval_df = pd.concat(ascvd_pgs_crs_pxs_pval, axis = 1)

afib_ascvd_pgs_crs_wt_pxs_beta_df = pd.concat(ascvd_pgs_crs_wt_pxs_beta, axis = 1)
afib_ascvd_pgs_crs_wt_pxs_pval_df = pd.concat(ascvd_pgs_crs_wt_pxs_pval, axis = 1)

afib_heart_failure_crs_beta_df = pd.concat(heart_failure_crs_beta, axis = 1)
afib_heart_failure_crs_pval_df = pd.concat(heart_failure_crs_pval, axis = 1)

afib_heart_failure_pgs_crs_beta_df = pd.concat(heart_failure_pgs_crs_beta, axis = 1)
afib_heart_failure_pgs_crs_pval_df = pd.concat(heart_failure_pgs_crs_pval, axis = 1)

afib_heart_failure_crs_pxs_beta_df = pd.concat(heart_failure_crs_pxs_beta, axis = 1)
afib_heart_failure_crs_pxs_pval_df = pd.concat(heart_failure_crs_pxs_pval, axis = 1)

afib_heart_failure_crs_wt_pxs_beta_df = pd.concat(heart_failure_crs_wt_pxs_beta, axis = 1)
afib_heart_failure_crs_wt_pxs_pval_df = pd.concat(heart_failure_crs_wt_pxs_pval, axis = 1)

afib_heart_failure_pgs_crs_pxs_beta_df = pd.concat(heart_failure_pgs_crs_pxs_beta, axis = 1)
afib_heart_failure_pgs_crs_pxs_pval_df = pd.concat(heart_failure_pgs_crs_pxs_pval, axis = 1)

afib_heart_failure_pgs_crs_wt_pxs_beta_df = pd.concat(heart_failure_pgs_crs_wt_pxs_beta, axis = 1)
afib_heart_failure_pgs_crs_wt_pxs_pval_df = pd.concat(heart_failure_pgs_crs_wt_pxs_pval, axis = 1)

afib_stroke_crs_beta_df = pd.concat(stroke_crs_beta, axis = 1)
afib_stroke_crs_pval_df = pd.concat(stroke_crs_pval, axis = 1)

afib_stroke_pgs_crs_beta_df = pd.concat(stroke_pgs_crs_beta, axis = 1)
afib_stroke_pgs_crs_pval_df = pd.concat(stroke_pgs_crs_pval, axis = 1)

afib_stroke_crs_pxs_beta_df = pd.concat(stroke_crs_pxs_beta, axis = 1)
afib_stroke_crs_pxs_pval_df = pd.concat(stroke_crs_pxs_pval, axis = 1)

afib_stroke_crs_wt_pxs_beta_df = pd.concat(stroke_crs_wt_pxs_beta, axis = 1)
afib_stroke_crs_wt_pxs_pval_df = pd.concat(stroke_crs_wt_pxs_pval, axis = 1)

afib_stroke_pgs_crs_pxs_beta_df = pd.concat(stroke_pgs_crs_pxs_beta, axis = 1)
afib_stroke_pgs_crs_pxs_pval_df = pd.concat(stroke_pgs_crs_pxs_pval, axis = 1)

afib_stroke_pgs_crs_wt_pxs_beta_df = pd.concat(stroke_pgs_crs_wt_pxs_beta, axis = 1)
afib_stroke_pgs_crs_wt_pxs_pval_df = pd.concat(stroke_pgs_crs_wt_pxs_pval, axis = 1)

afib_chd_crs_beta_df = pd.concat(chd_crs_beta, axis = 1)
afib_chd_crs_pval_df = pd.concat(chd_crs_pval, axis = 1)

afib_chd_pgs_crs_beta_df = pd.concat(chd_pgs_crs_beta, axis = 1)
afib_chd_pgs_crs_pval_df = pd.concat(chd_pgs_crs_pval, axis = 1)

afib_chd_crs_pxs_beta_df = pd.concat(chd_crs_pxs_beta, axis = 1)
afib_chd_crs_pxs_pval_df = pd.concat(chd_crs_pxs_pval, axis = 1)

afib_chd_crs_wt_pxs_beta_df = pd.concat(chd_crs_wt_pxs_beta, axis = 1)
afib_chd_crs_wt_pxs_pval_df = pd.concat(chd_crs_wt_pxs_pval, axis = 1)

afib_chd_pgs_crs_pxs_beta_df = pd.concat(chd_pgs_crs_pxs_beta, axis = 1)
afib_chd_pgs_crs_pxs_pval_df = pd.concat(chd_pgs_crs_pxs_pval, axis = 1)

afib_chd_pgs_crs_wt_pxs_beta_df = pd.concat(chd_pgs_crs_wt_pxs_beta, axis = 1)
afib_chd_pgs_crs_wt_pxs_pval_df = pd.concat(chd_pgs_crs_wt_pxs_pval, axis = 1)

afib_c2hest_crs_beta_df = pd.concat(c2hest_crs_beta, axis = 1)
afib_c2hest_crs_pval_df = pd.concat(c2hest_crs_pval, axis = 1)

afib_c2hest_pgs_crs_beta_df = pd.concat(c2hest_pgs_crs_beta, axis = 1)
afib_c2hest_pgs_crs_pval_df = pd.concat(c2hest_pgs_crs_pval, axis = 1)

afib_c2hest_crs_pxs_beta_df = pd.concat(c2hest_crs_pxs_beta, axis = 1)
afib_c2hest_crs_pxs_pval_df = pd.concat(c2hest_crs_pxs_pval, axis = 1)

afib_c2hest_crs_wt_pxs_beta_df = pd.concat(c2hest_crs_wt_pxs_beta, axis = 1)
afib_c2hest_crs_wt_pxs_pval_df = pd.concat(c2hest_crs_wt_pxs_pval, axis = 1)

afib_c2hest_pgs_crs_pxs_beta_df = pd.concat(c2hest_pgs_crs_pxs_beta, axis = 1)
afib_c2hest_pgs_crs_pxs_pval_df = pd.concat(c2hest_pgs_crs_pxs_pval, axis = 1)

afib_c2hest_pgs_crs_wt_pxs_beta_df = pd.concat(c2hest_pgs_crs_wt_pxs_beta, axis = 1)
afib_c2hest_pgs_crs_wt_pxs_pval_df = pd.concat(c2hest_pgs_crs_wt_pxs_pval, axis = 1)

### HF

In [ ]:
!gsutil -m cp -R ${WORKSPACE_BUCKET}/Cardio_IRM/HF/output/regressions/eval/ HF/

In [ ]:
pheno = 'HF'

total_cvd_auroc = []
total_cvd_auprc = []
total_cvd_f1 = []
total_cvd_balanced_acc = []
ascvd_auroc = []
ascvd_auprc = []
ascvd_f1 = []
ascvd_balanced_acc = []
heart_failure_auroc = []
heart_failure_auprc = []
heart_failure_f1 = []
heart_failure_balanced_acc = []
chd_auroc = []
chd_auprc = []
chd_f1 = []
chd_balanced_acc = []
stroke_auroc = []
stroke_auprc = []
stroke_f1 = []
stroke_balanced_acc = []

for iter in list(range(1, 1001)):
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    total_cvd_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    total_cvd_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    total_cvd_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    total_cvd_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    ascvd_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    ascvd_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    ascvd_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    ascvd_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    heart_failure_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    heart_failure_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    heart_failure_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    heart_failure_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    chd_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    chd_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    chd_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    chd_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))
    
    auroc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.AUROC.ITER_' + str(iter) + '.txt'
    auprc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.AUPRC.ITER_' + str(iter) + '.txt'
    f1_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.F1_SCORE.ITER_' + str(iter) + '.txt'
    balanced_acc_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.BALANCED_ACCURACY.ITER_' + str(iter) + '.txt'
    
    stroke_auroc.append(pd.read_csv(auroc_filename, sep = '\t', index_col = 0))
    stroke_auprc.append(pd.read_csv(auprc_filename, sep = '\t', index_col = 0))
    stroke_f1.append(pd.read_csv(f1_filename, sep = '\t', index_col = 0))
    stroke_balanced_acc.append(pd.read_csv(balanced_acc_filename, sep = '\t', index_col = 0))


hf_total_cvd_auroc_df = pd.concat(total_cvd_auroc, axis = 1)
hf_total_cvd_auprc_df = pd.concat(total_cvd_auprc, axis = 1)
hf_total_cvd_f1_df = pd.concat(total_cvd_f1, axis = 1)
hf_total_cvd_balanced_acc_df = pd.concat(total_cvd_balanced_acc, axis = 1)
hf_ascvd_auroc_df = pd.concat(ascvd_auroc, axis = 1)
hf_ascvd_auprc_df = pd.concat(ascvd_auprc, axis = 1)
hf_ascvd_f1_df = pd.concat(ascvd_f1, axis = 1)
hf_ascvd_balanced_acc_df = pd.concat(ascvd_balanced_acc, axis = 1)
hf_heart_failure_auroc_df = pd.concat(heart_failure_auroc, axis = 1)
hf_heart_failure_auprc_df = pd.concat(heart_failure_auprc, axis = 1)
hf_heart_failure_f1_df = pd.concat(heart_failure_f1, axis = 1)
hf_heart_failure_balanced_acc_df = pd.concat(heart_failure_balanced_acc, axis = 1)
hf_chd_auroc_df = pd.concat(chd_auroc, axis = 1)
hf_chd_auprc_df = pd.concat(chd_auprc, axis = 1)
hf_chd_f1_df = pd.concat(chd_f1, axis = 1)
hf_chd_balanced_acc_df = pd.concat(chd_balanced_acc, axis = 1)
hf_stroke_auroc_df = pd.concat(stroke_auroc, axis = 1)
hf_stroke_auprc_df = pd.concat(stroke_auprc, axis = 1)
hf_stroke_f1_df = pd.concat(stroke_f1, axis = 1)
hf_stroke_balanced_acc_df = pd.concat(stroke_balanced_acc, axis = 1)

In [ ]:
pheno = 'HF'

pgs_beta = []
pgs_pval = []
pxs_beta = []
pxs_pval = []
wt_pxs_beta = []
wt_pxs_pval = []
pgs_pxs_beta = []
pgs_pxs_pval = []
pgs_wt_pxs_beta = []
pgs_wt_pxs_pval = []

total_cvd_crs_beta = []
total_cvd_crs_pval = []
total_cvd_pgs_crs_beta = []
total_cvd_pgs_crs_pval = []
total_cvd_crs_pxs_beta = []
total_cvd_crs_pxs_pval = []
total_cvd_crs_wt_pxs_beta = []
total_cvd_crs_wt_pxs_pval = []
total_cvd_pgs_crs_pxs_beta = []
total_cvd_pgs_crs_pxs_pval = []
total_cvd_pgs_crs_wt_pxs_beta = []
total_cvd_pgs_crs_wt_pxs_pval = []

ascvd_crs_beta = []
ascvd_crs_pval = []
ascvd_pgs_crs_beta = []
ascvd_pgs_crs_pval = []
ascvd_crs_pxs_beta = []
ascvd_crs_pxs_pval = []
ascvd_crs_wt_pxs_beta = []
ascvd_crs_wt_pxs_pval = []
ascvd_pgs_crs_pxs_beta = []
ascvd_pgs_crs_pxs_pval = []
ascvd_pgs_crs_wt_pxs_beta = []
ascvd_pgs_crs_wt_pxs_pval = []

heart_failure_crs_beta = []
heart_failure_crs_pval = []
heart_failure_pgs_crs_beta = []
heart_failure_pgs_crs_pval = []
heart_failure_crs_pxs_beta = []
heart_failure_crs_pxs_pval = []
heart_failure_crs_wt_pxs_beta = []
heart_failure_crs_wt_pxs_pval = []
heart_failure_pgs_crs_pxs_beta = []
heart_failure_pgs_crs_pxs_pval = []
heart_failure_pgs_crs_wt_pxs_beta = []
heart_failure_pgs_crs_wt_pxs_pval = []

stroke_crs_beta = []
stroke_crs_pval = []
stroke_pgs_crs_beta = []
stroke_pgs_crs_pval = []
stroke_crs_pxs_beta = []
stroke_crs_pxs_pval = []
stroke_crs_wt_pxs_beta = []
stroke_crs_wt_pxs_pval = []
stroke_pgs_crs_pxs_beta = []
stroke_pgs_crs_pxs_pval = []
stroke_pgs_crs_wt_pxs_beta = []
stroke_pgs_crs_wt_pxs_pval = []

chd_crs_beta = []
chd_crs_pval = []
chd_pgs_crs_beta = []
chd_pgs_crs_pval = []
chd_crs_pxs_beta = []
chd_crs_pxs_pval = []
chd_crs_wt_pxs_beta = []
chd_crs_wt_pxs_pval = []
chd_pgs_crs_pxs_beta = []
chd_pgs_crs_pxs_pval = []
chd_pgs_crs_wt_pxs_beta = []
chd_pgs_crs_wt_pxs_pval = []

for iter in list(range(1, 1001)):
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_total_cvd.PVAL.ITER_' + str(iter) + '.txt'
    
    pgs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 1]).dropna().rename(columns = {'PGS_HF' : ('ITER_' + str(iter))}))
    pgs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 1]).dropna().rename(columns = {'PGS_HF' : ('ITER_' + str(iter))}))
    
    pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 3]).dropna().rename(columns = {'PXS_AVG' : ('ITER_' + str(iter))}))
    pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 3]).dropna().rename(columns = {'PXS_AVG' : ('ITER_' + str(iter))}))
    
    wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 4]).dropna().rename(columns = {'PXS_WEIGHTED_AVG' : ('ITER_' + str(iter))}))
    wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 4]).dropna().rename(columns = {'PXS_WEIGHTED_AVG' : ('ITER_' + str(iter))}))
    
    pgs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 8]).dropna().rename(columns = {"('PGS_HF', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    pgs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 8]).dropna().rename(columns = {"('PGS_HF', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    pgs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 11]).dropna().rename(columns = {"('PGS_HF', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    pgs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 11]).dropna().rename(columns = {"('PGS_HF', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'HF_PREVENT_CRS_total_cvd' : ('ITER_' + str(iter))}))
    total_cvd_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'HF_PREVENT_CRS_total_cvd' : ('ITER_' + str(iter))}))
    
    total_cvd_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_total_cvd')" : ('ITER_' + str(iter))}))
    total_cvd_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_total_cvd')" : ('ITER_' + str(iter))}))
    
    total_cvd_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('HF_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('HF_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('HF_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('HF_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_total_cvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    total_cvd_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    total_cvd_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_total_cvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_ascvd.PVAL.ITER_' + str(iter) + '.txt'
    
    ascvd_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'HF_PREVENT_CRS_ascvd' : ('ITER_' + str(iter))}))
    ascvd_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'HF_PREVENT_CRS_ascvd' : ('ITER_' + str(iter))}))
    
    ascvd_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_ascvd')" : ('ITER_' + str(iter))}))
    ascvd_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_ascvd')" : ('ITER_' + str(iter))}))
    
    ascvd_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('HF_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    ascvd_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('HF_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    ascvd_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('HF_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    ascvd_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('HF_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    ascvd_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    ascvd_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_ascvd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    ascvd_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    ascvd_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_ascvd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))

    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_heart_failure.PVAL.ITER_' + str(iter) + '.txt'
    
    heart_failure_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'HF_PREVENT_CRS_heart_failure' : ('ITER_' + str(iter))}))
    heart_failure_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'HF_PREVENT_CRS_heart_failure' : ('ITER_' + str(iter))}))
    
    heart_failure_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_heart_failure')" : ('ITER_' + str(iter))}))
    heart_failure_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_heart_failure')" : ('ITER_' + str(iter))}))
    
    heart_failure_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('HF_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('HF_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    heart_failure_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('HF_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('HF_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    heart_failure_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_heart_failure', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    heart_failure_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    heart_failure_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_heart_failure', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_stroke.PVAL.ITER_' + str(iter) + '.txt'
    
    stroke_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'HF_PREVENT_CRS_stroke' : ('ITER_' + str(iter))}))
    stroke_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'HF_PREVENT_CRS_stroke' : ('ITER_' + str(iter))}))
    
    stroke_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_stroke')" : ('ITER_' + str(iter))}))
    stroke_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_stroke')" : ('ITER_' + str(iter))}))
    
    stroke_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('HF_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    stroke_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('HF_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    stroke_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('HF_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    stroke_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('HF_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    stroke_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    stroke_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_stroke', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    stroke_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    stroke_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_stroke', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    beta_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.BETA.ITER_' + str(iter) + '.txt'
    pval_filename = pheno + '/eval/' + pheno + '.PREVENT_CRS_chd.PVAL.ITER_' + str(iter) + '.txt'
    
    chd_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'HF_PREVENT_CRS_chd' : ('ITER_' + str(iter))}))
    chd_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 2]).dropna().rename(columns = {'HF_PREVENT_CRS_chd' : ('ITER_' + str(iter))}))
    
    chd_pgs_crs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_chd')" : ('ITER_' + str(iter))}))
    chd_pgs_crs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 7]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_chd')" : ('ITER_' + str(iter))}))
    
    chd_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('HF_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    chd_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 6]).dropna().rename(columns = {"('HF_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    chd_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('HF_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    chd_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 10]).dropna().rename(columns = {"('HF_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
    chd_pgs_crs_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    chd_pgs_crs_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 5]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_chd', 'PXS_AVG')" : ('ITER_' + str(iter))}))
    
    chd_pgs_crs_wt_pxs_beta.append(pd.read_csv(beta_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    chd_pgs_crs_wt_pxs_pval.append(pd.read_csv(pval_filename, sep = '\t', index_col = 0, usecols = [0, 9]).dropna().rename(columns = {"('PGS_HF', 'HF_PREVENT_CRS_chd', 'PXS_WEIGHTED_AVG')" : ('ITER_' + str(iter))}))
    
hf_pgs_beta_df = pd.concat(pgs_beta, axis = 1)
hf_pgs_pval_df = pd.concat(pgs_pval, axis = 1)

hf_pxs_beta_df = pd.concat(pxs_beta, axis = 1)
hf_pxs_pval_df = pd.concat(pxs_pval, axis = 1)

hf_wt_pxs_beta_df = pd.concat(wt_pxs_beta, axis = 1)
hf_wt_pxs_pval_df = pd.concat(wt_pxs_pval, axis = 1)

hf_pgs_pxs_beta_df = pd.concat(pgs_pxs_beta, axis = 1)
hf_pgs_pxs_pval_df = pd.concat(pgs_pxs_pval, axis = 1)

hf_pgs_wt_pxs_beta_df = pd.concat(pgs_wt_pxs_beta, axis = 1)
hf_pgs_wt_pxs_pval_df = pd.concat(pgs_wt_pxs_pval, axis = 1)

hf_total_cvd_crs_beta_df = pd.concat(total_cvd_crs_beta, axis = 1)
hf_total_cvd_crs_pval_df = pd.concat(total_cvd_crs_pval, axis = 1)

hf_total_cvd_pgs_crs_beta_df = pd.concat(total_cvd_pgs_crs_beta, axis = 1)
hf_total_cvd_pgs_crs_pval_df = pd.concat(total_cvd_pgs_crs_pval, axis = 1)

hf_total_cvd_crs_pxs_beta_df = pd.concat(total_cvd_crs_pxs_beta, axis = 1)
hf_total_cvd_crs_pxs_pval_df = pd.concat(total_cvd_crs_pxs_pval, axis = 1)

hf_total_cvd_crs_wt_pxs_beta_df = pd.concat(total_cvd_crs_wt_pxs_beta, axis = 1)
hf_total_cvd_crs_wt_pxs_pval_df = pd.concat(total_cvd_crs_wt_pxs_pval, axis = 1)

hf_total_cvd_pgs_crs_pxs_beta_df = pd.concat(total_cvd_pgs_crs_pxs_beta, axis = 1)
hf_total_cvd_pgs_crs_pxs_pval_df = pd.concat(total_cvd_pgs_crs_pxs_pval, axis = 1)

hf_total_cvd_pgs_crs_wt_pxs_beta_df = pd.concat(total_cvd_pgs_crs_wt_pxs_beta, axis = 1)
hf_total_cvd_pgs_crs_wt_pxs_pval_df = pd.concat(total_cvd_pgs_crs_wt_pxs_pval, axis = 1)

hf_ascvd_crs_beta_df = pd.concat(ascvd_crs_beta, axis = 1)
hf_ascvd_crs_pval_df = pd.concat(ascvd_crs_pval, axis = 1)

hf_ascvd_pgs_crs_beta_df = pd.concat(ascvd_pgs_crs_beta, axis = 1)
hf_ascvd_pgs_crs_pval_df = pd.concat(ascvd_pgs_crs_pval, axis = 1)

hf_ascvd_crs_pxs_beta_df = pd.concat(ascvd_crs_pxs_beta, axis = 1)
hf_ascvd_crs_pxs_pval_df = pd.concat(ascvd_crs_pxs_pval, axis = 1)

hf_ascvd_crs_wt_pxs_beta_df = pd.concat(ascvd_crs_wt_pxs_beta, axis = 1)
hf_ascvd_crs_wt_pxs_pval_df = pd.concat(ascvd_crs_wt_pxs_pval, axis = 1)

hf_ascvd_pgs_crs_pxs_beta_df = pd.concat(ascvd_pgs_crs_pxs_beta, axis = 1)
hf_ascvd_pgs_crs_pxs_pval_df = pd.concat(ascvd_pgs_crs_pxs_pval, axis = 1)

hf_ascvd_pgs_crs_wt_pxs_beta_df = pd.concat(ascvd_pgs_crs_wt_pxs_beta, axis = 1)
hf_ascvd_pgs_crs_wt_pxs_pval_df = pd.concat(ascvd_pgs_crs_wt_pxs_pval, axis = 1)

hf_heart_failure_crs_beta_df = pd.concat(heart_failure_crs_beta, axis = 1)
hf_heart_failure_crs_pval_df = pd.concat(heart_failure_crs_pval, axis = 1)

hf_heart_failure_pgs_crs_beta_df = pd.concat(heart_failure_pgs_crs_beta, axis = 1)
hf_heart_failure_pgs_crs_pval_df = pd.concat(heart_failure_pgs_crs_pval, axis = 1)

hf_heart_failure_crs_pxs_beta_df = pd.concat(heart_failure_crs_pxs_beta, axis = 1)
hf_heart_failure_crs_pxs_pval_df = pd.concat(heart_failure_crs_pxs_pval, axis = 1)

hf_heart_failure_crs_wt_pxs_beta_df = pd.concat(heart_failure_crs_wt_pxs_beta, axis = 1)
hf_heart_failure_crs_wt_pxs_pval_df = pd.concat(heart_failure_crs_wt_pxs_pval, axis = 1)

hf_heart_failure_pgs_crs_pxs_beta_df = pd.concat(heart_failure_pgs_crs_pxs_beta, axis = 1)
hf_heart_failure_pgs_crs_pxs_pval_df = pd.concat(heart_failure_pgs_crs_pxs_pval, axis = 1)

hf_heart_failure_pgs_crs_wt_pxs_beta_df = pd.concat(heart_failure_pgs_crs_wt_pxs_beta, axis = 1)
hf_heart_failure_pgs_crs_wt_pxs_pval_df = pd.concat(heart_failure_pgs_crs_wt_pxs_pval, axis = 1)

hf_stroke_crs_beta_df = pd.concat(stroke_crs_beta, axis = 1)
hf_stroke_crs_pval_df = pd.concat(stroke_crs_pval, axis = 1)

hf_stroke_pgs_crs_beta_df = pd.concat(stroke_pgs_crs_beta, axis = 1)
hf_stroke_pgs_crs_pval_df = pd.concat(stroke_pgs_crs_pval, axis = 1)

hf_stroke_crs_pxs_beta_df = pd.concat(stroke_crs_pxs_beta, axis = 1)
hf_stroke_crs_pxs_pval_df = pd.concat(stroke_crs_pxs_pval, axis = 1)

hf_stroke_crs_wt_pxs_beta_df = pd.concat(stroke_crs_wt_pxs_beta, axis = 1)
hf_stroke_crs_wt_pxs_pval_df = pd.concat(stroke_crs_wt_pxs_pval, axis = 1)

hf_stroke_pgs_crs_pxs_beta_df = pd.concat(stroke_pgs_crs_pxs_beta, axis = 1)
hf_stroke_pgs_crs_pxs_pval_df = pd.concat(stroke_pgs_crs_pxs_pval, axis = 1)

hf_stroke_pgs_crs_wt_pxs_beta_df = pd.concat(stroke_pgs_crs_wt_pxs_beta, axis = 1)
hf_stroke_pgs_crs_wt_pxs_pval_df = pd.concat(stroke_pgs_crs_wt_pxs_pval, axis = 1)

hf_chd_crs_beta_df = pd.concat(chd_crs_beta, axis = 1)
hf_chd_crs_pval_df = pd.concat(chd_crs_pval, axis = 1)

hf_chd_pgs_crs_beta_df = pd.concat(chd_pgs_crs_beta, axis = 1)
hf_chd_pgs_crs_pval_df = pd.concat(chd_pgs_crs_pval, axis = 1)

hf_chd_crs_pxs_beta_df = pd.concat(chd_crs_pxs_beta, axis = 1)
hf_chd_crs_pxs_pval_df = pd.concat(chd_crs_pxs_pval, axis = 1)

hf_chd_crs_wt_pxs_beta_df = pd.concat(chd_crs_wt_pxs_beta, axis = 1)
hf_chd_crs_wt_pxs_pval_df = pd.concat(chd_crs_wt_pxs_pval, axis = 1)

hf_chd_pgs_crs_pxs_beta_df = pd.concat(chd_pgs_crs_pxs_beta, axis = 1)
hf_chd_pgs_crs_pxs_pval_df = pd.concat(chd_pgs_crs_pxs_pval, axis = 1)

hf_chd_pgs_crs_wt_pxs_beta_df = pd.concat(chd_pgs_crs_wt_pxs_beta, axis = 1)
hf_chd_pgs_crs_wt_pxs_pval_df = pd.concat(chd_pgs_crs_wt_pxs_pval, axis = 1)

## compute mean metrics

### CAD

In [ ]:
cad_total_cvd_auroc_df['AUROC'] = cad_total_cvd_auroc_df.mean(axis = 1)
cad_ascvd_auroc_df['AUROC'] = cad_ascvd_auroc_df.mean(axis = 1)
cad_heart_failure_auroc_df['AUROC'] = cad_heart_failure_auroc_df.mean(axis = 1)
cad_chd_auroc_df['AUROC'] = cad_chd_auroc_df.mean(axis = 1)
cad_stroke_auroc_df['AUROC'] = cad_stroke_auroc_df.mean(axis = 1)

In [ ]:
cad_total_cvd_auprc_df['AUPRC'] = cad_total_cvd_auprc_df.mean(axis = 1)
cad_ascvd_auprc_df['AUPRC'] = cad_ascvd_auprc_df.mean(axis = 1)
cad_heart_failure_auprc_df['AUPRC'] = cad_heart_failure_auprc_df.mean(axis = 1)
cad_chd_auprc_df['AUPRC'] = cad_chd_auprc_df.mean(axis = 1)
cad_stroke_auprc_df['AUPRC'] = cad_stroke_auprc_df.mean(axis = 1)

In [ ]:
cad_total_cvd_f1_df['F1_SCORE'] = cad_total_cvd_f1_df.mean(axis = 1)
cad_ascvd_f1_df['F1_SCORE'] = cad_ascvd_f1_df.mean(axis = 1)
cad_heart_failure_f1_df['F1_SCORE'] = cad_heart_failure_f1_df.mean(axis = 1)
cad_chd_f1_df['F1_SCORE'] = cad_chd_f1_df.mean(axis = 1)
cad_stroke_f1_df['F1_SCORE'] = cad_stroke_f1_df.mean(axis = 1)

In [ ]:
cad_total_cvd_balanced_acc_df['BALANCED_ACCURACY'] = cad_total_cvd_balanced_acc_df.mean(axis = 1)
cad_ascvd_balanced_acc_df['BALANCED_ACCURACY'] = cad_ascvd_balanced_acc_df.mean(axis = 1)
cad_heart_failure_balanced_acc_df['BALANCED_ACCURACY'] = cad_heart_failure_balanced_acc_df.mean(axis = 1)
cad_chd_balanced_acc_df['BALANCED_ACCURACY'] = cad_chd_balanced_acc_df.mean(axis = 1)
cad_stroke_balanced_acc_df['BALANCED_ACCURACY'] = cad_stroke_balanced_acc_df.mean(axis = 1)

In [ ]:
cad_pgs_beta_df['PGS'] = cad_pgs_beta_df.mean(axis = 1)
cad_pgs_pval_df['PGS'] = cad_pgs_pval_df.mean(axis = 1)

In [ ]:
cad_pxs_beta_df['PXS_AVG'] = cad_pxs_beta_df.mean(axis = 1)
cad_pxs_pval_df['PXS_AVG'] = cad_pxs_pval_df.mean(axis = 1)

In [ ]:
cad_wt_pxs_beta_df['PXS_WEIGHTED_AVG'] = cad_wt_pxs_beta_df.mean(axis = 1)
cad_wt_pxs_pval_df['PXS_WEIGHTED_AVG'] = cad_wt_pxs_pval_df.mean(axis = 1)

In [ ]:
cad_pgs_pxs_beta_df['PGS_PXS_AVG'] = cad_pgs_pxs_beta_df.mean(axis = 1)
cad_pgs_pxs_pval_df['PGS_PXS_AVG'] = cad_pgs_pxs_pval_df.mean(axis = 1)

In [ ]:
cad_pgs_wt_pxs_beta_df['PGS_PXS_WEIGHTED_AVG'] = cad_pgs_wt_pxs_beta_df.mean(axis = 1)
cad_pgs_wt_pxs_pval_df['PGS_PXS_WEIGHTED_AVG'] = cad_pgs_wt_pxs_pval_df.mean(axis = 1)

In [ ]:
cad_total_cvd_crs_beta_df['CRS'] = cad_total_cvd_crs_beta_df.mean(axis = 1)
cad_ascvd_crs_beta_df['CRS'] = cad_ascvd_crs_beta_df.mean(axis = 1)
cad_heart_failure_crs_beta_df['CRS'] = cad_heart_failure_crs_beta_df.mean(axis = 1)
cad_chd_crs_beta_df['CRS'] = cad_chd_crs_beta_df.mean(axis = 1)
cad_stroke_crs_beta_df['CRS'] = cad_stroke_crs_beta_df.mean(axis = 1)

cad_total_cvd_crs_pval_df['CRS'] = cad_total_cvd_crs_pval_df.mean(axis = 1)
cad_ascvd_crs_pval_df['CRS'] = cad_ascvd_crs_pval_df.mean(axis = 1)
cad_heart_failure_crs_pval_df['CRS'] = cad_heart_failure_crs_pval_df.mean(axis = 1)
cad_chd_crs_pval_df['CRS'] = cad_chd_crs_pval_df.mean(axis = 1)
cad_stroke_crs_pval_df['CRS'] = cad_stroke_crs_pval_df.mean(axis = 1)

In [ ]:
cad_total_cvd_pgs_crs_beta_df['PGS_CRS'] = cad_total_cvd_pgs_crs_beta_df.mean(axis = 1)
cad_ascvd_pgs_crs_beta_df['PGS_CRS'] = cad_ascvd_pgs_crs_beta_df.mean(axis = 1)
cad_heart_failure_pgs_crs_beta_df['PGS_CRS'] = cad_heart_failure_pgs_crs_beta_df.mean(axis = 1)
cad_chd_pgs_crs_beta_df['PGS_CRS'] = cad_chd_pgs_crs_beta_df.mean(axis = 1)
cad_stroke_pgs_crs_beta_df['PGS_CRS'] = cad_stroke_pgs_crs_beta_df.mean(axis = 1)

cad_total_cvd_pgs_crs_pval_df['PGS_CRS'] = cad_total_cvd_pgs_crs_pval_df.mean(axis = 1)
cad_ascvd_pgs_crs_pval_df['PGS_CRS'] = cad_ascvd_pgs_crs_pval_df.mean(axis = 1)
cad_heart_failure_pgs_crs_pval_df['PGS_CRS'] = cad_heart_failure_pgs_crs_pval_df.mean(axis = 1)
cad_chd_pgs_crs_pval_df['PGS_CRS'] = cad_chd_pgs_crs_pval_df.mean(axis = 1)
cad_stroke_pgs_crs_pval_df['PGS_CRS'] = cad_stroke_pgs_crs_pval_df.mean(axis = 1)

In [ ]:
cad_total_cvd_crs_pxs_beta_df['CRS_PXS_AVG'] = cad_total_cvd_crs_pxs_beta_df.mean(axis = 1)
cad_ascvd_crs_pxs_beta_df['CRS_PXS_AVG'] = cad_ascvd_crs_pxs_beta_df.mean(axis = 1)
cad_heart_failure_crs_pxs_beta_df['CRS_PXS_AVG'] = cad_heart_failure_crs_pxs_beta_df.mean(axis = 1)
cad_chd_crs_pxs_beta_df['CRS_PXS_AVG'] = cad_chd_crs_pxs_beta_df.mean(axis = 1)
cad_stroke_crs_pxs_beta_df['CRS_PXS_AVG'] = cad_stroke_crs_pxs_beta_df.mean(axis = 1)

cad_total_cvd_crs_pxs_pval_df['CRS_PXS_AVG'] = cad_total_cvd_crs_pxs_pval_df.mean(axis = 1)
cad_ascvd_crs_pxs_pval_df['CRS_PXS_AVG'] = cad_ascvd_crs_pxs_pval_df.mean(axis = 1)
cad_heart_failure_crs_pxs_pval_df['CRS_PXS_AVG'] = cad_heart_failure_crs_pxs_pval_df.mean(axis = 1)
cad_chd_crs_pxs_pval_df['CRS_PXS_AVG'] = cad_chd_crs_pxs_pval_df.mean(axis = 1)
cad_stroke_crs_pxs_pval_df['CRS_PXS_AVG'] = cad_stroke_crs_pxs_pval_df.mean(axis = 1)

In [ ]:
cad_total_cvd_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = cad_total_cvd_crs_wt_pxs_beta_df.mean(axis = 1)
cad_ascvd_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = cad_ascvd_crs_wt_pxs_beta_df.mean(axis = 1)
cad_heart_failure_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = cad_heart_failure_crs_wt_pxs_beta_df.mean(axis = 1)
cad_chd_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = cad_chd_crs_wt_pxs_beta_df.mean(axis = 1)
cad_stroke_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = cad_stroke_crs_wt_pxs_beta_df.mean(axis = 1)

cad_total_cvd_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = cad_total_cvd_crs_wt_pxs_pval_df.mean(axis = 1)
cad_ascvd_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = cad_ascvd_crs_wt_pxs_pval_df.mean(axis = 1)
cad_heart_failure_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = cad_heart_failure_crs_wt_pxs_pval_df.mean(axis = 1)
cad_chd_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = cad_chd_crs_wt_pxs_pval_df.mean(axis = 1)
cad_stroke_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = cad_stroke_crs_wt_pxs_pval_df.mean(axis = 1)

In [ ]:
cad_total_cvd_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = cad_total_cvd_pgs_crs_pxs_beta_df.mean(axis = 1)
cad_ascvd_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = cad_ascvd_pgs_crs_pxs_beta_df.mean(axis = 1)
cad_heart_failure_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = cad_heart_failure_pgs_crs_pxs_beta_df.mean(axis = 1)
cad_chd_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = cad_chd_pgs_crs_pxs_beta_df.mean(axis = 1)
cad_stroke_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = cad_stroke_pgs_crs_pxs_beta_df.mean(axis = 1)

cad_total_cvd_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = cad_total_cvd_pgs_crs_pxs_pval_df.mean(axis = 1)
cad_ascvd_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = cad_ascvd_pgs_crs_pxs_pval_df.mean(axis = 1)
cad_heart_failure_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = cad_heart_failure_pgs_crs_pxs_pval_df.mean(axis = 1)
cad_chd_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = cad_chd_pgs_crs_pxs_pval_df.mean(axis = 1)
cad_stroke_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = cad_stroke_pgs_crs_pxs_pval_df.mean(axis = 1)

In [ ]:
cad_total_cvd_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = cad_total_cvd_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
cad_ascvd_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = cad_ascvd_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
cad_heart_failure_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = cad_heart_failure_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
cad_chd_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = cad_chd_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
cad_stroke_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = cad_stroke_pgs_crs_wt_pxs_beta_df.mean(axis = 1)

cad_total_cvd_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = cad_total_cvd_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
cad_ascvd_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = cad_ascvd_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
cad_heart_failure_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = cad_heart_failure_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
cad_chd_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = cad_chd_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
cad_stroke_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = cad_stroke_pgs_crs_wt_pxs_pval_df.mean(axis = 1)

### AFIB

In [ ]:
afib_total_cvd_auroc_df['AUROC'] = afib_total_cvd_auroc_df.mean(axis = 1)
afib_ascvd_auroc_df['AUROC'] = afib_ascvd_auroc_df.mean(axis = 1)
afib_heart_failure_auroc_df['AUROC'] = afib_heart_failure_auroc_df.mean(axis = 1)
afib_chd_auroc_df['AUROC'] = afib_chd_auroc_df.mean(axis = 1)
afib_stroke_auroc_df['AUROC'] = afib_stroke_auroc_df.mean(axis = 1)
afib_c2hest_auroc_df['AUROC'] = afib_c2hest_auroc_df.mean(axis = 1)

In [ ]:
afib_total_cvd_auprc_df['AUPRC'] = afib_total_cvd_auprc_df.mean(axis = 1)
afib_ascvd_auprc_df['AUPRC'] = afib_ascvd_auprc_df.mean(axis = 1)
afib_heart_failure_auprc_df['AUPRC'] = afib_heart_failure_auprc_df.mean(axis = 1)
afib_chd_auprc_df['AUPRC'] = afib_chd_auprc_df.mean(axis = 1)
afib_stroke_auprc_df['AUPRC'] = afib_stroke_auprc_df.mean(axis = 1)
afib_c2hest_auprc_df['AUPRC'] = afib_c2hest_auprc_df.mean(axis = 1)

In [ ]:
afib_total_cvd_f1_df['F1_SCORE'] = afib_total_cvd_f1_df.mean(axis = 1)
afib_ascvd_f1_df['F1_SCORE'] = afib_ascvd_f1_df.mean(axis = 1)
afib_heart_failure_f1_df['F1_SCORE'] = afib_heart_failure_f1_df.mean(axis = 1)
afib_chd_f1_df['F1_SCORE'] = afib_chd_f1_df.mean(axis = 1)
afib_stroke_f1_df['F1_SCORE'] = afib_stroke_f1_df.mean(axis = 1)
afib_c2hest_f1_df['F1_SCORE'] = afib_c2hest_f1_df.mean(axis = 1)

In [ ]:
afib_total_cvd_balanced_acc_df['BALANCED_ACCURACY'] = afib_total_cvd_balanced_acc_df.mean(axis = 1)
afib_ascvd_balanced_acc_df['BALANCED_ACCURACY'] = afib_ascvd_balanced_acc_df.mean(axis = 1)
afib_heart_failure_balanced_acc_df['BALANCED_ACCURACY'] = afib_heart_failure_balanced_acc_df.mean(axis = 1)
afib_chd_balanced_acc_df['BALANCED_ACCURACY'] = afib_chd_balanced_acc_df.mean(axis = 1)
afib_stroke_balanced_acc_df['BALANCED_ACCURACY'] = afib_stroke_balanced_acc_df.mean(axis = 1)
afib_c2hest_balanced_acc_df['BALANCED_ACCURACY'] = afib_c2hest_balanced_acc_df.mean(axis = 1)

In [ ]:
afib_pgs_beta_df['PGS'] = afib_pgs_beta_df.mean(axis = 1)
afib_pgs_pval_df['PGS'] = afib_pgs_pval_df.mean(axis = 1)

In [ ]:
afib_pxs_beta_df['PXS_AVG'] = afib_pxs_beta_df.mean(axis = 1)
afib_pxs_pval_df['PXS_AVG'] = afib_pxs_pval_df.mean(axis = 1)

In [ ]:
afib_wt_pxs_beta_df['PXS_WEIGHTED_AVG'] = afib_wt_pxs_beta_df.mean(axis = 1)
afib_wt_pxs_pval_df['PXS_WEIGHTED_AVG'] = afib_wt_pxs_pval_df.mean(axis = 1)

In [ ]:
afib_pgs_pxs_beta_df['PGS_PXS_AVG'] = afib_pgs_pxs_beta_df.mean(axis = 1)
afib_pgs_pxs_pval_df['PGS_PXS_AVG'] = afib_pgs_pxs_pval_df.mean(axis = 1)

In [ ]:
afib_pgs_wt_pxs_beta_df['PGS_PXS_WEIGHTED_AVG'] = afib_pgs_wt_pxs_beta_df.mean(axis = 1)
afib_pgs_wt_pxs_pval_df['PGS_PXS_WEIGHTED_AVG'] = afib_pgs_wt_pxs_pval_df.mean(axis = 1)

In [ ]:
afib_total_cvd_crs_beta_df['CRS'] = afib_total_cvd_crs_beta_df.mean(axis = 1)
afib_ascvd_crs_beta_df['CRS'] = afib_ascvd_crs_beta_df.mean(axis = 1)
afib_heart_failure_crs_beta_df['CRS'] = afib_heart_failure_crs_beta_df.mean(axis = 1)
afib_chd_crs_beta_df['CRS'] = afib_chd_crs_beta_df.mean(axis = 1)
afib_stroke_crs_beta_df['CRS'] = afib_stroke_crs_beta_df.mean(axis = 1)
afib_c2hest_crs_beta_df['CRS'] = afib_c2hest_crs_beta_df.mean(axis = 1)

afib_total_cvd_crs_pval_df['CRS'] = afib_total_cvd_crs_pval_df.mean(axis = 1)
afib_ascvd_crs_pval_df['CRS'] = afib_ascvd_crs_pval_df.mean(axis = 1)
afib_heart_failure_crs_pval_df['CRS'] = afib_heart_failure_crs_pval_df.mean(axis = 1)
afib_chd_crs_pval_df['CRS'] = afib_chd_crs_pval_df.mean(axis = 1)
afib_stroke_crs_pval_df['CRS'] = afib_stroke_crs_pval_df.mean(axis = 1)
afib_c2hest_crs_pval_df['CRS'] = afib_c2hest_crs_pval_df.mean(axis = 1)

In [ ]:
afib_total_cvd_pgs_crs_beta_df['PGS_CRS'] = afib_total_cvd_pgs_crs_beta_df.mean(axis = 1)
afib_ascvd_pgs_crs_beta_df['PGS_CRS'] = afib_ascvd_pgs_crs_beta_df.mean(axis = 1)
afib_heart_failure_pgs_crs_beta_df['PGS_CRS'] = afib_heart_failure_pgs_crs_beta_df.mean(axis = 1)
afib_chd_pgs_crs_beta_df['PGS_CRS'] = afib_chd_pgs_crs_beta_df.mean(axis = 1)
afib_stroke_pgs_crs_beta_df['PGS_CRS'] = afib_stroke_pgs_crs_beta_df.mean(axis = 1)
afib_c2hest_pgs_crs_beta_df['PGS_CRS'] = afib_c2hest_pgs_crs_beta_df.mean(axis = 1)

afib_total_cvd_pgs_crs_pval_df['PGS_CRS'] = afib_total_cvd_pgs_crs_pval_df.mean(axis = 1)
afib_ascvd_pgs_crs_pval_df['PGS_CRS'] = afib_ascvd_pgs_crs_pval_df.mean(axis = 1)
afib_heart_failure_pgs_crs_pval_df['PGS_CRS'] = afib_heart_failure_pgs_crs_pval_df.mean(axis = 1)
afib_chd_pgs_crs_pval_df['PGS_CRS'] = afib_chd_pgs_crs_pval_df.mean(axis = 1)
afib_stroke_pgs_crs_pval_df['PGS_CRS'] = afib_stroke_pgs_crs_pval_df.mean(axis = 1)
afib_c2hest_pgs_crs_pval_df['PGS_CRS'] = afib_c2hest_pgs_crs_pval_df.mean(axis = 1)

In [ ]:
afib_total_cvd_crs_pxs_beta_df['CRS_PXS_AVG'] = afib_total_cvd_crs_pxs_beta_df.mean(axis = 1)
afib_ascvd_crs_pxs_beta_df['CRS_PXS_AVG'] = afib_ascvd_crs_pxs_beta_df.mean(axis = 1)
afib_heart_failure_crs_pxs_beta_df['CRS_PXS_AVG'] = afib_heart_failure_crs_pxs_beta_df.mean(axis = 1)
afib_chd_crs_pxs_beta_df['CRS_PXS_AVG'] = afib_chd_crs_pxs_beta_df.mean(axis = 1)
afib_stroke_crs_pxs_beta_df['CRS_PXS_AVG'] = afib_stroke_crs_pxs_beta_df.mean(axis = 1)
afib_c2hest_crs_pxs_beta_df['CRS_PXS_AVG'] = afib_c2hest_crs_pxs_beta_df.mean(axis = 1)

afib_total_cvd_crs_pxs_pval_df['CRS_PXS_AVG'] = afib_total_cvd_crs_pxs_pval_df.mean(axis = 1)
afib_ascvd_crs_pxs_pval_df['CRS_PXS_AVG'] = afib_ascvd_crs_pxs_pval_df.mean(axis = 1)
afib_heart_failure_crs_pxs_pval_df['CRS_PXS_AVG'] = afib_heart_failure_crs_pxs_pval_df.mean(axis = 1)
afib_chd_crs_pxs_pval_df['CRS_PXS_AVG'] = afib_chd_crs_pxs_pval_df.mean(axis = 1)
afib_stroke_crs_pxs_pval_df['CRS_PXS_AVG'] = afib_stroke_crs_pxs_pval_df.mean(axis = 1)
afib_c2hest_crs_pxs_pval_df['CRS_PXS_AVG'] = afib_c2hest_crs_pxs_pval_df.mean(axis = 1)

In [ ]:
afib_total_cvd_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = afib_total_cvd_crs_wt_pxs_beta_df.mean(axis = 1)
afib_ascvd_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = afib_ascvd_crs_wt_pxs_beta_df.mean(axis = 1)
afib_heart_failure_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = afib_heart_failure_crs_wt_pxs_beta_df.mean(axis = 1)
afib_chd_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = afib_chd_crs_wt_pxs_beta_df.mean(axis = 1)
afib_stroke_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = afib_stroke_crs_wt_pxs_beta_df.mean(axis = 1)
afib_c2hest_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = afib_c2hest_crs_wt_pxs_beta_df.mean(axis = 1)

afib_total_cvd_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = afib_total_cvd_crs_wt_pxs_pval_df.mean(axis = 1)
afib_ascvd_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = afib_ascvd_crs_wt_pxs_pval_df.mean(axis = 1)
afib_heart_failure_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = afib_heart_failure_crs_wt_pxs_pval_df.mean(axis = 1)
afib_chd_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = afib_chd_crs_wt_pxs_pval_df.mean(axis = 1)
afib_stroke_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = afib_stroke_crs_wt_pxs_pval_df.mean(axis = 1)
afib_c2hest_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = afib_c2hest_crs_wt_pxs_pval_df.mean(axis = 1)

In [ ]:
afib_total_cvd_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = afib_total_cvd_pgs_crs_pxs_beta_df.mean(axis = 1)
afib_ascvd_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = afib_ascvd_pgs_crs_pxs_beta_df.mean(axis = 1)
afib_heart_failure_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = afib_heart_failure_pgs_crs_pxs_beta_df.mean(axis = 1)
afib_chd_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = afib_chd_pgs_crs_pxs_beta_df.mean(axis = 1)
afib_stroke_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = afib_stroke_pgs_crs_pxs_beta_df.mean(axis = 1)
afib_c2hest_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = afib_c2hest_pgs_crs_pxs_beta_df.mean(axis = 1)

afib_total_cvd_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = afib_total_cvd_pgs_crs_pxs_pval_df.mean(axis = 1)
afib_ascvd_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = afib_ascvd_pgs_crs_pxs_pval_df.mean(axis = 1)
afib_heart_failure_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = afib_heart_failure_pgs_crs_pxs_pval_df.mean(axis = 1)
afib_chd_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = afib_chd_pgs_crs_pxs_pval_df.mean(axis = 1)
afib_stroke_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = afib_stroke_pgs_crs_pxs_pval_df.mean(axis = 1)
afib_c2hest_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = afib_c2hest_pgs_crs_pxs_pval_df.mean(axis = 1)

In [ ]:
afib_total_cvd_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_total_cvd_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
afib_ascvd_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_ascvd_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
afib_heart_failure_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_heart_failure_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
afib_chd_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_chd_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
afib_stroke_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_stroke_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
afib_c2hest_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_c2hest_pgs_crs_wt_pxs_beta_df.mean(axis = 1)

afib_total_cvd_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_total_cvd_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
afib_ascvd_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_ascvd_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
afib_heart_failure_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_heart_failure_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
afib_chd_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_chd_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
afib_stroke_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_stroke_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
afib_c2hest_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = afib_c2hest_pgs_crs_wt_pxs_pval_df.mean(axis = 1)

### HF

In [ ]:
hf_total_cvd_auroc_df['AUROC'] = hf_total_cvd_auroc_df.mean(axis = 1)
hf_ascvd_auroc_df['AUROC'] = hf_ascvd_auroc_df.mean(axis = 1)
hf_heart_failure_auroc_df['AUROC'] = hf_heart_failure_auroc_df.mean(axis = 1)
hf_chd_auroc_df['AUROC'] = hf_chd_auroc_df.mean(axis = 1)
hf_stroke_auroc_df['AUROC'] = hf_stroke_auroc_df.mean(axis = 1)

In [ ]:
hf_total_cvd_auprc_df['AUPRC'] = hf_total_cvd_auprc_df.mean(axis = 1)
hf_ascvd_auprc_df['AUPRC'] = hf_ascvd_auprc_df.mean(axis = 1)
hf_heart_failure_auprc_df['AUPRC'] = hf_heart_failure_auprc_df.mean(axis = 1)
hf_chd_auprc_df['AUPRC'] = hf_chd_auprc_df.mean(axis = 1)
hf_stroke_auprc_df['AUPRC'] = hf_stroke_auprc_df.mean(axis = 1)

In [ ]:
hf_total_cvd_f1_df['F1_SCORE'] = hf_total_cvd_f1_df.mean(axis = 1)
hf_ascvd_f1_df['F1_SCORE'] = hf_ascvd_f1_df.mean(axis = 1)
hf_heart_failure_f1_df['F1_SCORE'] = hf_heart_failure_f1_df.mean(axis = 1)
hf_chd_f1_df['F1_SCORE'] = hf_chd_f1_df.mean(axis = 1)
hf_stroke_f1_df['F1_SCORE'] = hf_stroke_f1_df.mean(axis = 1)

In [ ]:
hf_total_cvd_balanced_acc_df['BALANCED_ACCURACY'] = hf_total_cvd_balanced_acc_df.mean(axis = 1)
hf_ascvd_balanced_acc_df['BALANCED_ACCURACY'] = hf_ascvd_balanced_acc_df.mean(axis = 1)
hf_heart_failure_balanced_acc_df['BALANCED_ACCURACY'] = hf_heart_failure_balanced_acc_df.mean(axis = 1)
hf_chd_balanced_acc_df['BALANCED_ACCURACY'] = hf_chd_balanced_acc_df.mean(axis = 1)
hf_stroke_balanced_acc_df['BALANCED_ACCURACY'] = hf_stroke_balanced_acc_df.mean(axis = 1)

In [ ]:
hf_pgs_beta_df['PGS'] = hf_pgs_beta_df.mean(axis = 1)
hf_pgs_pval_df['PGS'] = hf_pgs_pval_df.mean(axis = 1)

In [ ]:
hf_pxs_beta_df['PXS_AVG'] = hf_pxs_beta_df.mean(axis = 1)
hf_pxs_pval_df['PXS_AVG'] = hf_pxs_pval_df.mean(axis = 1)

In [ ]:
hf_wt_pxs_beta_df['PXS_WEIGHTED_AVG'] = hf_wt_pxs_beta_df.mean(axis = 1)
hf_wt_pxs_pval_df['PXS_WEIGHTED_AVG'] = hf_wt_pxs_pval_df.mean(axis = 1)

In [ ]:
hf_pgs_pxs_beta_df['PGS_PXS_AVG'] = hf_pgs_pxs_beta_df.mean(axis = 1)
hf_pgs_pxs_pval_df['PGS_PXS_AVG'] = hf_pgs_pxs_pval_df.mean(axis = 1)

In [ ]:
hf_pgs_wt_pxs_beta_df['PGS_PXS_WEIGHTED_AVG'] = hf_pgs_wt_pxs_beta_df.mean(axis = 1)
hf_pgs_wt_pxs_pval_df['PGS_PXS_WEIGHTED_AVG'] = hf_pgs_wt_pxs_pval_df.mean(axis = 1)

In [ ]:
hf_total_cvd_crs_beta_df['CRS'] = hf_total_cvd_crs_beta_df.mean(axis = 1)
hf_ascvd_crs_beta_df['CRS'] = hf_ascvd_crs_beta_df.mean(axis = 1)
hf_heart_failure_crs_beta_df['CRS'] = hf_heart_failure_crs_beta_df.mean(axis = 1)
hf_chd_crs_beta_df['CRS'] = hf_chd_crs_beta_df.mean(axis = 1)
hf_stroke_crs_beta_df['CRS'] = hf_stroke_crs_beta_df.mean(axis = 1)

hf_total_cvd_crs_pval_df['CRS'] = hf_total_cvd_crs_pval_df.mean(axis = 1)
hf_ascvd_crs_pval_df['CRS'] = hf_ascvd_crs_pval_df.mean(axis = 1)
hf_heart_failure_crs_pval_df['CRS'] = hf_heart_failure_crs_pval_df.mean(axis = 1)
hf_chd_crs_pval_df['CRS'] = hf_chd_crs_pval_df.mean(axis = 1)
hf_stroke_crs_pval_df['CRS'] = hf_stroke_crs_pval_df.mean(axis = 1)

In [ ]:
hf_total_cvd_pgs_crs_beta_df['PGS_CRS'] = hf_total_cvd_pgs_crs_beta_df.mean(axis = 1)
hf_ascvd_pgs_crs_beta_df['PGS_CRS'] = hf_ascvd_pgs_crs_beta_df.mean(axis = 1)
hf_heart_failure_pgs_crs_beta_df['PGS_CRS'] = hf_heart_failure_pgs_crs_beta_df.mean(axis = 1)
hf_chd_pgs_crs_beta_df['PGS_CRS'] = hf_chd_pgs_crs_beta_df.mean(axis = 1)
hf_stroke_pgs_crs_beta_df['PGS_CRS'] = hf_stroke_pgs_crs_beta_df.mean(axis = 1)

hf_total_cvd_pgs_crs_pval_df['PGS_CRS'] = hf_total_cvd_pgs_crs_pval_df.mean(axis = 1)
hf_ascvd_pgs_crs_pval_df['PGS_CRS'] = hf_ascvd_pgs_crs_pval_df.mean(axis = 1)
hf_heart_failure_pgs_crs_pval_df['PGS_CRS'] = hf_heart_failure_pgs_crs_pval_df.mean(axis = 1)
hf_chd_pgs_crs_pval_df['PGS_CRS'] = hf_chd_pgs_crs_pval_df.mean(axis = 1)
hf_stroke_pgs_crs_pval_df['PGS_CRS'] = hf_stroke_pgs_crs_pval_df.mean(axis = 1)

In [ ]:
hf_total_cvd_crs_pxs_beta_df['CRS_PXS_AVG'] = hf_total_cvd_crs_pxs_beta_df.mean(axis = 1)
hf_ascvd_crs_pxs_beta_df['CRS_PXS_AVG'] = hf_ascvd_crs_pxs_beta_df.mean(axis = 1)
hf_heart_failure_crs_pxs_beta_df['CRS_PXS_AVG'] = hf_heart_failure_crs_pxs_beta_df.mean(axis = 1)
hf_chd_crs_pxs_beta_df['CRS_PXS_AVG'] = hf_chd_crs_pxs_beta_df.mean(axis = 1)
hf_stroke_crs_pxs_beta_df['CRS_PXS_AVG'] = hf_stroke_crs_pxs_beta_df.mean(axis = 1)

hf_total_cvd_crs_pxs_pval_df['CRS_PXS_AVG'] = hf_total_cvd_crs_pxs_pval_df.mean(axis = 1)
hf_ascvd_crs_pxs_pval_df['CRS_PXS_AVG'] = hf_ascvd_crs_pxs_pval_df.mean(axis = 1)
hf_heart_failure_crs_pxs_pval_df['CRS_PXS_AVG'] = hf_heart_failure_crs_pxs_pval_df.mean(axis = 1)
hf_chd_crs_pxs_pval_df['CRS_PXS_AVG'] = hf_chd_crs_pxs_pval_df.mean(axis = 1)
hf_stroke_crs_pxs_pval_df['CRS_PXS_AVG'] = hf_stroke_crs_pxs_pval_df.mean(axis = 1)

In [ ]:
hf_total_cvd_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = hf_total_cvd_crs_wt_pxs_beta_df.mean(axis = 1)
hf_ascvd_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = hf_ascvd_crs_wt_pxs_beta_df.mean(axis = 1)
hf_heart_failure_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = hf_heart_failure_crs_wt_pxs_beta_df.mean(axis = 1)
hf_chd_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = hf_chd_crs_wt_pxs_beta_df.mean(axis = 1)
hf_stroke_crs_wt_pxs_beta_df['CRS_PXS_WEIGHTED_AVG'] = hf_stroke_crs_wt_pxs_beta_df.mean(axis = 1)

hf_total_cvd_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = hf_total_cvd_crs_wt_pxs_pval_df.mean(axis = 1)
hf_ascvd_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = hf_ascvd_crs_wt_pxs_pval_df.mean(axis = 1)
hf_heart_failure_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = hf_heart_failure_crs_wt_pxs_pval_df.mean(axis = 1)
hf_chd_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = hf_chd_crs_wt_pxs_pval_df.mean(axis = 1)
hf_stroke_crs_wt_pxs_pval_df['CRS_PXS_WEIGHTED_AVG'] = hf_stroke_crs_wt_pxs_pval_df.mean(axis = 1)

In [ ]:
hf_total_cvd_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = hf_total_cvd_pgs_crs_pxs_beta_df.mean(axis = 1)
hf_ascvd_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = hf_ascvd_pgs_crs_pxs_beta_df.mean(axis = 1)
hf_heart_failure_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = hf_heart_failure_pgs_crs_pxs_beta_df.mean(axis = 1)
hf_chd_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = hf_chd_pgs_crs_pxs_beta_df.mean(axis = 1)
hf_stroke_pgs_crs_pxs_beta_df['PGS_CRS_PXS_AVG'] = hf_stroke_pgs_crs_pxs_beta_df.mean(axis = 1)

hf_total_cvd_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = hf_total_cvd_pgs_crs_pxs_pval_df.mean(axis = 1)
hf_ascvd_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = hf_ascvd_pgs_crs_pxs_pval_df.mean(axis = 1)
hf_heart_failure_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = hf_heart_failure_pgs_crs_pxs_pval_df.mean(axis = 1)
hf_chd_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = hf_chd_pgs_crs_pxs_pval_df.mean(axis = 1)
hf_stroke_pgs_crs_pxs_pval_df['PGS_CRS_PXS_AVG'] = hf_stroke_pgs_crs_pxs_pval_df.mean(axis = 1)

In [ ]:
hf_total_cvd_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = hf_total_cvd_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
hf_ascvd_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = hf_ascvd_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
hf_heart_failure_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = hf_heart_failure_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
hf_chd_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = hf_chd_pgs_crs_wt_pxs_beta_df.mean(axis = 1)
hf_stroke_pgs_crs_wt_pxs_beta_df['PGS_CRS_PXS_WEIGHTED_AVG'] = hf_stroke_pgs_crs_wt_pxs_beta_df.mean(axis = 1)

hf_total_cvd_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = hf_total_cvd_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
hf_ascvd_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = hf_ascvd_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
hf_heart_failure_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = hf_heart_failure_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
hf_chd_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = hf_chd_pgs_crs_wt_pxs_pval_df.mean(axis = 1)
hf_stroke_pgs_crs_wt_pxs_pval_df['PGS_CRS_PXS_WEIGHTED_AVG'] = hf_stroke_pgs_crs_wt_pxs_pval_df.mean(axis = 1)

## make combined df

### cad

In [ ]:
cad_total_cvd_all_metrics = pd.concat([cad_total_cvd_auroc_df[['AUROC']],
                                       cad_total_cvd_auprc_df[['AUPRC']],
                                       cad_total_cvd_f1_df[['F1_SCORE']],
                                       cad_total_cvd_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
cad_total_cvd_all_metrics.index = 'Total CVD CRS: ' + cad_total_cvd_all_metrics.index
cad_total_cvd_all_metrics

In [ ]:
cad_ascvd_all_metrics = pd.concat([cad_ascvd_auroc_df[['AUROC']],
                                       cad_ascvd_auprc_df[['AUPRC']],
                                       cad_ascvd_f1_df[['F1_SCORE']],
                                       cad_ascvd_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
cad_ascvd_all_metrics.index = 'ASCVD CRS: ' + cad_ascvd_all_metrics.index
cad_ascvd_all_metrics

In [ ]:
cad_heart_failure_all_metrics = pd.concat([cad_heart_failure_auroc_df[['AUROC']],
                                       cad_heart_failure_auprc_df[['AUPRC']],
                                       cad_heart_failure_f1_df[['F1_SCORE']],
                                       cad_heart_failure_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
cad_heart_failure_all_metrics.index = 'Heart Failure CRS: ' + cad_heart_failure_all_metrics.index
cad_heart_failure_all_metrics

In [ ]:
cad_chd_all_metrics = pd.concat([cad_chd_auroc_df[['AUROC']],
                                       cad_chd_auprc_df[['AUPRC']],
                                       cad_chd_f1_df[['F1_SCORE']],
                                       cad_chd_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
cad_chd_all_metrics.index = 'CHD CRS: ' + cad_chd_all_metrics.index
cad_chd_all_metrics

In [ ]:
cad_stroke_all_metrics = pd.concat([cad_stroke_auroc_df[['AUROC']],
                                       cad_stroke_auprc_df[['AUPRC']],
                                       cad_stroke_f1_df[['F1_SCORE']],
                                       cad_stroke_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
cad_stroke_all_metrics.index = 'Stroke CRS: ' + cad_stroke_all_metrics.index
cad_stroke_all_metrics

In [ ]:
cad_total_cvd_beta = pd.concat([cad_pgs_beta_df[['PGS']],
                                cad_total_cvd_crs_beta_df[['CRS']],
                                cad_pxs_beta_df[['PXS_AVG']],
                                cad_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                cad_total_cvd_pgs_crs_beta_df[['PGS_CRS']],
                                cad_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                cad_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                cad_total_cvd_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                cad_total_cvd_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                cad_total_cvd_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                cad_total_cvd_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
cad_total_cvd_beta

In [ ]:
cad_total_cvd_pval = pd.concat([cad_pgs_pval_df[['PGS']],
                                cad_total_cvd_crs_pval_df[['CRS']],
                                cad_pxs_pval_df[['PXS_AVG']],
                                cad_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                cad_total_cvd_pgs_crs_pval_df[['PGS_CRS']],
                                cad_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                cad_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                cad_total_cvd_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                cad_total_cvd_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                cad_total_cvd_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                cad_total_cvd_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
cad_total_cvd_pval

In [ ]:
cad_ascvd_beta = pd.concat([cad_pgs_beta_df[['PGS']],
                                cad_ascvd_crs_beta_df[['CRS']],
                                cad_pxs_beta_df[['PXS_AVG']],
                                cad_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                cad_ascvd_pgs_crs_beta_df[['PGS_CRS']],
                                cad_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                cad_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                cad_ascvd_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                cad_ascvd_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                cad_ascvd_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                cad_ascvd_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
cad_ascvd_beta

In [ ]:
cad_ascvd_pval = pd.concat([cad_pgs_pval_df[['PGS']],
                                cad_ascvd_crs_pval_df[['CRS']],
                                cad_pxs_pval_df[['PXS_AVG']],
                                cad_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                cad_ascvd_pgs_crs_pval_df[['PGS_CRS']],
                                cad_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                cad_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                cad_ascvd_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                cad_ascvd_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                cad_ascvd_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                cad_ascvd_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
cad_ascvd_pval

In [ ]:
cad_heart_failure_beta = pd.concat([cad_pgs_beta_df[['PGS']],
                                cad_heart_failure_crs_beta_df[['CRS']],
                                cad_pxs_beta_df[['PXS_AVG']],
                                cad_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                cad_heart_failure_pgs_crs_beta_df[['PGS_CRS']],
                                cad_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                cad_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                cad_heart_failure_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                cad_heart_failure_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                cad_heart_failure_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                cad_heart_failure_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
cad_heart_failure_beta

In [ ]:
cad_heart_failure_pval = pd.concat([cad_pgs_pval_df[['PGS']],
                                cad_heart_failure_crs_pval_df[['CRS']],
                                cad_pxs_pval_df[['PXS_AVG']],
                                cad_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                cad_heart_failure_pgs_crs_pval_df[['PGS_CRS']],
                                cad_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                cad_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                cad_heart_failure_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                cad_heart_failure_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                cad_heart_failure_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                cad_heart_failure_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
cad_heart_failure_pval

In [ ]:
cad_stroke_beta = pd.concat([cad_pgs_beta_df[['PGS']],
                                cad_stroke_crs_beta_df[['CRS']],
                                cad_pxs_beta_df[['PXS_AVG']],
                                cad_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                cad_stroke_pgs_crs_beta_df[['PGS_CRS']],
                                cad_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                cad_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                cad_stroke_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                cad_stroke_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                cad_stroke_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                cad_stroke_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
cad_stroke_beta

In [ ]:
cad_stroke_pval = pd.concat([cad_pgs_pval_df[['PGS']],
                                cad_stroke_crs_pval_df[['CRS']],
                                cad_pxs_pval_df[['PXS_AVG']],
                                cad_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                cad_stroke_pgs_crs_pval_df[['PGS_CRS']],
                                cad_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                cad_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                cad_stroke_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                cad_stroke_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                cad_stroke_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                cad_stroke_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
cad_stroke_pval

In [ ]:
cad_chd_beta = pd.concat([cad_pgs_beta_df[['PGS']],
                                cad_chd_crs_beta_df[['CRS']],
                                cad_pxs_beta_df[['PXS_AVG']],
                                cad_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                cad_chd_pgs_crs_beta_df[['PGS_CRS']],
                                cad_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                cad_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                cad_chd_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                cad_chd_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                cad_chd_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                cad_chd_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
cad_chd_beta

In [ ]:
cad_chd_pval = pd.concat([cad_pgs_pval_df[['PGS']],
                                cad_chd_crs_pval_df[['CRS']],
                                cad_pxs_pval_df[['PXS_AVG']],
                                cad_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                cad_chd_pgs_crs_pval_df[['PGS_CRS']],
                                cad_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                cad_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                cad_chd_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                cad_chd_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                cad_chd_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                cad_chd_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
cad_chd_pval

### afib

In [ ]:
afib_total_cvd_all_metrics = pd.concat([afib_total_cvd_auroc_df[['AUROC']],
                                       afib_total_cvd_auprc_df[['AUPRC']],
                                       afib_total_cvd_f1_df[['F1_SCORE']],
                                       afib_total_cvd_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
afib_total_cvd_all_metrics.index = 'Total CVD CRS: ' + afib_total_cvd_all_metrics.index
afib_total_cvd_all_metrics

In [ ]:
afib_ascvd_all_metrics = pd.concat([afib_ascvd_auroc_df[['AUROC']],
                                       afib_ascvd_auprc_df[['AUPRC']],
                                       afib_ascvd_f1_df[['F1_SCORE']],
                                       afib_ascvd_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
afib_ascvd_all_metrics.index = 'ASCVD CRS: ' + afib_ascvd_all_metrics.index
afib_ascvd_all_metrics

In [ ]:
afib_heart_failure_all_metrics = pd.concat([afib_heart_failure_auroc_df[['AUROC']],
                                       afib_heart_failure_auprc_df[['AUPRC']],
                                       afib_heart_failure_f1_df[['F1_SCORE']],
                                       afib_heart_failure_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
afib_heart_failure_all_metrics.index = 'Heart Failure CRS: ' + afib_heart_failure_all_metrics.index
afib_heart_failure_all_metrics

In [ ]:
afib_chd_all_metrics = pd.concat([afib_chd_auroc_df[['AUROC']],
                                       afib_chd_auprc_df[['AUPRC']],
                                       afib_chd_f1_df[['F1_SCORE']],
                                       afib_chd_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
afib_chd_all_metrics.index = 'CHD CRS: ' + afib_chd_all_metrics.index
afib_chd_all_metrics

In [ ]:
afib_stroke_all_metrics = pd.concat([afib_stroke_auroc_df[['AUROC']],
                                       afib_stroke_auprc_df[['AUPRC']],
                                       afib_stroke_f1_df[['F1_SCORE']],
                                       afib_stroke_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
afib_stroke_all_metrics.index = 'Stroke CRS: ' + afib_stroke_all_metrics.index
afib_stroke_all_metrics

In [ ]:
afib_c2hest_all_metrics = pd.concat([afib_c2hest_auroc_df[['AUROC']],
                                       afib_c2hest_auprc_df[['AUPRC']],
                                       afib_c2hest_f1_df[['F1_SCORE']],
                                       afib_c2hest_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
afib_c2hest_all_metrics.index = 'C2HEST CRS: ' + afib_c2hest_all_metrics.index
afib_c2hest_all_metrics

In [ ]:
afib_total_cvd_beta = pd.concat([afib_pgs_beta_df[['PGS']],
                                afib_total_cvd_crs_beta_df[['CRS']],
                                afib_pxs_beta_df[['PXS_AVG']],
                                afib_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                afib_total_cvd_pgs_crs_beta_df[['PGS_CRS']],
                                afib_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_total_cvd_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                afib_total_cvd_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_total_cvd_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                afib_total_cvd_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_total_cvd_beta

In [ ]:
afib_total_cvd_pval = pd.concat([afib_pgs_pval_df[['PGS']],
                                afib_total_cvd_crs_pval_df[['CRS']],
                                afib_pxs_pval_df[['PXS_AVG']],
                                afib_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                afib_total_cvd_pgs_crs_pval_df[['PGS_CRS']],
                                afib_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_total_cvd_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                afib_total_cvd_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_total_cvd_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                afib_total_cvd_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_total_cvd_pval

In [ ]:
afib_ascvd_beta = pd.concat([afib_pgs_beta_df[['PGS']],
                                afib_ascvd_crs_beta_df[['CRS']],
                                afib_pxs_beta_df[['PXS_AVG']],
                                afib_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                afib_ascvd_pgs_crs_beta_df[['PGS_CRS']],
                                afib_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_ascvd_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                afib_ascvd_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_ascvd_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                afib_ascvd_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_ascvd_beta

In [ ]:
afib_ascvd_pval = pd.concat([afib_pgs_pval_df[['PGS']],
                                afib_ascvd_crs_pval_df[['CRS']],
                                afib_pxs_pval_df[['PXS_AVG']],
                                afib_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                afib_ascvd_pgs_crs_pval_df[['PGS_CRS']],
                                afib_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_ascvd_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                afib_ascvd_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_ascvd_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                afib_ascvd_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_ascvd_pval

In [ ]:
afib_heart_failure_beta = pd.concat([afib_pgs_beta_df[['PGS']],
                                afib_heart_failure_crs_beta_df[['CRS']],
                                afib_pxs_beta_df[['PXS_AVG']],
                                afib_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                afib_heart_failure_pgs_crs_beta_df[['PGS_CRS']],
                                afib_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_heart_failure_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                afib_heart_failure_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_heart_failure_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                afib_heart_failure_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_heart_failure_beta

In [ ]:
afib_heart_failure_pval = pd.concat([afib_pgs_pval_df[['PGS']],
                                afib_heart_failure_crs_pval_df[['CRS']],
                                afib_pxs_pval_df[['PXS_AVG']],
                                afib_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                afib_heart_failure_pgs_crs_pval_df[['PGS_CRS']],
                                afib_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_heart_failure_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                afib_heart_failure_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_heart_failure_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                afib_heart_failure_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_heart_failure_pval

In [ ]:
afib_stroke_beta = pd.concat([afib_pgs_beta_df[['PGS']],
                                afib_stroke_crs_beta_df[['CRS']],
                                afib_pxs_beta_df[['PXS_AVG']],
                                afib_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                afib_stroke_pgs_crs_beta_df[['PGS_CRS']],
                                afib_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_stroke_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                afib_stroke_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_stroke_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                afib_stroke_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_stroke_beta

In [ ]:
afib_stroke_pval = pd.concat([afib_pgs_pval_df[['PGS']],
                                afib_stroke_crs_pval_df[['CRS']],
                                afib_pxs_pval_df[['PXS_AVG']],
                                afib_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                afib_stroke_pgs_crs_pval_df[['PGS_CRS']],
                                afib_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_stroke_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                afib_stroke_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_stroke_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                afib_stroke_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_stroke_pval

In [ ]:
afib_chd_beta = pd.concat([afib_pgs_beta_df[['PGS']],
                                afib_chd_crs_beta_df[['CRS']],
                                afib_pxs_beta_df[['PXS_AVG']],
                                afib_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                afib_chd_pgs_crs_beta_df[['PGS_CRS']],
                                afib_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_chd_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                afib_chd_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_chd_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                afib_chd_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_chd_beta

In [ ]:
afib_chd_pval = pd.concat([afib_pgs_pval_df[['PGS']],
                                afib_chd_crs_pval_df[['CRS']],
                                afib_pxs_pval_df[['PXS_AVG']],
                                afib_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                afib_chd_pgs_crs_pval_df[['PGS_CRS']],
                                afib_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_chd_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                afib_chd_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_chd_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                afib_chd_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_chd_pval

In [ ]:
afib_c2hest_beta = pd.concat([afib_pgs_beta_df[['PGS']],
                                afib_c2hest_crs_beta_df[['CRS']],
                                afib_pxs_beta_df[['PXS_AVG']],
                                afib_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                afib_c2hest_pgs_crs_beta_df[['PGS_CRS']],
                                afib_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_c2hest_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                afib_c2hest_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_c2hest_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                afib_c2hest_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_c2hest_beta

In [ ]:
afib_c2hest_pval = pd.concat([afib_pgs_pval_df[['PGS']],
                                afib_c2hest_crs_pval_df[['CRS']],
                                afib_pxs_pval_df[['PXS_AVG']],
                                afib_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                afib_c2hest_pgs_crs_pval_df[['PGS_CRS']],
                                afib_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                afib_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                afib_c2hest_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                afib_c2hest_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                afib_c2hest_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                afib_c2hest_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
afib_c2hest_pval

### HF

In [ ]:
hf_total_cvd_all_metrics = pd.concat([hf_total_cvd_auroc_df[['AUROC']],
                                       hf_total_cvd_auprc_df[['AUPRC']],
                                       hf_total_cvd_f1_df[['F1_SCORE']],
                                       hf_total_cvd_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
hf_total_cvd_all_metrics.index = 'Total CVD CRS: ' + hf_total_cvd_all_metrics.index
hf_total_cvd_all_metrics

In [ ]:
hf_ascvd_all_metrics = pd.concat([hf_ascvd_auroc_df[['AUROC']],
                                       hf_ascvd_auprc_df[['AUPRC']],
                                       hf_ascvd_f1_df[['F1_SCORE']],
                                       hf_ascvd_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
hf_ascvd_all_metrics.index = 'ASCVD CRS: ' + hf_ascvd_all_metrics.index
hf_ascvd_all_metrics

In [ ]:
hf_heart_failure_all_metrics = pd.concat([hf_heart_failure_auroc_df[['AUROC']],
                                       hf_heart_failure_auprc_df[['AUPRC']],
                                       hf_heart_failure_f1_df[['F1_SCORE']],
                                       hf_heart_failure_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
hf_heart_failure_all_metrics.index = 'Heart Failure CRS: ' + hf_heart_failure_all_metrics.index
hf_heart_failure_all_metrics

In [ ]:
hf_chd_all_metrics = pd.concat([hf_chd_auroc_df[['AUROC']],
                                       hf_chd_auprc_df[['AUPRC']],
                                       hf_chd_f1_df[['F1_SCORE']],
                                       hf_chd_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
hf_chd_all_metrics.index = 'CHD CRS: ' + hf_chd_all_metrics.index
hf_chd_all_metrics

In [ ]:
hf_stroke_all_metrics = pd.concat([hf_stroke_auroc_df[['AUROC']],
                                       hf_stroke_auprc_df[['AUPRC']],
                                       hf_stroke_f1_df[['F1_SCORE']],
                                       hf_stroke_balanced_acc_df[['BALANCED_ACCURACY']]], axis = 1)
hf_stroke_all_metrics.index = 'Stroke CRS: ' + hf_stroke_all_metrics.index
hf_stroke_all_metrics

In [ ]:
hf_total_cvd_beta = pd.concat([hf_pgs_beta_df[['PGS']],
                                hf_total_cvd_crs_beta_df[['CRS']],
                                hf_pxs_beta_df[['PXS_AVG']],
                                hf_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                hf_total_cvd_pgs_crs_beta_df[['PGS_CRS']],
                                hf_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                hf_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                hf_total_cvd_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                hf_total_cvd_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                hf_total_cvd_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                hf_total_cvd_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
hf_total_cvd_beta

In [ ]:
hf_total_cvd_pval = pd.concat([hf_pgs_pval_df[['PGS']],
                                hf_total_cvd_crs_pval_df[['CRS']],
                                hf_pxs_pval_df[['PXS_AVG']],
                                hf_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                hf_total_cvd_pgs_crs_pval_df[['PGS_CRS']],
                                hf_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                hf_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                hf_total_cvd_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                hf_total_cvd_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                hf_total_cvd_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                hf_total_cvd_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
hf_total_cvd_pval

In [ ]:
hf_ascvd_beta = pd.concat([hf_pgs_beta_df[['PGS']],
                                hf_ascvd_crs_beta_df[['CRS']],
                                hf_pxs_beta_df[['PXS_AVG']],
                                hf_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                hf_ascvd_pgs_crs_beta_df[['PGS_CRS']],
                                hf_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                hf_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                hf_ascvd_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                hf_ascvd_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                hf_ascvd_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                hf_ascvd_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
hf_ascvd_beta

In [ ]:
hf_ascvd_pval = pd.concat([hf_pgs_pval_df[['PGS']],
                                hf_ascvd_crs_pval_df[['CRS']],
                                hf_pxs_pval_df[['PXS_AVG']],
                                hf_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                hf_ascvd_pgs_crs_pval_df[['PGS_CRS']],
                                hf_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                hf_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                hf_ascvd_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                hf_ascvd_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                hf_ascvd_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                hf_ascvd_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
hf_ascvd_pval

In [ ]:
hf_heart_failure_beta = pd.concat([hf_pgs_beta_df[['PGS']],
                                hf_heart_failure_crs_beta_df[['CRS']],
                                hf_pxs_beta_df[['PXS_AVG']],
                                hf_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                hf_heart_failure_pgs_crs_beta_df[['PGS_CRS']],
                                hf_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                hf_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                hf_heart_failure_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                hf_heart_failure_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                hf_heart_failure_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                hf_heart_failure_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
hf_heart_failure_beta

In [ ]:
hf_heart_failure_pval = pd.concat([hf_pgs_pval_df[['PGS']],
                                hf_heart_failure_crs_pval_df[['CRS']],
                                hf_pxs_pval_df[['PXS_AVG']],
                                hf_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                hf_heart_failure_pgs_crs_pval_df[['PGS_CRS']],
                                hf_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                hf_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                hf_heart_failure_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                hf_heart_failure_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                hf_heart_failure_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                hf_heart_failure_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
hf_heart_failure_pval

In [ ]:
hf_stroke_beta = pd.concat([hf_pgs_beta_df[['PGS']],
                                hf_stroke_crs_beta_df[['CRS']],
                                hf_pxs_beta_df[['PXS_AVG']],
                                hf_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                hf_stroke_pgs_crs_beta_df[['PGS_CRS']],
                                hf_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                hf_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                hf_stroke_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                hf_stroke_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                hf_stroke_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                hf_stroke_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
hf_stroke_beta

In [ ]:
hf_stroke_pval = pd.concat([hf_pgs_pval_df[['PGS']],
                                hf_stroke_crs_pval_df[['CRS']],
                                hf_pxs_pval_df[['PXS_AVG']],
                                hf_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                hf_stroke_pgs_crs_pval_df[['PGS_CRS']],
                                hf_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                hf_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                hf_stroke_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                hf_stroke_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                hf_stroke_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                hf_stroke_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
hf_stroke_pval

In [ ]:
hf_chd_beta = pd.concat([hf_pgs_beta_df[['PGS']],
                                hf_chd_crs_beta_df[['CRS']],
                                hf_pxs_beta_df[['PXS_AVG']],
                                hf_wt_pxs_beta_df[['PXS_WEIGHTED_AVG']],
                                hf_chd_pgs_crs_beta_df[['PGS_CRS']],
                                hf_pgs_pxs_beta_df[['PGS_PXS_AVG']],
                                hf_pgs_wt_pxs_beta_df[['PGS_PXS_WEIGHTED_AVG']],
                                hf_chd_crs_pxs_beta_df[['CRS_PXS_AVG']],
                                hf_chd_crs_wt_pxs_beta_df[['CRS_PXS_WEIGHTED_AVG']],
                                hf_chd_pgs_crs_pxs_beta_df[['PGS_CRS_PXS_AVG']],
                                hf_chd_pgs_crs_wt_pxs_beta_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
hf_chd_beta

In [ ]:
hf_chd_pval = pd.concat([hf_pgs_pval_df[['PGS']],
                                hf_chd_crs_pval_df[['CRS']],
                                hf_pxs_pval_df[['PXS_AVG']],
                                hf_wt_pxs_pval_df[['PXS_WEIGHTED_AVG']],
                                hf_chd_pgs_crs_pval_df[['PGS_CRS']],
                                hf_pgs_pxs_pval_df[['PGS_PXS_AVG']],
                                hf_pgs_wt_pxs_pval_df[['PGS_PXS_WEIGHTED_AVG']],
                                hf_chd_crs_pxs_pval_df[['CRS_PXS_AVG']],
                                hf_chd_crs_wt_pxs_pval_df[['CRS_PXS_WEIGHTED_AVG']],
                                hf_chd_pgs_crs_pxs_pval_df[['PGS_CRS_PXS_AVG']],
                                hf_chd_pgs_crs_wt_pxs_pval_df[['PGS_CRS_PXS_WEIGHTED_AVG']]], axis = 1)
hf_chd_pval

## combine results per pheno

### cad

In [ ]:
cad_ascvd_sub = cad_ascvd_all_metrics.drop(index = ['ASCVD CRS: PGS_CAD',
                                                    'ASCVD CRS: PXS_AVG',
                                                    'ASCVD CRS: PXS_WEIGHTED_AVG',
                                                    "ASCVD CRS: ('PGS_CAD', 'PXS_AVG')",
                                                    "ASCVD CRS: ('PGS_CAD', 'PXS_WEIGHTED_AVG')"])

In [ ]:
cad_heart_failure_sub = cad_heart_failure_all_metrics.drop(index = ['Heart Failure CRS: PGS_CAD',
                                                    'Heart Failure CRS: PXS_AVG',
                                                    'Heart Failure CRS: PXS_WEIGHTED_AVG',
                                                    "Heart Failure CRS: ('PGS_CAD', 'PXS_AVG')",
                                                    "Heart Failure CRS: ('PGS_CAD', 'PXS_WEIGHTED_AVG')"])

In [ ]:
cad_chd_sub = cad_chd_all_metrics.drop(index = ['CHD CRS: PGS_CAD',
                                                    'CHD CRS: PXS_AVG',
                                                    'CHD CRS: PXS_WEIGHTED_AVG',
                                                    "CHD CRS: ('PGS_CAD', 'PXS_AVG')",
                                                    "CHD CRS: ('PGS_CAD', 'PXS_WEIGHTED_AVG')"])

In [ ]:
cad_stroke_sub = cad_stroke_all_metrics.drop(index = ['Stroke CRS: PGS_CAD',
                                                    'Stroke CRS: PXS_AVG',
                                                    'Stroke CRS: PXS_WEIGHTED_AVG',
                                                    "Stroke CRS: ('PGS_CAD', 'PXS_AVG')",
                                                    "Stroke CRS: ('PGS_CAD', 'PXS_WEIGHTED_AVG')"])

In [ ]:
cad_all_crs = pd.concat([cad_total_cvd_all_metrics,
                         cad_ascvd_sub,
                         cad_heart_failure_sub,
                         cad_chd_sub,
                         cad_stroke_sub], axis = 0)

cad_all_crs.index = cad_all_crs.index.str.replace('Total CVD CRS: PGS_CAD', 'PGS_CAD')
cad_all_crs.index = cad_all_crs.index.str.replace('Total CVD CRS: PXS_AVG', 'PXS_AVG')
cad_all_crs.index = cad_all_crs.index.str.replace('Total CVD CRS: PXS_WEIGHTED_AVG', 'PXS_WEIGHTED_AVG')
cad_all_crs.index = cad_all_crs.index.str.replace("Total CVD CRS: ('PGS_CAD', 'PXS_AVG')", "('PGS_CAD', 'PXS_AVG')")
cad_all_crs.index = cad_all_crs.index.str.replace("Total CVD CRS: ('PGS_CAD', 'PXS_WEIGHTED_AVG')", "('PGS_CAD', 'PXS_WEIGHTED_AVG')")

cad_all_crs.index = cad_all_crs.index.str.replace('CAD_PREVENT_CRS_heart_failure', 'CRS')
cad_all_crs.index = cad_all_crs.index.str.replace('CAD_PREVENT_CRS_stroke', 'CRS')
cad_all_crs.index = cad_all_crs.index.str.replace('CAD_PREVENT_CRS_chd', 'CRS')
cad_all_crs.index = cad_all_crs.index.str.replace('CAD_PREVENT_CRS_ascvd', 'CRS')
cad_all_crs.index = cad_all_crs.index.str.replace('CAD_PREVENT_CRS_total_cvd', 'CRS')

cad_all_crs.index = cad_all_crs.index.str.replace('PGS_CAD', 'PGS')
cad_all_crs.index = cad_all_crs.index.str.replace("'", '')
cad_all_crs.index = cad_all_crs.index.str.replace("(", '')
cad_all_crs.index = cad_all_crs.index.str.replace(")", '')
cad_all_crs.index = cad_all_crs.index.str.replace(", ", ' + ')

cad_all_crs = cad_all_crs.sort_values(by = ['AUROC'], ascending = False)
cad_all_crs

### AFIB

In [ ]:
afib_ascvd_sub = afib_ascvd_all_metrics.drop(index = ['ASCVD CRS: PGS_AFIB',
                                                    'ASCVD CRS: PXS_AVG',
                                                    'ASCVD CRS: PXS_WEIGHTED_AVG',
                                                    "ASCVD CRS: ('PGS_AFIB', 'PXS_AVG')",
                                                    "ASCVD CRS: ('PGS_AFIB', 'PXS_WEIGHTED_AVG')"])

In [ ]:
afib_heart_failure_sub = afib_heart_failure_all_metrics.drop(index = ['Heart Failure CRS: PGS_AFIB',
                                                    'Heart Failure CRS: PXS_AVG',
                                                    'Heart Failure CRS: PXS_WEIGHTED_AVG',
                                                    "Heart Failure CRS: ('PGS_AFIB', 'PXS_AVG')",
                                                    "Heart Failure CRS: ('PGS_AFIB', 'PXS_WEIGHTED_AVG')"])

In [ ]:
afib_chd_sub = afib_chd_all_metrics.drop(index = ['CHD CRS: PGS_AFIB',
                                                    'CHD CRS: PXS_AVG',
                                                    'CHD CRS: PXS_WEIGHTED_AVG',
                                                    "CHD CRS: ('PGS_AFIB', 'PXS_AVG')",
                                                    "CHD CRS: ('PGS_AFIB', 'PXS_WEIGHTED_AVG')"])

In [ ]:
afib_stroke_sub = afib_stroke_all_metrics.drop(index = ['Stroke CRS: PGS_AFIB',
                                                    'Stroke CRS: PXS_AVG',
                                                    'Stroke CRS: PXS_WEIGHTED_AVG',
                                                    "Stroke CRS: ('PGS_AFIB', 'PXS_AVG')",
                                                    "Stroke CRS: ('PGS_AFIB', 'PXS_WEIGHTED_AVG')"])

In [ ]:
afib_c2hest_sub = afib_c2hest_all_metrics.drop(index = ['C2HEST CRS: PGS_AFIB',
                                                    'C2HEST CRS: PXS_AVG',
                                                    'C2HEST CRS: PXS_WEIGHTED_AVG',
                                                    "C2HEST CRS: ('PGS_AFIB', 'PXS_AVG')",
                                                    "C2HEST CRS: ('PGS_AFIB', 'PXS_WEIGHTED_AVG')"])

In [ ]:
afib_all_crs = pd.concat([afib_total_cvd_all_metrics,
                         afib_ascvd_sub,
                         afib_heart_failure_sub,
                         afib_chd_sub,
                         afib_stroke_sub,
                         afib_c2hest_sub], axis = 0)

afib_all_crs.index = afib_all_crs.index.str.replace('Total CVD CRS: PGS_AFIB', 'PGS_AFIB')
afib_all_crs.index = afib_all_crs.index.str.replace('Total CVD CRS: PXS_AVG', 'PXS_AVG')
afib_all_crs.index = afib_all_crs.index.str.replace('Total CVD CRS: PXS_WEIGHTED_AVG', 'PXS_WEIGHTED_AVG')
afib_all_crs.index = afib_all_crs.index.str.replace("Total CVD CRS: ('PGS_AFIB', 'PXS_AVG')", "('PGS_AFIB', 'PXS_AVG')")
afib_all_crs.index = afib_all_crs.index.str.replace("Total CVD CRS: ('PGS_AFIB', 'PXS_WEIGHTED_AVG')", "('PGS_AFIB', 'PXS_WEIGHTED_AVG')")

afib_all_crs.index = afib_all_crs.index.str.replace('AFIB_PREVENT_CRS_heart_failure', 'CRS')
afib_all_crs.index = afib_all_crs.index.str.replace('AFIB_PREVENT_CRS_stroke', 'CRS')
afib_all_crs.index = afib_all_crs.index.str.replace('AFIB_PREVENT_CRS_chd', 'CRS')
afib_all_crs.index = afib_all_crs.index.str.replace('AFIB_PREVENT_CRS_ascvd', 'CRS')
afib_all_crs.index = afib_all_crs.index.str.replace('AFIB_PREVENT_CRS_total_cvd', 'CRS')
afib_all_crs.index = afib_all_crs.index.str.replace('AFIB_C2HEST_score', 'CRS')


afib_all_crs.index = afib_all_crs.index.str.replace('PGS_AFIB', 'PGS')
afib_all_crs.index = afib_all_crs.index.str.replace("'", '')
afib_all_crs.index = afib_all_crs.index.str.replace("(", '')
afib_all_crs.index = afib_all_crs.index.str.replace(")", '')
afib_all_crs.index = afib_all_crs.index.str.replace(", ", ' + ')

afib_all_crs = afib_all_crs.sort_values(by = ['AUROC'], ascending = False)
afib_all_crs

### HF

In [ ]:
hf_ascvd_sub = hf_ascvd_all_metrics.drop(index = ['ASCVD CRS: PGS_HF',
                                                    'ASCVD CRS: PXS_AVG',
                                                    'ASCVD CRS: PXS_WEIGHTED_AVG',
                                                    "ASCVD CRS: ('PGS_HF', 'PXS_AVG')",
                                                    "ASCVD CRS: ('PGS_HF', 'PXS_WEIGHTED_AVG')"])

In [ ]:
hf_heart_failure_sub = hf_heart_failure_all_metrics.drop(index = ['Heart Failure CRS: PGS_HF',
                                                    'Heart Failure CRS: PXS_AVG',
                                                    'Heart Failure CRS: PXS_WEIGHTED_AVG',
                                                    "Heart Failure CRS: ('PGS_HF', 'PXS_AVG')",
                                                    "Heart Failure CRS: ('PGS_HF', 'PXS_WEIGHTED_AVG')"])

In [ ]:
hf_chd_sub = hf_chd_all_metrics.drop(index = ['CHD CRS: PGS_HF',
                                                    'CHD CRS: PXS_AVG',
                                                    'CHD CRS: PXS_WEIGHTED_AVG',
                                                    "CHD CRS: ('PGS_HF', 'PXS_AVG')",
                                                    "CHD CRS: ('PGS_HF', 'PXS_WEIGHTED_AVG')"])

In [ ]:
hf_stroke_sub = hf_stroke_all_metrics.drop(index = ['Stroke CRS: PGS_HF',
                                                    'Stroke CRS: PXS_AVG',
                                                    'Stroke CRS: PXS_WEIGHTED_AVG',
                                                    "Stroke CRS: ('PGS_HF', 'PXS_AVG')",
                                                    "Stroke CRS: ('PGS_HF', 'PXS_WEIGHTED_AVG')"])

In [ ]:
hf_all_crs = pd.concat([hf_total_cvd_all_metrics,
                         hf_ascvd_sub,
                         hf_heart_failure_sub,
                         hf_chd_sub,
                         hf_stroke_sub], axis = 0)

hf_all_crs.index = hf_all_crs.index.str.replace('Total CVD CRS: PGS_HF', 'PGS_HF')
hf_all_crs.index = hf_all_crs.index.str.replace('Total CVD CRS: PXS_AVG', 'PXS_AVG')
hf_all_crs.index = hf_all_crs.index.str.replace('Total CVD CRS: PXS_WEIGHTED_AVG', 'PXS_WEIGHTED_AVG')
hf_all_crs.index = hf_all_crs.index.str.replace("Total CVD CRS: ('PGS_HF', 'PXS_AVG')", "('PGS_HF', 'PXS_AVG')")
hf_all_crs.index = hf_all_crs.index.str.replace("Total CVD CRS: ('PGS_HF', 'PXS_WEIGHTED_AVG')", "('PGS_HF', 'PXS_WEIGHTED_AVG')")

hf_all_crs.index = hf_all_crs.index.str.replace('HF_PREVENT_CRS_heart_failure', 'CRS')
hf_all_crs.index = hf_all_crs.index.str.replace('HF_PREVENT_CRS_stroke', 'CRS')
hf_all_crs.index = hf_all_crs.index.str.replace('HF_PREVENT_CRS_chd', 'CRS')
hf_all_crs.index = hf_all_crs.index.str.replace('HF_PREVENT_CRS_ascvd', 'CRS')
hf_all_crs.index = hf_all_crs.index.str.replace('HF_PREVENT_CRS_total_cvd', 'CRS')

hf_all_crs.index = hf_all_crs.index.str.replace('PGS_HF', 'PGS')
hf_all_crs.index = hf_all_crs.index.str.replace("'", '')
hf_all_crs.index = hf_all_crs.index.str.replace("(", '')
hf_all_crs.index = hf_all_crs.index.str.replace(")", '')
hf_all_crs.index = hf_all_crs.index.str.replace(", ", ' + ')

hf_all_crs = hf_all_crs.sort_values(by = ['AUROC'], ascending = False)
hf_all_crs

## reformat for plots

### extract pgs + pxs metrics

In [ ]:
cad_pgs_pxs = cad_total_cvd_all_metrics[cad_total_cvd_all_metrics.index.isin(['Total CVD CRS: PGS_CAD',
                                                                              'Total CVD CRS: PXS_AVG',
                                                                              'Total CVD CRS: PXS_WEIGHTED_AVG',
                                                                              "Total CVD CRS: ('PGS_CAD', 'PXS_AVG')",
                                                                              "Total CVD CRS: ('PGS_CAD', 'PXS_WEIGHTED_AVG')"])]
cad_pgs_pxs

In [ ]:
afib_pgs_pxs = afib_total_cvd_all_metrics[afib_total_cvd_all_metrics.index.isin(['Total CVD CRS: PGS_AFIB',
                                                                              'Total CVD CRS: PXS_AVG',
                                                                              'Total CVD CRS: PXS_WEIGHTED_AVG',
                                                                              "Total CVD CRS: ('PGS_AFIB', 'PXS_AVG')",
                                                                              "Total CVD CRS: ('PGS_AFIB', 'PXS_WEIGHTED_AVG')"])]
afib_pgs_pxs

In [ ]:
hf_pgs_pxs = hf_total_cvd_all_metrics[hf_total_cvd_all_metrics.index.isin(['Total CVD CRS: PGS_HF',
                                                                              'Total CVD CRS: PXS_AVG',
                                                                              'Total CVD CRS: PXS_WEIGHTED_AVG',
                                                                              "Total CVD CRS: ('PGS_HF', 'PXS_AVG')",
                                                                              "Total CVD CRS: ('PGS_HF', 'PXS_WEIGHTED_AVG')"])]
hf_pgs_pxs

### combine with CRS

In [ ]:
cad_clean = pd.concat([cad_pgs_pxs, cad_chd_sub], axis = 0)
cad_clean

In [ ]:
afib_clean = pd.concat([afib_pgs_pxs, afib_c2hest_sub], axis = 0)
afib_clean

In [ ]:
hf_clean = pd.concat([hf_pgs_pxs, hf_heart_failure_sub], axis = 0)
hf_clean

### clean

In [ ]:
cad_clean.index = cad_clean.index.str.replace("'", "")
cad_clean.index = cad_clean.index.str.replace("(", "")
cad_clean.index = cad_clean.index.str.replace(")", "")
cad_clean.index = cad_clean.index.str.replace(",", " +")
cad_clean.index = cad_clean.index.str.replace('CHD CRS: ', '')
cad_clean.index = cad_clean.index.str.replace('Total CVD CRS: ', '')
cad_clean.index = cad_clean.index.str.replace('CAD_PREVENT_CRS_chd', 'CRS')
cad_clean.index = cad_clean.index.str.replace('_CAD', '')
cad_clean

In [ ]:
afib_clean.index = afib_clean.index.str.replace("'", "")
afib_clean.index = afib_clean.index.str.replace("(", "")
afib_clean.index = afib_clean.index.str.replace(")", "")
afib_clean.index = afib_clean.index.str.replace(",", " +")
afib_clean.index = afib_clean.index.str.replace('C2HEST CRS: ', '')
afib_clean.index = afib_clean.index.str.replace('Total CVD CRS: ', '')
afib_clean.index = afib_clean.index.str.replace('AFIB_C2HEST_score', 'CRS')
afib_clean.index = afib_clean.index.str.replace('_AFIB', '')
afib_clean

In [ ]:
hf_clean.index = hf_clean.index.str.replace("'", "")
hf_clean.index = hf_clean.index.str.replace("(", "")
hf_clean.index = hf_clean.index.str.replace(")", "")
hf_clean.index = hf_clean.index.str.replace(",", " +")
hf_clean.index = hf_clean.index.str.replace('Heart Failure CRS: ', '')
hf_clean.index = hf_clean.index.str.replace('Total CVD CRS: ', '')
hf_clean.index = hf_clean.index.str.replace('HF_PREVENT_CRS_heart_failure', 'CRS')
hf_clean.index = hf_clean.index.str.replace('_HF', '')
hf_clean

### reorder index

In [ ]:
index_order = ['PGS',
               'CRS',
               'PXS_AVG',
               'PXS_WEIGHTED_AVG',
               'PGS + CRS',
               'PGS + PXS_AVG',
               'PGS + PXS_WEIGHTED_AVG',
               'CRS + PXS_AVG',
               'CRS + PXS_WEIGHTED_AVG',
               'PGS + CRS + PXS_AVG',
               'PGS + CRS + PXS_WEIGHTED_AVG']
print(len(index_order))

In [ ]:
cad_clean = cad_clean.reindex(index_order)
cad_clean

In [ ]:
afib_clean = afib_clean.reindex(index_order)
afib_clean

In [ ]:
hf_clean = hf_clean.reindex(index_order)
hf_clean

### add model number

In [ ]:
cad_clean.index = [f"Model {i+1}: {idx}" for i, idx in enumerate(cad_clean.index)]
cad_clean

In [ ]:
afib_clean.index = [f"Model {i+1}: {idx}" for i, idx in enumerate(afib_clean.index)]
afib_clean

In [ ]:
hf_clean.index = [f"Model {i+1}: {idx}" for i, idx in enumerate(hf_clean.index)]
hf_clean

## export

### cad

In [ ]:
cad_all_crs.to_csv('AOU.CAD.IRM.population_performance.csv')

In [ ]:
cad_clean.to_csv('AOU.CAD.IRM.population_performance.reformat.csv')

In [ ]:
cad_total_cvd_beta.to_csv('AOU.CAD.IRM.PREVENT_CRS_total_cvd.beta.csv', na_rep = 'NaN')

In [ ]:
cad_total_cvd_pval.to_csv('AOU.CAD.IRM.PREVENT_CRS_total_cvd.pval.csv', na_rep = 'NaN')

In [ ]:
cad_ascvd_beta.to_csv('AOU.CAD.IRM.PREVENT_CRS_ascvd.beta.csv', na_rep = 'NaN')

In [ ]:
cad_ascvd_pval.to_csv('AOU.CAD.IRM.PREVENT_CRS_ascvd.pval.csv', na_rep = 'NaN')

In [ ]:
cad_heart_failure_beta.to_csv('AOU.CAD.IRM.PREVENT_CRS_heart_failure.beta.csv', na_rep = 'NaN')

In [ ]:
cad_heart_failure_pval.to_csv('AOU.CAD.IRM.PREVENT_CRS_heart_failure.pval.csv', na_rep = 'NaN')

In [ ]:
cad_stroke_beta.to_csv('AOU.CAD.IRM.PREVENT_CRS_stroke.beta.csv', na_rep = 'NaN')

In [ ]:
cad_stroke_pval.to_csv('AOU.CAD.IRM.PREVENT_CRS_stroke.pval.csv', na_rep = 'NaN')

In [ ]:
cad_chd_beta.to_csv('AOU.CAD.IRM.PREVENT_CRS_chd.beta.csv', na_rep = 'NaN')

In [ ]:
cad_chd_pval.to_csv('AOU.CAD.IRM.PREVENT_CRS_chd.pval.csv', na_rep = 'NaN')

### AFIB

In [ ]:
afib_all_crs.to_csv('AOU.AFIB.IRM.population_performance.csv')

In [ ]:
afib_clean.to_csv('AOU.AFIB.IRM.population_performance.reformat.csv')

In [ ]:
afib_total_cvd_beta.to_csv('AOU.AFIB.IRM.PREVENT_CRS_total_cvd.beta.csv', na_rep = 'NaN')

In [ ]:
afib_total_cvd_pval.to_csv('AOU.AFIB.IRM.PREVENT_CRS_total_cvd.pval.csv', na_rep = 'NaN')

In [ ]:
afib_ascvd_beta.to_csv('AOU.AFIB.IRM.PREVENT_CRS_ascvd.beta.csv', na_rep = 'NaN')

In [ ]:
afib_ascvd_pval.to_csv('AOU.AFIB.IRM.PREVENT_CRS_ascvd.pval.csv', na_rep = 'NaN')

In [ ]:
afib_heart_failure_beta.to_csv('AOU.AFIB.IRM.PREVENT_CRS_heart_failure.beta.csv', na_rep = 'NaN')

In [ ]:
afib_heart_failure_pval.to_csv('AOU.AFIB.IRM.PREVENT_CRS_heart_failure.pval.csv', na_rep = 'NaN')

In [ ]:
afib_stroke_beta.to_csv('AOU.AFIB.IRM.PREVENT_CRS_stroke.beta.csv', na_rep = 'NaN')

In [ ]:
afib_stroke_pval.to_csv('AOU.AFIB.IRM.PREVENT_CRS_stroke.pval.csv', na_rep = 'NaN')

In [ ]:
afib_chd_beta.to_csv('AOU.AFIB.IRM.PREVENT_CRS_chd.beta.csv', na_rep = 'NaN')

In [ ]:
afib_chd_pval.to_csv('AOU.AFIB.IRM.PREVENT_CRS_chd.pval.csv', na_rep = 'NaN')

In [ ]:
afib_c2hest_beta.to_csv('AOU.AFIB.IRM.C2HEST_CRS.beta.csv', na_rep = 'NaN')

In [ ]:
afib_c2hest_pval.to_csv('AOU.AFIB.IRM.C2HEST_CRS_chd.pval.csv', na_rep = 'NaN')

### hf

In [ ]:
hf_all_crs.to_csv('AOU.HF.IRM.population_performance.csv')

In [ ]:
hf_clean.to_csv('AOU.HF.IRM.population_performance.reformat.csv')

In [ ]:
hf_total_cvd_beta.to_csv('AOU.HF.IRM.PREVENT_CRS_total_cvd.beta.csv', na_rep = 'NaN')

In [ ]:
hf_total_cvd_pval.to_csv('AOU.HF.IRM.PREVENT_CRS_total_cvd.pval.csv', na_rep = 'NaN')

In [ ]:
hf_ascvd_beta.to_csv('AOU.HF.IRM.PREVENT_CRS_ascvd.beta.csv', na_rep = 'NaN')

In [ ]:
hf_ascvd_pval.to_csv('AOU.HF.IRM.PREVENT_CRS_ascvd.pval.csv', na_rep = 'NaN')

In [ ]:
hf_heart_failure_beta.to_csv('AOU.HF.IRM.PREVENT_CRS_heart_failure.beta.csv', na_rep = 'NaN')

In [ ]:
hf_heart_failure_pval.to_csv('AOU.HF.IRM.PREVENT_CRS_heart_failure.pval.csv', na_rep = 'NaN')

In [ ]:
hf_stroke_beta.to_csv('AOU.HF.IRM.PREVENT_CRS_stroke.beta.csv', na_rep = 'NaN')

In [ ]:
hf_stroke_pval.to_csv('AOU.HF.IRM.PREVENT_CRS_stroke.pval.csv', na_rep = 'NaN')

In [ ]:
hf_chd_beta.to_csv('AOU.HF.IRM.PREVENT_CRS_chd.beta.csv', na_rep = 'NaN')

In [ ]:
hf_chd_pval.to_csv('AOU.HF.IRM.PREVENT_CRS_chd.pval.csv', na_rep = 'NaN')